In [1]:
from typing import List, Any
import os
import weaviate
import json
import pandas as pd
from langchain_weaviate import WeaviateVectorStore
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
from dotenv import load_dotenv


In [2]:
load_dotenv('../.env.example/.env')

True

In [3]:
# weaviate Keys
WEAVIATE_URL = os.environ["WEAVIATE_URL"]
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"]

In [4]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY,
)

In [5]:
weaviate_client.is_ready()

True

In [5]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [6]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
v = embedding_model.embed_query("rape sentencing guidelines, punishment for rape, statutory penalties for rape, judicial discretion in rape cases")

In [15]:
Euro_Laws = weaviate_client.collections.use("Euro_Laws")

In [16]:
response = Euro_Laws.query.near_vector(
    near_vector= v,
    limit=5
)

In [11]:
for obj in response.objects:
    print (obj.properties)

{'act_name': 'Directive 2012/29/EU of the European Parliament and of the Council of 25Ã\x82 October 2012 establishing minimum standards on the rights, support and protection of victims of crime, and replacing Council Framework Decision 2001/220/JHA', 'total_chunks': 31, 'status': 'In Force', 'legal_basis': '12010E082; 12010E294', 'eurovoc': 'crime against individuals; restorative justice; aid for victims; access to justice; AFSJ', 'act_type': 'Directive', 'celex': '32012L0029', 'chunk_number': 9, 'document_length': 78554, 'authors': 'European Parliament; European Council', 'subject_matter': 'criminal law;  justice;  European construction', 'cites': 'dec_framw/2002/475; 52012XX0209%2802%29; 32001R45; dec_framw/2008/977; 42000A0712%2801%29; 52010XG0504%2801%29; dec_framw/2009/948; 52009IP0098%2801%29; 32011L99; 32011L93; 52011IP0127; 32011G0628%2801%29; 32011L36', 'text': "competent authorities are aware of the victim and throughout criminal proceedings and for an appropriate time after 

In [17]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32011L0093', '32012L0029', '32005F0214', '32019D0417', '32012L0029']

In [13]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32012L0029', '32005F0214', '32011L0093', '32019D0417', '32008F0947']

In [21]:
from weaviate.classes.query import Filter

In [23]:
eur_docs = weaviate_client.collections.get("Euro_Law_Documents")

In [20]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Please provide an overview of European Union aviation law, specifically focusing on the regulation of European airspace, including key legal frameworks, relevant EU regulations, and the roles of governing bodies such as the European Union Aviation Safety Agency (EASA) and Eurocontrol.")

ValueError: Error during query: Query call with protocol GRPC search failed with message Deadline Exceeded.

In [22]:
def get_full_doc_weaviate(celex_id):

    response = eur_docs.query.fetch_objects(
    filters=Filter.by_property("celex").equal(celex_id),
    limit=1
)
    r = response.objects[0].properties

    # r is a dict with key, values for each doc
    return r    

In [28]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    weaviate_client.connect()

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):
        f_doc_meta = get_full_doc_weaviate(celex_id)

        full_doc_info += f"""

        doc {i} :

        'celex': {f_doc_meta['celex']}
        'status': {f_doc_meta['status']}
        'act_type': {f_doc_meta['act_type']}
        'treaty': {f_doc_meta['treaty']}

        full_doc :

        {f_doc_meta['full_doc']}
{"=="*15} "END OF DOC" {"=="*15}
        """
    weaviate_client.close()
    
    return full_doc_info    

In [26]:
weaviate_client.close()

In [ ]:
display(Markdown(search_docs("Sexual Assault Rape sentence")))

 

        doc 0 :

        'celex': 32011L0093
        'status': In Force
        'act_type': Directive
        'treaty': TFEU (2008)

        full_doc :

        17.12.2011 EN Official Journal of the European Union L 335/1 DIRECTIVE 2011/92/EU OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 13 December 2011 on combating the sexual abuse and sexual exploitation of children and child pornography, and replacing Council Framework Decision 2004/68/JHA THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 82(2) and Article 83(1) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Sexual abuse and sexual exploitation of children, including child pornography, constitute serious violations of fundamental rights, in particular of the rights of children to the protection and care necessary for their well-being, as provided for by the 1989 United Nations Convention on the Rights of the Child and by the Charter of Fundamental Rights of the European Union (3). (2) In accordance with Article 6(1) of the Treaty on European Union, the Union recognises the rights, freedoms and principles set out in the Charter of Fundamental Rights of the European Union, in which Article 24(2) provides that in all actions relating to children, whether taken by public authorities or private institutions, the childs best interests must be a primary consideration. Moreover, the Stockholm Programme  An Open and Secure Europe Serving and Protecting Citizens (4) gives a clear priority to combating the sexual abuse and sexual exploitation of children and child pornography. (3) Child pornography, which consists of images of child sexual abuse, and other particularly serious forms of sexual abuse and sexual exploitation of children are increasing and spreading through the use of new technologies and the Internet. (4) Council Framework Decision 2004/68/JHA of 22 December 2003 on combating the sexual exploitation of children and child pornography (5) approximates Member States legislation to criminalise the most serious forms of child sexual abuse and sexual exploitation, to extend domestic jurisdiction, and to provide for a minimum level of assistance for victims. Council Framework Decision 2001/220/JHA of 15 March 2001 on the standing of victims in criminal proceedings (6) establishes a set of victims rights in criminal proceedings, including the right to protection and compensation. Moreover, the coordination of prosecution of cases of sexual abuse, sexual exploitation of children and child pornography will be facilitated by the implementation of Council Framework Decision 2009/948/JHA of 30 November 2009 on prevention and settlement of conflicts of exercise of jurisdiction in criminal proceedings (7). (5) In accordance with Article 34 of the United Nations Convention on the Rights of the Child, States Parties undertake to protect the child from all forms of sexual exploitation and sexual abuse. The 2000 United Nations Optional Protocol to the Convention on the Rights of the Child on the sale of children, child prostitution and child pornography and, in particular, the 2007 Council of Europe Convention on the Protection of Children against Sexual Exploitation and Sexual Abuse are crucial steps in the process of enhancing international cooperation in this field. (6) Serious criminal offences such as the sexual exploitation of children and child pornography require a comprehensive approach covering the prosecution of offenders, the protection of child victims, and prevention of the phenomenon. The childs best interests must be a primary consideration when carrying out any measures to combat these offences in accordance with the Charter of Fundamental Rights of the European Union and the United Nations Convention on the Rights of the Child. Framework Decision 2004/68/JHA should be replaced by a new instrument providing such comprehensive legal framework to achieve that purpose. (7) This Directive should be fully complementary with Directive 2011/36/EU of the European Parliament and of the Council of 5 April 2011 on preventing and combating trafficking in human beings and protecting its victims, and replacing Council Framework Decision 2002/629/JHA (8), as some victims of human trafficking have also been child victims of sexual abuse or sexual exploitation. (8) In the context of criminalising acts related to pornographic performance, this Directive refers to such acts which consist of an organised live exhibition, aimed at an audience, thereby excluding personal face-to-face communication between consenting peers, as well as children over the age of sexual consent and their partners from the definition. (9) Child pornography frequently includes images recording the sexual abuse of children by adults. It may also include images of children involved in sexually explicit conduct, or of their sexual organs, where such images are produced or used for primarily sexual purposes and exploited with or without the childs knowledge. Furthermore, the concept of child pornography also covers realistic images of a child, where a child is engaged or depicted as being engaged in sexually explicit conduct for primarily sexual purposes. (10) Disability, by itself, does not automatically constitute an impossibility to consent to sexual relations. However, the abuse of the existence of such a disability in order to engage in sexual activities with a child should be criminalised. (11) In adopting legislation on substantive criminal law, the Union should ensure consistency of such legislation in particular with regard to the level of penalties. The Council conclusions of 24 and 25 April 2002 on the approach to apply regarding approximation of penalties, which indicate four levels of penalties, should be kept in mind in the light of the Lisbon Treaty. This Directive, because it contains an exceptionally high number of different offences, requires, in order to reflect the various degrees of seriousness, a differentiation in the level of penalties which goes further than what should usually be provided in Union legal instruments. (12) Serious forms of sexual abuse and sexual exploitation of children should be subject to effective, proportionate and dissuasive penalties. This includes, in particular, various forms of sexual abuse and sexual exploitation of children which are facilitated by the use of information and communication technology, such as the online solicitation of children for sexual purposes via social networking websites and chat rooms. The definition of child pornography should also be clarified and brought closer to that contained in international instruments. (13) The maximum term of imprisonment provided for in this Directive for the offences referred to therein should apply at least to the most serious forms of such offences. (14) In order to reach the maximum term of imprisonment provided for in this Directive for offences concerning sexual abuse and sexual exploitation of children and child pornography, Member States may combine, taking into account their national law, the imprisonment terms provided for in national legislation in respect of those offences. (15) This Directive obliges Member States to provide for criminal penalties in their national legislation in respect of the provisions of Union law on combating sexual abuse, sexual exploitation of children and child pornography. This Directive creates no obligations regarding the application of such penalties, or any other available system of law enforcement, in individual cases. (16) Especially for those cases where the offences referred to in this Directive are committed with the purpose of financial gain, Member States are invited to consider providing for the possibility to impose financial penalties in addition to imprisonment. (17) In the context of child pornography, the term without right allows Member States to provide a defence in respect of conduct relating to pornographic material having for example, a medical, scientific or similar purpose. It also allows activities carried out under domestic legal powers, such as the legitimate possession of child pornography by the authorities in order to conduct criminal proceedings or to prevent, detect or investigate crime. Furthermore, it does not exclude legal defences or similar relevant principles that relieve a person of responsibility under specific circumstances, for example where telephone or Internet hotlines carry out activities to report those cases. (18) Knowingly obtaining access, by means of information and communication technology, to child pornography should be criminalised. To be liable, the person should both intend to enter a site where child pornography is available and know that such images can be found there. Penalties should not be applied to persons inadvertently accessing sites containing child pornography. The intentional nature of the offence may notably be deduced from the fact that it is recurrent or that the offence was committed via a service in return for payment. (19) Solicitation of children for sexual purposes is a threat with specific characteristics in the context of the Internet, as the latter provides unprecedented anonymity to users because they are able to conceal their real identity and personal characteristics, such as their age. At the same time, Member States acknowledge the importance of also combating the solicitation of a child outside the context of the Internet, in particular where such solicitation is not carried out by using information and communication technology. Member States are encouraged to criminalise the conduct where the solicitation of a child to meet the offender for sexual purposes takes place in the presence or proximity of the child, for instance in the form of a particular preparatory offence, attempt to commit the offences referred to in this Directive or as a particular form of sexual abuse. Whichever legal solution is chosen to criminalise off-line grooming, Member States should ensure that they prosecute the perpetrators of such offences one way or another. (20) This Directive does not govern Member States policies with regard to consensual sexual activities in which children may be involved and which can be regarded as the normal discovery of sexuality in the course of human development, taking account of the different cultural and legal traditions and of new forms of establishing and maintaining relations among children and adolescents, including through information and communication technologies. These issues fall outside of the scope of this Directive. Member States which avail themselves of the possibilities referred to in this Directive do so in the exercise of their competences. (21) Member States should provide for aggravating circumstances in their national law in accordance with the applicable rules established by their legal systems on aggravating circumstances. They should ensure that those aggravating circumstances are available for judges to consider when sentencing offenders, although there is no obligation on judges to apply those aggravating circumstances. The aggravating circumstances should not be provided for in Member States law when irrelevant taking into account the nature of the specific offence. The relevance of the various aggravating circumstances provided for in this Directive should be evaluated at national level for each of the offences referred to in this Directive. (22) Physical or mental incapacity under this Directive should be understood as also including the state of physical or mental incapacity caused by the influence of drugs and alcohol. (23) In combating sexual exploitation of children, full use should be made of existing instruments on the seizure and confiscation of the proceeds of crime, such as the United Nations Convention against Transnational Organized Crime and the Protocols thereto, the 1990 Council of Europe Convention on Laundering, Search, Seizure and Confiscation of the Proceeds from Crime, Council Framework Decision 2001/500/JHA of 26 June 2001 on money laundering, the identification, tracing, freezing, seizing and confiscation of instrumentalities and the proceeds of crime (9), and Council Framework Decision 2005/212/JHA of 24 February 2005 on Confiscation of Crime Related Proceeds, Instrumentalities and Property (10). The use of seized and confiscated instrumentalities and the proceeds from the offences referred to in this Directive to support victims assistance and protection should be encouraged. (24) Secondary victimisation should be avoided for victims of offences referred to in this Directive. In Member States where prostitution or the appearance in pornography is punishable under national criminal law, it should be possible not to prosecute or impose penalties under those laws where the child concerned has committed those acts as a result of being victim of sexual exploitation or where the child was compelled to participate in child pornography. (25) As an instrument of approximation of criminal law, this Directive provides for levels of penalties which should apply without prejudice to the specific criminal policies of the Member States concerning child offenders. (26) Investigating offences and bringing charges in criminal proceedings should be facilitated, to take into account the difficulty for child victims of denouncing sexual abuse and the anonymity of offenders in cyberspace. To ensure successful investigations and prosecutions of the offences referred to in this Directive, their initiation should not depend, in principle, on a report or accusation made by the victim or by his or her representative. The length of the sufficient period of time for prosecution should be determined in accordance with national law. (27) Effective investigatory tools should be made available to those responsible for the investigation and prosecutions of the offences referred to in this Directive. Those tools could include interception of communications, covert surveillance including electronic surveillance, monitoring of bank accounts or other financial investigations, taking into account, inter alia, the principle of proportionality and the nature and seriousness of the offences under investigation. Where appropriate, and in accordance with national law, such tools should also include the possibility for law enforcement authorities to use a concealed identity on the Internet. (28) Member States should encourage any person who has knowledge or suspicion of the sexual abuse or sexual exploitation of a child to report to the competent services. It is the responsibility of each Member State to determine the competent authorities to which such suspicions may be reported. Those competent authorities should not be limited to child protection services or relevant social services. The requirement of suspicion in good faith should be aimed at preventing the provision being invoked to authorise the denunciation of purely imaginary or untrue facts carried out with malicious intent. (29) Rules on jurisdiction should be amended to ensure that sexual abusers or sexual exploiters of children from the Union face prosecution even if they commit their crimes outside the Union, in particular via so-called sex tourism. Child sex tourism should be understood as the sexual exploitation of children by a person or persons who travel from their usual environment to a destination abroad where they have sexual contact with children. Where child sex tourism takes place outside the Union, Member States are encouraged to seek to increase, through the available national and international instruments including bilateral or multilateral treaties on extradition, mutual assistance or a transfer of the proceedings, cooperation with third countries and international organisations with a view to combating sex tourism. Member States should foster open dialogue and communication with countries outside the Union in order to be able to prosecute perpetrators, under the relevant national legislation, who travel outside the Union borders for the purposes of child sex tourism. (30) Measures to protect child victims should be adopted in their best interest, taking into account an assessment of their needs. Child victims should have easy access to legal remedies and measures to address conflicts of interest where sexual abuse or sexual exploitation of a child occurs within the family. When a special representative should be appointed for a child during a criminal investigation or proceeding, this role may be also carried out by a legal person, an institution or an authority. Moreover, child victims should be protected from penalties, for example under national legislation on prostitution, if they bring their case to the attention of competent authorities. Furthermore, participation in criminal proceedings by child victims should not cause additional trauma to the extent possible, as a result of interviews or visual contact with offenders. A good understanding of children and how they behave when faced with traumatic experiences will help to ensure a high quality of evidence-taking and also reduce the stress placed on children when carrying out the necessary measures. (31) Member States should consider giving short and long term assistance to child victims. Any harm caused by the sexual abuse and sexual exploitation of a child is significant and should be addressed. Because of the nature of the harm caused by sexual abuse and sexual exploitation, such assistance should continue for as long as necessary for the childs physical and psychological recovery and may last into adulthood if necessary. Assistance and advice should be considered to be extended to parents or guardians of the child victims where they are not involved as suspects in relation to the offence concerned, in order to help them to assist child victims throughout the proceedings. (32) Framework Decision 2001/220/JHA establishes a set of victims rights in criminal proceedings, including the right to protection and compensation. In addition child victims of sexual abuse, sexual exploitation and child pornography should be given access to legal counselling and, in accordance with the role of victims in the relevant justice systems, to legal representation, including for the purpose of claiming compensation. Such legal counselling and legal representation could also be provided by the competent authorities for the purpose of claiming compensation from the State. The purpose of legal counselling is to enable victims to be informed and receive advice about the various possibilities open to them. Legal counselling should be provided by a person having received appropriate legal training without necessarily being a lawyer. Legal counselling and, in accordance with the role of victims in the relevant justice systems, legal representation should be provided free of charge, at least when the victim does not have sufficient financial resources, in a manner consistent with the internal procedures of Member States. (33) Member States should undertake action to prevent or prohibit acts related to the promotion of sexual abuse of children and child sex tourism. Different preventative measures could be considered, such as the drawing up and reinforcement of a code of conduct and self-regulatory mechanisms in the tourism industry, the setting-up of a code of ethics or quality labels for tourist organisations combating child sex tourism, or establishing an explicit policy to tackle child sex tourism. (34) Member States should establish and/or strengthen policies to prevent sexual abuse and sexual exploitation of children, including measures to discourage and reduce the demand that fosters all forms of sexual exploitation of children, and measures to reduce the risk of children becoming victims, by means of, information and awareness-raising campaigns, and research and education programmes. In such initiatives, Member States should adopt a child-rights based approach. Particular care should be taken to ensure that awareness-raising campaigns aimed at children are appropriate and sufficiently easy to understand. The establishment of help-lines or hotlines should be considered. (35) Regarding the system of reporting sexual abuse and sexual exploitation of children and helping children in need, hotlines under the number 116 000 for missing children, 116 006 for victims of crime and 116 111 for children, as introduced by Commission Decision 2007/116/EC of 15 February 2007 on reserving the national numbering beginning with 116 for harmonised numbers for harmonised services of social value (11), should be promoted and experience regarding their functioning should be taken into account. (36) Professionals likely to come into contact with child victims of sexual abuse and sexual exploitation should be adequately trained to identify and deal with such victims. That training should be promoted for members of the following categories when they are likely to come into contact with child victims: police officers, public prosecutors, lawyers, members of the judiciary and court officials, child and health care personnel, but could also involve other groups of persons who are likely to encounter child victims of sexual abuse and sexual exploitation in their work. (37) In order to prevent the sexual abuse and sexual exploitation of children, intervention programmes or measures targeting sex offenders should be proposed to them. Those intervention programmes or measures should meet a broad, flexible approach focusing on the medical and psycho-social aspects and have a non-obligatory character. Those intervention programmes or measures are without prejudice to intervention programmes or measures imposed by the competent judicial authorities. (38) Intervention programmes or measures are not provided as an automatic right. It is for the Member State to decide which intervention programmes or measures are appropriate. (39) To prevent and minimise recidivism, offenders should be subject to an assessment of the danger posed by the offenders and the possible risks of repetition of sexual offences against children. Arrangements for such assessment, such as the type of authority competent to order and carry out the assessment or the moment in or after the criminal proceedings when that assessment should take place as well as arrangements for effective intervention programmes or measures offered following that assessment should be consistent with the internal procedures of Member States. For the same objective of preventing and minimising recidivism, offenders should also have access to effective intervention programmes or measures on a voluntary basis. Those intervention programmes or measures should not interfere with national schemes set up to deal with the treatment of persons suffering from mental disorders. (40) Where the danger posed by the offenders and the possible risks of repetition of the offences make it appropriate, convicted offenders should be temporarily or permanently prevented from exercising at least professional activities involving direct and regular contacts with children. Employers when recruiting for a post involving direct and regular contact with children are entitled to be informed of existing convictions for sexual offences against children entered in the criminal record, or of existing disqualifications. For the purposes of this Directive, the term employers should also cover persons running an organisation that is active in volunteer work related to the supervision and/or care of children involving direct and regular contact with children. The manner in which such information is delivered, such as for example access via the person concerned, and the precise content of the information, the meaning of organised voluntary activities and direct and regular contact with children should be laid down in accordance with national law. (41) With due regard to the different legal traditions of the Member States, this Directive takes into account the fact that access to criminal records is allowed only either by the competent authorities or by the person concerned. This Directive does not establish an obligation to modify the national systems governing criminal records or the means of access to those records. (42) The aim of this Directive is not to harmonise rules concerning consent of the person concerned when exchanging information from the criminal registers, i.e. whether or not to require such consent. Whether the consent is required or not under national law, this Directive does not establish any new obligation to change the national law and national procedures in this respect. (43) Member States may consider adopting additional administrative measures in relation to perpetrators, such as the registration in sex offender registers of persons convicted of offences referred to in this Directive. Access to those registers should be subject to limitation in accordance with national constitutional principles and applicable data protection standards, for instance by limiting access to the judiciary and/or law enforcement authorities. (44) Member States are encouraged to create mechanisms for data collection or focal points, at the national or local levels and in collaboration with civil society, for the purpose of observing and evaluating the phenomenon of sexual abuse and sexual exploitation of children. In order to be able to properly evaluate the results of actions to combat sexual abuse and sexual exploitation of children and child pornography, the Union should continue to develop its work on methodologies and data collection methods to produce comparable statistics. (45) Member States should take appropriate action for setting up information services to provide information on how to recognise the signs of sexual abuse and sexual exploitation. (46) Child pornography, which constitutes child sexual abuse images, is a specific type of content which cannot be construed as the expression of an opinion. To combat it, it is necessary to reduce the circulation of child sexual abuse material by making it more difficult for offenders to upload such content onto the publicly accessible web. Action is therefore necessary to remove the content and apprehend those guilty of making, distributing or downloading child sexual abuse images. With a view to supporting the Unions efforts to combat child pornography, Member States should use their best endeavours to cooperate with third countries in seeking to secure the removal of such content from servers within their territory. (47) However, despite such efforts, the removal of child pornography content at its source is often not possible when the original materials are not located within the Union, either because the State where the servers are hosted is not willing to cooperate or because obtaining removal of the material from the State concerned proves to be particularly long. Mechanisms may also be put in place to block access from the Unions territory to Internet pages identified as containing or disseminating child pornography. The measures undertaken by Member States in accordance with this Directive in order to remove or, where appropriate, block websites containing child pornography could be based on various types of public action, such as legislative, non-legislative, judicial or other. In that context, this Directive is without prejudice to voluntary action taken by the Internet industry to prevent the misuse of its services or to any support for such action by Member States. Whichever basis for action or method is chosen, Member States should ensure that it provides an adequate level of legal certainty and predictability to users and service providers. Both with a view to the removal and the blocking of child abuse content, cooperation between public authorities should be established and strengthened, particularly in the interests of ensuring that national lists of websites containing child pornography material are as complete as possible and of avoiding duplication of work. Any such developments must take account of the rights of the end users and comply with existing legal and judicial procedures and the European Convention for the Protection of Human Rights and Fundamental Freedoms and the Charter of Fundamental Rights of the European Union. The Safer Internet Programme has set up a network of hotlines the goal of which is to collect information and to ensure coverage and exchange of reports on the major types of illegal content online. (48) This Directive aims to amend and expand the provisions of Framework Decision 2004/68/JHA. Since the amendments to be made are of substantial number and nature, the Framework Decision should, in the interests of clarity, be replaced in its entirety in relation to Member States participating in the adoption of this Directive. (49) Since the objective of this Directive, namely to combat sexual abuse, sexual exploitation of children and child pornography, cannot be sufficiently achieved by the Member States alone and can therefore, by reasons of the scale and effects, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty on European Union. In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary to achieve that objective. (50) This Directive respects fundamental rights and observes the principles recognised in particular by the Charter of Fundamental Rights of the European Union and in particular the right to the protection of human dignity, the prohibition of torture and inhuman or degrading treatment or punishment, the rights of the child, the right to liberty and security, the right to freedom of expression and information, the right to the protection of personal data, the right to an effective remedy and to a fair trial and the principles of legality and proportionality of criminal offences and penalties. This Directive seeks to ensure full respect for those rights and principles and must be implemented accordingly. (51) In accordance with Article 3 of the Protocol (No 21) on the position of United Kingdom and Ireland in respect of the area of freedom, security and justice, annexed to the Treaty on European Union and the Treaty on the Functioning of the European Union, the United Kingdom and Ireland have notified their wish to take part in the adoption and application of this Directive. (52) In accordance with Articles 1 and 2 of the Protocol (No 22) on the position of Denmark annexed to the Treaty on European Union and to the Treaty on the Functioning of the European Union, Denmark is not taking part in the adoption of this Directive and is not bound by it or subject to its application, HAVE ADOPTED THIS DIRECTIVE: Article 1 Subject matter This Directive establishes minimum rules concerning the definition of criminal offences and sanctions in the area of sexual abuse and sexual exploitation of children, child pornography and solicitation of children for sexual purposes. It also introduces provisions to strengthen the prevention of those crimes and the protection of the victims thereof. Article 2 Definitions For the purposes of this Directive, the following definitions apply: (a) child means any person below the age of 18 years; (b) age of sexual consent means the age below which, in accordance with national law, it is prohibited to engage in sexual activities with a child; (c) child pornography means: (i) any material that visually depicts a child engaged in real or simulated sexually explicit conduct; (ii) any depiction of the sexual organs of a child for primarily sexual purposes; (iii) any material that visually depicts any person appearing to be a child engaged in real or simulated sexually explicit conduct or any depiction of the sexual organs of any person appearing to be a child, for primarily sexual purposes; or (iv) realistic images of a child engaged in sexually explicit conduct or realistic images of the sexual organs of a child, for primarily sexual purposes; (d) child prostitution means the use of a child for sexual activities where money or any other form of remuneration or consideration is given or promised as payment in exchange for the child engaging in sexual activities, regardless of whether that payment, promise or consideration is made to the child or to a third party; (e) pornographic performance means a live exhibition aimed at an audience, including by means of information and communication technology, of: (i) a child engaged in real or simulated sexually explicit conduct; or (ii) the sexual organs of a child for primarily sexual purposes; (f) legal person means an entity having legal personality under the applicable law, except for States or public bodies in the exercise of State authority and for public international organisations. Article 3 Offences concerning sexual abuse 1. Member States shall take the necessary measures to ensure that the intentional conduct referred to in paragraphs 2 to 6 is punishable. 2. Causing, for sexual purposes, a child who has not reached the age of sexual consent to witness sexual activities, even without having to participate, shall be punishable by a maximum term of imprisonment of at least 1 year. 3. Causing, for sexual purposes, a child who has not reached the age of sexual consent to witness sexual abuse, even without having to participate, shall be punishable by a maximum term of imprisonment of at least 2 years. 4. Engaging in sexual activities with a child who has not reached the age of sexual consent shall be punishable by a maximum term of imprisonment of at least 5 years. 5. Engaging in sexual activities with a child, where: (i) abuse is made of a recognised position of trust, authority or influence over the child, shall be punishable by a maximum term of imprisonment of at least 8 years if the child has not reached the age of sexual consent, and of at least 3 years of imprisonment, if the child is over that age; or (ii) abuse is made of a particularly vulnerable situation of the child, in particular because of a mental or physical disability or a situation of dependence, shall be punishable by a maximum term of imprisonment of at least 8 years if the child has not reached the age of sexual consent, and of at least 3 years of imprisonment if the child is over that age; or (iii) use is made of coercion, force or threats shall be punishable by a maximum term of imprisonment of at least 10 years if the child has not reached the age of sexual consent, and of at least 5 years of imprisonment if the child is over that age. 6. Coercing, forcing or threatening a child into sexual activities with a third party shall be punishable by a maximum term of imprisonment of at least 10 years if the child has not reached the age of sexual consent, and of at least 5 years of imprisonment if the child is over that age. Article 4 Offences concerning sexual exploitation 1. Member States shall take the necessary measures to ensure that the intentional conduct referred to in paragraphs 2 to 7 is punishable. 2. Causing or recruiting a child to participate in pornographic performances, or profiting from or otherwise exploiting a child for such purposes shall be punishable by a maximum term of imprisonment of at least 5 years if the child has not reached the age of sexual consent and of at least 2 years of imprisonment if the child is over that age. 3. Coercing or forcing a child to participate in pornographic performances, or threatening a child for such purposes shall be punishable by a maximum term of imprisonment of at least 8 years if the child has not reached the age of sexual consent, and of at least 5 years of imprisonment if the child is over that age. 4. Knowingly attending pornographic performances involving the participation of a child shall be punishable by a maximum term of imprisonment of at least 2 years if the child has not reached the age of sexual consent, and of at least 1 year of imprisonment if the child is over that age. 5. Causing or recruiting a child to participate in child prostitution, or profiting from or otherwise exploiting a child for such purposes shall be punishable by a maximum term of imprisonment of at least 8 years if the child has not reached the age of sexual consent, and of at least 5 years of imprisonment if the child is over that age. 6. Coercing or forcing a child into child prostitution, or threatening a child for such purposes shall be punishable by a maximum term of imprisonment of at least 10 years if the child has not reached the age of sexual consent, and of at least 5 years of imprisonment if the child is over that age. 7. Engaging in sexual activities with a child, where recourse is made to child prostitution shall be punishable by a maximum term of imprisonment of at least 5 years if the child has not reached the age of sexual consent, and of at least 2 years of imprisonment if the child is over that age. Article 5 Offences concerning child pornography 1. Member States shall take the necessary measures to ensure that the intentional conduct, when committed without right, referred to in paragraphs 2 to 6 is punishable. 2. Acquisition or possession of child pornography shall be punishable by a maximum term of imprisonment of at least 1 year. 3. Knowingly obtaining access, by means of information and communication technology, to child pornography shall be punishable by a maximum term of imprisonment of at least 1 year. 4. Distribution, dissemination or transmission of child pornography shall be punishable by a maximum term of imprisonment of at least 2 years. 5. Offering, supplying or making available child pornography shall be punishable by a maximum term of imprisonment of at least 2 years. 6. Production of child pornography shall be punishable by a maximum term of imprisonment of at least 3 years. 7. It shall be within the discretion of Member States to decide whether this Article applies to cases involving child pornography as referred to in Article 2(c)(iii), where the person appearing to be a child was in fact 18 years of age or older at the time of depiction. 8. It shall be within the discretion of Member States to decide whether paragraphs 2 and 6 of this Article apply to cases where it is established that pornographic material as referred to in Article 2(c)(iv) is produced and possessed by the producer solely for his or her private use in so far as no pornographic material as referred to in Article 2(c)(i), (ii) or (iii) has been used for the purpose of its production and provided that the act involves no risk of dissemination of the material. Article 6 Solicitation of children for sexual purposes 1. Member States shall take the necessary measures to ensure that the following intentional conduct is punishable: the proposal, by means of information and communication technology, by an adult to meet a child who has not reached the age of sexual consent, for the purpose of committing any of the offences referred to in Article 3(4) and Article 5(6), where that proposal was followed by material acts leading to such a meeting, shall be punishable by a maximum term of imprisonment of at least 1 year. 2. Member States shall take the necessary measures to ensure that an attempt, by means of information and communication technology, to commit the offences provided for in Article 5(2) and (3) by an adult soliciting a child who has not reached the age of sexual consent to provide child pornography depicting that child is punishable. Article 7 Incitement, aiding and abetting, and attempt 1. Member States shall take the necessary measures to ensure that inciting or aiding and abetting to commit any of the offences referred to in Articles 3 to 6 is punishable. 2. Member States shall take the necessary measures to ensure that an attempt to commit any of the offences referred to in Article 3(4), (5) and (6), Article 4(2), (3), (5), (6) and (7), and Article 5(4), (5) and (6) is punishable. Article 8 Consensual sexual activities 1. It shall be within the discretion of Member States to decide whether Article 3(2) and (4) apply to consensual sexual activities between peers, who are close in age and degree of psychological and physical development or maturity, in so far as the acts did not involve any abuse. 2. It shall be within the discretion of Member States to decide whether Article 4(4) applies to a pornographic performance that takes place in the context of a consensual relationship where the child has reached the age of sexual consent or between peers who are close in age and degree of psychological and physical development or maturity, in so far as the acts did not involve any abuse or exploitation and no money or other form of remuneration or consideration is given as payment in exchange for the pornographic performance. 3. It shall be within the discretion of Member States to decide whether Article 5(2) and (6) apply to the production, acquisition or possession of material involving children who have reached the age of sexual consent where that material is produced and possessed with the consent of those children and only for the private use of the persons involved, in so far as the acts did not involve any abuse. Article 9 Aggravating circumstances In so far as the following circumstances do not already form part of the constituent elements of the offences referred to in Articles 3 to 7, Member States shall take the necessary measures to ensure that the following circumstances may, in accordance with the relevant provisions of national law, be regarded as aggravating circumstances, in relation to the relevant offences referred to in Articles 3 to 7: (a) the offence was committed against a child in a particularly vulnerable situation, such as a child with a mental or physical disability, in a situation of dependence or in a state of physical or mental incapacity; (b) the offence was committed by a member of the childs family, a person cohabiting with the child or a person who has abused a recognised position of trust or authority; (c) the offence was committed by several persons acting together; (d) the offence was committed within the framework of a criminal organisation within the meaning of Council Framework Decision 2008/841/JHA of 24 October 2008 on the fight against organised crime (12); (e) the offender has previously been convicted of offences of the same nature; (f) the offender has deliberately or recklessly endangered the life of the child; or (g) the offence involved serious violence or caused serious harm to the child. Article 10 Disqualification arising from convictions 1. In order to avoid the risk of repetition of offences, Member States shall take the necessary measures to ensure that a natural person who has been convicted of any of the offences referred to in Articles 3 to 7 may be temporarily or permanently prevented from exercising at least professional activities involving direct and regular contacts with children. 2. Member States shall take the necessary measures to ensure that employers, when recruiting a person for professional or organised voluntary activities involving direct and regular contacts with children, are entitled to request information in accordance with national law by way of any appropriate means, such as access upon request or via the person concerned, of the existence of criminal convictions for any of the offences referred to in Articles 3 to 7 entered in the criminal record or of the existence of any disqualification from exercising activities involving direct and regular contacts with children arising from those criminal convictions. 3. Member States shall take the necessary measures to ensure that, for the application of paragraphs 1 and 2 of this Article, information concerning the existence of criminal convictions for any of the offences referred to in Articles 3 to 7, or of any disqualification from exercising activities involving direct and regular contacts with children arising from those criminal convictions, is transmitted in accordance with the procedures set out in Council Framework Decision 2009/315/JHA of 26 February 2009 on the organisation and content of the exchange of information extracted from the criminal record between Member States (13) when requested under Article 6 of that Framework Decision with the consent of the person concerned. Article 11 Seizure and confiscation Member States shall take the necessary measures to ensure that their competent authorities are entitled to seize and confiscate instrumentalities and proceeds from the offences referred to in Articles 3, 4 and 5. Article 12 Liability of legal persons 1. Member States shall take the necessary measures to ensure that legal persons may be held liable for any of the offences referred to in Articles 3 to 7 committed for their benefit by any person, acting either individually or as part of an organ of the legal person, and having a leading position within the legal person, based on: (a) a power of representation of the legal person; (b) an authority to take decisions on behalf of the legal person; or (c) an authority to exercise control within the legal person. 2. Member States shall also take the necessary measures to ensure that legal persons may be held liable where the lack of supervision or control by a person referred to in paragraph 1 has made possible the commission, by a person under its authority, of any of the offences referred to in Articles 3 to 7 for the benefit of that legal person. 3. Liability of legal persons under paragraphs 1 and 2 shall be without prejudice to criminal proceedings against natural persons who are perpetrators, inciters or accessories to the offences referred to in Articles 3 to 7. Article 13 Sanctions on legal persons 1. Member States shall take the necessary measures to ensure that a legal person held liable pursuant to Article 12(1) is punishable by effective, proportionate and dissuasive sanctions, which shall include criminal or non-criminal fines and may include other sanctions, such as: (a) exclusion from entitlement to public benefits or aid; (b) temporary or permanent disqualification from the practice of commercial activities; (c) placing under judicial supervision; (d) judicial winding-up; or (e) temporary or permanent closure of establishments which have been used for committing the offence. 2. Member States shall take the necessary measures to ensure that a legal person held liable pursuant to Article 12(2) is punishable by sanctions or measures which are effective, proportionate and dissuasive. Article 14 Non-prosecution or non-application of penalties to the victim Member States shall, in accordance with the basic principles of their legal systems take the necessary measures to ensure that competent national authorities are entitled not to prosecute or impose penalties on child victims of sexual abuse and sexual exploitation for their involvement in criminal activities, which they have been compelled to commit as a direct consequence of being subjected to any of the acts referred to in Article 4(2), (3), (5) and (6), and in Article 5(6). Article 15 Investigation and prosecution 1. Member States shall take the necessary measures to ensure that investigations into or the prosecution of the offences referred to in Articles 3 to 7 are not dependent on a report or accusation being made by the victim or by his or her representative, and that criminal proceedings may continue even if that person has withdrawn his or her statements. 2. Member States shall take the necessary measures to enable the prosecution of any of the offences referred to in Article 3, Article 4(2), (3), (5), (6) and (7) and of any serious offences referred to in Article 5(6) when child pornography as referred to in Article 2(c)(i) and (ii) has been used, for a sufficient period of time after the victim has reached the age of majority and which is commensurate with the gravity of the offence concerned. 3. Member States shall take the necessary measures to ensure that effective investigative tools, such as those which are used in organised crime or other serious crime cases are available to persons, units or services responsible for investigating or prosecuting offences referred to in Articles 3 to 7. 4. Member States shall take the necessary measures to enable investigative units or services to attempt to identify the victims of the offences referred to in Articles 3 to 7, in particular by analysing child pornography material, such as photographs and audiovisual recordings transmitted or made available by means of information and communication technology. Article 16 Reporting suspicion of sexual abuse or sexual exploitation 1. Member States shall take the necessary measures to ensure that the confidentiality rules imposed by national law on certain professionals whose main duty is to work with children do not constitute an obstacle to the possibility, for those professionals, of their reporting to the services responsible for child protection any situation where they have reasonable grounds for believing that a child is the victim of offences referred to in Articles 3 to 7. 2. Member States shall take the necessary measures to encourage any person who knows about or suspects, in good faith that any of the offences referred to in Articles 3 to 7 have been committed, to report this to the competent services. Article 17 Jurisdiction and coordination of prosecution 1. Member States shall take the necessary measures to establish their jurisdiction over the offences referred to in Articles 3 to 7 where: (a) the offence is committed in whole or in part within their territory; or (b) the offender is one of their nationals. 2. A Member State shall inform the Commission where it decides to establish further jurisdiction over an offence referred to in Articles 3 to 7 committed outside its territory, inter alia, where: (a) the offence is committed against one of its nationals or a person who is an habitual resident in its territory; (b) the offence is committed for the benefit of a legal person established in its territory; or (c) the offender is an habitual resident in its territory. 3. Member States shall ensure that their jurisdiction includes situations where an offence referred to in Articles 5 and 6, and in so far as is relevant, in Articles 3 and 7, is committed by means of information and communication technology accessed from their territory, whether or not it is based on their territory. 4. For the prosecution of any of the offences referred to in Article 3(4), (5) and (6), Article 4(2), (3), (5), (6) and (7) and Article 5(6) committed outside the territory of the Member State concerned, as regards paragraph 1(b) of this Article, each Member State shall take the necessary measures to ensure that its jurisdiction is not subordinated to the condition that the acts are a criminal offence at the place where they were performed. 5. For the prosecution of any of the offences referred to in Articles 3 to 7 committed outside the territory of the Member State concerned, as regards paragraph 1(b) of this Article, each Member State shall take the necessary measures to ensure that its jurisdiction is not subordinated to the condition that the prosecution can only be initiated following a report made by the victim in the place where the offence was committed, or a denunciation from the State of the place where the offence was committed. Article 18 General provisions on assistance, support and protection measures for child victims 1. Child victims of the offences referred to in Articles 3 to 7 shall be provided assistance, support and protection in accordance with Articles 19 and 20, taking into account the best interests of the child. 2. Member States shall take the necessary measures to ensure that a child is provided with assistance and support as soon as the competent authorities have a reasonable-grounds indication for believing that a child might have been subject to any of the offences referred to in Articles 3 to 7. 3. Member States shall ensure that, where the age of a person subject to any of the offences referred to in Articles 3 to 7 is uncertain and there are reasons to believe that the person is a child, that person is presumed to be a child in order to receive immediate access to assistance, support and protection in accordance with Articles 19 and 20. Article 19 Assistance and support to victims 1. Member States shall take the necessary measures to ensure that assistance and support are provided to victims before, during and for an appropriate period of time after the conclusion of criminal proceedings in order to enable them to exercise the rights set out in Framework Decision 2001/220/JHA, and in this Directive. Member States shall, in particular, take the necessary steps to ensure protection for children who report cases of abuse within their family. 2. Member States shall take the necessary measures to ensure that assistance and support for a child victim are not made conditional on the child victims willingness to cooperate in the criminal investigation, prosecution or trial. 3. Member States shall take the necessary measures to ensure that the specific actions to assist and support child victims in enjoying their rights under this Directive, are undertaken following an individual assessment of the special circumstances of each particular child victim, taking due account of the childs views, needs and concerns. 4. Child victims of any of the offences referred to in Articles 3 to 7 shall be considered as particularly vulnerable victims pursuant to Article 2(2), Article 8(4) and Article 14(1) of Framework Decision 2001/220/JHA. 5. Member States shall take measures, where appropriate and possible, to provide assistance and support to the family of the child victim in enjoying the rights under this Directive when the family is in the territory of the Member States. In particular, Member States shall, where appropriate and possible, apply Article 4 of Framework Decision 2001/220/JHA to the family of the child victim. Article 20 Protection of child victims in criminal investigations and proceedings 1. Member States shall take the necessary measures to ensure that in criminal investigations and proceedings, in accordance with the role of victims in the relevant justice system, competent authorities appoint a special representative for the child victim where, under national law, the holders of parental responsibility are precluded from representing the child as a result of a conflict of interest between them and the child victim, or where the child is unaccompanied or separated from the family. 2. Member States shall ensure that child victims have, without delay, access to legal counselling and, in accordance with the role of victims in the relevant justice system, to legal representation, including for the purpose of claiming compensation. Legal counselling and legal representation shall be free of charge where the victim does not have sufficient financial resources. 3. Without prejudice to the rights of the defence, Member States shall take the necessary measures to ensure that in criminal investigations relating to any of the offences referred to in Articles 3 to 7: (a) interviews with the child victim take place without unjustified delay after the facts have been reported to the competent authorities; (b) interviews with the child victim take place, where necessary, in premises designed or adapted for this purpose; (c) interviews with the child victim are carried out by or through professionals trained for this purpose; (d) the same persons, if possible and where appropriate, conduct all interviews with the child victim; (e) the number of interviews is as limited as possible and interviews are carried out only where strictly necessary for the purpose of criminal investigations and proceedings; (f) the child victim may be accompanied by his or her legal representative or, where appropriate, by an adult of his or her choice, unless a reasoned decision has been made to the contrary in respect of that person. 4. Member States shall take the necessary measures to ensure that in criminal investigations of any of the offences referred to in Articles 3 to 7 all interviews with the child victim or, where appropriate, with a child witness, may be audio-visually recorded and that such audio-visually recorded interviews may be used as evidence in criminal court proceedings, in accordance with the rules under their national law. 5. Member States shall take the necessary measures to ensure that in criminal court proceedings relating to any of the offences referred to in Articles 3 to 7, that it may be ordered that: (a) the hearing take place without the presence of the public; (b) the child victim be heard in the courtroom without being present, in particular through the use of appropriate communication technologies. 6. Member States shall take the necessary measures, where in the interest of child victims and taking into account other overriding interests, to protect the privacy, identity and image of child victims, and to prevent the public dissemination of any information that could lead to their identification. Article 21 Measures against advertising abuse opportunities and child sex tourism Member States shall take appropriate measures to prevent or prohibit: (a) the dissemination of material advertising the opportunity to commit any of the offences referred to in Articles 3 to 6; and (b) the organisation for others, whether or not for commercial purposes, of travel arrangements with the purpose of committing any of the offences referred to in Articles 3 to 5. Article 22 Preventive intervention programmes or measures Member States shall take the necessary measures to ensure that persons who fear that they might commit any of the offences referred to in Articles 3 to 7 may have access, where appropriate, to effective intervention programmes or measures designed to evaluate and prevent the risk of such offences being committed. Article 23 Prevention 1. Member States shall take appropriate measures, such as education and training, to discourage and reduce the demand that fosters all forms of sexual exploitation of children. 2. Member States shall take appropriate action, including through the Internet, such as information and awareness-raising campaigns, research and education programmes, where appropriate in cooperation with relevant civil society organisations and other stakeholders, aimed at raising awareness and reducing the risk of children, becoming victims of sexual abuse or exploitation. 3. Member States shall promote regular training for officials likely to come into contact with child victims of sexual abuse or exploitation, including front-line police officers, aimed at enabling them to identify and deal with child victims and potential child victims of sexual abuse or exploitation. Article 24 Intervention programmes or measures on a voluntary basis in the course of or after criminal proceedings 1. Without prejudice to intervention programmes or measures imposed by the competent judicial authorities under national law, Member States shall take the necessary measures to ensure that effective intervention programmes or measures are made available to prevent and minimise the risks of repeated offences of a sexual nature against children. Such programmes or measures shall be accessible at any time during the criminal proceedings, inside and outside prison, in accordance with national law. 2. The intervention programmes or measures, referred to in paragraph 1 shall meet the specific developmental needs of children who sexually offend. 3. Member States shall take the necessary measures to ensure that the following persons may have access to the intervention programmes or measures referred to in paragraph 1: (a) persons subject to criminal proceedings for any of the offences referred to in Articles 3 to 7, under conditions which are neither detrimental nor contrary to the rights of the defence or to the requirements of a fair and impartial trial, and, in particular, in compliance with the principle of the presumption of innocence; and (b) persons convicted of any of the offences referred to in Articles 3 to 7. 4. Member States shall take the necessary measures to ensure that the persons referred to in paragraph 3 are subject to an assessment of the danger that they present and the possible risks of repetition of any of the offences referred to in Articles 3 to 7, with the aim of identifying appropriate intervention programmes or measures. 5. Member States shall take the necessary measures to ensure that the persons referred to in paragraph 3 to whom intervention programmes or measures in accordance with paragraph 4 have been proposed: (a) are fully informed of the reasons for the proposal; (b) consent to their participation in the programmes or measures with full knowledge of the facts; (c) may refuse and, in the case of convicted persons, are made aware of the possible consequences of such a refusal. Article 25 Measures against websites containing or disseminating child pornography 1. Member States shall take the necessary measures to ensure the prompt removal of web pages containing or disseminating child pornography hosted in their territory and to endeavour to obtain the removal of such pages hosted outside of their territory. 2. Member States may take measures to block access to web pages containing or disseminating child pornography towards the Internet users within their territory. These measures must be set by transparent procedures and provide adequate safeguards, in particular to ensure that the restriction is limited to what is necessary and proportionate, and that users are informed of the reason for the restriction. Those safeguards shall also include the possibility of judicial redress. Article 26 Replacement of Framework Decision 2004/68/JHA Framework Decision 2004/68/JHA is hereby replaced in relation to Member States participating in the adoption of this Directive without prejudice to the obligations of those Member States relating to the time limits for transposition of the Framework Decision into national law. In relation to Member States participating in the adoption of this Directive, references to Framework Decision 2004/68/JHA shall be construed as references to this Directive. Article 27 Transposition 1. Member States shall bring into force the laws, regulations and administrative provisions necessary to comply with this Directive by 18 December 2013. 2. Member States shall transmit to the Commission the text of the provisions transposing into their national law the obligations imposed on them under this Directive. 3. When Member States adopt those measures, they shall contain a reference to this Directive or be accompanied by such a reference on the occasion of their official publication. The methods of making such reference shall be laid down by the Member States. Article 28 Reporting 1. The Commission shall, by 18 December 2015, submit a report to the European Parliament and the Council assessing the extent to which the Member States have taken the necessary measures in order to comply with this Directive, accompanied, if necessary, by a legislative proposal. 2. The Commission shall, by 18 December 2015, submit a report to the European Parliament and the Council assessing the implementation of the measures referred to in Article 25. Article 29 Entry into force This Directive shall enter into force on the day of its publication in the Official Journal of the European Union. Article 30 Addressees This Directive is addressed to the Member States in accordance with the Treaties. Done at Strasbourg, 13 December 2011. For the European Parliament The President J. BUZEK For the Council The President M. SZPUNAR (1) OJ C 48, 15.2.2011, p. 138. (2) Position of the European Parliament of 27 October 2011 (not yet published in the Official Journal) and decision of the Council of 15 November 2011. (3) OJ C 364, 18.12.2000, p. 1. (4) OJ C 115, 4.5.2010, p. 1. (5) OJ L 13, 20.1.2004, p. 44. (6) OJ L 82, 22.3.2001, p. 1. (7) OJ L 328, 15.12.2009, p. 42. (8) OJ L 101, 15.4.2011, p. 1. (9) OJ L 182, 5.7.2001, p. 1. (10) OJ L 68, 15.3.2005, p. 49. (11) OJ L 49, 17.2.2007, p. 30. (12) OJ L 300, 11.11.2008, p. 42. (13) OJ L 93, 7.4.2009, p. 23.
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32019D0417
        'status': In Force
        'act_type': Decision
        'treaty': TFEU

        full_doc :

        15.3.2019 EN Official Journal of the European Union L 73/121 COMMISSION IMPLEMENTING DECISION (EU) 2019/417 of 8 November 2018 laying down guidelines for the management of the European Union Rapid Information System RAPEX established under Article 12 of Directive 2001/95/EC on general product safety and its notification system (notified under document C(2018) 7334) THE EUROPEAN COMMISSION, Having regard to the Treaty on the Functioning of the European Union, Having regard to Directive 2001/95/EC of the European Parliament and of the Council of 3 December 2001 on general product safety (1), and in particular the third subparagraph of Article 11(1) and point 8 of Annex II thereof, Having regard to Regulation (EC) No 765/2008 of the European Parliament and of the Council of 9 July 2008, setting out the requirements for accreditation and market surveillance on the marketing of products and repealing Regulation (EEC) No 339/93 (2), After consulting the Advisory Committee, set up by Article 15 of Directive 2001/95/EC, Whereas: (1) Article 12 of Directive 2001/95/EC establishes a European Union Rapid Information System (RAPEX) for the rapid exchange of information between the Member States and the Commission on measures and action taken on products posing a serious risk to the health and safety of consumers. (2) Point 8 of Annex II to Directive 2001/95/EC requires the guidelines to be regularly updated in the light of new developments and experience. Commission Decision 2010/15/EU (3) was the first and only update of the guidelines. (3) In view of new developments and in order to ensure more efficient and effective notification procedures in line with best practice, a further update of the guidelines is required. (4) Terminology and references have become obsolete as has the means of communication between the Commission and the Member States authorities and between the authorities themselves. (5) New tools that have been developed over the last years for the proper functioning of RAPEX (wiki's, interface between RAPEX and other market surveillance systems) have to be taken into consideration in the guidelines. (6) Criteria for the RAPEX notification, following the new developments, have become unclear and need to be clarified. (7) Cross-border online sales of goods have increased. This development needs to be reflected in the notification techniques as well as in the follow-up instruments to be used. (8) Regulation (EC) No 765/2008 extends the application of RAPEX provided for in Article 12 of Directive 2001/95/EC also to products covered by that legislation. Extending the application of RAPEX raises some issues that need to be clarified in the guidelines. (9) Regulation (EC) No 765/2008 applies to consumer products and to professional products such as some medical devices. That Regulation covers a broader scope of risks, other than those related to the health and safety of consumers, such as security and environmental risks. Therefore, a risk can concern not only consumers but also an indeterminate group of people referred to as end-users. (10) Article 22 of Regulation (EC) No 765/2008 provides therefore that measures taken against products presenting a serious risk to health and safety or other relevant public interests should be notified via RAPEX. (11) Directive 2001/95/EC and Regulation (EC) No 765/2008 are complementary and provide a system to improve the safety of non-food products. (12) RAPEX helps to prevent and restrict the supply of products posing a serious risk to health and safety or, in the case of products covered by Regulation (EC) No 765/2008, also to other relevant public interests. It enables the Commission to monitor the effectiveness and consistency of market surveillance and enforcement activities in the Member States. (13) RAPEX provides a basis for identifying the need for action at EU level and makes for consistent enforcement of EU product safety requirements and therefore contributes to the smooth functioning of the single market. (14) The notification procedure established under Article 11 of Directive 2001/95/EC provides for an exchange of information between the Member States and the Commission on measures adopted on products posing a less than serious risk to the health and safety of consumers. It helps to ensure a consistent, high level of consumer health and to preserve the single market. (15) Article 23 of Regulation (EC) No 765/2008 provides for an information support system where Member States make available to the Commission the information required by the same Article on products presenting a less than serious risk. (16) According to the applicable legislation, Member States are not obliged to provide such information in the RAPEX system. (17) Article 16 of the GPSD provides an obligation for Member States and the Commission to make available to the public information relating to risks to consumer health and safety posed by products. (18) To ensure a coherent system of information for products posing a risk to the health and safety of consumers or, in case of products covered by Regulation (EC) No 765/2008, also to other relevant public interests, it would be desirable that available information concerning dangerous products covered by Article 23 of Regulation (EC) No 765/2008 could also be made available in the RAPEX system. (19) In order to enable the functioning of the RAPEX system, guidelines should be drawn up on the various aspects of these notification procedures and, in particular, to establish the content of notifications. These should include the information to be contained in the notification, criteria for notifications involving risks that do not or cannot go beyond the territory of the Member State and criteria for the classification of notifications according to the degree of urgency. The guidelines should also lay down operating arrangements, including deadlines for the various steps of the notification and follow-up notification procedures as well as confidentiality rules. (20) To ensure that notification procedures are properly applied, the guidelines should also set out risk assessment methods with criteria for identifying risks taking into consideration also the management of risks. (21) In the light of point 2 in Annex II to Directive 2001/95/EC, the new guidelines include a set of risk assessment guidelines for consumer products and refer also to professional products, which specify the criteria for identifying serious risks. (22) The guidelines should be addressed to all Member States authorities participating in the RAPEX network pursuant to Directive 2001/95/EC and Regulation EC No 765/2008, including market surveillance authorities responsible for monitoring the compliance of products with safety requirements and authorities in charge of external border controls, HAS ADOPTED THIS DECISION: Article 1 The guidelines for the management of the European Union Rapid Information System RAPEX established under Article 12 of Directive 2001/95/EC and its notification system are set out in the Annex to this Decision. Article 2 Decision 2010/15/EU is repealed. Article 3 This Decision is addressed to the Member States. Done at Brussels, 8 November 2018. For the Commission VÃ ra JOUROVÃ  Member of the Commission (1) OJ L 11, 15.1.2002, p. 4. (2) OJ L 218, 13.8.2008, p. 30. (3) Commission Decision 2010/15/EU of 16 December 2009 laying down guidelines for the management of the Community Rapid Information System RAPEX established under Article 12 and of the notification procedure established under Article 11 of Directive 2001/95/EC (the General Product Safety Directive) (OJ L 22, 26.1.2010, p. 1). ANNEX GUIDELINES FOR THE MANAGEMENT OF THE EUROPEAN UNION RAPID INFORMATION SYSTEM RAPEX ESTABLISHED UNDER ARTICLE 12 OF DIRECTIVE 2001/95/EC (THE GENERAL PRODUCT SAFETY DIRECTIVE) AND ITS NOTIFICATION SYSTEM PART I SCOPE AND ADDRESSEES OF THE GUIDELINES 1. Scope, objectives and update 1.1. Scope The Guidelines for the management of the European Union Rapid Information System RAPEX established under Article 12 of Directive 2001/95/EC on general product safety (the Guidelines) are adopted by the Commission (1) under Article 11(1) and Annex II, point 8, of Directive 2001/95/EC (the GPSD). The Commission is assisted by an advisory committee composed of the representatives from EU Member States and established under Article 15(3) of the GPSD. Point 8 of Annex II to the GPSD states that: The Commission shall prepare and regularly update, in accordance with the procedure laid down in Article 15(3), guidelines concerning the management of RAPEX by the Commission and Member States.. Article 11 of the GPSD prescribes that Member States should inform the Commission of measures taken which restrict the placing on the market of products  or require their withdrawal or recall  to the extent that such information is not eligible for the type of notification Article 12 of the GPSD provides for, nor does it qualify for any other notification under any specific Community legislation. Article 22 of Regulation (EC) No 765/2008, provides that, where a Member State takes or intends to take a measure that prevents, restricts or imposes specific conditions on the marketing and use of products posing a serious risk to the health, safety and other relevant public interests of the end-users, it must immediately notify such a measure to the Commission using RAPEX. Article 23 of Regulation (EC) No 765/2008 provides that Member States must make available to the Commission the information at their disposal, and not already provided under Article 22, on products presenting a (less than serious) risk. Article 16 of the GPSD provides an obligation for Member States and the Commission to make available to the public information relating to risks to consumer health and safety posed by products. It would therefore be opportune that all information on measures adopted against products posing a risk, insofar as product safety is at stake, are contained in the system intended for this purpose. Member States are therefore encouraged to provide RAPEX with the measures adopted against products posing a risk and entering into the scope of application of the GPSD or Regulation (EC) No 765/2008. The information can be provided directly in RAPEX. In case the information has to be notified in another information system according to Regulation (EC) No 765/2008 (2), the Member State can generate a RAPEX notification from within the information system (see Part II, Chapters 1.2(h) and 2.2 of these Guidelines). Whereas the GPSD applies only to consumer products posing a risk to the health and safety of consumers, Regulation (EC) No 765/2008 applies to consumer products but also professional products covered by EU harmonisation legislation (such as certain medical devices and marine equipment). It also covers a broader scope of risk, in addition to those related to the health and safety of consumers, such as security and environmental risks. Therefore, a risk can concern not only consumers but also, where Regulation (EC) No 765/2008 applies, other end-users. Risk Assessment Guidelines of Appendix 6 on Part III are an integral part of the RAPEX Guidelines. They are the instruments that enable determining the level of risk of a product and therefore help to identify the measures to be adopted. The Risk Assessment Guidelines refer to the level of risk as well as to the possible injuries caused by a single product. The risk assessment for a single product must be accompanied by sound risk management. For example, the risk level for a defective household electrical appliance posing a risk of fire may be only low, meaning that the probability of a single appliance causing a fatal fire during the lifetime of the appliance is less than one in a million. Nevertheless, if millions of the defective appliances have been placed on the market, it is almost inevitable that fatal fires will occur if appropriate measures are not taken. Member States (3), applicant countries, countries which are parties to the European Economic Area (EEA) Agreement as well as other non-EU countries and international organisations that are granted access to RAPEX (on the conditions defined in Article 12(4) of the GPSD), participate in the system according to the rules provided for in the GPSD and these Guidelines (4). 1.2. Objectives The objectives of these Guidelines are to: (a) streamline the processes for the notification mechanisms; (b) set out the notification criteria for the notification mechanisms; (c) define the content of notifications and follow-up notifications sent under the notification mechanism, in particular what data are required and which forms are to be used; (d) establish follow-up activities to be taken by Member States upon receipt of a notification and the type of information to be provided; (e) describe the handling of notifications and follow-up notifications by the Commission; (f) set deadlines for the various types of action taken under the notification mechanisms; (g) set out the practical and technical arrangements needed at Commission and Member State level for the notification mechanisms to be employed effectively and efficiently; and (h) establish risk assessment methods and, in particular, criteria for identifying serious risks. 1.3. Update The Guidelines will be regularly updated by the Commission in accordance with the advisory procedure on the basis of experience and new developments in the product safety area. 2. Addressees of the Guidelines The Guidelines are addressed to all Member States authorities acting on product safety and participating in the RAPEX network, including market surveillance authorities responsible for monitoring the compliance of products with safety requirements and authorities in charge of external border controls. 3. Products 3.1. Products covered by these Guidelines These Guidelines cover two sets of products: the products covered by the GPSD and the products covered by Regulation (EC) No 765/2008. 3.1.1. Products covered by the GPSD Under Article 2(a) of the GPSD, consumer products for the purpose of these Guidelines are: (a) products intended for consumers  products that are designed and manufactured for and made available to consumers; (b) migrating products (5)  products that are designed and manufactured for professionals, which are likely, however, under reasonably foreseeable conditions, to be used by consumers. These are products manufactured for professionals that are made available to consumers, who can purchase and operate them without any special knowledge or training, e.g. a power drill, an angle grinder and a table saw designed and manufactured for professionals, but also supplied on the consumer market (i.e. consumers can readily purchase them in shops and operate them on their own without any special training). Both products intended for consumers and migrating products can be given to consumers free of charge, can be purchased by consumers and can be provided to consumers in the context of a service. All three situations are covered by RAPEX. According to Article 2 (a) of the GPSD, products provided to consumers in the context of a service are to be considered as including: (a) products supplied to consumers that are taken away and used outside the premises of a service provider, such as cars and lawn-mowing machines rented or leased in rental shops, and tattoo inks and implants (that are not classified as medical devices) implanted beneath the skin of a consumer by a service provider; (b) products used on the premises of a service provider, provided that consumers themselves actively operate a product (e.g. start the machine, have the option of stopping it, and affect its operation by changing its position or intensity during use). Sun-beds used in tanning salons and fitness centres are examples of such products. Use of the products by consumers must be active, and involve a significant degree of control. Merely passive use, such as the use of a shampoo by a person whose hair is washed by a hairdresser, or the use of a bus by its passengers, does not qualify as use by consumers. 3.1.2. Products covered by Regulation (EC) No 765/2008 Under Regulation (EC) No 765/2008, products for the purpose of RAPEX are to be considered the products according to the scope and definitions contained in Article 15 of the same Regulation whether intended for consumers or for professional users. 3.2. Products not covered by these Guidelines These Guidelines do not cover: (a) Products that are covered by specific and equivalent notification mechanisms established by other EU legislation, notably: (i) food and feed and other products covered by Regulation (EC) No 178/2002 of the European Parliament and of the Council (6); (ii) medicinal products covered by Directive 2001/83/EC of the European Parliament and of the Council (7), and Directive 2001/82/EC of the European Parliament and of the Council (8); (iii) medical devices covered by Regulation (EU) 2017/745 of the European Parliament and of the Council (9); (iv) active implantable medical devices covered by Council Directive 90/385/EEC (10). (b) Products that are not covered by the definition of a product as laid down in Article 2(a) of the GPSD, notably: (i) second-hand products or products supplied as antiques or as products to be repaired or reconditioned prior to being used, provided that the supplier clearly informs the person to whom he supplies the product to that effect (Article 2(a) of the GPSD); (ii) equipment used or operated by a professional service provider to supply a service, e.g. equipment on which consumers ride or travel and equipment which is operated by a service provider and not by the consumer (recital 9 of the GPSD); (c) Products which do not enter into the definition of product contained in Article 15(4) of Regulation (EC) No 765/2008. 4. Measures 4.1. Types of measures Preventive and restrictive measures can be taken in relation to products posing a risk either on the initiative of the economic operator who placed and/or distributed it on the market (voluntary measures), or as ordered by an authority of a Member State competent to monitor the compliance of products with the safety requirements (compulsory measures). For the purpose of these Guidelines, the compulsory measures and voluntary measures are defined as follows: (a) Compulsory measures: measures adopted or decided to be adopted by Member State authorities, often in the form of an administrative decision, which oblige an economic operator to take preventive, corrective or restrictive action in relation to a specific product that they made available on the market. (b) Voluntary measures: (i) preventive and restrictive measures adopted on a voluntary basis by an economic operator, i.e. without any intervention of an authority of a Member State; (ii) recommendations and agreements with economic operators in their respective activities concluded by Member State authorities; this includes agreements which are not in written form and result in preventive or restrictive action taken by economic operators in their respective activities in relation to products posing a serious risk that they made available on the market. 4.2. Categories of measures Article 8(1)(b) to (f) of the GPSD provides a list of the different categories of measures that are notifiable under RAPEX when the conditions for notification are fulfilled, including the following measures: (a) marking a product with appropriate warnings on the risk(s) it may present; (b) making the marketing of a product subject to prior conditions; (c) warning consumers and end-users of the risks that could be posed by a product; (d) temporary ban on the supply, offer to supply and display of a product; (e) ban on the marketing of a product and any accompanying measures, i.e. measures required to ensure compliance with the ban; (f) withdrawal of a product from the market; (g) recall of a product from consumers; (h) destruction of a withdrawn or recalled product. For the purpose of RAPEX, the term withdrawal is used exclusively for measures aimed at preventing the distribution, display and offer of a product posing a risk to consumers or other end-users, while the term recall is used only for measures aimed at achieving the return of such a product that has already been made available to consumers or other end-users by a producer or distributor. 4.3. Requirements of the measures Under Article 12(1) of the GPSD and Article 22 of Regulation (EC) No 765/2008 concerning serious risks, both compulsory and voluntary measures are to be notified in RAPEX. Preventive and restrictive measures adopted on a voluntary basis by an economic operator, i.e. without any intervention of an authority of a Member State concerning a product posing a serious risk and the related preventive or restrictive measures initiated by an economic operator should be immediately notified to the competent authorities of Member States as indicated in Article 5(3) of the GPSD and in Article 22(2) and (3) of Regulation (EC) No 765/2008. All categories of preventive and restrictive measures taken in relation to the marketing and use of consumer products posing a serious risk to the health and safety of consumers or, in the case of products covered by Regulation (EC) No 765/2008, posing a serious risk to the health, safety or other relevant public interests of the end-users are subject to the notification obligation under RAPEX. 4.4. Exclusion of generally applicable compulsory measures Generally applicable acts adopted at national level and aimed at preventing or restricting the marketing and use of (a) generally described category(ies) of consumer products due to the serious risk they pose to the health and safety of consumers should not be notified to the Commission through the RAPEX application. All such national measures that apply to only generally defined categories of products, such as all products in general or all products serving the same purpose  and not to (categories of) products specifically identified by their brand, specific look, producer, trader, model name or number, etc.  are notified to the Commission under Directive (EU) 2015/1535 of the European Parliament and of the Council (11). 5. Risk Levels 5.1. Serious risk Before an authority of a Member State decides to submit a RAPEX notification, it always performs an appropriate risk assessment (see Part III, Appendix 6 of these Guidelines or the complementary EU general risk assessment methodology for products covered by Regulation (EC) No 765/2008 (12)) in order to assess whether the product to be notified poses a serious risk to the health and safety of consumers or, in the case of products covered by Regulation (EC) No 765/2008, a serious risk to the health, safety or to other relevant public interests (for example, security or the environment) of the end-users, and thus whether one of the RAPEX notification criteria is met. 5.2. Less than serious risk Notifications sent in accordance with Article 11 of the GPSD or Article 23 of Regulation (EC) No 765/2008 are generally considered as notifications for products posing a less than serious risk. Notifications of such products, contrary to notifications for products presenting a serious risk, do not necessarily involve an obligation for follow-up activities by other Member States unless the nature of the product or of the risk so requires (see Part II Chapter 3.4.6.1). 5.3. Risk assessment method Part III, Appendix 6 to these Guidelines sets out a risk assessment method that can be used by Member State authorities to assess the level of risks posed by consumer products to the health and safety of consumers and to decide whether a RAPEX notification is necessary. Equally, you may need to consult the complementary EU general risk assessment methodology as referred to in Chapter 5.1 in case the product concerned is covered by Regulation (EC) No 765/2008. A specific tool (RAG or Risk Assessment Guidelines (13)) is available on the RAPEX website and in the RAPEX application to perform risk assessments, which takes accounts of the principles provided for in Appendix 6. 5.4. Assessing authority The risk assessment is always performed or checked by the authority of a Member State that either carried out the investigation and took appropriate measures, or which monitored the voluntary action taken with regard to a product posing a risk by an economic operator. Any unclear issues are resolved by the RAPEX Contact Point (see Part II, Chapter 5.1) with the authority responsible before a notification is transmitted through the RAPEX application. 6. Cross-border effects 6.1. International event Under Article 12 of the GPSD and Article 22 of Regulation (EC) No 765/2008, a Member State submits a RAPEX notification only if it considers that the effects of the risk(s) posed by a product go or can go beyond its territory (cross-border effects or international event). In the light of the free movement of products in the internal market, and the fact that products are imported into the EU through different distribution channels and that consumers buy products during stays abroad and via the internet, national authorities are encouraged to interpret the cross-border effects criterion in a fairly broad sense. An Article 12 of the GPSD or Article 22 notification of Regulation (EC) No 765/2008, therefore, is submitted where: (a) it cannot be excluded that a product posing a risk has been sold in more than one EU Member State; or (b) it cannot be excluded that a product posing a risk has been sold via the internet; or (c) the product originates from a third country and is likely to have been imported into the EU through multiple distribution channels. 6.2. Local event Measures adopted in relation to a product posing a serious risk that can only have a local effect (Local event) are not notified under Article 12 of the GPSD. This applies in situations where an authority of a Member State has concrete and strong reasons to exclude the possibility that a product has been and or will be made available (by any means) in other Member States, e.g. measures taken with regard to a local product manufactured and distributed only in one Member State. In its evaluation, the authorities of the Member State have to take carefully into consideration the possibility that a product could be sold online or through new emerging distribution channels. A notification in relation to a product posing a serious risk involving a local event only requires to be submitted to the Commission insofar as it involves information likely to be of interest to Member States from the product safety standpoint, and in particular if they are in response to a new type of risk which has not yet been notified, a new type of risk arising from a combination of products or a new type or category of products. Such notification is to be submitted under Article 11 with reference to the second subparagraph of Article 11(1), of the GPSD. PART II EU RAPID INFORMATION SYSTEM RAPEX ESTABLISHED UNDER ARTICLE 12 OF THE GENERAL PRODUCT SAFETY DIRECTIVE 1. Introduction 1.1. Objectives of RAPEX Article 12 of the GPSD establishes an EU Rapid Information System (RAPEX). RAPEX plays an important role in the area of product safety. It complements other actions taken both at national and at EU level to ensure a high level of product safety in the EU. RAPEX data helps to: (a) prevent and restrict the supply of dangerous products; (b) monitor the effectiveness and consistency of market surveillance and enforcement activities carried out by Member State authorities; (c) identify needs and provide a basis for action at EU level; and (d) make for consistent enforcement of the EU product safety requirements and therefore contribute to the smooth functioning of the single market. 1.2. Components of RAPEX RAPEX consists of several complementary components, which are crucial for its effective and efficient operation. The most important are: (a) the legal framework that regulates how the system operates (i.e. the GPSD and the Guidelines); (b) the online application (the RAPEX application), which allows Member States and the Commission to exchange information rapidly via a web-based platform; (c) the RAPEX Contact Points network, which consists of the single RAPEX Contact Points responsible for operating RAPEX in all Member States (see Part II, Chapter 5.1); (d) the national RAPEX networks established in all Member States, which include the RAPEX Contact Point (see Part II, Chapter 5.1) and all the authorities involved in ensuring product safety; (e) the Commission RAPEX team in the department responsible for the GPSD, which examines and validates documents submitted through the RAPEX application, and maintains and ensures correct operation of RAPEX; (f) the RAPEX website (14), which provides summaries of RAPEX notifications as well as weekly updates; (g) RAPEX publications, such as RAPEX statistics, RAPEX annual reports and other promotional materials; and (h) the interface between RAPEX and ICSMS, which consists on a link between both systems that facilitates the encoding of RAPEX notifications based on investigation data already available in ICSMS. By filling in the appropriate fields in ICSMS, a RAPEX notification can be automatically submitted. 2. Notification criteria RAPEX applies to measures which prevent, restrict or impose specific conditions on the marketing and use of products posing a serious risk to the health and safety of consumers or, in the case of products covered by Regulation (EC) No 765/2008, to measures which prevent, restrict or impose specific conditions on the marketing and use of products posing a serious risk to the health, safety or other relevant public interests (for example, security or the environment) of the end-users. 2.1. Mandatory participation in RAPEX: Article 12 of the GPSD and Article 22 of Regulation (EC) No 765/2008 Under the GPSD and Regulation (EC) No 765/2008, the participation of Member States in RAPEX is mandatory. According to Article 12 of the GPSD and Article 22 of Regulation (EC) No 765/2008 Member States have a legal obligation to notify the Commission both compulsory and voluntary measures when the following four notification criteria are met: (a) the product falls under the scope of application of the GPSD or under the scope of application of Regulation (EC) No 765/2008; (b) the product is subject to measures that prevent, restrict or impose specific conditions on its possible marketing or use (preventive and restrictive measures); (c) the product poses a serious risk to the health and safety of consumers or, in case of products covered by Regulation (EC) No 765/2008, also to other relevant public interests of the end-users; (d) it cannot be ruled out that the effect of the serious risk to the health and safety of consumers or, in case of products covered by Regulation (EC) No 765/2008, also to other relevant public interests of the end-users, goes beyond the territory of the notifying Member State. 2.2. Non-mandatory participation in RAPEX: Article 11 of the GPSD and Article 23 of Regulation (EC) No 765/2008 According to Article 11 of the GPSD, Member States should inform the Commission of measures taken which restrict the placing on the market of products  or require their withdrawal or recall  insofar such information does not qualify for an Article 12 nor any other notification set out in any specific Community legislation. For the sake of simplification and efficiency gains, Member States may also make use of the RAPEX application to notify measures taken against products which would not qualify for submitting an Article 12 notification in the terms outlined herein. Where the following four notification criteria are met, Member States have a legal obligation to notify the Commission under Article 11 of the GPSD: (a) the product concerned is a consumer product; (b) it is subject to restrictive measures adopted by national authorities (compulsory measures); (c) it poses a less than serious risk to the health and safety of consumers and the effects of which can or do go beyond the territory of one Member State or, it poses a serious risk to the health and safety of consumers and the effect of which do not or cannot go beyond its territory yet the measures adopted involve information likely to be of interest to other Member States from a product safety standpoint (15); (d) The measures adopted do not have to be notified under any other notification procedure established by EU law. Notwithstanding the fact that Article 11 of the GPSD does not contain an explicit obligation to notify voluntary measures adopted against products posing a less than serious risk, Article 16 of the GPSD requires Member States and the Commission to make information relating to risks to consumer health and safety available to the public. Therefore, for the sake of coherence in the notification system and to effectively implement the obligations both Member States and the Commission have according to Article 16 of the GPSD, Member States are recommended to notify in RAPEX also voluntary measures adopted by the producers and distributors against products posing a less than serious risk. According to Article 23 of Regulation (EC) No 765/2008 Member States provide the Commission with information at their disposal, and not already provided under Article 22, on products presenting a (less than serious) risk. Contrary to Article 22 of this Regulation, Article 23 does not oblige Member States to submit a notification to RAPEX with this information. Article 16 of the GPSD obliges, though, the Commission and the Member States to make public the information they may have relating to risks to consumer health and safety. For the sake of coherence and to effectively implement the obligations contained in Article 16 of the GPSD, the most pragmatic solution could be for RAPEX to contain all measures adopted against products presenting serious and less than serious risks to consumer health and safety both for GPSD products and products covered by Regulation (EC) No 765/2008, and in the latter case, also to other relevant public interests of the end-users. Therefore, when measures are adopted and provided through ICSMS according to Article 23 of Regulation (EC) No 765/2008, Member States are encouraged to notify such information in RAPEX. This can be done either by submitting a separate notification in RAPEX or through ICSMS. A link between both systems facilitates the encoding of notifications based on investigation data already available in ICSMS. (See Part II, Chapter 1.2(h)). Type of risk Product covered by the GPSD Product covered by Regulation (EC) 765/2008 Measure adopted Cross-border effect Unsufficient Identification Information Information Involving new risk NOTIFICATION TYPE Serious risk Article 12 of the GPSD Article 11 of the GPSD Article 22 of Regulation (EC) No 765/2008 Indistinctly For information Indistinctly Information to ICSMS RAPEX notification encouraged Less than serious risk Compulsory measures Article 11 of the GPSD Voluntary measures For information Article 23 of Regulation (EC) No 765/2008 RAPEX notification encouraged Pending For information (if relevant) A notification scheme is included in Part III, Appendix 3 of these Guidelines providing further clarification on the notification criteria referred to in Part II Chapter 2 of these Guidelines. 3. Notifications 3.1. Types of notification 3.1.1. Notifications The Authorities of the Member States are required to submit a notification to the RAPEX system in the following cases: (a) where all the RAPEX notification criteria laid down in Article 12 of the GPSD (16) are met, a Member State prepares and submits to the Commission a RAPEX notification classified in the RAPEX application as an Article 12 notification. (b) where all the RAPEX notification criteria are met and, in addition, a product poses a life-threatening risk and/or there have been fatal accidents, and in other cases where a RAPEX notification requires emergency action by all Member States, the notifying Member State prepares and submits to the Commission a RAPEX notification classified in the RAPEX application as a Notification requiring emergency action. (c) where all RAPEX notification criteria laid down in Article 22 of Regulation (EC) No 765/2008 (17) are met, a Member State prepares and submits to the Commission a RAPEX notification classified in the RAPEX application as an Article 22 notification. Where all notification criteria laid own in Article 11 of the GPSD (18) are met, a Member State prepares and submits to the Commission a notification, which, when notified in RAPEX is classified as an Article 11 notification. Moreover, Member States are encouraged to submit a notification where the criteria laid down in Article 23 of Regulation (EC) No 765/2008 are met (19). Following the abovementioned reasoning in Part II Chapter 2, Member States are encouraged to prepare and submit, either directly or indirectly, to the Commission a notification classified in RAPEX as an Article 23 notification when the criteria laid down in the same article are met. Before sending a notification to the Commission, the RAPEX Contact Point (see Part II, Chapter 5.1) of the notifying Member State checks that all notification criteria are met. 3.1.2. Notifications for information If the criteria laid down in these Guidelines for the notifications listed in Part II Chapters 2.1 and 2.2 of these Guidelines are not met, the RAPEX Contact Point (see Part II, Chapter 5.1) may choose to use the RAPEX application to send the information concerned for information purposes. Such notifications are classified in RAPEX as Notifications for information and they may be sent in the following situations: (a) Where all the RAPEX notification criteria laid down in Article 12 of the GPSD or in Article 22 of Regulation (EC) No 765/2008 are met but a notification does not contain all the information (mainly on product identification and distribution channels) necessary for other Member States to ensure follow-up (20) to such a notification. A notification where the product name, brand and picture are missing and thus the notified product cannot be correctly identified and it cannot be distinguished from other products of the same category or type that are available on the market, is an example of a notification that can be distributed through the RAPEX application as Notification for information. Assessment as to whether a notification contains sufficient information for other Member States to ensure follow-up activities is always on a case-by-case basis. (b) Where a Member State is aware of the fact that a consumer product that is available on the EU market poses a serious risk to the health and safety of consumers or, in the case of products covered by Regulation (EC) 765/2008, is aware of the fact that a consumer or a professional product poses a serious risk to the health and safety or other relevant public interests of the end-users, but preventive and restrictive measures have not yet been taken by the producer or distributor or adopted or decided to be adopted by an authority of a Member State. If information on such a product is distributed through the RAPEX application before measures are taken, the notifying Member State subsequently informs the Commission (as soon as possible and not later than the deadlines specified in Appendix 4 to these Guidelines) of the final decision taken with regard to the notified product (mainly, what type of preventive or restrictive measures were taken or why such measures were not taken). Where the notifying Member State takes measures at a later stage, it informs the Commission, who will update the notification in application of Article 12 of the GPSD or Article 22 of Regulation (EC) No 765/2008. (c) Where a Member State decides to notify preventive and restrictive measures taken in relation to a consumer product posing a serious risk to the health and safety of consumers which has only local effects (local event). If, however, as explained in Part I, Chapter 6.2, a notification by local event involves information on product safety likely to be of interest for other Member States, it should be sent as if it were a notification under Article 11 of the GPSD. (d) Where a notification concerns a product whose safety aspects (especially the level of risk posed to the health and safety of consumers) are subject to discussion at EU level to ensure a common approach between Member States to risk assessment and/or enforcement action (21). (e) Where a decision cannot be taken with certainty that one or more of the notification criteria are met, but a notification involves information on product safety likely to be of interest for other Member States. When sending a Notification for information, the RAPEX Contact Point (see Part II, Chapter 5.1) clearly states the reasons for so doing. 3.2. Content of notifications 3.2.1. Scope of data Notifications sent to the Commission through the RAPEX application include the following types of data: (a) Information enabling the notified product to be identified, i.e. product category, product name, brand, model and/or type number, barcode, batch or serial number, customs code, description of the product and its packaging accompanied by pictures showing the product, its packaging and labels. Detailed and accurate product identification is a key element for market surveillance and enforcement, as it allows national authorities to identify the notified product, to distinguish it from other products of the same or similar type or category that are available on the market and to find it on the market and take or agree on appropriate measures. (b) Information establishing the product's origin, i.e. country of origin, name, address and contact details, such as telephone number and e-mail address, of a manufacturer and exporters. In particular, Member States provide all available information on manufacturers and exporters located in third countries that cooperate closely with the EU on product safety. The following documents are also to be attached to the form where available: copies of orders, sales contracts, invoices, shipping documents, customs declarations, etc. These documents should be transmitted in pdf format or any other format accepted by the application. Detailed information on third country producers allows the Commission to promote more effective enforcement in those countries and helps to reduce the number of products posing a risk to consumers exported into the EU. (c) Wherever possible, information about where exactly the product has been made available (a major store, local shop or market, online, etc.). (d) Information on the safety requirements applicable to the notified product, including the reference number and name of the applicable legislation and standards. (e) A risk description of the notified product, including a description of the results of laboratory or visual tests, test reports and certificates proving non-compliance of the notified product with the safety requirements, a complete risk assessment with conclusions and information on known accidents or incidents (see Part I Chapter 3.3.1 of these Guidelines). (f) Information on the supply chains of the notified product in the Member States and, in particular, information on the countries of destination, plus information on importers and also, if available, on distributors of the notified product in Europe. (g) Information on measures taken, in particular, the type (compulsory or voluntary), category (e.g. withdrawal from the market, recall from consumers), scope (e.g. national, local), and date of entry into force and duration of the measure (e.g. permanent, temporary). (h) Indication of whether a notification, part of it and/or attachment(s) are covered by confidentiality. Requests for confidentiality are always accompanied by a justification clearly stating the reasons for such a request. (i) Information on whether the product is counterfeit, when available. For this purpose, the Commission will provide Member States with any specific tools available at European level to facilitate the identification of counterfeit products. (j) Information on reported accidents related to the product, indicating when possible the reasons for the accident (risk related to the use made by the user or inherent to the product). (k) Additional information on whether the notification has been submitted in the context of a coordinated enforcement activity at European level. (l) Information on whether the authorities of a Member State envisage sending other notifications related to the same product or similar products. This should be indicated in the original notification. Member States are encouraged to look for and provide information on the supply chains of the notified product in non-EU countries that cooperate closely with the EU on product safety. 3.2.2. Completeness of data Notifications should be as complete as possible. The elements to be contained in the notification are listed in Appendix 1 to these Guidelines and are included in the RAPEX application. All fields of the notification template should be completed with the required data. Where the required information is not available at the time a notification is submitted, this is clearly indicated and explained on the form by the notifying Member State. Once the missing information becomes available, the notifying Member State updates its notification. The updated notification is examined by the Commission before being validated and distributed through the system. RAPEX Contact Points provide all national authorities that participate in the RAPEX network with instructions on the scope of data required to complete the notification. This helps to ensure that the information provided by these authorities to the RAPEX Contact Point is correct and complete (see Part II, Chapter 5.1). Where part of the information required by these Guidelines is not yet available, Member States should nonetheless comply with the established deadlines and not delay sending a RAPEX notification on a product that poses a life-threatening risk to the health and safety of consumers or other end-users and/or where a RAPEX notification requires emergency action by Member States. Before submitting a notification, the RAPEX Contact Point checks (to avoid any unnecessary duplication) that the product concerned has not already been notified through the RAPEX application by another Member State. If the product has already been notified, rather than creating a new notification, the RAPEX Contact Point submits a follow-up notification to the existing notification and provides any additional information that may be relevant for authorities in other Member States, such as additional vehicle identification numbers, a detailed list of importers and distributors, additional test reports, etc. (See also Part II, Chapter 5.1). 3.2.3. Updating of data The notifying Member State informs the Commission (as soon as possible and not later than by the deadlines specified in Appendix 4 to these Guidelines) of any developments that require changes to a notification transmitted through the RAPEX application. In particular, Member States inform the Commission of any changes (e.g. following a ruling by a court during an appeal procedure) to the status of the notified measures, to the risk assessment and to new decisions regarding confidentiality. The Commission examines the information provided by the notifying Member State and updates the information concerned in the RAPEX application and on the RAPEX website, where necessary. 3.2.4. Responsibility for the information transmitted Responsibility for the information provided lies with the notifying Member State (22). The notifying Member State and the national authority responsible ensure that all data provided through the RAPEX application are accurate so as to avoid any confusion with similar products of the same category or type that are available on the EU market. The authority(ies) involved in the notification procedure (e.g. by performing the risk assessment of the notified product or by providing information on distribution channels) take responsibility for the information provided through the RAPEX application. The RAPEX Contact Point checks and validates all notifications received from the authorities responsible before transmitting them to the Commission (See also Part II, Chapter 5.1). Any action taken by the Commission, such as examining notifications, validating and distributing them through the RAPEX application and publishing them on the RAPEX website, does not imply any assumption of responsibility for the information transmitted, which remains with the notifying Member State. 3.3. Actors and roles involved in the notification process The parties involved in the notification process and their responsibilities therein are the following: 3.3.1. Economic operators Economic operators are not directly involved in the submission of notifications in the RAPEX application. However, in case of a product posing a risk, economic operators shall immediately inform the competent authorities in all Member States where the product was made available. The conditions and details for providing such information are laid down in Annex I to the GPSD. Such information will be dealt with by the Member State where the notifying producer/distributor is established (Main Member State). The transmission of information on products posing a risk can be submitted by economic operators through the Product Safety Business Alert Gateway, a tool available on the RAPEX website (see Part II Chapter 5.3.2). Economic operators should include a detailed description of the risk of the product and can make use of the RAG tool available for this purpose (see Part I Chapter 5.3). Risk assessments carried out by economic operators are not binding on Member State authorities who are responsible for carrying out their own risk assessment. It is therefore possible for an authority of a Member State to come to a different conclusion regarding the risk assessment provided in an alert submitted via the Business Gateway. 3.3.2. Member States authorities Member States authorities notify the Commission through the RAPEX application about both compulsory and voluntary measures taken on their own territory against products posing a risk. Member States establish the roles for the creation, submission and follow-up of notifications in RAPEX. 3.3.3. Authorities in charge of external border controls Measures adopted by the authorities in charge of external border controls that prevent the marketing in the EU of a consumer product posing a serious risk to the health and safety of consumers (e.g. decisions to stop the import at the EU border) should be notified to the Commission through the RAPEX application in the same manner as measures adopted by market surveillance authorities that restrict the marketing or use of a product. 3.3.4. European Commission The Commission may inform the RAPEX Contact Points (see Part II, Chapter 5.1) regarding products posing serious risks, imported into or exported from the Community and the European Economic Area (23). The Commission may transmit information to the Member States about products of EU and non-EU origin posing a risk that, according to the information available, are likely to be on the EU market. This mainly concerns information that the Commission receives from third countries, international organisations, businesses or other rapid alert systems. This information might be circulated amongst Member States by means other than the RAPEX application. 3.4. Workflow 3.4.1. Creation of a notification 3.4.1.1. By a national authority According to the national arrangements, different national authorities involved in the RAPEX process (local/regional market surveillance authorities, external border control authorities, etc.) may be allowed to create a notification. 3.4.1.2. By the Commission In certain cases, the Commission may create a notification as explained in point 3.3.4. 3.4.2. Submission of notifications to the Commission The RAPEX Contact Point is responsible for the submission of all notifications for validation by the Commission. (See Part II, Chapter 5.1). 3.4.3. Examination of notifications by the Commission The Commission checks all notifications received through the RAPEX application before transmitting them to Member States to ensure that they are correct and complete. 3.4.3.1. Correctness When assessing the correctness of a notification, the Commission checks in particular that: (a) The notification meets all the relevant requirements set out in the GPSD or in Article 22 of Regulation (EC) No 765/2008 and in these Guidelines; (b) the notified product has not already been notified (to avoid any unnecessary duplication, including between ICSMS and RAPEX); (c) the notification submitted for validation by the notifying Member State is classified in accordance with the criteria set out in Part II Chapter 2 of these Guidelines; (d) the information provided including the risk assessment takes due account of the applicable legislation and the relevant standards; (e) the correct notification procedure has been used. 3.4.3.2. Completeness Once a notification is confirmed as correct, the Commission checks that it is complete. Part II, Chapters 3.2.1 and 3.2.2 of these Guidelines act as a point of reference. Special attention is given to the parts of a notification concerning product identification, risk description, measures, traceability and distribution channels. The Commission is not responsible for performing a risk assessment of the product, but only for checking that the notification includes an appropriate risk assessment containing all the elements listed in Part II Chapter 3.2.1 of these Guidelines (with the exceptions referred to in point 3.4.3.3). See also Part I Chapter 5.1 of these Guidelines. 3.4.3.3. Validation of notifications without a detailed risk assessment Member States should submit a risk assessment for every notification but in certain cases, the Commission may validate notifications that are submitted without a detailed and individual risk assessment: (a) Notifications of products posing chemical risks The risk level of a product may be considered to be serious if it contains a chemical substance either banned or in a concentration above the limit established by European legislation. Therefore, in cases where measures are taken against products containing a chemical substance subject to a restriction contained in EU Legislation, a notification may be submitted without a detailed risk assessment. (b) Notifications of cosmetic products Validation of notifications that do not include a detailed risk assessment may equally be possible for cosmetic products containing banned or restricted substances, which are backed up by an EU scientific committee opinion supporting that such presence of substances above the established limits poses a risk to the health and safety of consumers. For this specific product sector, other factors (e.g. concentration or time of exposure) may need to be taken into consideration. Nevertheless, if measures have been taken against a product containing not authorised chemical substances for which no scientific opinion has been issued confirming that the product poses a risk, a proper risk assessment may be required depending on a case-by-case analysis to prove that the product poses a serious or less than serious risk. In cases where the risk assessment is needed, if such risk assessment is not provided, these cases shall only be validated for information in RAPEX. As regards products that are subject to restrictive measures by market surveillance authorities based on the presence of a chemical substance mentioned in the list of ingredients which is subject to restrictions contained in EU Legislation and where there is no scientific data assessing the risk, notifications need to be assessed on a case-by-case basis. In case where the risk assessment is needed, if such risk assessment is not provided, these cases shall only be validated for information in RAPEX. (c) Notification of other products Where there is well-documented evidence that certain features of certain products consistently lead to a specific risk and risk level (e.g., the presence of any drawstrings or functional cords in the head, neck or upper chest on garments intended for young children always implies a serious risk), no further risk assessment is required for that given product. 3.4.3.4. Requests for additional information Should, during examination, the Commission have questions regarding a notification, it may suspend validation of the notification and ask the notifying Member State for additional information or clarification. This additional information is provided by the notifying Member State by the deadline specified in the Commission's request for information. 3.4.3.5. Investigation Where necessary, the Commission may carry out an investigation to assess the safety of a product. This investigation may be conducted in particular where there are serious doubts as to the risks posed by the product notified via the RAPEX application. These doubts can either arise during the examination of a notification by the Commission, or be brought to the attention of the Commission by a Member State (e.g. through a follow-up notification) or by a third party (e.g. a producer). As part of such investigations, the Commission may, in particular: (a) ask any Member State to provide information or clarification; (b) ask for an independent risk assessment and independent testing (laboratory or visual) of the product under investigation; (c) consult the Scientific Committees, the Joint Research Centre or any other institution specialising in the safety of consumer products; (d) convene the GPSD Committee, Consumer Safety Network and/or RAPEX Contact Points meetings, as well as consult the relevant Working Groups to discuss developments in an investigation. Where an investigation concerns a product notified through the RAPEX application, the Commission may suspend validation of a notification or, where such a notification has already been validated and distributed through the RAPEX application, temporarily remove the overview published on the RAPEX website. After an investigation, and depending on the outcome, the Commission (after consulting the notifying Member State, where necessary) may in particular validate and distribute through the RAPEX application the previously suspended notification, uphold the validated notification in the RAPEX application (with any changes) or permanently withdraw the notification from RAPEX. The Commission informs all Member States of the following: (a) its decision to launch an investigation, clearly stating the reasons for its decision; (b) its decision to close an investigation, presenting its conclusions and changes to the investigated notification(s) (if any); (c) all the relevant developments during an investigation. 3.4.4. Validation and distribution of notifications The Commission validates and distributes through the RAPEX application, by the deadlines specified in Appendix 5 to these Guidelines, all notifications assessed as correct and complete during the examination. Where, during an examination, a request for additional information or clarification was sent to the notifying Member State (followed by a reminder, if necessary), the Commission may take the following decisions: (a) where the additional information or clarification requested has been provided, the Commission re-examines the notification and may validate it with the changed classification where necessary (e.g. from a Notification for information to an Article 12 notification) or keep it on hold until further clarification; (b) where the additional information or clarification requested has not been provided within a specified deadline or it is insufficient, the Commission takes a decision on the basis of the information provided and, depending on the circumstances, may either validate it after changing the classification (e.g. from an Article 12 notification to Notification for information) or decide not to validate it. Once a common approach to risk assessment and/or enforcement has been agreed between Member States, depending on the circumstances and the views of the Member States, the Commission may take one of the following actions: (a) keep the notifications concerned in the RAPEX application; (b) change the classification of the notifications stored in the RAPEX application; (c) withdraw notifications from RAPEX (24). 3.4.5. Publication of notifications 3.4.5.1. Disclosure of information as a general rule The public has the right to be informed about products posing a risk. To meet this obligation, the Commission publishes overviews of new notifications on the RAPEX website (25). For external communication reasons, the RAPEX website will in future be called Safety Gate. Member States equally provide the public with information in the national languages on products posing a serious risk to consumers and on measures taken to address this risk. Such information may be distributed via the internet, on paper, by electronic media, etc. The information made available to the public is a summary of a notification and includes in particular the elements which allow the identification of the product, as well as the information about the risks and measures taken to prevent or restrict those risks. The Commission and the Member States may decide to disclose other elements of the notifications to the public, only when this information, due to its nature, is not confidential (professional secrets) and does not need to be protected. The following notifications are made available on the RAPEX website, in line with the requirements laid down in Article 16 of the GPSD: (a) notifications submitted falling under the scope of Article 12 of the GPSD; (b) notifications submitted falling under the scope of Article 22 of Regulation (EC) No 765/2008; (c) notifications submitted falling under the scope of Article 11 of the GPSD for products posing less than serious risk, the cross-border effect of which has also been recognised. As Chapter 3.4 provides for, the cross-border effect ascertains whether such a scenario is to be notified under Article 11; (d) notifications submitted falling under the scope of Article 23 of Regulation (EC) No 765/2008 concerning products presenting risks that are less than serious and regardless of whether the measures taken were compulsory or voluntary (26); (e) notifications submitted for information only if the notifying Member State so requests by ticking the ad hoc box in RAPEX, especially when voluntary measures are adopted and the products concerned are sufficiently identified. The publication of these notifications might need to be considered from the standpoint of securing an appropriate risk management. 3.4.5.2. Exceptions to the general rule Member States and the Commission should not disclose to the public any information about a product notified through the RAPEX application if such disclosure undermines the protection of court proceedings, monitoring and investigation activities or professional secrecy, except for information relating to the safety properties of products which must be made public if circumstances so require to protect the health and safety of consumers, or, in case of products covered by Regulation (EC) No 765/2008, also to protect other relevant public interests of the end-users (27). 3.4.5.3. Requests for confidentiality A notifying Member State may request confidentiality of a notification. Such a request clearly indicates the part(s) of the notification that should be kept confidential. Furthermore, each request for confidentiality is accompanied by a justification clearly stating the reasons (28). Requests for confidentiality are subject to examination by the Commission. The Commission checks that the request is complete (i.e. that it states which parts of the notification are covered by confidentiality and that it contains a justification) and justified (i.e. that it is in line with the provisions of the GPSD and these Guidelines). A decision as to the validity of the request is taken by the Commission after consulting the respective RAPEX Contact Point. (See Part II, Chapter 5.1). 3.4.5.4. Handling of notifications covered by confidentiality Article 16(2) of the GPSD states that the protection of professional secrecy or confidentiality shall not prevent the dissemination to the competent authorities of information relevant for ensuring the effectiveness of market monitoring and surveillance activities. Notifications covered partially or fully by confidentiality are examined by the Commission and, after being validated and distributed through the RAPEX application, they are subject to the usual follow-up activities by the Member States. The confidentiality of a notification or parts of it does not prevent it from being handled and distributed through the RAPEX application to the competent national authorities. The only significant difference in the handling and follow-up procedures is that the Commission and Member States should not disclose any parts of a notification that are confidential to the public. These parts have to remain confidential and thus they should not be published in any form. Member State authorities that receive confidential information through the RAPEX application ensure that it is protected when performing their activities. 3.4.5.5. Withdrawal of request for confidentiality The notifying Member State withdraws its request for confidentiality immediately after the authority in that Member State becomes aware that the justification for such a request is no longer valid, and informs the Commission accordingly. The Commission informs all Member States of the withdrawal of confidentiality on receipt of such a request by the notifying Member State. A notification that is no longer covered by full or partial confidentiality is made available to the public in line with the general rules applying to publication of notifications set out in these Guidelines. 3.4.6. Follow-up to notifications 3.4.6.1. Follow-up to the different types of notification Member States ensure appropriate follow-up to Article 12 notifications, Article 12 notifications requiring emergency action, notifications under Article 22 of Regulation (EC) No 765/2008 and to information on products posing a risk sent by the Commission (Chapter 3.3.4) as soon as possible and by the deadlines specified in Appendix 4 to these Guidelines at the latest. Notifications for information as well as notifications under Article 11 of the GPSD and notifications under Article 23 of Regulation (EC) No 765/2008 (notification for less than serious risks) do not require any specific follow-up activities. These notifications often do not contain the data needed for effective and efficient enforcement regarding the notified product (e.g. the notified product and/or measures are not sufficiently identified) or the level of the risk is not considered to be serious. Although there is no specific need for a follow-up in the referred cases, it is still important that Member States verify whether they disagree with the consideration of the risk as less than serious so they may eventually make a follow-up upon the information of a different risk assessment. Member States are therefore encouraged to ensure follow-up to such notifications where the notified product is likely to have been made available to consumers on their market and product identification allows measures to be taken. 3.4.6.2. Objectives of the follow-up activities On receipt of a notification, a Member State examines the information provided in the notification and takes appropriate action in order to: (a) establish whether the product was marketed on its territory; (b) assess what preventive or restrictive measures should be taken with regard to the notified product found on its market, taking into account the measures taken by the notifying Member State and any special circumstances that could justify different types of measures or no action being taken; (c) perform additional risk assessment and testing of the notified product, if necessary; (d) collect any additional information that may be relevant for other Member States (e.g. information on distribution channels of the notified product in other Member States). 3.4.6.3. Follow-up techniques To ensure efficient and effective follow-up, best practice follow-up techniques should be employed by national authorities, including: (a) Checks on the market National authorities organise regular (planned and random) checks on the market in order to establish whether consumer products notified through the RAPEX application are made available to consumers. When the Member State is mentioned as a country of destination, reinforced checks on the market shall be carried out, notably by contacting the economic operator(s) indicated in the notification. (b) Cooperation with business associations National authorities provide, when necessary, business associations with overviews of the most recent notifications and enquire whether any of the notified products were produced or distributed by their members. National authorities provide businesses only with summaries of notifications, such as the weekly overviews published on the RAPEX website. Whole notifications should not be transmitted to third parties, as certain information (e.g. details of the risk description or information on distribution channels) is often confidential and should be protected. (c) Publication of RAPEX data via the internet or other electronic and paper media National authorities regularly alert consumers and businesses about consumer products notified through the RAPEX application via their websites and/or other media, e.g. referring consumers and business to the RAPEX website. Information published in this way allows consumers to check whether they have and use products posing a risk and often provides the authority with useful feedback. (d) Online checks National authorities regularly perform online checks to try to identify whether products notified via RAPEX are available on online markets. Online check techniques may include web-crawling, data mining, data scraping, etc. National authorities apply various follow-up techniques in parallel and ideally do not limit their activities to only one of them. The Member State in which a manufacturer, a representative or an importer of the notified product is established (Main Member State) ensures appropriate follow-up to notifications distributed through the RAPEX application. The Main Member State often has better legal and technical means of obtaining information on the notified case, which will help other Member States to undertake effective follow-up activities. 3.4.7. Withdrawal/removal of notifications 3.4.7.1. Permanent withdrawal of a notification from RAPEX Notifications distributed through the RAPEX application are kept in the system for an unlimited period of time. The Commission may, however, in the situations presented in this Chapter, permanently withdraw a notification from RAPEX. 3.4.7.1.1. Situations where withdrawal of a submitted or validated notification is possible (a) There is proof that one or more of the notification criteria (29) are not met and thus a notification is not justified. This concerns cases in particular where it is established that the original risk assessment was performed incorrectly and that the notified product does not pose a risk. It also covers situations where the notified measures were successfully challenged in court or in other proceedings and they are no longer valid. (b) No measures have been taken with regard to a product notified through the RAPEX application (for information) before it was decided to adopt measures or take action (30). (c) After a discussion held at EU level, Member States agree that it is not useful to exchange information on certain safety aspects that have been notified through the RAPEX application (31). (d) There is proof that products covered by a notification are no longer marketed and there is proof that all items that had been made available have already been withdrawn from the market and retrieved in all Member States. Withdrawal of a notification that has been submitted or validated cannot be requested on the basis of the fact that the notified product has been subject to changes needed for it to comply with all the applicable safety requirements, unless proof is provided that all the products (items) concerned that had been made available have been withdrawn and retrieved in all Member States and that they are no longer marketed. 3.4.7.1.2. Request for permanent or temporary withdrawal by Member States The Commission may withdraw notifications from RAPEX only at the request of the notifying Member State, as the latter takes full responsibility for the information transmitted through the system. Other Member States, however, are encouraged to inform the Commission of any facts that may justify withdrawal. 3.4.7.1.3. Content of the request for permanent or temporary withdrawal Every request for withdrawal is accompanied by a justification stating the reasons and by all available documents supporting those reasons. The Commission examines each request and checks the justification and the supporting documents in particular. The Commission may request additional information, clarification or the opinion of the notifying Member State and/or other Member States before taking any decision. 3.4.7.1.4. Decision to withdraw Should, on the basis of the justification provided, the Commission decide to withdraw a notification from RAPEX, it removes it from: (a) the RAPEX application (or makes it otherwise invisible to all users of the system); (b) the RAPEX website (if necessary). The Commission informs all Member States of the withdrawal of a notification by mail or through other equally effective means and, if necessary, also the public by publishing a corrigendum on the RAPEX website. 3.4.7.2. Temporary removal of a notification from the RAPEX website 3.4.7.2.1. Situations where temporary removal is possible Where justified, the Commission may temporarily remove a notification from the RAPEX website, especially where the notifying Member State suspects that a risk assessment submitted in a notification has been performed incorrectly and thus the notified product may not pose a risk. A notification can be temporarily removed from the RAPEX website until the risk assessment of the notified product has been clarified. 3.4.7.2.2. Request for temporary removal by Member States The Commission may temporary remove notifications from the RAPEX application only at the request of the notifying Member State, as the latter takes full responsibility for the information transmitted through the application. Other Member States, however, are encouraged to inform the Commission of any facts that may justify such removal. 3.4.7.2.3. Content of the request for temporary removal Every request for temporary removal is accompanied by a justification stating the reasons and by all available documents supporting those reasons. The Commission examines each request and checks the justification and the supporting documents in particular. The Commission may request additional information, clarification or the opinion of the notifying Member State and/or other Member States before taking any decision. 3.4.7.2.4. Decision to remove Should, on the basis of the justification provided, the Commission decide to remove a notification from the RAPEX website, it informs all Member States by e-mail or by other equally effective means and, if necessary, also the public by publishing a corrigendum on the RAPEX website. 3.4.7.2.5. Re-publishing of a notification temporarily removed The notifying Member State immediately informs the Commission when the reasons for the removal of a notification from the RAPEX website are no longer valid. In particular, it informs the Commission of the results of any new risk assessment to enable the Commission to determine whether to maintain a notification in the RAPEX application and to re-publish it on the RAPEX website or to withdraw it permanently from RAPEX (following a request from the notifying Member State). The Commission may re-publish a notification on the RAPEX website following a justified request from the notifying Member State after the risk assessment has been clarified. The Commission informs the other Member States of the re-publishing of a notification on the RAPEX website by e-mail or by other equally effective means and also the public by replacing the corrigendum with a new one on the RAPEX website. 3.4.8. Notifications older than ten years The Commission will place all notifications older than ten years in a separate section of the RAPEX website. These notifications will still be available for public consultation. 3.5. Timing and deadlines for notifications 3.5.1. Timing of the notification Article 12(1) of the GPSD and Article 22 of Regulation (EC) No 765/2008 require Member States to immediately notify the Commission through the RAPEX application of preventive and restrictive measures concerning products posing serious risks. This provision applies to both compulsory and voluntary measures, although the timing of the notification is different. (a) Compulsory measures These measures are notified through the RAPEX application immediately after being adopted or after the decision to adopt them has been taken, even if an appeal against them at national level is likely, if they are already under appeal or they are subject to publication requirements. This approach is consistent with the objective of RAPEX, i.e. to ensure the rapid exchange of information between Member States and the Commission in order to prevent the supply and use of products that pose a risk. (b) Voluntary measures Under Article 5(3) of the GPSD and Article 22 of Regulation (EC) No 765/2008, economic operators are obliged to notify the competent Member State authorities of voluntary action and measures taken to prevent risks to consumers posed by products they have made available on the market (ideally by means of a Business Gateway notification). The authority of a Member State receiving this kind of notification uses this information as the basis for a notification (if all the notification criteria are met) and sends it immediately after receipt of the Business Gateway notification. Where voluntary measures are adopted in the form of an agreement between an economic operator and an authority of a Member State or on the basis of a recommendation from an authority to a producer or distributor, a notification is submitted immediately after the conclusion of such an agreement or the adoption of such a recommendation. To ensure common application of the notification obligation, Part III, Appendix 4 to these Guidelines lays down specific deadlines for submitting notifications to the Commission via the RAPEX application (32). 3.5.2. Deadlines (33) Member States notify the Commission of preventive and restrictive measures adopted as soon as possible and by the deadlines specified in Part III, Appendix 4 to these Guidelines at the latest. Appropriate arrangements are in place at national level concerning the transmission of information between national authorities in charge of product safety and the RAPEX Contact Point to ensure that the deadlines are met. (See Part II, Chapter 5.1). The deadlines provided apply irrespective of any appeal procedure or official publication requirement. 3.5.3. Emergency situations All notifications concerning products posing a serious risk requiring emergency action are preceded by a telephone call from the RAPEX Contact Point to the Commission RAPEX Team's mobile telephone number to facilitate immediate action and follow-up. This rule applies in particular to notifications transmitted at weekends or during holiday periods. (See also Part II, Chapter 5.1). 4. Follow-up activities 4.1. Communication of follow-up activities Member States notify the Commission of any findings subsequent to their follow-up activities in relation to RAPEX notifications (i.e. Article 12 notifications and Notifications requiring emergency action as well as notifications under Article 22 of Regulation (EC) No 765/2008) and information on products posing a risk sent by the Commission (Chapter 3.3.4). In addition, Member States are encouraged to notify the Commission of any follow-up activities regarding notifications for less than serious risks and for information. 4.2. Content of follow-up notifications 4.2.1. Scope of data Findings resulting from follow-up activities are communicated to the Commission in the form of follow-up notifications. To harmonise the type of information and to keep the workload to a minimum, Member States submit follow-up notifications in particular in the following situations: (a) A notified product has been found on the market A follow-up notification is sent when national authorities find the notified product on the market or at the external border. This follow-up notification contains the full details of the product in question (e.g. name, brand, model number, bar code, batch number) plus information on the total number of items found on the market. Furthermore, the following details of the measures taken are communicated: type (compulsory or voluntary), category (e.g. withdrawal from the market, recall from consumers), scope (e.g. country-wide, local), date of entry into force and duration (e.g. permanent, temporary). If the notified product was found on the market but no measures were adopted, specific reasons justifying no measures being taken should be given in the follow-up notification. To reduce the burden on the national authorities as regards their follow-up practice, Member States do not need to inform the Commission (unless the Commission asks to be informed) of the conclusions of follow-up activities by means of a follow-up notification when the notified product is not found on the market. (b) Different risk assessment A follow-up notification is sent when the conclusions of a risk assessment performed by an authority of the reacting Member State differ from the conclusions set out in the original notification. This follow-up notification contains a detailed risk description (including the results of tests, a risk assessment and information on known accidents and incidents), accompanied by supporting documents (test reports, certificates, etc.). Furthermore, the reacting Member State should prove that the risk assessment submitted with its follow-up notification was performed on the same product as the one notified, i.e. the same brand, name, model number, batch number, origin, etc. (c) Additional information A follow-up notification is sent when national authorities collect additional information (during their follow-up activities) that may be useful for market surveillance and enforcement in other Member States. Member States are encouraged to collect additional information that may be relevant for authorities both in other Member States and in third countries that cooperate closely with the EU on product safety. Details include product origin (e.g. information on the country of origin, manufacturer and/or exporters) and information on the supply chains (e.g. information on the countries of destination, importers and distributors). The country carrying out the follow-up activities attaches all available supporting documents to the follow-up notification, such as copies of orders, sales contracts, invoices, customs declarations, etc. Member States may also indicate whether certain follow-up actions have been performed although the product has not been found in their territory. 4.2.2. Completeness of follow-up notifications The RAPEX Contact Point of the reacting Member State, together with the responsible authority, ensures that all data provided in their follow-up notification is accurate and complete and that there is no confusion with other similar products that are available on the EU market. (See also Part II, Chapter 5.1). The standard follow-up notification template is provided in Part III, Appendix 2 to these Guidelines. Should certain relevant information not be available when a follow-up notification is submitted, the reacting Member State indicates this on the follow-up form. Once this information becomes available, the reacting Member State may request that its follow-up notification be updated. The updated follow-up notification is examined by the Commission before it is validated and distributed through the system. The RAPEX Contact Point provides all authorities in its own Member State that participate in the RAPEX network with instructions on the scope of the data required to complete the follow-up notification template correctly. This helps to ensure that information provided by these authorities to the Contact Point is correct and complete. (See Part II, Chapter 5.1). 4.2.3. Updating of validated follow-up notifications The reacting Member State informs the Commission (as soon as possible and by the deadlines specified in Part III, Appendix 4 to these Guidelines at the latest) of any developments that may require changes to a follow-up notification distributed through the RAPEX application. In particular, Member States inform the Commission of changes in the status of the measures taken or in the risk assessment submitted with their follow-up notification. The Commission examines the information provided by the reacting Member State and if necessary updates the information concerned. 4.2.4. Responsibility for follow-up notifications Responsibility for the information provided in follow-up notifications lies with the notifying Member State (34). The authority(ies) involved in the follow-up activities (e.g. by carrying out the risk assessment or by adopting restrictive measures) take responsibility for the information provided in follow-up notifications. The RAPEX Contact Point checks and validates all follow-up notifications prepared by the respective authorities before transmitting them to the Commission. (See also Part II, Chapter 5.1). Any action taken by the Commission, such as examining and validating follow-up notifications, does not imply any assumption of responsibility for the information transmitted, which remains with the Member State submitting the follow-up notification. 4.2.5. Response to follow-up notifications Member States may respond to any follow-up notifications regarding their own notification(s) by starting a discussion on the online collaborative space put at the disposal of Member States for the exchange of information (see Part II Chapter 5.3.2). This ensures that the response is visible to all members of RAPEX. 4.3. Actors and roles involved in follow-up activities The parties involved in the follow-up notification process and their responsibilities therein are the following: 4.3.1. Economic operators (35) Economic operators are not directly involved in the submission of follow-up notifications. However, economic operators must cooperate with national authorities and provide them with any information concerning a product which is the subject of an existing notification in order to facilitate the creation and submission of follow-up notifications via the RAPEX application. 4.3.2. Market surveillance authorities Market surveillance authorities notify the European Commission through the RAPEX application about any follow-up activities or other information regarding notifications. 4.3.3. European Commission The European Commission examines and validates follow-up notifications according to the specifications included in Part II, Chapter 4.2. 4.4. Workflow 4.4.1. Creation and submission of a follow-up notification by a Member State The RAPEX Contact Point is responsible for the submission of follow-up notifications via the RAPEX application. (See Part II, Chapter 5.1). 4.4.2. Examination of follow-up notifications by the Commission 4.4.2.1. Correctness and completeness The Commission checks all follow-up notifications received through the RAPEX application before they are validated and transmitted to the Member States. These checks focus on the correctness and completeness of the information provided. The Commission checks if a follow-up notification meets all the relevant requirements set out in the GPSD and in these Guidelines and if the correct procedure was applied. Once the correctness of a follow-up notification is confirmed, the Commission checks its completeness. Chapter 4.2.2 of these Guidelines is to be used as a point of reference for this examination. The Commission pays special attention to follow-up notifications containing risk assessments. It verifies, in particular, that the risk description is complete, clearly presented and well documented, and that the risk assessment clearly relates to the product covered by a notification. 4.4.2.2. Requests for additional information Before validating a follow-up notification, the Commission may request the reacting Member State to provide additional information or clarification within a given deadline. Validation of a follow-up notification may be conditional upon receipt of the data requested. The Commission may request the opinion of any Member State and, in particular, the notifying Member State on a validated follow-up notification. The Member State submits its opinion to the Commission within a deadline specified by the latter. Furthermore, the notifying Member State informs the Commission whether any changes to the notification (e.g. to the risk assessment) or to its status (e.g. permanent withdrawal from the system) are necessary. 4.4.3. Validation and distribution of follow-up notifications All follow-up notifications assessed as correct and complete are validated and distributed by the Commission according to the deadlines specified in Appendix 5 to these Guidelines. The Commission does not validate follow-up notifications with a risk assessment different from that of the notification they refer to, if the risk assessment is not complete, clearly presented and well documented, or if it is not shown that the risk assessment was performed in relation to the product covered by the notification. 4.4.4. Permanent withdrawal of a follow-up notification from RAPEX Follow-up notifications distributed through the RAPEX application are kept in the system as long as the notification to which they are attached. The Commission may permanently withdraw a validated follow-up notification from the RAPEX application if a notification to which this follow-up notification is attached has been withdrawn from RAPEX (in accordance with Part II, Chapter 3.4.7.1.1 of these Guidelines). Furthermore, the Commission may withdraw a validated follow-up notification where it clearly provides incorrect information, and in particular where: (a) the product found on the market by the reacting Member State is different from the product covered by the notification; (b) the measures adopted by the reacting Member State are successfully challenged in court or in other proceedings and subsequently withdrawn; (c) the risk assessment performed by the reacting Member State is proven to be incorrect or relates to a different product from the one covered by the notification. The provisions of Chapters 3.4.7.1.2 and 3.4.7.1.3 apply. Once the Commission decides to withdraw a follow-up notification it is removed from RAPEX (or otherwise made invisible to users of the system). The Commission informs all Member States of the withdrawal of a follow-up notification via the online collaborative space referred to in Part II, Chapter 5.3.2 or through other equally effective means. 4.5. Deadlines for submitting follow-up notifications Member States submit follow-up notifications to the Commission as soon as possible and by the deadlines specified in Appendix 4 to these Guidelines at the latest. Appropriate arrangements are established at national level concerning the transmission of information between all competent authorities and the RAPEX Contact Point to ensure that the deadlines are met. (See Part II, Chapter 5.1). The deadlines apply irrespective of any appeal procedure or official publication requirement. 4.6. Requests for confidentiality A reacting Member State may request confidentiality in its follow-up notification. Such requests clearly state which part(s) of the follow-up notification should be kept confidential. Furthermore, all requests for confidentiality are accompanied by justification clearly stating the reasons. Requests for confidentiality are examined by the Commission to determine that they are justified (i.e. in line with the provisions of the GPSD and these Guidelines) and complete (i.e. it states which parts of the form that it covers and if it contains a justification). The final decision on confidentiality is taken by the Commission after consultation of the responsible RAPEX Contact Point. (See Part II, Chapter 5.1). The Commission and the Member States treat follow-up notifications with requests for confidentiality in the same way as the other follow-up notifications. The confidentiality of a follow-up notification or parts of it does not prevent it from being distributed through the RAPEX application to the competent national authorities. However, neither the Commission nor the Member States should disclose any parts of a follow-up notification that are confidential to the public. This information is confidential and therefore cannot be published in any form. The Member State submitting the follow-up notification withdraws its request for confidentiality immediately after it becomes aware that the reasons for such a request are no longer valid. The Commission informs all Member States of the withdrawal of the confidentiality after the receipt of such a request from the reacting Member State. 5. RAPEX networks 5.1. RAPEX National Contact Points Each Member State establishes a single RAPEX Contact Point to operate RAPEX at national level. The Member States decide within which national authority to set up the RAPEX contact point. Each Member State also organises its national RAPEX network to ensure the efficient flow of information between the national contact point and the various authorities participating in RAPEX. (See Part I Chapter 5.4 and Part II, Chapter 1.2). 5.1.1. Organisation Each Member State gives the national Contact Point the resources and information it needs to perform its tasks and, in particular, to operate the system with effective back-up/business continuity. The RAPEX Contact Point has a separate email account for RAPEX, accessible to all officials in that contact point (e.g. rapex@ ¦). Professional or private email accounts of officials in charge of the RAPEX Contact Point should not be used as the email account of the RAPEX Contact Point. The RAPEX Contact Point also has a direct phone number through which it can be reached during and outside working hours. 5.1.2. Tasks The main tasks of the RAPEX Contact Point are to: (a) organise and steer the work of the national RAPEX network, in accordance with the rules set out in these Guidelines; (b) train and assist all authorities in the network in the use of RAPEX; (c) ensure that all RAPEX tasks stemming from the GPSD and these Guidelines are performed correctly and, in particular, that all required information (i.e. notifications, follow-up notifications, additional information, etc.) is provided to the Commission without delay; (d) transmit information between the Commission and the national market surveillance authorities and authorities in charge of external border controls; (e) check and validate the completeness of the information received from all authorities before transmission to the Commission through the RAPEX application; (f) check before submitting a notification whether a product has already been notified or information on that product has been exchanged through the RAPEX application (to avoid any duplication); (g) participate in RAPEX Contact Point Working Group meetings and other events on the operation of RAPEX; (h) suggest possible improvements to the operation of the system; (i) inform the Commission immediately of any technical problems with the functioning of the RAPEX application; (j) coordinate all national activities and initiatives carried out in relation to RAPEX; (k) explain to stakeholders how RAPEX operates and clarify their obligations, particularly for the business notification obligation set out in Article 5(3) of the GPSD. 5.2. RAPEX networks established at EU and national levels 5.2.1. The RAPEX Contact Point Network The Commission organises and steers the work of the RAPEX Contact Point Network. This network consists of all RAPEX Contact Points appointed in the Member States and European Economic Area (EEA) countries. The Commission regularly convenes meetings of the RAPEX Contact Point Network to discuss the operation of the system (e.g. to communicate the latest developments concerning RAPEX, to exchange experience and know-how), and to improve cooperation between the RAPEX Contact Points. 5.2.2. RAPEX networks established at national level The RAPEX Contact Points organise and steer the work of their own RAPEX national network. The network consists of: (a) the RAPEX Contact Point; (b) market surveillance authorities responsible for monitoring the safety of products; and (c) authorities in charge of external border controls. RAPEX Contact Points are encouraged to provide for the organisation and operation of the RAPEX national network so as to ensure that all the authorities involved are aware of their roles and responsibilities as regards the operation of RAPEX. This should be consistent with the information contained in these Guidelines. The RAPEX Contact Points are encouraged to facilitate regular and continuous exchange of information and discussion with their national network in order to discuss with all the authorities involved how RAPEX is organised, how it operates and, if necessary, to give training courses. 5.3. RAPEX internal communication tools, practical and technical arrangements for RAPEX and best practice 5.3.1. Languages The use of languages in notifications and follow-up notifications, as well as communications between the RAPEX Contact Points and the Commission, must take account of the objectives of RAPEX and must ensure a rapid exchange of information between Member States and the Commission on products posing serious risks. To facilitate the work of the network, Member States authorities are encouraged to use the existing EC eTranslation webpage to ensure all Member States understand what is being communicated through RAPEX. A link to this translation tool to submit documents or extracts from texts for translation from and into all EU languages (36) is available in the collaborative space. (See Part II, Chapter 5.3.2). 5.3.2. RAPEX online tools (a) RAPEX system The Commission has established and maintains a web-based application for use as a communication tool for the purpose of RAPEX. Member States use this system to create and submit notifications and follow-up notifications through the RAPEX application, and the Commission uses it to validate and distribute the documents it receives. The Commission provides access to the system to all RAPEX Contact Points, competent national authorities and the relevant Commission departments. The Commission lays down the rules for granting access to the system and gives access to as many users as possible, taking into account needs and technical limitations. Where the RAPEX system is temporarily not operational (for reasons other than regular and planned maintenance work), Member States should only submit notifications of serious risks to the Commission (i.e. Article 12 notifications, Article 12 notifications requiring emergency action or Article 22 of Regulation (EC) No 765/2008) notifications. The submission of other notifications and follow-up notifications is suspended until the RAPEX system is re-established. While the system is not operational, RAPEX notifications should be sent to the Commission by email to: just-rapex@ec.europa.eu or to another email address communicated in advance. If email transmission is not possible, RAPEX notifications are sent to the Commission by any other means considered appropriate (37). (b) Product Safety Business Alert Gateway The Product Safety Business Alert Gateway (also known as the Business Gateway) is intended to simplify the practical aspects of the obligation on producers and distributors, or their authorised representative, under Article 5(3) of the GPSD to notify the competent national authorities of the Member States if they know or ought to know, on the basis of the information in their possession and as professionals, that a product they have placed on the market is dangerous. The Business Gateway consists of two elements: (i) the notification template and (ii) the online database. The notification template is reserved for use by producers and distributors to inform the competent national authorities of the Member States that a product they have placed on the market is dangerous, in line with their obligation under Article 5(3) of the GPSD. The online database is intended for use by Member States national authorities responsible for receiving notifications of dangerous consumer products submitted by producers and distributors. The competent national authority may use the information provided to submit a RAPEX notification if all criteria for this are met. (c) Collaborative space The Commission also manages a collaborative space to exchange information between the Commission and the Member States competent national authorities. This includes the EU Consumer Product Safety platform, open to the RAPEX Contact Points and their colleagues working on product safety issues in the competent national authorities for all RAPEX-related issues. Requests for access to the space must be made by the RAPEX Contact Points in the relevant Member State and authorised by the Commission. This space also includes a section, managed by the Commission, containing useful tips and information on the functioning of RAPEX and input from the Member States. (d) RAG tool (38) The Commission has developed this tool available on the RAPEX website to facilitate the risk assessment of products notified through the RAPEX system, in accordance with the principles laid down in Appendix 6. 5.3.3. Contact details The Commission provides the RAPEX Contact Points with the contact details of the Commission's RAPEX team, including names, email addresses and telephone numbers. The RAPEX Contact Points provide the Commission with their contact details, including the names of officials working within the Contact Point, the name and address of the authority where the RAPEX Contact Point is established, the email addresses and phone numbers of officials. Any changes to the contact details are immediately communicated to the Commission by the RAPEX Contact Point. The Commission publishes and updates a list of contact details of the RAPEX Contact Points on the RAPEX website. Member States process contact details including personal data in application of the EU's data protection legislation. On the information exchange through RAPEX, Member States should process personal data ensuring that it circulates and is distributed only as far as it is strictly necessary. 5.3.4. Operation of RAPEX outside regular working hours RAPEX operates non-stop. The Commission and the RAPEX Contact Points ensure that officials responsible for operating RAPEX can always be contacted (by phone, e-mail or other equally effective means) and that they can take whatever action is necessary, including in an emergency and outside regular working hours, such as weekends and holidays. The Commission provides the RAPEX Contact Points with an emergency telephone number, which should be used to contact the Commission RAPEX team outside of working hours, with priority over any other communication channels. The RAPEX Contact Points provide the Commission with their contact details, including the phone numbers of officials who can be contacted during and outside working hours. Any changes to the contact details are immediately communicated to the Commission by the RAPEX Contact Points. PART III APPENDICES 1. Fields and information included in notifications (39) Fields that will be published on the web are shaded. Notification Form Section 1: General information Case number Creation date Validation/distribution date Notification type * Notifying country Full Contact details of the Notifying Authority * Section 2: Product Professional / Consumer Product Product category * OECD Portal category (if known) Product (what the product is) * Name * Brand * Type/number of model: * Batch number/Bar code * Customs code * Product and packaging description * Total number of items covered by the notification (if known) * Photos: Section 3: Regulations and standards applicable Legal provisions (directive, decision, regulation, etc.) * Standards * Proof of conformity * Is the product counterfeit? * Certificates Section 4: Traceability Country of origin (where the product manufactured) * Countries of destination * Full Contact details of the manufacturer or its representative(s) * Full Contact details of the exporter(s) * Full Contact details of the importer(s) * Full Contact details of the distributor(s) * Full Contact details of the retailer(s) * Is the product (also) sold online? Please give details: URL Section 5: Risk assessment Risk category * Risk level Summary of test results * Description of the technical issue that leads to the highest risk level Risk description (how the technical defect leads to the risk) * EU Legal provisions and /or Standards against which the product was tested and did not comply with * Information on known incidents and accidents * Section 6: Measures Type of measures adopted * If Voluntary: Type of economic operator taking notified measure(s) * Name of economic operator taking notified measure(s) * If Compulsory: Name of authority ordering the notified measure(s) * Type of economic operator to whom the measure(s) were ordered * Category of measures * Date of entry into force * Duration * Scope * Has the notification been sent by a producer or a distributor under Article 5(3) of the GPSD? * URL link to company recall page (if available) Section 7: Confidentiality Is the notification confidential? * Scope of confidentiality Justification Section 8: Other Additional information Justification for sending Notification for information Annexes Photos (products, packaging and label) Certificates Test report and risk assessment Notification sent by an economic operator through Business Gateway Adopted measures * Indicates a mandatory field. 2. Fields and information included in follow-up notifications (40) Fields that will be published on the web are shaded. Section 1: General information Case number Validated notification type Notifying country Creation date Validation/distribution date Submission number Follow up notification number Reacting country Full contact details of the notifying authority Validated notification product category Notified product Notified name Product (what the product is) Name (on the product or the packaging) Brand (on the product or the packaging) Type/number of model Batch number/Bar code (or other information to identify which products are affected) Photos (products, packaging and label) Section 2: Type of follow-up notification Product found * Total number of items found (if known) * Measures adopted / Measures not adopted Type of measures adopted * If Voluntary: Type of economic operator taking notified measure(s) * Name of economic operator taking notified measure(s) * If Compulsory: Name of authority ordering the notified measure(s) * Type of economic operator to whom the measure(s) were ordered * Category of measures * Date of entry into force * Duration * Scope * Adopted measures URL link to company recall page (if available): Different risk assessment * Risk category * Summary of the test results (description of technical defects) * Indication of legal provisions and standards (with clauses) against which the product was tested * Different risk assessment * Information on known incidents and accidents * Attachments (certificates, test report and risk assessment ¦) Additional information * Complementary information on distribution channels and/or product's origin Complementary information on the risk assessment Other complementary information Section 3: Confidentiality Is the follow-up confidential? * Scope of confidentiality Justification Annexes Photos (product, packaging and label) Test reports and risk assessments Certificates Adopted measures * Indicates a mandatory field 3. Notification scheme Notification for information (if relevant) (1) Enough identification information? Notification encouraged Product covered by Regulation (EC) No 765/2008? (3) Information involving new risk? Notification for information encouraged Product covered by Regulation (EC) No 765/2008 (3) Notification for information recommended Article 12 GPSD/22 Regulation (EC) No 765/2008 notification Risk level established and measure adopted? Enough identification information? Article 11 GPSD notification Notification for information Cross-border effect? (2) Voluntary means? Cross-border effect? (2) Serious risk? No Yes No No No Yes No No No ongoing Yes No Yes Yes No Yes Yes Yes Yes Yes (1) To be upgraded when a measure is adopted. (2) The notion of cross-border effect should be interpreted in a broad sense (see Part II Chapter 6.1 of these Guidelines). (3) See Part I, Chapter 3.1 of these Guidelines. 4. Deadlines for member states Member States are required to act within the deadlines indicated unless duly justified Notification procedure Action Deadline Notifications Send Article 12 notification requiring emergency action Within 3 days after:  adoption or decision to adopt Compulsory measures, or  receipt of information on Voluntary measures. Send Article 12 notification or Article 22 Regulation (EC) No 765/2008 notification Within 10 days after:  adoption or decision to adopt Compulsory measures, or  receipt of information on Voluntary measures. Confirm measures if the notification was sent before deciding to adopt measures Within 45 days after submission of the notification Update to a notification Within 5 days after receipt of the information on developments requiring changes to a notification Follow-up notifications Ensure follow-up activities to: Article 12 notification requiring emergency action Within 20 days after receipt of a notification Article 12 notification and to Notification sent by the European Commission as well as Article 22 of Regulation (EC) No 765/2008 notification Within 45 days after receipt of a notification Send follow-up notification to: Article 12 notification requiring emergency action Within 3 days after:  the notified product was found on the market, or  the completion of a risk assessment with different results, or  receipt of additional information Article 12 notification and to Notification sent by the European Commission as well as Article 22 of Regulation (EC) No 765/2008 notification Within 5 days after:  the notified product was found on the market, or  the completion of a risk assessment with different results, or  receipt of additional information Update to a follow-up notification Within 5 days after receipt of information or developments requiring changes to a follow-up notification Notification procedure established under Article 11 of the GPSD Notifications Send Article 11 notification Within 10 days after adoption of Compulsory measures Update to the notification Within 5 days after receipt of information on developments requiring changes to the notification 5. Deadlines for the Commission Notification procedure Action Deadline EU Rapid Information System RAPEX established under Article 12 of the GPSD Notifications Validate Article 12 notification requiring emergency action Within 3 days after receipt of a notification Validate Article 12 notification as well as Article 22 of Regulation (EC) No 765/2008 notification Within 5 days after receipt of a notification Validate Notification for information Within 10 days after receipt of a notification Follow-up notifications Validate follow-up notification sent to Article 12 notification requiring emergency action Within 3 days after receipt of a follow-up notification Validate follow-up notification sent to Article 12 notification and to Notification sent by the European Commission as well as Article 22 of Regulation (EC) No 765/2008 notification Within 5 days after receipt of a follow-up notification Validate follow-up notification sent to Notification for information Within 10 days after receipt of a follow-up notification Notification procedure established under Article 11 of the GPSD Notifications Validate Article 11 notification Within 10 days after receipt of a notification Follow-up notifications Validate follow-up notifications sent to Article 11 notification Within 10 days after receipt of a follow-up notification RISK ASSESSMENT GUIDELINES FOR CONSUMER PRODUCTS (41) 1. Introduction Consumer products may cause harm when used, e.g. a hot flat-iron that can cause burns, scissors or knives that can cause cuts, or a household cleaner that can damage the skin. This kind of damage is not a usual occurrence because general knowledge or instructions teach how to use consumer products safely. Nevertheless, the risk of damage remains. This risk can be assessed in different ways. A range of methods have been used to quantify risk for consumer products, such as a nomograph method (42), a matrix method (43), and the method previously recommended for the EU's RAPEX rapid alert system (44). While the general principles for risk assessment have always been agreed, how to quantify risks has been under permanent development. This has led to diverging results and ensuing discussions, as well as to consideration of what the best possible practice might be. The purpose of these risk assessment guidelines is therefore to improve the situation and, within the framework of the Directive on General Product Safety (45), to provide a transparent and practicable method for appropriate use by Member States' competent authorities when they assess the risks of non-food consumer products. These guidelines are based on a risk assessment method developed for other purposes, adapted to the specific requirements of non-food consumer products. A certain amount of training will of course be needed before these guidelines can be put into practice, but expertise in risk assessment will greatly facilitate this task. This will be backed by exchanges of views between risk assessors, since expertise and experience accumulated through the years is invaluable. In building up a risk assessment method in small, manageable steps, these guidelines help to focus on the relevant issues of a product, its user(s) and its use(s), and to identify possible divergences of views between risk assessors from the onset, thus avoiding time-consuming discussions. They should thus lead to consistent and robust risk assessment results based on evidence and science, and consequently to widely acceptable consensus on the risks that the many non-food consumer products may present. A quick overview and a flow chart on how to prepare a risk assessment pursuant to these guidelines is provided in section 5  Consumer products mean non-food consumer products throughout these guidelines. These guidelines do not set out to replace other guidelines that may address very specific products or may be specifically provided for in legislation, such as in the area of chemicals, cosmetics, pharmaceuticals or medical devices. It is highly recommended to use this specific guidance, since it is tailor-made, but it will always be for the risk assessor to decide how best to assess the risks of a product. Nor are these guidelines to be used by manufacturers just to avoid serious risks when designing and manufacturing products. Consumer products have to be safe, and these guidelines aim at helping authorities to identify serious risks when, despite the best efforts of the manufacturer, a product is not safe. 2. Risk assessment  an overview 2.1. Risk  Combination of hazard and probability Risk is generally understood as something that threatens the health or even the lives of people, or that may cause considerable material damage. Nevertheless, people take risks while being aware of the possible damage, because the damage does not always happen. For example:  Climbing a ladder always includes the possibility of falling off and injuring oneself. Falling off is therefore built into the ladder; it is an intrinsic part of using a ladder and cannot be excluded. Falling off is thus called the intrinsic hazard of a ladder.  This hazard, however, does not always materialise, since many people climb ladders without falling off and injuring themselves. This suggests that there is a certain likelihood (or probability), but no certainty, of the intrinsic hazard materialising. Whereas the hazard always exists, the probability of it materialising can be minimised, for example by the person climbing the ladder being careful.  Using a household cleaner with sodium hydroxide to free blocked sewage water pipes always entails the possibility of very severe damage to the skin, if the product comes into contact with skin, or even of permanent blindness if drops of the product get into the eye. This is because sodium hydroxide is very corrosive, meaning that the cleaner is intrinsically hazardous. Nevertheless, when the cleaner is handled properly, the hazard does not materialise. Proper handling may include wearing plastic gloves and protective glasses. Skin and eyes are then protected, and the probability of damage is much reduced. Risk is thus the combination of the severity of possible damage to the consumer and the probability that this damage should occur. 2.2. A risk assessment in three steps It takes three steps to determine the risk: 1. Anticipate an injury scenario in which the intrinsic product hazard harms the consumer (see table 1). Determine how severe the consumer's injury is. A yardstick for quantifying the intrinsic product hazard is the extent of the adverse effect that it can cause to the health of a consumer. The risk assessor therefore anticipates an injury scenario that describes step by step how the hazard leads to the injury of a consumer (see table 2). In short, the injury scenario describes the accident that the consumer has with the product in question, and the severity of the consumer's injury caused by that accident. An injury can vary in severity, depending on the hazard of the product, on the way the product is used by the consumer, on the type of consumer who uses the product, and much more (see section 3). The more severe the injury, the more severe the hazard that caused it, and vice versa. The severity of the injury is therefore a means of quantifying the hazard. These guidelines propose 4 levels of severity, from injuries that are normally completely reversible to very serious injuries that cause more than approximately 10 % of permanent disability or even death (see table 3). 2. Determine the probability of the consumer being injured in practice by the intrinsic product hazard. While the injury scenario describes how the consumer is injured by the hazard, the scenario only happens with a certain probability. The probability can be expressed as a fraction, such as > 50 % or > 1/1 000 (see left-hand side of table 4). 3. Combine the hazard (in terms of severity of the injury) with the probability (in terms of a fraction) to obtain the risk. This combination can be made by looking up both values in the appropriate table (see table 4); the table will provide the level of risk in terms of serious, high, medium and low risk. Where different injury scenarios are foreseeable, the risk for each of those scenarios should be determined the highest risk being labelled as the risk of the product. The highest risk is normally crucial because only action on the highest risk can effectively provide a high level of protection. On the other hand, an identified risk may be lower than the highest risk, but require specific risk reduction action. It is then important also to take measures against that risk so that all risks are effectively reduced. Once the three steps have been carried out, the risk assessment is basically complete. A flow chart on building a risk assessment is at the end of section 5. 2.3. Some useful tips Seek information As can be seen from the examples of Chapter 2.1, each of the three steps of a risk assessment (see point 2.2) requires anticipation of what might happen and how likely it is to happen, since the product under consideration will normally not have caused an accident, and thus the risk will not have materialised (yet). Previous experience with similar products will help in this exercise, as will any other information about the product, such as design, mechanical stability, chemical composition, operation, instructions for use, including possible risk management advice, type of consumers it is intended for (and those for which it is not), test reports, accident statistics, the EU Injury Database (IDB) (46), information about consumer complaints, about the behaviour of different consumers when they are using the product, and about product recalls. Product requirements laid down in legislation, in product standards or in checklists (such as in ISO 14121: Safety of machinery  Risk assessment) can also be useful sources of information. Nevertheless, the products to be assessed may be quite specific and thus these sources may not contain the information required. The information collected may also be incomplete, inconsistent, or not fully plausible. This may be the case in particular for accident statistics, when only the product category is registered. The absence of an accident history, a small number of accidents or low severity of accidents should not be taken as a presumption of low risk. Product-specific statistics also have to be viewed with great care, since the product may have changed over time, be it in design or composition. The information must always be critically assessed. Feedback from expert colleagues can be particularly useful, since they can draw from their real-life experience and pro vide suggestions that are not immediately obvious when assessing a product risk. They may also give advice when assessing the risk for different types of consumers, including vulnerable consumers such as children (see table 1), since the latter may handle a product differently. They may also help to assess the risk for different injuries that a product may cause, and the way in which those injuries emerge through the use of the product. They can also judge whether an injury scenario is totally unperceived, too unlikely, and then guide the risk assessor towards more realistic assumptions. Thus, feedback from experienced colleagues, although not an obligation, can be helpful in several aspects. A risk assessor from an authority could seek advice from colleagues in that same authority, in other authorities, in industry, in other countries, in scientific groupings, and elsewhere. Conversely, any risk assessor in industry could use his contacts with authorities and others when a new or improved product is to be assessed before it is placed on the market. New information obtained should of course be used to update any existing risk assessment. Make a sensitivity analysis of your risk assessment If all information searches and queries to expert colleagues do not provide the required, very specific data, a so-called sensitivity analysis might help. In this analysis a lower and a higher value than previously chosen is assumed for each parameter of the risk assessment, and taken through the entire risk assessment procedure. The resulting risk levels will show how sensitive the risk level reacts to the input of lower and higher values. In this way the range in which the real risk of the product will be can be estimated. If the most likely value of each parameter can be estimated, then those most likely values should be taken through the procedure, and the resulting risk level will be the most likely risk. An example of a sensitivity analysis is illustrated in section 6. Let others check your risk assessment Feedback from colleagues will also help when finalising the risk assessment. They will be able to provide advice on the assumptions and estimations made during the three steps referred to in point 2.2. They will feed in their experience and thus help to generate a more robust, more solid, more transparent and ultimately more acceptable risk assessment. It is therefore recommended that, ideally, advice be sought from expert colleagues, possibly in the form of a group discussion, before concluding a risk assessment. These groups, of perhaps 3 to 5 members, should include a combination of expertise appropriate to the product under assessment: engineers, chemists, (micro-)biologists, statisticians, product safety managers, and others. Group discussion will be particularly useful when a product is new on the market and has never been assessed before. Risk assessments should be solid and realistic. However, since they require a number of assumptions, different risk assessors may come to different conclusions in view of the data and other evidence they have been able to find or because of their diverging experience. It is thus necessary for risk assessors to talk to one another in order to reach agreement or, at least, consensus. The step-by-step risk assessment described in these guidelines, however, should make such discussions more productive. Each step in a risk assessment must be clearly described in detail. Thus, any point of disagreement can be quickly identified, and consensus can more easily be reached. This will make risk assessments more acceptable. Document your risk assessment It is important to document your risk assessment, describing the product and all the parameters that you chose while developing it, such as test results, the type(s) of consumers you chose for your injury scenario(s), and the probabilities with the underlying data and assumptions. This will enable you to demonstrate unambiguously how you estimated the level of risk, and it will also help you to update your assessment while keeping track of all changes. Several hazards, several injuries  but only one risk When several hazards, several injury scenarios or differing severities of injuries or probabilities have been identified, each of those should be carried through the entire risk assessment procedure in order to determine the risk for each. As a result, the product may have several risk levels. The overall risk of the product is then the highest risk level identified, because action on the highest risk level is normally the most effective way of risk reduction. Only in special cases may a less-than-highest risk be considered particularly important, since it may require specific risk management measures. As an example of several risks, a hammer may have a weak head and a weak grip, each of which may break when the hammer is used, and the consumer may be injured. If the relevant scenarios lead to different risk levels, the highest risk should be reported as the risk of the hammer. It could be argued that:  the apparently most significant hazard should be decisive, since it would lead to the most severe injuries. In the example of the hammer in point 2.1, this could be the hammer head breaking, since pieces of the broken head could fly into one's eye, possibly blinding the user. The hammer grip breaking, on the other hand, would never split into small pieces that could do as much damage to the eyes;  However, this would be a hazard assessment, not a risk assessment. A risk assessment also looks at the probability of an injury actually happening. Thus, the most significant hazard might cause an injury that is much less likely than a lesser hazard, and therefore present a lower risk. Conversely, a scenario leading to a less severe injury may be much more likely than a scenario resulting in death, and the less severe injury may therefore present a higher risk;  the highest probability for an injury scenario to happen should be the decisive factor for the risk of the product. In the example of the hammer in point 2.1, if the hammer grip is very weak, the most likely injury scenario would be from the grip breaking, and that should therefore be decisive. However, this would not consider the seriousness of eye injuries that the hammer head breaking could cause. Looking at probability alone would not therefore give the whole picture. In conclusion, risk is a balanced combination of both the hazard and the probability of the injury that the hazard can cause. Risk describes neither the hazard, nor the probability, but both at the same time. Taking the highest risk as the risk of the product will ensure the most effective product safety (apart from specific risks requiring specific risk management, as referred to at the beginning of this section. Can risks cumulate? Several injury scenarios leading to several risks can be developed for virtually every product. For example, an angle grinder may present the risk of an electric shock, because electrical wires may be too exposed, and the risk of fire, because the machine may overheat and ignite during normal use. If both risks are considered to be high, do they add up to the grinder posing an overall serious risk? Where several risks are linked to the same product, one of them is obviously more likely to materialise and causes an injury. The overall likelihood of an injury is therefore greater. This does not mean that the overall risk is automatically higher, however:  The overall probability is not calculated by simply adding up probabilities. More complex calculations are necessary, and these always result in a probability that is lower than the sum of all probabilities.  There is difference of a factor of 10 between two succeeding probability levels (table 4). This means that a lot of different scenarios of the same level would be needed to result in higher overall probability (and possibly risk).  Probability values are estimations which may not be totally accurate, as they often err on the safe side in order to ensure a high level of protection. It is therefore more useful to look at a more accurate estimation of the probability of a scenario leading to the highest risk than to add up rough estimations of probabilities of all sorts of scenarios.  With a little effort hundreds of injury scenarios could be developed. If risks were simply added together, the overall risk would depend on the number of injury scenarios generated and could increase endlessly. This does not make sense. Thus, risks are not simply cumulated. However, if more than one relevant risk exists, action to manage the risks may need to be taken more rapidly or may need to be more pronounced. For example, with two risks, a product may need to be immediately taken off the market and recalled, whereas, with a single risk, halting sales could be sufficient. Risk management depends on many factors, not only on the number of risks that a product may present at one and the same time. Thus, consideration is given to the link between risk and risk management (section 4). Compliance with limit values in legislation and standards In market surveillance, consumer products are often tested against limit values or requirements laid down in legislation and in product safety standards. A product that complies with the limit value(s) or requirement(s) (47) is presumed to be safe in terms of the safety characteristics covered by those value(s) or requirement(s). This assumption can be made because the risks of a product from its intended and reasonably foreseeable use are taken into account when establishing the limit value(s) or requirement(s). Manufacturers thus need their products to comply with these values or requirements, because they then only have to look at risks with their products that are not be covered by those limit value(s) or requirement(s). An example of a limit value in:  legislation is the limit of 5 mg/kg benzene in toys which must not be exceeded, as per point 5 of Annex XVII, to the REACH Regulation (48), as amended by Commission Regulation (EC) No 552/2009 (49);  a standard is the small parts cylinder: small parts of a toy for children under 36 months must not fit entirely into the cylinder described in the Toys Standard (50). If they do, they present a risk;  The product is presumed not to be safe where it fails to comply with established limit values. For limit values laid down in:  legislation, such as on cosmetics or restrictions on marketing and use, the product must not be made available on the market;  standards, the manufacturer may nevertheless try to provide evidence that his product is as safe as if it were compliant with the standard's limit value by way of a fully-fledged risk assessment on his product. However, this may require more effort, and may be impossible in cases such as the small parts cylinder referred to in the first bullet of this list, than actually manufacturing the product in compliance with the standard's limit value. Non-compliance with limit values does not automatically mean that the product presents a serious risk (which is the highest risk level covered by these guidelines). Therefore, to ensure appropriate risk reduction measures, a risk assessment will be required for those parts of a product that do not comply with or are not covered by legislation or a standard. Furthermore, some products, such as cosmetics, require a risk assessment even when they are compliant with the limit values laid down in legislation. This risk assessment should provide evidence of the safety of the whole product (51). In conclusion, compliance with limit values in legislation or in standards provides presumption of safety, but such compliance may not be sufficient. Specific risk assessment guidelines in specific cases For chemicals there are specific instructions on how to prepare a risk assessment (52), and therefore they are not dealt with in detail in these guidelines. Nevertheless, they follow the same principles as for normal consumer products:  hazard identification and assessment  this is the same as determining the severity of the injury, as described in section 2.2;  exposure assessment  in this step, exposure is expressed as the likely dose of the chemical that the consumer may take up via oral, inhalation or dermal routes, separately or jointly, when using the product as anticipated in the injury scenario. This step is the same as determining the probability that the injury will indeed occur;  risk characterisation  this step basically consists of comparing the dose of the chemical that the consumer is likely to take up (= exposure) with the derived no-effect level (DNEL) of that chemical. Should the exposure be sufficiently lower than the DNEL, in other words, should the risk characterisation ratio (RCR) be clearly below 1, risk is considered to be adequately controlled. This is the same as determining the risk level. Risk management measures may not be needed if the level of risk is sufficiently low. Since a chemical may possess several hazards, risk is normally determined for the leading health effect, which is the health effect (or endpoint such as acute toxicity, irritation, sensitisation, carcinogenicity, mutagenicity, toxicity for reproduction) considered to be the most important. For cosmetics, there is also specific guidance (53), and there may be specific guidance for other products or purposes. It is highly recommended to use such specific guidance, since it is tailored to the specific cases in question. Nevertheless, where the data required by the specific guidance do not exist or cannot be estimated the present guidelines may be used for a preliminary risk assessment. This risk assessment will have to be carried out with due care and attention in order to avoid any misinterpretation. 3. Building a risk assessment step by step This section describes in detail what points have to be taken into account and what questions have to be asked when preparing a risk assessment. 3.1. The product The product should be identified unambiguously. This includes the product name, the brand, the model name, the type number, a possible production lot number, any certificate that may come with the product, a child-resistant fastening if there is one, the identity of the person who placed it on the market, and the country of origin. A picture of the product, the packaging and the marking plate (if appropriate) and a test report(s) identifying the product hazard(s) can also be considered to be part of the product description. In particular cases, the hazard may be limited to a distinct part of the product, which can be separate from it and also separately available to consumers. In such cases, it is sufficient only to assess the distinct part of the product. Recharge able batteries of notebook computers which may overheat are an example of this. The description of the product includes any label that may be relevant for risk assessment, in particular warning labels. Instructions for use may also contain relevant information on the risk of the product and how to keep it as low as possible, for example by using personal protective equipment or by excluding children from using the product. An example of this is a chain saw. Products may also need to be self-assembled by consumers before use, such as self-assembled furniture. Are the assembly instructions clear enough for the ready-to-use product to meet all the relevant safety requirements? Or could consumers make mistakes when putting the product together that could lead to unforeseen risks? A risk assessment should always consider the entire life time of a product. This is particularly important when a new product has been developed and its risks are assessed. Will age and usage change the type or the extent of the hazard? Will new hazards appear with increasing product age or perhaps through reasonably foreseeable inappropriate use? How long is the time to product failure? What is the product's lifetime, including shelf life? How long is the product used in practice by the consumer before it becomes waste? Additional considerations may need to be taken into account when a product becomes unusable after a certain time period, even though it has never been used. Examples are electric blankets or heating pads. The electric cords in the products are usually thin and become fragile after ten years, even if the product has never been used. The heating cords can come into contact with each other, can cause a short-circuit and set the bedclothes on fire. Finally, the packaging of the product should also be included in any risk assessment. 3.2. The product hazard Hazard is the intrinsic property of the product that may cause an injury to the consumer who uses the product. It can appear in different forms:  mechanical hazard, such as sharp edges that can cut fingers, or tight openings in which someone can trap their fingers;  choking hazard, such as from small parts that come loose from a toy, which may be swallowed by a child and make the child choke;  suffocation hazard, such as from the drawstrings of an anorak hood which may lead to strangulation;  electrical hazard, such as from live electrical parts that can cause an electric shock;  heat or fire hazard, such as a heater fan that overheats, catches fire and causes burns;  thermal hazard, such as the hot outer surface of an oven that can cause a burn;  chemical hazard, such as a toxic substance that can poison a consumer immediately upon ingestion, or a carcinogenic substance that can cause cancer in the long term. Some chemicals may damage the consumer only after repeated exposure;  microbiological hazard, such as a bacteriological contamination of cosmetics which may cause a skin infection;  noise hazard, such as ring tones from toy mobile phones that are much too loud and can damage children's hearing capacity;  other hazards, such as explosion, implosion, sonic and ultrasonic pressure, fluid pressure, or radiation from laser sources. For the purpose of these guidelines, hazards have been grouped, linked to the size, shape and surface of a product, to potential, kinetic or electric energy, to extreme temperatures, and others, as shown in table 2. The table is for guidance only, and any risk assessor should adapt the scenario to the product under consideration. Of course not every type of hazard applies to every product. Nevertheless, table 2 should help risk assessors to look for and identify all possible hazards in consumer products that are being assessed. Where a product has several hazards, each hazard should be taken separately with its own risk assessment and the highest risk identified as the risk of the product. Of course, risks requiring specific risk management measures should also be reported, to ensure that all risks can be reduced. Note that a single hazard may lead to several injuries in the same scenario. For example, malfunctioning brakes on a motor cycle could cause an accident and result in damage to the driver's head, hands and legs, and could even cause burns if the petrol bursts into flames in the accident. In this case, all injuries would belong to the same injury scenario, and the severity of all injuries together would have to be estimated. Of course, these injuries together are very serious. Several injuries in different scenarios should, however, not be added. In the daily practice of market surveillance, it may be sufficient to assess the risk from even a single hazard. If the risk from that hazard provides for risk management action, that action can be taken without further ado. Nevertheless, the risk assessor should be sure that the risk identified is (one of) the highest risk(s), to ensure that the risk management action is sufficiently effective. This is always the case when the risk is serious, since this is the highest possible risk level proposed in these guidelines. In cases of less than serious risk, however, further risk assessments might be necessary and possibly specific risk management at a later stage. In conclusion, experience with risk assessment in market surveillance practice will limit the number of required risk assessments to a minimum. Hazard identification by tests and standards Hazards are often identified and quantified by tests. These tests and how to carry them out may be laid down in European or international product standards. Compliance of a product with a harmonised European standard (EN ¦), of which the references have been published in the Official Journal, provides presumption of safety (albeit only for the safety characteristics covered by the value(s) or standard(s)). It can be presumed in such cases that the product presents only a minimum risk and a high level of protection with regard to the specific hazard tested. Nevertheless, there may be instances where presumption of safety is not the case, and in such cases a particularly well-documented risk assessment will have to be prepared, including a call for amendment to the harmonised standard. On the other hand, if a product fails the test, a risk can normally be assumed, unless the manufacturer can provide evidence that the product is safe. Products may still present a risk even though they do not cause injuries Products may not be hazardous but can nevertheless cause a risk, due to not being fit for their intended use. Examples of this can be observed in the area of personal protective equipment or life-saving equipment, such as reflective jackets that car drivers put on after an accident. These jackets are meant to get the attention of oncoming drivers and traffic participants to warn them of the accident, in particular at night. However, they might not be seen if the reflector stripes are too small or do not reflect sufficiently, and do not therefore protect users as they should. These jackets therefore pose a risk even though they are not hazardous in themselves. Another example is a sunscreen product which displays high protection (sun protection factor of 30) on the label but provides only low protection (factor of 6). This can lead to severe sunburn. 3.3. The consumer The abilities and behaviour of the consumer using the product may greatly influence the level of risk. It is therefore of prime importance to have a clear idea of the type of consumer pictured in the injury scenario. It may be necessary to generate injury scenarios with different types of consumers in order to identify the highest risk and thus the risk of the product. It is not enough, for example, to consider only the most vulnerable consumers, because the probability of their suffering adverse effects in the scenario may be so low that the risk is lower than in an injury scenario with a non-vulnerable consumer. Consideration should also be given to people who are not actually using the product, but who may be in the vicinity of the user. For example, a chain saw may cause splinters to fly around and hit a bystander in the eye. Thus, although the risk from the chain saw may be effectively managed by the user him- or herself wearing protective equipment and complying with any other risk management measures specified by the manufacturer, bystanders may be under serious threat. Consequently, warnings should be given, for example in the chain saw instructions for use, about the risks to bystanders and how to minimise such risks. Thus, when developing an injury scenario, the following aspects should be taken into account regarding the type of consumer and how they use the product. This is not a complete list, but it should encourage risk assessors to describe their injury scenarios with the necessary level of detail. It should be noted that consumer also means people who are not actually using the product, but who may be affected by virtue of being nearby:  Intended/non-intended user: The intended user of a product may use the product with ease because he goes by the instructions or because he is familiar with this kind of product, including its apparent and non-apparent hazard(s). The hazard of the product may not then materialise, and the product risk could be minor. The non-intended user may not be familiar with the product and may not recognise the hazard(s). He therefore runs the risk of injury, and the consumer risk is thus higher. Thus, the risk may be different for an intended and a non-intended user, depending on the product and the way it is used.  Vulnerable consumers: Several categories of vulnerable and very vulnerable consumers can be distinguished: children (0 to 36 months, > 36 months to < 8 years, 8 to 14 years) and others such as the elderly (see table 1). They all have less capacity to recognise a hazard, for example children who, when touching a hot surface, notice the heat only after some 8 seconds (and then are already burnt), whereas adults notice heat immediately. Vulnerable consumers may also have problems taking account of warning labels, or may have particular problems using a product they have never used before. They may also act in a way that makes them more exposed, for example young children crawling and mouthing. Children may also be attracted to products because of their appeal, which makes them a high risk in the hands of children. On the other hand, supervision by parents or other adults should normally prevent children from running straight into trouble. Furthermore, consumers who are not usually vulnerable may become vulnerable in specific situations, for example when the instructions or warnings on a product are in a foreign language that the consumer does not understand. Finally, in the particular case of chemicals, children may be more susceptible to the toxicity of chemicals than the average adult. Therefore, children should not be treated as if they were small adults. In conclusion, a product that is normally safe for an average adult may not be safe for vulnerable consumers. This has to be taken into account when determining the severity and probability of an injury (see section 3.5) and thus the risk.  Intended and reasonably foreseeable use: Consumers may use a product for other purposes than the one for which it is intended, although the instructions are clearly understandable, including any warnings. Therefore, as warnings may not be fully effective, other uses than the intended ones also have to be taken into account in a risk assessment. This aspect is particularly important for the manufacturer of a product, since he has to ensure that the product is safe under any reasonably foreseeable conditions of use. Reasonably foreseeable use may have to be based on experience, because there may be no information available in official accident statistics or other sources of information. It may then be difficult to draw the line between reasonably foreseeable and totally unperceived scenarios. Nevertheless, even totally unperceived scenarios can be considered under these guidelines, even when they lead to very severe injuries, because such scenarios will always have very low probability. This possibly safeguards against such scenarios having too much of an influence in determining the overall risk of the product.  Frequency and duration of use: Different consumers may use a product often or not so often, and for longer or shorter periods of time. This depends on the attractiveness of the product and the ease with which it can be used. Daily or long-term use could make a consumer entirely familiar with a product and its specifics, including its hazards, instructions and warning labels, thus making the risk minor. On the other hand, daily or long-term use may make the consumer too used to the product and lead to user fatigue where he recklessly ignores instructions and warnings, thus increasing the risk. Finally, daily or long-term use may also accelerate product ageing, and any parts that cannot withstand such frequent use may quickly fail and cause a hazard, and possibly an injury, which also increases the risk.  Hazard recognition and protective behaviour and equipment: Some products are known for their hazards, such as scissors, knives, do-it-yourself drilling machines, chain saws, roller blades, bicycles, motor bikes and cars. In all these cases, the product hazard is clearly known or readily recognisable, or described in the instructions, which will include risk management measures. The consumer can then act carefully or use personal protective equipment such as gloves, helmets or seat-belts, thereby using the product in a way that minimises the risk. In other cases, the product hazard may not be so readily recognisable, such as a short-circuit within an electric iron, warning labels may be overlooked or misunderstood, and consumers will only rarely be able to take preventive measures.  Consumer behaviour in the event of an incident: Where the hazard impinges on the consumer it may cause injury. It is thus important for a risk assessment to consider how the consumer may react. Will he put the product to one side calmly and take preventive action, such as combating a fire caused by the product, or will he throw it away in a panic? Vulnerable consumers, especially children, may after all not behave the same as other, non-vulnerable consumers.  The consumer's cultural background and the way a product is used in his home country may influence the risk of a product. Manufacturers in particular have to take account of these cultural differences when launching a new product on a market. Manufacturers' experience in this area can thus be a valuable source of information for authorities preparing a risk assessment. 3.4. Injury scenario: Steps leading to injury(ies) Most injury scenarios consist of the following three main steps: 1. the product has a defect or can lead to a dangerous situation during its foreseeable lifetime; 2. the defect or dangerous situation results in an accident; 3. the accident results in an injury. These three main steps can be divided into further steps to show how the product hazard can lead to injury and the like. Nevertheless, these steps to injury have to be clear and concise, and not exaggerate the detail or the number of steps. With experience, it will be increasingly easier to identify the conditions for the occurrence of any given injury and the shortest path to injury (or critical path to injury). It is probably easiest to start with a scenario with the consumer for whom the product is intended where the consumer uses the product as per the instructions or, if there are none, according to normal handling and use. If this assessment produces the highest risk level, there is normally no need to carry out further assessments, and appropriate risk reduction measures can be taken. Similarly, where an incident is reported in a specific consumer complaint, a single injury scenario may be sufficient to conclude as to appropriate risk reduction measures. Otherwise, further scenarios could be developed to include vulnerable consumers, in particular children (see table 1), slight or more pronounced deviations from normal use, use under different climate conditions, such as very cold or very hot, unfavourable conditions of use, such as without proper daylight or illumination, use as suggested when the product was sold (for instance, a lamp sold in a toy shop should also be assessed for its risk when used by a child), use over the entire life-time (including wear and tear), etc. Each scenario should be considered through the entire risk assessment procedure. Where the product displays several hazards, injury and thus risk scenarios should be developed for each of them. Nevertheless, a plausibility check as to whether an injury scenario might lead to a risk requiring action can limit the number of injury scenarios. From all the scenarios generated, the scenario providing the highest risk (= the risk of the product) will normally be decisive for the risk reduction measures to be taken, because action on the highest risk reduces the risk most effectively. An exception to the rule might be a specific, less-than-highest risk stemming from a different hazard, which could be managed by specific measures and should, of course, also cover the highest risk. As a rule of thumb, injury scenarios can lead to the highest risk level when:  the injury(ies) considered are in the highest severity levels (levels 4 or 3);  the overall probability of an injury scenario is quite high (at least > 1/100). Table 4 provides further guidance in this respect. This might help to limit the number of scenarios. Of course, the number of injury scenarios remains the responsibility of the risk assessor, and it depends on the number of factors that need to be taken into account when determining the risk of the product. It is therefore impossible to give a specific number of injury scenarios that may be necessary in a specific case. To help develop a suitable number of scenarios, these guidelines provide a table with typical injury scenarios (table 2). These should be adapted to the specific product, consumer type and other circumstances. 3.5. Severity of injury The injury that a hazard can cause to the consumer can have different degrees of severity. The severity of the injury thus reflects the effect the hazard has on the consumer under the conditions described in the injury scenario. The severity of the injury can depend on:  the type of hazard (see list of hazards of section 3.2 in table 2). A mechanical hazard, such as sharp edges, can cause cuts to the fingers; these are immediately noticed, and the consumer will take action to heal his injuries. On the other hand, a chemical hazard may cause cancer. This normally passes unnoticed, and the illness may appear only after many years, and is considered to be very severe since cancer is very difficult to cure, if at all;  how powerful the hazard is. For example, a surface heated to 50 °C may cause slight burns, whereas a surface at 180 °C will cause severe burns;  how long the hazard impinges on the consumer. A short contact time with an abrasion hazard may scratch the consumer's skin only superficially, whereas a longer time may take off large parts of the skin;  what body part is injured. For example, penetration by a sharp point into the skin of the arm is painful, but penetration into an eye is a more serious and perhaps a life-affecting injury;  what impact the hazard has on one or several body parts. An electrical hazard may cause an electric shock with unconsciousness and, subsequently, a fire which may damage the lungs when the unconscious person inhales the smoke;  the type and behaviour of the consumer. A product labelled with a warning message can be used, without harm, by an adult consumer, because the consumer adjusts to using the product. On the other hand, a child or other vulnerable consumer (see table 1) who cannot read or understand the warning label may be very seriously injured. To quantify the severity of injury(ies), table 3 in these guidelines shows how to classify injuries into four categories, depending on the reversibility of an injury, i.e. whether recovery from an injury is possible and to what extent. This categorisation is for guidance only, and a risk assessor should change the category if necessary, and report it in the risk assessment. Where several injury scenarios are considered in the risk assessment, the severity of each injury should be classified separately, and considered throughout the entire risk assessment process. An example: A consumer uses a hammer to knock a nail into a wall. The hammer head is too weak (due to incorrect material) and it breaks, one of the pieces flying into the eye of the consumer so hard that it causes blindness. The injury is thus an eye injury, foreign body in eye: permanent loss of sight (one eye), which is a level 3 injury in table 3. 3.6. Probability of injury The probability of injury is the probability that injury scenario may indeed materialise during the expected lifetime of the product. This probability is not easy to estimate; but when a scenario is described in distinct steps, each step can be given a certain probability, and multiplying these partial probabilities together gives the overall probability of the scenario. This stepwise approach should make it easier to estimate the overall probability. Of course, where several scenarios are developed, each scenario requires its own overall probability. Where an injury scenario is nevertheless described in a single step, the probability of the scenario can also only be determined in a single overall step. This would only be a guesstimate, however, which could be severely criticised and thus call the entire risk assessment into question. A more transparent assignment of probabilities to a several-step scenario is therefore preferable, especially as the partial probabilities can be built on undisputable evidence. These guidelines distinguish between 8 levels of probability to classify overall probability: from < 1/1 000 000 to > 50 % (see left-hand side of table 4). The following example of a hammer head that breaks when the user knocks a nail into a wall should illustrate how to assign a probability to each step, and how to classify overall probability: Step 1: The hammer head breaks when the user tries to knock a nail into a wall because the material of the hammer head is too weak. The weakness was determined in a test, and with the reported weakness the probability of the hammer head breaking during the otherwise expected lifetime of the hammer is put at 1/10. Step 2: One of the pieces of the hammer hits the user when it breaks. The probability of this happening is put at 1/10, since the area of upper body exposed to the pieces flying off is considered to be 1/10 of the half-sphere in front of the wall. Of course, if the user were standing very close to the wall, his body would take a larger share of the half-sphere, and the probability would be higher. Step 3: The piece hits the user on the head. The head is estimated to be about 1/3 of the upper body, and the probability is therefore 1/3. Step 4: The piece hits the user in the eye. The eyes are considered to be about 1/20 of the area of the head, and therefore the probability is 1/20. Multiplying the probabilities of these steps together gives an overall probability for the scenario of 1/10 Ã  1/10 Ã  1/3 Ã  1/20 = 1/6 000. This translates into > 1/10 000 (see left-hand side of table 4). Once the overall probability has been calculated for an injury scenario, it should be checked for plausibility. This requires rather a lot of experience, thus suggesting that the assistance of persons experienced in risk assessment should be sought (see section Let others check your risk assessment). As experience is gained with these guidelines estimating probability should become easier, and an increasing number of examples will become available to facilitate this task. Assigning probabilities to different injury scenarios for the same product may lead to the following:  When the product is used by more vulnerable consumers in a scenario, the probability may have to be raised in general because more vulnerable consumers can be injured more easily. This applies in particular to children, since children do not normally have the experience to take preventive action, on the contrary (see also Vulnerable consumers in section 3.3).  When the risk is readily recognisable, including through warning labels, the probability may have to be lowered because the user will use the product more carefully in order to avoid injury as far as possible. This may not apply to an injury scenario with a (young) child or other vulnerable user (see table 1) who cannot read.  When accidents have been reported that fit into the injury scenario, the probability for that scenario could increase. In cases where accidents have only rarely been reported, or are not known at all, it may be useful to ask the manufacturer of the product whether he is aware of any accident or adverse effect caused by the product.  When a fairly large number of conditions are needed for the injury to occur, the overall probability of the scenario would normally be lower.  When the conditions needed for the injury to occur are easily met, this may increase the probability.  When the test results of the product fail by a large margin to come within the limit values required (by the relevant standard or legislation), the probability of the injury (scenario) occurring may be higher than if the product performed close to the limit values. The probability of injury in this instance is the probability that the injury scenario may actually happen. Probability does not therefore describe the general exposure of the population to the product, calculated, for example, by considering the millions of product items sold on the market and then considering that a few of them might fail. Considerations of this kind do, however, play a role when determining the appropriate risk reduction measures (see section 4). Also, accident statistics, even if product-specific, have to be considered with care when used for to estimate probability. The circumstances of the accident may not be reported in sufficient detail, the product may have changed over time, or the manufacturer may be different, and so on. In addition, light accidents may not have been reported to those collecting the data for the statistics. None the less, accident statistics can shed light on injury scenarios and their probability. 3.7. Determination of risk Once the severity of the injury and the probability have been determined, if possible for several injury scenarios, the risk level then needs to be looked up in table 4. Table 4 combines both the severity of the injury and the probability, and the highest risk is the risk of the product. Risks requiring specific risk management measures should also be reported, to ensure that all risks are reduced to a minimum. These guidelines distinguish between 4 levels of risk: serious, high, medium and low. The risk level between neighbouring severities of injury or probability normally changes by 1 level. This is consistent with the general experience that risk does not increase incrementally when input factors change gradually. However, where the severity of injury increases from level 1 to level 2 (on the right-hand side of table 4), some risk levels increase by 2 levels, namely from medium to serious and from low to high. This is due to the fact that these guidelines include 4 graduations of severity of injury, whereas the original method (see Introduction) included 5. Nevertheless, 4 graduations are considered normal for consumer products, since they make for a sufficiently robust estimation of severity; 5 levels would be too sophisticated since neither the severity of the injury nor the probability can be determined with very high precision. At the end of the risk assessment, be it for an individual injury scenario or for the overall risk of the product, the plausibility of the risk level and uncertainties in the estimates should be considered. This may mean verifying that the risk assessor has used the best information available to make his estimations and assumptions. Feedback from colleagues and other experts can also be helpful. A sensitivity analysis can also be very valuable (see example in section 6.3). How does the risk level change when the severity of injury or probability changes by 1 level up or down? If the risk level does not change at all, it is quite plausible that it has been estimated correctly. If it changes, however, the risk level may be borderline. It is then necessary to reconsider the injury scenarios and the assigned severity of injury(ies) and probability(ies). At the end of the sensitivity analysis the risk assessor should be confident that the risk level is sufficiently plausible and that he can document it and pass the information on. 4. From risk to action: how to manage risk responsibly Once the risk assessment is complete it will normally be used to decide whether action needs to be taken to reduce the risk and thus prevent harm to a consumer's health. Although action is separate from risk assessment, some points are raised here to illustrate the possible follow-up of identified risks. Within market surveillance, action will often be taken in contact between the authority and the manufacturer, importer or distributor. This can help the authority to determine the most effective and efficient way of managing the risk. With a serious risk in a consumer product, measures to reduce the risk may include withdrawal from the market or recall. Lower levels of risk normally lead to less rigorous measures. It may then be sufficient to add warning labels on the product or to improve the instructions to make the product safe. Thus, whatever the level of risk, the authority should consider whether to take action, and if so, what action. Nevertheless, there is no automatic link from risk to action. When a product shows several less-than-serious risks, and its overall risk is thus not serious, urgent action may be necessary since any of the risks may materialise quite quickly. The pattern of risks in the product may indicate a lack of quality control in production (54). It is also important to take account of exposure of the population as a whole. Where there are a large number of products on the market and the product is therefore used by a large number of consumers, even a single less-than-serious risk may require quick action to avoid adverse effects to the health of consumers. Less-than-serious risks may also require action when the product concerned could cause fatal accidents, even though such accidents may be extremely unlikely. This could be the case with a fastening on a beverage container, which could come loose and be swallowed by a child, causing the child to choke to death. A simple change of design to the lid could eliminate the risk, and no further action might be required. Even a selling-off period may be granted if the risk of a fatal accident were indeed extremely small. Other risk-related aspects may be the public perception of risk and its likely consequences, cultural and political sensitivities and how it is portrayed in the media. These aspects may be especially relevant when the consumers concerned are vulnerable, in particular children. It will be up to the national market surveillance authority(ies) to determine what measures are required. Taking action to counteract a risk may also depend on the product itself and the minimum risks compatible with the product's use, considered to be acceptable and consistent with a high level of protection (55). This minimum risk will probably be much lower for toys, where children are involved, than for a chain-saw, which is known to be so high-risk that solid protective equipment is required to keep the risk at a manageable level. Finally, even if there is no risk, action may be necessary, for example, when a product is non-compliant with the applicable regulation/legislation (e.g. incomplete markings). In conclusion, there is no automatic link from risk to action. Surveillance authorities will take account of a range of factors such as those indicated in section 3.3. The principle of proportionality always has to be considered, and action has to be effective. 5. How to prepare a risk assessment  in brief 1. Describe the product and its hazard. Describe the product unambiguously. Does the hazard concern the entire product or only a (separable) part of the product? Is there only one hazard within the product? Are there several hazards? See table 2 for guidance. Identify the standard(s) or legislation applicable to the product. Identify the standard(s) or legislation applicable to the product. 2. Identify the type of consumer you want to include in your injury scenario with the hazardous product. Start with the intended user and the intended use of the product for your first injury scenario. Take other consumers (See table 1) and uses for further scenarios. 3. Describe an injury scenario in which the product hazard(s) you have selected causes an injury(ies) or adverse health effect(s) to the consumer you selected. Describe the steps to the injury(ies) clearly and concisely, without exaggerating the details (shortest path to injury, critical path to injury). If there are several concurrent injuries in your scenario, include them all in that same scenario. When you describe the injury scenario, consider the frequency and duration of use, hazard recognition by the consumer, whether the consumer is vulnerable (in particular children), protective equipment, the consumer's behaviour in the case of an accident, the consumer's cultural background, and other factors that you consider important for the risk assessment. See section 3.3 and table 2 for guidance. 4. Determine the severity of the injury. Determine the level of severity (1 to 4) of the injury to the consumer. If the consumer suffers from several injuries in your injury scenario, estimate the severity of all those injuries together. See table 3 for guidance. 5. Determine the probability of the injury scenario. Assign a probability to each step of your injury scenario. Multiply the probabilities to calculate the overall probability of your injury scenario. See left-hand side of table 4 for guidance. 6. Determine the risk level. Combine the severity of the injury and the overall probability of the injury scenario and check the risk level in table 4. 7. Check whether the risk level is plausible. If the risk level does not seem plausible, or if you are uncertain about the severity of injury(ies) or about the probability(ies), move them one level up and down and recalculate the risk. This sensitivity analysis will show you whether the risk changes when your input changes. If the risk level remains the same, you can be quite confident of your risk assessment. If it changes easily, you may want to err on the safe side and take the higher risk level as the risk of the consumer product. You could also discuss the plausibility of the risk level with experienced colleagues. 8. Develop several injury scenarios to identify the highest risk of the product. If your first injury scenario identifies a risk level below the highest risk level set out in these guidelines, and if you think that the product may pose a higher risk than the one identified,  select other consumers (including vulnerable consumers, in particular children);  identify other uses (including reasonably foreseeable uses), in order to determine which injury scenario puts the product at its highest risk. The highest risk is normally the risk of the product that allows the most effective risk management measures. In specific cases, a particular hazard may lead to a less-than-highest risk and require specific risk management measures. This has to be taken duly into account. As a rule of thumb, injury scenarios may lead to the highest risk level set out in these guidelines where:  the injury(ies) considered are at least at levels 3 or 4;   the overall probability of an injury scenario is at least > 1/100. See table 4 for guidance. 9. Document and pass on your risk assessment. Be transparent and also set out all the uncertainties that you encountered when making your risk assessment. Examples for reporting risk assessments are provided in section 6 of these guidelines. Schematic flow of risk assessment Pass on the risk assessment Highest risk identified? Yes No 6. Look up the Risk in Table 4 See table 4: Probability levels from high (&gt; 50 %) to low (&lt; 1/1 000 000) See table 3: Severity of injury  Laceration, cut  Bruising  Concussion  Entrapment/pinching  Sprain, strain, musculoskeletal disorder  Dislocation  Fracture  Crushing  Amputation  etc. 5. Determine the probability Assign a probability to each step. Multiply to get the overall probability 4. Determine the severity of the injury 3. Describe the Injury scenario in several steps: Shortest path to injury See table 1: Consumer types, incl. vulnerable consumers (in particular children)  Intended/non-intended user  Intended and reasonably foreseeable use  Frequency and duration of use  Hazard recognition/protective behaviour ¦  Consumer behaviour in the case of an incident  Consumer's cultural background See table 2: Hazards ¦  Size, shape and surface  Potential energy  Kinetic energy  Electrical energy  Extreme temperatures  Radiation  Fire and explosion  etc. 2. Identify consumer(s) 1. Describe the product unambiguously, and its hazard(s) 6. Examples 6.1. Folding chair A folding chair has a folding mechanism constructed in such a way that the user's fingers can get trapped between the seat and the folding mechanism. This can lead to fractures or even loss of one or more fingers. Determination of risk(s) Injury scenario Injury type and location Severity of injury Probability of injury Overall probability Risk Person unfolds the chair, grips seat close to the back corner by mistake (Person inattentive/ distracted), finger gets caught between seat and backrest Minor pinching of finger 1 Unfolding the chair 1 1/500 Low risk Gripping the seat at back corner while unfolding 1/50 Finger gets caught 1/10 > 1/1 000 Minor pinching 1 Person unfolds the chair, grips seat at the side by mistake (Person inattentive/ distracted), finger gets caught between seat and link Minor pinching of finger 1 Unfolding the chair 1 1/500 Low risk Gripping the seat at the side while unfolding 1/50 Finger gets caught 1/10 > 1/1 000 Minor pinching 1 Person unfolds the chair, chair is clamped, person tries to push down the seat and grips seat close to the corner by mistake (Person inattentive/ distracted), finger gets caught between seat and backrest Fracture of finger 2 Unfolding the chair 1 1/500 000 Low risk Chair clamps 1/1 000 Gripping the seat at corners while unfolding 1/50 Finger gets caught 1/10 > 1/1 000 000 Fracture of finger 1 Person unfolds the chair, chair is clamped, person tries to push down the seat and grips seat at the side by mistake (Person inattentive/ distracted), finger gets caught between seat and link Fracture of finger 2 Unfolding the chair 1 1/500 000 Low risk Chair clamps 1/1 000 Gripping the seat at the side while unfolding 1/50 Finger gets caught 1/10 > 1/1 000 000 Fracture of finger 1 Person is sitting on chair, wants to move the chair and tries to lift it by gripping the chair at the rear part of the seat, finger gets caught between seat and backrest Loss of digit 3 Sitting on chair 1 1/6 000 High risk Moves the chair while sitting 1/2 Grips chair at rear part while moving 1/2 Chair partially folds, creating a gap between the backrest and seat 1/3 > 1/10 000 Finger is between backrest and seat 1/5 Finger gets caught 1/10 Loss of (part of) finger 1/10 Person is sitting on chair, wants to move the chair and tries to lift it by gripping the chair at the rear part of the seat, finger gets caught between seat and link Loss of digit 3 Sitting on chair 1 1/6 000 High risk Moves the chair while sitting 1/2 Grips chair at rear part while moving 1/2 Chair partially folds, creating a gap between the backrest and seat 1/3 > 1/10 000 Finger is between backrest and seat 1/5 Finger gets caught 1/10 Loss of (part of) finger 1/10 The overall risk of the folding chair is thus high risk. 6.2. Socket protectors This case deals with socket protectors. These are devices that users (parents) put into the electrical socket outlets to stop small children from accessing live parts by putting a long metal object into one of the holes in the outlet and getting a (fatal) electric shock. The holes in this particular protector (where the pins of the plug go through) are so narrow that the pins can get stuck. This means that the user may pull the protector off the outlet when the plug is pulled out. The user may not notice this happening. Determination of risk(s) Injury scenario Injury type and location Severity of injury Probability of injury Overall probability Risk Protector is removed from the socket, which becomes unprotected. Child is playing with thin conductible object, which can be inserted into the socket, accessing high voltage and is electrocuted. Electrocution 4 Removal of protector 9/10 27/160 000 Serious risk Not noticing the removal of protector 1/10 Child is playing with thin conductible object 1/10 Child is unattended when playing 1/2 > 1/10 000 Child inserts the object into the socket 3/10 Access to voltage 1/2 Electrocution due to voltage (without circuit interrupter) 1/4 Protector is removed from the socket, which becomes unprotected. Child is playing with thin conductible object, which can be inserted into the socket, accessing high voltage and sustains shock. Burns 2nd degree 1 Removal of protector 9/10 81/160 000 Low risk Not noticing the removal of protector 1/10 Child is playing with thin conductible object 1/10 Child inserts the object into the socket 3/10 Access to voltage 1/2 > 1/10 000 Child is unattended when playing 1/2 Burn due to electric current (without circuit interrupter) 3/4 Socket unprotected. Child is playing with thin conductible object, which can be inserted into the socket, accessing high voltage and is electrocuted. Electrocution 4 Child is playing with thin conductible object 1/10 3/80 000 High risk Child is unattended when playing 1/100 Child inserts the object into the socket 3/10 Access to voltage 1/2 > 1/100 000 Electrocution due to voltage (without circuit interrupter) 1/4 The overall risk of the socket protectors is thus serious. 6.3. Sensitivity analysis The factors used to calculate the risk of an injury scenario, namely the severity of the injury and the probability, often have to be estimated. This creates uncertainty. Probability in particular can be difficult to estimate, since the behaviour of consumers, for example, can be difficult to predict. Does a person perform a certain action often or only occasionally? It is therefore important to consider the level of uncertainty of the two factors and to make a sensitivity analysis. The purpose of this analysis is to establish how much the risk level varies when the estimated factors vary. The example provided on the table below only shows the variation of probability, since the severity of the injury is usually predicted with more certainty. A practical way of performing the sensitivity analysis is to repeat the risk assessment for a certain scenario, but to use a different probability for one or more steps in the scenario. For example, a candle containing seeds could cause a fire, because the seeds can catch fire and generate high flames. Furniture or curtains can catch fire and persons not in the room could inhale toxic fumes and suffer fatal poisoning: Injury scenario Injury type and location Severity of injury Probability of injury Resulting probability Risk Seeds or beans catch fire generating high flames. Furniture or curtains catch fire. Persons are not in room, but inhale toxic fumes. Fatal poisoning 4  Seeds or beans catch fire: 90 % (0,9)  People not in the room for some time: 30 % (0,3)  Furniture or curtains catch fire: 50 % (0,5) (depends on surface on which candle is placed)  Persons inhale toxic fumes: 5 % (0,05) 0 00675 > 1/1 000 Serious The probability levels for the steps in the scenario were estimated as shown in the table. The overall probability is 0,00675, which corresponds to > 1/1 000 in table 4. This leads to the conclusion of serious risk. Note that the exact probability is closer to 1/100 than to 1/1 000, which already gives some confidence in the risk level because it is a little deeper in the serious risk area of table 4 than the > 1/1 000 row suggests. Suppose we are uncertain about the 5 % probability that persons inhale the toxic fumes. We could put it at a much lower 0,1 % (0 001 = 1 in a thousand). If we recalculate with that assumption, the overall probability is 0,000135, which translates into > 1/10 000. Nevertheless, the risk is still serious. Even if for some reason the probability were to be a factor of 10 lower, the risk would still be high. Therefore, although the probability may vary 10- or 100-fold, we still find a serious or high risk (the latter being quite close to serious). Thus, this sensitivity analysis lets us confidently assess the risk as serious. In general, however, risk assessment should be based on reasonable worst cases: not too pessimistic on every factor, but certainly not too optimistic. Table 1 Consumers Consumers Description Very vulnerable consumers Very young children: 0 to 36 months Others: Persons with extensive and complex disabilities Vulnerable consumers Young children: Children older than 36 months and younger than 8 years. Older children: Children 8 to 14 years Others: Persons with reduced physical, sensory or mental capabilities (e.g. partially disabled, elderly, including those over 65, with some reduction in their physical and mental capabilities), or lack of experience and knowledge Other consumers Consumers other than very vulnerable or vulnerable consumers Table 2 Hazards, typical injury scenarios and typical injuries Hazard group Hazard (product property) Typical injury scenario Typical injury Size, shape and surface Product is obstacle Person trips over product and falls; or person bumps into product Bruising; fracture, concussion Product is impermeable to air Product covers mouth and/or nose of a person (typically a child), or covers internal airway Suffocation Product is or contains small part Person (child) swallows small part; the part gets stuck in larynx and blocks airways Choking, internal airway obstruction Possible to bite off small part from product Person (child) swallows small part; the part gets stuck in the digestive tract Digestive tract obstruction Sharp corner or point Person bumps into sharp corner or is hit by moving sharp object; this causes a puncture or penetration injury Puncture; blinding, foreign body in eye; hearing, foreign body in ear Sharp edge Person touches sharp edge; this lacerates the skin or cuts through tissues Laceration, cut; amputation Slippery surface Person walks on surface, slips and falls Bruising; fracture, concussion Rough surface Person slides along rough surface; this causes friction and/or abrasion Abrasion Gap or opening between parts Person puts a limb or body in opening and finger, arm, neck, head, body or clothing is trapped; injury occurs due to gravity or movement Crushing, fracture, amputation, strangulation Potential energy Low mechanical stability Product tips; person on top of product falls from height, or person near product is hit by the product; electrical product tips, breaks and gives access to live parts, or continues to work heating nearby surfaces Bruising; dislocation; sprain; fracture, concussion; crushing; electric shock; burns Low mechanical strength Product collapses by overloading; person on top of product falls from height, or person near product is hit by the product; electrical product tips, breaks and gives access to live parts, or continues to work heating nearby surfaces Bruising; dislocation; fracture, concussion; crushing; electric shock; burns High position of user Person at high position on the product loses balance, has no support to hold on to and falls from height Bruising; dislocation; fracture, concussion; crushing Elastic element or spring Elastic element or spring under tension is suddenly released; person in the line of movement is hit by the product Bruising; dislocation; fracture, concussion; crushing Pressurised liquid or gas, or vacuum Liquid or gas under pressure is suddenly released; person in the vicinity is hit; or implosion of the product produces flying objects Dislocation; fracture, concussion; crushing; cuts (see also under fire and explosion) Kinetic Energy Moving product Person in the line of movement of the product is hit by the product or run over Bruising; sprain; fracture, concussion; crushing Parts moving against one another Person puts a body part between the moving parts while they move together; the body part gets trapped and put under pressure (crushed) Bruising; dislocation; fracture; crushing Parts moving past one another Person puts a body part between the moving parts while they move close by (scissor movement); the body part gets trapped between the moving parts and put under pressure (shearing) Laceration, cut; amputation Rotating parts A body part, hair or clothing of a person is entangled by the rotating part; this causes a pulling force Bruising; fracture; laceration (skin of the head); strangulation Rotating parts close to one another A body part, hair or clothing of a person is drawn in by the rotating parts; this causes a pulling force and pressure on the body part Crushing, fracture, amputation, strangulation Acceleration Person on the accelerating product loses balance, has no support to hold on to and falls with some speed Dislocation; fracture, concussion; crushing Flying objects Person is hit by the flying object and depending on the energy sustains injuries Bruising; dislocation; fracture, concussion; crushing Vibration Person holding the product loses balance and falls; or prolonged contact with vibrating product causes neurological disorders, osteoarticular disorder, trauma of the spine, vascular disorder Bruising; dislocation; fracture; crushing Noise Person is exposed to noise from the product. Tinnitus and hearing loss may occur depending on sound level and distance Hearing injury Electrical Energy High/low voltage Person touches part of the product that is at high voltage; the person receives an electric shock and may be electrocuted Electric shock Heat production Product becomes hot; a person touching it may sustain burns; or the product may emit molten particles, steam, etc., that hits a person Burn, scald Live parts too close Electric arc or sparks occur between the live parts. This may cause a fire and intense radiation Eye injury; burn, scald Extreme temperatures Open flames A person near the flames may sustain burns, possibly after clothing catches fire Burn, scald Hot surfaces Person does not recognise the hot surface and touches it; the person sustains burns Burn Hot liquids Person handling a container of liquid spills some of it; the liquid falls on the skin and causes scalds Scald Hot gases Person breathes in the hot gases emitted from a product; this causes lung burn; or prolonged exposure to hot air causes dehydration Burn Cold surfaces Person does not recognise the cold surface and touches it; the person sustains frostbite Burn Radiation Ultraviolet radiation, laser Skin or eyes of a person are exposed to radiation emitted by the product Burn, scald; neurological disorders; eye injury; skin cancer, mutation High intensity electromagnetic field (EMF) source; low frequency or high frequency (microwave) Person is close to the electromagnetic field (EMF) source, body (central nervous system) is exposed Neurological (brain) damage, leukaemia (children) Fire and explosion Flammable sub stances Person is near the flammable substance; an ignition source sets the substance on fire; this causes injuries to the person Burn Explosive mixtures Person is near the explosive mixture; an ignition source causes an explosion; the person is hit by the shock wave, burning material and/or flames Burn, scald; eye injury, foreign body in eye; hearing injury, foreign body in ear Ignition sources The ignition source causes a fire; a person is injured by flames, or intoxicated by gases from the house fire Burn; poisoning Overheating Product overheats; fire, explosion Burn, scald; eye injury, foreign body in eye; hearing injury, foreign body in ear Toxicity Toxic solid or fluid Person ingests substance from product, e.g. by putting it in mouth, and/or substance gets on skin Acute poisoning; irritation, dermatitis Person breathes in solid or fluid, for example vomited material (pulmonary aspiration) Acute poisoning in lungs (aspiration pneumonia); infection Toxic gas, vapour or dust Person inhales substance from product; and/or substance gets on skin Acute poisoning in lungs; irritation, dermatitis Sensitising substance Person ingests substance from product, e.g. by putting it in mouth; and/or substance gets on skin; and/or person inhales gas, vapour or dust Sensitisation; allergic reaction Irritating or corrosive solid or fluid Person ingests substance from product, e.g. by putting it in mouth, and/or substance gets on skin or in eyes Irritation, dermatitis; skin burn; eye injury, foreign body in eye Irritating or corrosive gas or vapour Person inhales substance from product, and/or substance gets on skin or in eyes Irritation, dermatitis; skin burn; acute poisoning or corrosive effect in lungs or in eyes CMR substance Person ingests substance from product, e.g. by putting it in mouth, and/or substance gets onto skin; and/or person inhales substance as gas, vapour or dust Cancer, mutation, reproductive toxicity Microbiological contamination Microbiological contamination Person gets into contact with contaminated product by ingestion, inhalation or skin contact Infection, local or systemic Product operating hazards Unhealthy posture Design causes unhealthy posture of person when operating the product Strain; musculoskeletal disorder Overexertion Design requires use of considerable force when operating the product Sprain or strain; musculoskeletal disorder Anatomical unsuitability Design is not adapted to human anatomy, which makes it difficult or impossible to operate Sprain or strain Ignoring personal protection Design makes it difficult for a person wearing protection to handle or operate the product Various injuries Inadvertent (de)activation Person can easily (de)activate product, which leads to unwanted operation Various injuries Operational inadequacy Design provokes faulty operation by a person; or product with a protective function does not provide expected protection Various injuries Failure to stop Person wants to stop the product, but it continues to operate in situation where this is unwanted Various injuries Unexpected start Product shuts down during a power failure, but resumes operation in a hazardous way Various injuries Inability to stop In an emergency situation, person is not able to stop operation of the product Various injuries Inadequately fitting parts Person tries to fit a part, needs too much force to fit, product breaks; or part is too loosely fitted and becomes loose during use Sprain or strain; laceration, cut; bruising; entrapment Missing or incorrectly fitted protection Hazardous parts are reachable for a per son Various injuries Insufficient warning instructions, signs and symbols User does not notice warning instructions signs and/or does not understand symbols Various injuries Insufficient warning signals User does not see or hear warning signal (optical or audio), causing dangerous operation Various injuries NB: This table is for guidance only; the typical injury scenarios should be adapted when preparing a risk assessment. There is specific risk assessment guidance for chemicals, cosmetics and possibly others. It is highly recommended to use this specific guidance when assessing such products. See section 3.2. Table 3 Severity of injury Introduction These risk assessment guidelines distinguish between four levels of injury harm severity. It is important to realise that severity should be assessed completely objectively. The aim is to compare the severity of different scenarios and to set priorities, not to judge the acceptability of a single injury at this stage. Any injury harm that could easily have been avoided will be difficult to accept for a consumer. However, authorities can justifiably invest more effort into avoiding irreversible consequences than into preventing temporary discomfort. In order to assess the severity of the consequences (acute injury or other damage to health), objective criteria can be found, on the one hand, in the level of medical intervention, and, on the other hand, in the consequences to the further functioning of the victim. Both could be expressed as cost, but the costs of consequences of health damage may be difficult to quantify. Combining these criteria, the four levels may be defined as follows: 1. Harm or consequence that after basic treatment (first aid, normally not by a doctor) does not substantially hamper functioning or cause excessive pain; usually the consequences are completely reversible. 2. Harm or consequence for which a visit to A&E may be necessary, but in general, hospitalisation is not required. Functioning may be affected for a limited period, not more than about 6 months, and recovery is more or less complete. 3. Harm or consequence that normally requires hospitalisation and will affect functioning for more than 6 months or lead to a permanent loss of function. 4. Harm or consequence that is or could be fatal, including brain death; consequences that affect reproduction or offspring; severe loss of limbs and/or function, leading to more than approximately 10 % of disability. The following table, which should be considered as a guide rather than prescriptive or complete, provides examples of injuries at all four levels. National differences may exist, either cultural or caused by different systems of health care and financial arrangements. However, deviating from the proposed classification in the table will affect uniform assessment of risks in the EU; this should be clearly stated and explained in the risk assessment report, and reasons should be given. Type of injury Severity of injury 1 2 3 4 Laceration, cut Superficial External (deep) (> 10 cm long on body) (> 5 cm long on face) requiring stitches Tendon or into joint White of eye or cornea Optic nerve Neck artery Trachea Internal organs Bronchial tube Oesophagus Aorta Spinal cord (low) Deep laceration of internal organs Severed high spinal cord Brain (severe lesion/ dysfunction) Bruising (abrasion/ contusion, swelling, oedema) Superficial  ¤ 25 cm2 on face  ¤ 50 cm2 on body Major > 25 cm2 on face > 50 cm2 on body Trachea Internal organs (minor) Heart Brain Lung, with blood or air in chest Brain stem Spinal cord causing paralysis Concussion  Very short unconsciousness (minutes) Prolonged unconsciousness Coma Entrapment/ pinching Minor pinching  (Use as appropriate the final outcomes of bruising, crushing, fracture, dislocation, amputation, as applicable.) (Same outcome as for suffocation/ strangulation.) Sprain, strain, musculoskeletal disorder Extremities Joints Spine (no dislocation or fracture) Knee ligaments strain Ligament or tendon rupture/tear Muscle tear Whiplash  Dislocation  Extremities (finger, toe, hand, foot) Elbow Jaw Loosening of tooth Ankle Wrist Shoulder Hip Knee Spine Spinal column Fracture  Extremities (finger, toe, hand, foot) Wrist Arm Rib Sternum Nose Tooth Jaw Bones around eye Ankle Leg (femur and lower leg) Hip Thigh Skull Spine (minor compression fracture) Jaw (severe) Larynx Multiple rib fractures Blood or air in chest Neck Spinal column Crushing   Extremities (fingers, toe, hand, foot) Elbow Ankle Wrist Forearm Leg Shoulder Trachea Larynx Pelvis Spinal cord Mid-low neck Chest (massive crushing) Brain stem Amputation   Finger(s) Toe(s) Hand Foot (Part of) Arm Leg Eye Both extremities Piercing, puncturing Limited depth, only skin involved Deeper than skin Abdominal wall (no organ involvement) Eye Internal organs Chest wall Aorta Heart Bronchial tube Deep injuries in organs (liver, kidney, bowel, etc.) Ingestion   Internal organ injury (Refer also to internal airway obstruction where the ingested object gets stuck high in the oesophagus.) Permanent damage to internal organ Internal air way obstruction   Oxygen flow to brain blocked without permanent consequences Oxygen flow to brain blocked with permanent consequences Suffocation/ Strangulation   Oxygen flow to brain blocked without permanent consequences Fatal suffocation/ strangulation Submersion/ Drowning    Fatal drowning Burn/Scald (by heat, cold, or chemical substance) 1 °, up to 100 % of body surface 2 °, < 6 % of body surface 2 °, 6-15 % of body surface 2 °, 16-35 % of body surface, or 3 °, up to 35 % of body surface Inhalation burn 2 ° or 3 °, > 35 % of body surface Inhalation burn requiring respiratory assistance Electric shock (See also under burns as electric current can cause burns.) Local effects (temporary cramp or muscle paralysis)  Electrocution Neurological disorders   Triggered epileptic seizure  Eye injury, foreign body in eye Temporary pain in eye without need for treatment Temporary loss of sight Partial loss of sight Permanent loss of sight (one eye) Permanent loss of sight (both eyes) Hearing injury, foreign body in ear Temporary pain in ear without need for treatment Temporary impairment of hearing Partial loss of hearing Complete loss of hearing (one ear) Complete loss of hearing (both ears) Poisoning from substances (ingestion, inhalation, dermal) Diarrhoea, vomiting, local symptoms Reversible damage to internal organs, e.g. liver, kidney, slight haemolytic anaemia Irreversible damage to internal organs, e.g. oesophagus, stomach, liver, kidney, haemolytic anaemia, reversible damage to nerve system Irreversible damage to nerve system Fatality Irritation, dermatitis, inflammation or corrosive effect of substances (inhalation, dermal) Local slight irritation Reversible eye damage Reversible systemic effects Inflammatory effects Lungs, respiratory insufficiency, chemical pneumonia Irreversible systemic effects Partial loss of sight Corrosive effects Lungs, requiring respiratory assistance Asphyxia Allergic reaction or sensitisation Mild or local allergic reaction Allergic reaction, widespread allergic contact dermatitis Strong sensitisation, provoking allergies to multiple substances Anaphylactic reaction, shock Fatality Long-term damage from contact with substances or from exposure to radiation Diarrhoea, vomiting, local symptoms Reversible damage to internal organs, e.g. liver, kidney, slight haemolytic anaemia Damage to nervous system, e.g. Organic Psycho Syndrome (OPS; also called Chronic Toxic Encephalopathy, also known as painters' disease). Irreversible damage to internal organs, e.g. oesophagus, stomach, liver, kidney, haemolytic anaemia, reversible damage to nervous system Cancer (leukaemia) Effects on reproduction Effects on offspring CNS depression Microbiological infection Reversible damage Irreversible effects Infection requiring prolonged hospitalisation, antibiotics-resistant organisms Fatality Table 4 Risk level from the combination of the severity of injury and probability Probability of damage during foreseeable lifetime of the product Severity of injury 1 2 3 4 High Low >50 % H S S S > 1/10 M S S S > 1/100 M S S S > 1/1 000 L H S S > 1/10 000 L M H S > 1/100 000 L L M H > 1/1 000 000 L L L M < 1/1 000 000 L L L L S  Serious Risk H  High risk M  Medium risk L  Low risk Glossary of terms Hazard: Source of danger involving the chance of being injured or harmed. A means of quantifying the hazard in a risk assessment is the severity of the possible injury or harm. Product hazard: Hazard created by the properties of a product. Risk: Balanced combination of a hazard and the probability that damage will occur. Risk describes neither the hazard, nor the probability, but both at the same time. Risk assessment: Procedure for identifying and assessing hazards, consisting of three steps: 1. identification of the seriousness of a hazard; 2. determination of the probability that a consumer will be injured by that hazard; 3. combination of the hazard with the probability. Risk level: Degree of risk, which may be serious, high, medium and low. When the (highest) level of risk has been identified, the risk assessment is complete. Risk management: Follow-up action, which is separate from risk assessment and aims to reduce or eliminate a risk. (1) In other places of these Guidelines, the term Commission generally refers to the RAPEX team established in the Commission department responsible for Directive 2001/95/EC and to the relevant Commission services, where appropriate. (2) The Information and Communication System on Market Surveillance (ICSMS). This platform is aimed at facilitating communication between market surveillance bodies in the EU and in EFTA countries on non-compliant products. (3) In the context of this document, the term Member States must be interpreted as not precluding all other actors from being addressed by the provisions contained in these Guidelines. (4) See the latest EC Implementing Decision published on https://ec.europa.eu/consumers/consumers_safety/safety_products/rapex/alerts/repository/content/pages/rapex/index_en.htm (5) See recital 10 of Directive 2001/95/EC. (6) Regulation (EC) No 178/2002 of the European Parliament and of the Council of 28 January 2002 laying down the general principles and requirements of food law, establishing the European Food Safety Authority and laying down procedures in matters of food safety (OJ L 31, 1.2.2002, p. 1). (7) Directive 2001/83/EC of the European Parliament and of the Council of 6 November 2001 on the Community code relating to medicinal products for human use (OJ L 311, 28.11.2001, p. 67). (8) Directive 2001/82/EC of the European Parliament and of the Council of 6 November 2001 on the Community code relating to veterinary medicinal products (OJ L 311, 28.11.2001, p. 1). (9) Regulation (EU) 2017/745 of the European Parliament and of the Council of 5 April 2017 on medical devices, amending Directive 2001/83/EC, Regulation (EC) No 178/2002 and Regulation (EC) No 1223/2009 and repealing Council Directives 90/385/EEC and 93/42/EEC (OJ L 117, 5.5.2017, p. 1). (10) Council Directive 90/385/EEC of 20 June 1990 on the approximation of the laws of the Member States relating to active implantable medical devices (OJ L 189, 20.7.1990, p. 17). (11) Directive (EU) 2015/1535 of the European Parliament and of the Council of 9 September 2015 laying down a procedure for the provision of information in the field of technical regulations and of rules on Information Society services (OJ L 241, 17.9.2015, p. 1). (12) See EU general risk assessment methodology (Action 5 of Multi-Annual Action Plan for the surveillance of products in the EU (COM(2013)76) providing guidance to authorities with relation to Article 20(2) of Regulation (EC) No 765/2008: http://ec.europa.eu/DocsRoom/documents/17107/attachments/1/translations (13) See https://ec.europa.eu/consumers/consumer-safety/rag/#/screen/home (14) www.ec.europa.eu/rapex (15) See Part I, Chapter 6.2 of these Guidelines. (16) See Part II, Chapter 2.2.1 of these Guidelines. (17) See Part II, Chapter 2.2.1 of these Guidelines. (18) See Part II, Chapter 2.2.2 of these Guidelines. (19) See Part II, Chapter 2.2.2 of these Guidelines. (20) For more information on follow-up actions, see Part II Chapter 4.4.5 of these Guidelines. (21) For more information about notifications where safety aspects are subject to discussions at EU level, see Part II Chapters 3.4.4 and 3.4.7.1.1 of these Guidelines. (22) See point 10 of Annex II of Directive 2001/95/EC. (23) See point 9 of Annex II of Directive 2001/95/EC. (24) For more information on notifications on safety aspects subject to discussions at EU level, see Part II Chapters 3.1.2(d) and 3.4.7.1.1. (25) https://ec.europa.eu/consumers/consumers_safety/safety_products/rapex/alerts/?event=main.search (26) Practice already agreed at the GPSD Committee of 24 September 2012, of which RAPEX Contact Points were informed at the RAPEX Contact Point meeting of 4 October (agenda point 4) and applied since 2013. (27) Paragraph 1 of Article 16(1) of Directive 2001/95/EC and Article 23 paragraph 3 in relation to Article 19(5) of Regulation (EC) No 765/2008. (28) Article 16(1) and (2) of Directive 2001/95/EC. (29) For more information on the notification criteria, see Part I Chapter 2. (30) For more information on notifications sent through the RAPEX application before measures are taken, see Chapter 3.1.2(b). (31) For more information on notifications on safety aspects subject to discussions at EU level, see Part II Chapters 3.1.2.d and 3.4.4. (32) For more information about deadlines, see Part III Appendix 4 of these Guidelines. (33) All deadlines mentioned in these Guidelines are expressed in calendar days. (34) See point 10 of Annex II of Directive 2001/95/EC. (35) For the purpose of these Guidelines, economic operator refers to any natural of legal person defined as economic operator in Regulation (EC) No 765/2008 or as producer and distributor in the GPSD. (36) https://webgate.ec.europa.eu/etranslation/translateDocument.html (37) There is no need to send notifications via the Permanent Representation of a Member State to the EU. (38) See Part I, Chapter 5.3 of these Guidelines. (39) The fields contained in the template may be updated following developments agreed between the Commission and Member States. (40) The fields contained in the template may be updated following developments agreed between the Commission and Member States. (41) If you need more information on the Risk Assessment method for harmonised products (both consumer and professional products) in relation to broader categories of public risks protected under EU harmonisation legislation, please refer to Part I, Chapter 5.3. (42) Benis HG (1990): A Product Risk Assessment Nomograph, report prepared for the New Zealand Ministry of Consumer Affairs, dated February 1990. Cited in: European Commission (2005) Establishing a Comparative Inventory of Approaches and Methods Used by Enforcement Authorities for the Assessment of the Safety of Consumer Products Covered by Directive 2001/95/EC on General Product Safety and Identification of Best Practices. Report prepared by Risk & Policy Analysts (RPA), Loddon, Norfolk, UK. (43) Method used by the Belgian authorities. Cited in: European Commission (2005) Establishing a Comparative Inventory of Approaches and Methods Used by Enforcement Authorities for the Assessment of the Safety of Consumer Products Covered by Directive 2001/95/EC on General Product Safety and Identification of Best Practices. Report prepared by Risk & Policy Analysts (RPA), Loddon, Norfolk, UK. (44) Commission Decision 2004/418/EC of 29 April 2004 laying down guidelines for the management of the EU Rapid Information System (RAPEX) and for notifications presented in accordance with Article 11 of Directive 2001/95/EC (OJ L 151, 30.4.2004, p. 83). (45) Directive 2001/95/EC. (46) https://webgate.ec.europa.eu/idbpa/. (47) NB: uncertainty always has to be taken into account when comparing a test result with a limit. See, for example:  the Report on the relationship between analytical results, measurement uncertainty, recovery factors and the provisions of EU food and feed legislation ¦ https://ec.europa.eu/food/safety/chemical_safety/contaminants/catalogue_en  the Summary report on the Preparation of a working document in support of the uniform interpretation of legislative standards and the laboratory quality standards prescribed under Directive 93/99/EEC. http://ec.europa.eu/food/fs/scoop/9.1_sr_en.pdf (48) Regulation (EC) No 1907/2006 of the European Parliament and of the Council of 18 December 2006 concerning the Registration, Evaluation, Authorisation and Restriction of Chemicals (REACH), establishing a European Chemicals Agency, amending Directive 1999/45/EC and repealing Council Regulation (EEC) No 793/93 and Commission Regulation (EC) No 1488/94 as well as Council Directive 76/769/EEC and Commission Directives 91/155/EEC, 93/67/EEC, 93/105/EC and 2000/21/EC (OJ L 396, 30.12.2006, p. 1). (49) OJ L 164, 26.6.2009, p. 7. (50) Standard EN 71-1:2005, section 8.2 +A6:2008. (51) Article 10 of Regulation (EC) No 1223/2009 (OJ L 342, 22.12.2009, p. 59). (52) REACH Regulation and guidance documents on REACH, see http://echa.europa.eu/ European Chemicals Agency (2008). The Guidance on Information Requirements and Chemical Safety Assessment: http://guidance.echa.europa.eu/docs/guidance_document/information_requirements_en.htm (53) Commission Implementing Decision 2013/674/EU of 25 November 2013 on Guidelines on Annex I to Regulation (EC) No 1223/2009 of the European Parliament and of the Council on cosmetic products (OJ L 315, 26.11.2013, p. 82); SCCS (Scientific Committee on Consumer Safety), SCCS Notes of Guidance for the Testing of Cosmetic Ingredients and their Safety Evaluation 9th revision, 29 September 2015, SCCS/1564/15, revision of 25 April 2016: http://ec.europa.eu/health/scientific_committees/consumer_safety/docs/sccs_o_190.pdf (54) See Part I Chapter 1.1, penultimate paragraph. (55) This is taken from the definition of safe product in Article 2(b) of Directive 2001/95/EC.
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32008F0947
        'status': In Force
        'act_type': Decision_FRAMW
        'treaty': TEU (1992)

        full_doc :

        16.12.2008 EN Official Journal of the European Union L 337/102 COUNCIL FRAMEWORK DECISION 2008/947/JHA of 27 November 2008 on the application of the principle of mutual recognition to judgments and probation decisions with a view to the supervision of probation measures and alternative sanctions THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Article 31(1)(a) and (c) and Article 34(2)(b) thereof, Having regard to the initiative of the Federal Republic of Germany and of the French Republic (1), Having regard to the Opinion of the European Parliament (2), Whereas: (1) The European Union has set itself the objective of developing an area of freedom, security and justice. This presupposes that there is an understanding of freedom, security and justice on the part of the Member States which is identical in its essential elements and based on the principles of freedom, democracy, respect for human rights and fundamental freedoms, as well as the rule of law. (2) The aim of police and judicial cooperation in the European Union is to provide a high degree of security for all citizens. One of the cornerstones for this is the principle of mutual recognition of judicial decisions, established in the conclusions of the European Council meeting in Tampere on 15 and 16 October 1999 and reaffirmed in the Hague Programme of 4 and 5 November 2004 for strengthening freedom, security and justice in the European Union (3). In the programme of measures of 29 November 2000 adopted for the purpose of implementing the principle of mutual recognition of decisions in criminal matters, the Council pronounced itself in favour of cooperation in the area of suspended sentences and parole. (3) Council Framework Decision 2008/909/JHA of 27 November 2008 on the application of the principle of mutual recognition to judgments in criminal matters imposing custodial sentences or measures involving deprivation of liberty for the purpose of their enforcement in the European Union (4) concerns the mutual recognition and enforcement of custodial sentences or measures involving deprivation of liberty. Further common rules are required, in particular where a non-custodial sentence involving the supervision of probation measures or alternative sanctions has been imposed in respect of a person who does not have his lawful and ordinary residence in the State of conviction. (4) The Council of Europe Convention of 30 November 1964 on the Supervision of Conditionally Sentenced or Conditionally Released Offenders has been ratified by only 12 Member States, with, in some cases, numerous reservations. The present Framework Decision provides for a more effective instrument because it is based on the principle of mutual recognition and all Member States participate. (5) This Framework Decision respects fundamental rights and adheres to the principles recognised in Article 6 of the Treaty on European Union, which are also expressed in the Charter of Fundamental Rights of the European Union, especially in Chapter VI thereof. No provision of this Framework Decision should be interpreted as prohibiting refusal to recognise a judgment and/or supervise a probation measure or alternative sanction if there are objective reasons to believe that the probation measure or alternative sanction was imposed to punish a person because of his or her sex, race, religion, ethnic origin, nationality, language, political opinions or sexual orientation or that this person might be disadvantaged for one of these reasons. (6) This Framework Decision should not prevent any Member State from applying its constitutional rules relating to entitlement to due process, freedom of association, freedom of the press, freedom of expression in other media and freedom of religion. (7) The provisions of this Framework Decision should be applied in conformity with the right of the Unions citizens to move and reside freely within the territory of the Member States, pursuant to Article 18 of the Treaty establishing the European Community. (8) The aim of mutual recognition and supervision of suspended sentences, conditional sentences, alternative sanctions and decisions on conditional release is to enhance the prospects of the sentenced persons being reintegrated into society, by enabling that person to preserve family, linguistic, cultural and other ties, but also to improve monitoring of compliance with probation measures and alternative sanctions, with a view to preventing recidivism, thus paying due regard to the protection of victims and the general public. (9) There are several types of probation measures and alternative sanctions which are common among the Member States and which all Member States are in principle willing to supervise. The supervision of these types of measures and sanctions should be obligatory, subject to certain exceptions provided for in this Framework Decision. Member States may declare that, in addition, they are willing to supervise other types of probation measures and/or other types of alternative sanctions. (10) The probation measures and alternative sanctions that are, in principle, obligatory to supervise include, inter alia, orders relating to behaviour (such as an obligation to stop the consumption of alcohol), residence (such as an obligation to change residence for reasons of domestic violence), education and training (such as an obligation to follow a safe-driving course), leisure activities (such as an obligation to cease playing or attending a certain sport) and limitations on or modalities of carrying out a professional activity (such as an obligation to seek a professional activity in a different working environment; this obligation does not include the supervision of compliance with any professional disqualifications imposed on the person as part of the sanction). (11) Where appropriate, electronic monitoring could be used with a view to supervising probation measures or alternative sanctions, in accordance with national law and procedures. (12) The Member State where the person concerned is sentenced may forward a judgment and, where applicable, a probation decision to the Member State where the sentenced person is lawfully and ordinarily resident with a view to the recognition thereof and to the supervision of probation measures or alternative sanctions contained therein. (13) The decision on whether to forward the judgment and, where applicable, the probation decision to another Member State should be taken in each individual case by the competent authority of the issuing Member State, taking into account, inter alia, the declarations made in accordance with Articles 5(4), 10(4) and 14(3). (14) The judgment and, where applicable, the probation decision may also be forwarded to a Member State other than that where the sentenced person is residing, if the competent authority of that executing State, taking account of any conditions set out in the relevant declaration made by that State in accordance with this Framework Decision, consents to such forwarding. In particular, consent may be given, with a view to social rehabilitation, where the sentenced person, without losing his/her right of residence, intends to move to another Member State because he/she is granted an employment contract, if he/she is a family member of a lawful and ordinary resident person of that Member State, or if he/she intends to follow a study or training in that Member State, in accordance with Community law. (15) Member States should apply their own national law and procedures for the recognition of a judgment and, where applicable, a probation decision. In the case of a conditional sentence or alternative sanction where the judgment does not contain a custodial sentence or measure involving deprivation of liberty to be enforced in case of non-compliance with the obligations or instructions concerned, this could imply that having made the relevant declaration in accordance with this Framework Decision, Member States, when deciding to recognise, agree to supervise the probation measures or alternative sanctions concerned and to assume no other responsibility than just for taking the subsequent decisions consisting of the modification of obligations or instructions contained in the probation measure or alternative sanction, or modification of the duration of the probation period. Consequently, the recognition has, in such cases, no further effect than to enable the executing State to take those types of subsequent decisions. (16) A Member State may refuse to recognise a judgment and, where applicable, a probation decision, if the judgment concerned was issued against a person who has not been found guilty, such as in the case of a mentally ill person, and the judgment or, where applicable, the probation decision provides for medical/therapeutic treatment which the executing State cannot supervise in respect of such persons under its national law. (17) The ground for refusal relating to territoriality should be applied only in exceptional cases and with a view to cooperating to the greatest extent possible under the provisions of this Framework Decision, while taking into account of the objectives thereof. Any decision to apply this ground for refusal should be based on a case-by-case analysis and on consultations between the competent authorities of the issuing and executing States. (18) If the probation measures or alternative sanctions include community service, then the executing State should be entitled to refuse to recognise the judgment and, where applicable, the probation decision if the community service would normally be completed in less than six months. (19) The form of the certificate is drafted in such a way so that essential elements of the judgment and, where applicable, of the probation decision are comprised in the certificate, which should be translated into the official language or one of the official languages of the executing State. The certificate should assist the competent authorities in the executing State in taking decisions under this Framework Decision, including decisions on recognition and assumption of responsibility for supervision of probation measures and alternative sanctions, decisions on adaptation of probation measures and alternative sanctions, and subsequent decisions in case, notably, of non-compliance with a probation measure or alternative sanction. (20) In view of the principle of mutual recognition, on which this Framework Decision is based, issuing and executing Member States should promote direct contact between their competent authorities in the application of this Framework Decision. (21) All Member States should ensure that sentenced persons, in respect of whom decisions under this Framework Decision are taken, are subject to a set of legal rights and remedies in accordance with their national law, regardless of whether the competent authorities designated to take decisions under this Framework Decision are of a judicial or a non-judicial nature. (22) All subsequent decisions relating to a suspended sentence, a conditional sentence or an alternative sanction which result in the imposition of a custodial sentence or measure involving deprivation of liberty should be taken by a judicial authority. (23) Since all Member States have ratified the Council of Europe Convention of 28 January 1981 for the Protection of Individuals with regard to Automatic Processing of Personal Data, personal data processed when implementing this Framework Decision should be protected in accordance with the principles laid down in that Convention. (24) Since the objectives of this Framework Decision, namely facilitating the social rehabilitation of sentenced persons, improving the protection of victims and of the general public, and facilitating the application of suitable probation measures and alternative sanctions in case of offenders who do not live in the State of conviction, cannot be sufficiently achieved by the Member States themselves in view of the cross-border nature of the situations involved and can therefore, by reason of the scale of the action, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity as defined in Article 5 of the Treaty establishing the European Community as applied by the second paragraph of Article 2 of the Treaty on European Union. In accordance with the principle of proportionality, as set out in Article 5 of the Treaty establishing the European Community, this Framework Decision does not go beyond what is necessary in order to achieve those objectives, HAS ADOPTED THIS FRAMEWORK DECISION: Article 1 Objectives and scope 1. This Framework Decision aims at facilitating the social rehabilitation of sentenced persons, improving the protection of victims and of the general public, and facilitating the application of suitable probation measures and alternative sanctions, in case of offenders who do not live in the State of conviction. With a view to achieving these objectives, this Framework Decision lays down rules according to which a Member State, other than the Member State in which the person concerned has been sentenced, recognises judgments and, where applicable, probation decisions and supervises probation measures imposed on the basis of a judgment, or alternative sanctions contained in such a judgment, and takes all other decisions relating to that judgment, unless otherwise provided for in this Framework Decision. 2. This Framework Decision shall apply only to: (a) the recognition of judgments and, where applicable, probation decisions; (b) the transfer of responsibility for the supervision of probation measures and alternative sanctions; (c) all other decisions related to those under (a) and (b); as described and provided for in this Framework Decision. 3. This Framework Decision shall not apply to: (a) the execution of judgments in criminal matters imposing custodial sentences or measures involving deprivation of liberty which fall within the scope of Framework Decision 2008/909/JHA; (b) recognition and execution of financial penalties and confiscation orders which fall within the scope of Council Framework Decision 2005/214/JHA of 24 February 2005 on the application of the principle of mutual recognition to financial penalties (5) and Council Framework Decision 2006/783/JHA of 6 October 2006 on the application of the principle of mutual recognition to confiscation orders (6). 4. This Framework Decision shall not have the effect of modifying the obligation to respect fundamental rights and fundamental legal principles as enshrined in Article 6 of the Treaty on European Union. Article 2 Definitions For the purposes of this Framework Decision: 1. judgment shall mean a final decision or order of a court of the issuing State, establishing that a natural person has committed a criminal offence and imposing: (a) a custodial sentence or measure involving deprivation of liberty, if a conditional release has been granted on the basis of that judgment or by a subsequent probation decision; (b) a suspended sentence; (c) a conditional sentence; (d) an alternative sanction; 2. suspended sentence shall mean a custodial sentence or measure involving deprivation of liberty, the execution of which is conditionally suspended, wholly or in part, when the sentence is passed by imposing one or more probation measures. Such probation measures may be included in the judgment itself or determined in a separate probation decision taken by a competent authority; 3. conditional sentence shall mean a judgment in which the imposition of a sentence has been conditionally deferred by imposing one or more probation measures or in which one or more probation measures are imposed instead of a custodial sentence or measure involving deprivation of liberty. Such probation measures may be included in the judgment itself or determined in a separate probation decision taken by a competent authority; 4. alternative sanction shall mean a sanction, other than a custodial sentence, a measure involving deprivation of liberty or a financial penalty, imposing an obligation or instruction; 5. probation decision shall mean a judgment or a final decision of a competent authority of the issuing State taken on the basis of such judgment: (a) granting a conditional release; or (b) imposing probation measures; 6. conditional release shall mean a final decision of a competent authority or stemming from the national law on the early release of a sentenced person after part of the custodial sentence or measure involving deprivation of liberty has been served by imposing one or more probation measures; 7. probation measures shall mean obligations and instructions imposed by a competent authority on a natural person, in accordance with the national law of the issuing State, in connection with a suspended sentence, a conditional sentence or a conditional release; 8. issuing State shall mean the Member State in which a judgment is delivered; 9. executing State shall mean the Member State in which the probation measures and alternative sanctions are supervised following a decision in accordance with Article 8. Article 3 Designation of competent authorities 1. Each Member State shall inform the General Secretariat of the Council which authority or authorities, under its national law, are competent to act according to this Framework Decision in the situation where that Member State is the issuing State or the executing State. 2. Member States may designate non-judicial authorities as the competent authorities for taking decisions under this Framework Decision, provided that such authorities have competence for taking decisions of a similar nature under their national law and procedures. 3. If a decision under Article 14(1)(b) or (c) is taken by a competent authority other than a court, the Member States shall ensure that, upon request of the person concerned, such decision may be reviewed by a court or by another independent court-like body. 4. The General Secretariat of the Council shall make the information received available to all Member States and to the Commission. Article 4 Types of probation measures and alternative sanctions 1. This Framework Decision shall apply to the following probation measures or alternative sanctions: (a) an obligation for the sentenced person to inform a specific authority of any change of residence or working place; (b) an obligation not to enter certain localities, places or defined areas in the issuing or executing State; (c) an obligation containing limitations on leaving the territory of the executing State; (d) instructions relating to behaviour, residence, education and training, leisure activities, or containing limitations on or modalities of carrying out a professional activity; (e) an obligation to report at specified times to a specific authority; (f) an obligation to avoid contact with specific persons; (g) an obligation to avoid contact with specific objects, which have been used or are likely to be used by the sentenced person with a view to committing a criminal offence; (h) an obligation to compensate financially for the prejudice caused by the offence and/or an obligation to provide proof of compliance with such an obligation; (i) an obligation to carry out community service; (j) an obligation to cooperate with a probation officer or with a representative of a social service having responsibilities in respect of sentenced persons; (k) an obligation to undergo therapeutic treatment or treatment for addiction. 2. Each Member State shall notify the General Secretariat of the Council, when implementing this Framework Decision, which probation measures and alternative sanctions, apart from those referred to in paragraph 1, it is prepared to supervise. The General Secretariat of the Council shall make the information received available to all Member States and to the Commission. Article 5 Criteria for forwarding a judgment and, where applicable, a probation decision 1. The competent authority of the issuing State may forward a judgment and, where applicable, a probation decision to the competent authority of the Member State in which the sentenced person is lawfully and ordinarily residing, in cases where the sentenced person has returned or wants to return to that State. 2. The competent authority of the issuing State may, upon request of the sentenced person, forward the judgment and, where applicable, the probation decision to a competent authority of a Member State other than the Member State in which the sentenced person is lawfully and ordinarily residing, on condition that this latter authority has consented to such forwarding. 3. When implementing this Framework Decision, Member States shall determine under which conditions their competent authorities may consent to the forwarding of a judgment and, where applicable, a probation decision under paragraph 2. 4. Each Member State shall make a declaration to the General Secretariat of the Council of the determination made under paragraph 3. Member States may modify such a declaration at any time. The General Secretariat shall make the information received available to all Member States and to the Commission. Article 6 Procedure for forwarding a judgment and, where applicable, a probation decision 1. When, in application of Article 5(1) or (2), the competent authority of the issuing State forwards a judgment and, where applicable, a probation decision to another Member State, it shall ensure that it is accompanied by a certificate, the standard form for which is set out in Annex I. 2. The judgment and, where applicable, the probation decision, together with the certificate referred to in paragraph 1, shall be forwarded by the competent authority of the issuing State directly to the competent authority of the executing State by any means which leaves a written record under conditions allowing the executing State to establish their authenticity. The original of the judgment and, where applicable, the probation decision, or certified copies thereof, as well as the original of the certificate, shall be sent to the competent authority of the executing State if it so requires. All official communications shall also be made directly between the said competent authorities. 3. The certificate referred to in paragraph 1 shall be signed and its content certified as accurate by the competent authority of the issuing State. 4. Apart from the measures and sanctions referred to in Article 4(1), the certificate referred to in paragraph 1 of this Article shall include only such measures or sanctions as notified by the executing State in accordance with Article 4(2). 5. The competent authority of the issuing State shall forward the judgment and, where applicable, the probation decision, together with the certificate referred to in paragraph 1 only to one executing State at any one time. 6. If the competent authority of the executing State is not known to the competent authority of the issuing State, the latter shall make all necessary inquiries, including via the contact points of the European Judicial Network created by Council Joint Action 98/428/JHA (7), in order to obtain the information from the executing State. 7. When an authority of the executing State which receives a judgment and, where applicable, a probation decision, together with the certificate referred to in paragraph 1, has no competence to recognise it and take the ensuing necessary measures for the supervision of the probation measure or alternative sanction, it shall, ex officio, forward it to the competent authority and shall without delay inform the competent authority of the issuing State accordingly by any means which leaves a written record. Article 7 Consequences for the issuing State 1. Once the competent authority of the executing State has recognised the judgment and, where applicable, the probation decision forwarded to it and has informed the competent authority of the issuing State of such recognition, the issuing State shall no longer have competence in relation to the supervision of the probation measures or alternative sanctions imposed, nor to take subsequent measures referred to in Article 14(1). 2. The competence referred to in paragraph 1 shall revert to the issuing State: (a) as soon as its competent authority has notified withdrawal of the certificate referred to in Article 6(1), pursuant to Article 9(4), to the competent authority of the executing State; (b) in cases referred to in Article 14(3) in combination with 14(5); and (c) in cases referred to in Article 20. Article 8 Decision of the executing State 1. The competent authority of the executing State shall recognise the judgment and, where applicable, the probation decision forwarded in accordance with Article 5 and following the procedure laid down in Article 6 and shall without delay take all necessary measures for the supervision of the probation measures or alternative sanctions, unless it decides to invoke one of the grounds for refusing recognition and supervision referred to in Article 11. 2. The competent authority of the executing State may postpone the decision on recognition of the judgment and, where applicable, the probation decision in the situation where the certificate referred to in Article 6(1) is incomplete or obviously does not correspond to the judgment or, where applicable, the probation decision, until such reasonable deadline set for the certificate to be completed or corrected. Article 9 Adaptation of the probation measures or alternative sanctions 1. If the nature or duration of the relevant probation measure or alternative sanction, or the duration of the probation period, are incompatible with the law of the executing State, the competent authority of that State may adapt them in line with the nature and duration of the probation measures and alternative sanctions, or duration of the probation period, which apply, under the law of the executing State, to equivalent offences. The adapted probation measure, alternative sanction or duration of the probation period shall correspond as far as possible to that imposed in the issuing State. 2. Where the probation measure, the alternative sanction or the probation period has been adapted because its duration exceeds the maximum duration provided for under the law of the executing State, the duration of the adapted probation measure, alternative sanction or probation period shall not be below the maximum duration provided for equivalent offences under the law of the executing State. 3. The adapted probation measure, alternative sanction or probation period shall not be more severe or longer than the probation measure, alternative sanction or probation period which was originally imposed. 4. Following receipt of the information referred to in Articles 16(2) or 18(5), the competent authority of the issuing State may decide to withdraw the certificate referred to in Article 6(1) provided that supervision in the executing State has not yet begun. In such cases, the decision shall be taken and communicated as soon as possible and within ten days of the receipt of the information. Article 10 Double criminality 1. The following offences, if they are punishable in the issuing State by a custodial sentence or a measure involving deprivation of liberty for a maximum period of at least three years, and as they are defined by the law of the issuing State, shall, under the terms of this Framework Decision and without verification of the double criminality of the act, give rise to recognition of the judgment and, where applicable, the probation decision and to supervision of probation measures and alternative sanctions:  participation in a criminal organisation,  terrorism,  trafficking in human beings,  sexual exploitation of children and child pornography,  illicit trafficking in narcotic drugs and psychotropic substances,  illicit trafficking in weapons, munitions and explosives,  corruption,  fraud, including that affecting the financial interests of the European Communities within the meaning of the Convention of 26 July 1995 on the protection of the European Communities financial interests (8),  laundering of the proceeds of crime,  counterfeiting currency, including of the euro,  computer-related crime,  environmental crime, including illicit trafficking in endangered animal species and in endangered plant species and varieties,  facilitation of unauthorised entry and residence,  murder, grievous bodily injury,  illicit trade in human organs and tissue,  kidnapping, illegal restraint and hostage-taking,  racism and xenophobia,  organised or armed robbery,  illicit trafficking in cultural goods, including antiques and works of art,  swindling,  racketeering and extortion,  counterfeiting and piracy of products,  forgery of administrative documents and trafficking therein,  forgery of means of payment,  illicit trafficking in hormonal substances and other growth promoters,  illicit trafficking in nuclear or radioactive materials,  trafficking in stolen vehicles,  rape,  arson,  crimes within the jurisdiction of the International Criminal Court,  unlawful seizure of aircraft/ships,  sabotage. 2. The Council may decide to add other categories of offences to the list provided for in paragraph 1 of this Article at any time, acting unanimously after consultation of the European Parliament under the conditions laid down in Article 39(1) of the Treaty on European Union. The Council shall examine, in the light of the report submitted to it pursuant to Article 26(1) of this Framework Decision, whether the list should be extended or amended. 3. For offences other than those covered by paragraph 1, the executing State may make the recognition of the judgment and, where applicable, the probation decision and the supervision of probation measures and of alternative sanctions subject to the condition that the judgment relates to acts which also constitute an offence under the law of the executing State, whatever its constituent elements or however it is described. 4. Each Member State may, on the adoption of this Framework Decision or later, by a declaration notified to the General Secretariat of the Council, declare that it will not apply paragraph 1. Any such declaration may be withdrawn at any time. Such declarations or withdrawals of declarations shall be published in the Official Journal of the European Union. Article 11 Grounds for refusing recognition and supervision 1. The competent authority of the executing State may refuse to recognise the judgment or, where applicable, the probation decision and to assume responsibility for supervising probation measures or alternative sanctions if: (a) the certificate referred to in Article 6(1) is incomplete or manifestly does not correspond to the judgment or to the probation decision and has not been completed or corrected within a reasonable period set by the competent authority of the executing State; (b) the criteria set forth in Articles 5(1), 5(2) or 6(4) are not met; (c) recognition of the judgment and assumption of responsibility for supervising probation measures or alternative sanctions would be contrary to the principle of ne bis in idem; (d) in a case referred to in Article 10(3) and, where the executing State has made a declaration under Article 10(4), in a case referred to in Article 10(1), the judgment relates to acts which would not constitute an offence under the law of the executing State. However, in relation to taxes or duties, customs and exchange, execution of the judgment or, where applicable, the probation decision may not be refused on the grounds that the law of the executing State does not impose the same kind of tax or duty or does not contain the same type of rules as regards taxes or duties, customs and exchange regulations as the law of the issuing State; (e) the enforcement of the sentence is statute-barred according to the law of the executing State and relates to an act which falls within its competence according to that law; (f) there is immunity under the law of the executing State, which makes it impossible to supervise probation measures or alternative sanctions; (g) under the law of the executing State, the sentenced person cannot, owing to his or her age, be held criminally liable for the acts in respect of which the judgment was issued; (h) the judgment was rendered in absentia, unless the certificate states that the person was summoned personally or informed via a representative competent according to the national law of the issuing State of the time and place of the proceedings which resulted in the judgment being rendered in absentia, or that the person has indicated to a competent authority that he or she does not contest the case; (i) the judgment or, where applicable, the probation decision provides for medical/therapeutic treatment which, notwithstanding Article 9, the executing State is unable to supervise in view of its legal or health-care system; (j) the probation measure or alternative sanction is of less than six months duration; or (k) the judgment relates to criminal offences which under the law of the executing State are regarded as having been committed wholly or for a major or essential part within its territory, or in a place equivalent to its territory. 2. Any decision under paragraph 1(k) in relation to offences committed partly within the territory of the executing State, or in a place equivalent to its territory, shall be taken by the competent authority of the executing State only in exceptional circumstances and on a case-by case basis, having regard to the specific circumstances of the case, and in particular to whether a major or essential part of the conduct in question has taken place in the issuing State. 3. In the cases referred to in paragraph 1(a), (b), (c), (h), (i), (j) and (k), before deciding not to recognise the judgment or, where applicable, the probation decision and to assume responsibility for supervising probation measures and alternative sanctions, the competent authority of the executing State shall communicate, by appropriate means, with the competent authority of the issuing State and shall, as necessary, ask it to supply all additional information required without delay. 4. Where the competent authority of the executing State has decided to invoke a ground for refusal referred to in paragraph 1 of this Article, in particular the grounds referred to under paragraph 1(d) or (k), it may nevertheless, in agreement with the competent authority of the issuing State, decide to supervise the probation measures or alternative sanctions that are imposed in the judgment and, where applicable, the probation decision forwarded to it, without assuming the responsibility for taking any of the decisions referred to in Article 14(1)(a), (b) and (c). Article 12 Time limit 1. The competent authority of the executing State shall decide as soon as possible, and within 60 days of receipt of the judgment and, where applicable, the probation decision, together with the certificate referred to in Article 6(1), whether or not to recognise the judgment and, where applicable, the probation decision and assume responsibility for supervising the probation measures or alternative sanctions. It shall immediately inform the competent authority of the issuing State, by any means which leaves a written record, of its decision. 2. When in exceptional circumstances it is not possible for the competent authority of the executing State to comply with the time limit provided for in paragraph 1, it shall immediately inform the competent authority of the issuing State by any means, giving the reasons for the delay and indicating the estimated time needed for the final decision to be taken. Article 13 Governing law 1. The supervision and application of probation measures and alternative sanctions shall be governed by the law of the executing State. 2. The competent authority of the executing State may supervise an obligation as referred to in Article 4(1)(h) by requiring the sentenced person to provide proof of compliance with an obligation to compensate for the prejudice caused by the offence. Article 14 Jurisdiction to take all subsequent decisions and governing law 1. The competent authority of the executing State shall have jurisdiction to take all subsequent decisions relating to a suspended sentence, conditional release, conditional sentence and alternative sanction, in particular in case of non-compliance with a probation measure or alternative sanction or if the sentenced person commits a new criminal offence. Such subsequent decisions include notably: (a) the modification of obligations or instructions contained in the probation measure or alternative sanction, or the modification of the duration of the probation period; (b) the revocation of the suspension of the execution of the judgment or the revocation of the decision on conditional release; and (c) the imposition of a custodial sentence or measure involving deprivation of liberty in case of an alternative sanction or conditional sentence. 2. The law of the executing State shall apply to decisions taken pursuant to paragraph 1 and to all subsequent consequences of the judgment including, where applicable, the enforcement and, if necessary, the adaptation of the custodial sentence or measure involving deprivation of liberty. 3. Each Member State may, at the time of adoption of this Framework Decision or at a later stage, declare that as an executing State it will refuse to assume the responsibility provided for in paragraph 1(b) and (c) in cases or categories of cases to be specified by that Member State, in particular: (a) in cases relating to an alternative sanction, where the judgment does not contain a custodial sentence or measure involving deprivation of liberty to be enforced in case of non-compliance with the obligations or instructions concerned; (b) in cases relating to a conditional sentence; (c) in cases where the judgment relates to acts which do not constitute an offence under the law of the executing State, whatever its constituent elements or however it is described. 4. When a Member State makes use of any of the possibilities referred to in paragraph 3, the competent authority of the executing State shall transfer jurisdiction back to the competent authority of the issuing State in case of non-compliance with a probation measure or alternative sanction if the competent authority of the executing State is of the view that a subsequent decision as referred to in paragraph 1(b) or (c) needs to be taken. 5. In the cases referred to in paragraph 3 of this Article, the obligation to recognise the judgment and, where applicable, the probation decision, as well as the obligation to take without delay all necessary measures for the supervision of the probation measures or alternative sanctions, as referred to in Article 8(1), shall not be affected. 6. Declarations as mentioned in paragraph 3 shall be made by notification to the General Secretariat of the Council. Any such declaration may be withdrawn at any time. The declarations and withdrawals mentioned in this Article shall be published in the Official Journal of the European Union. Article 15 Consultations between competent authorities Where and whenever it is felt appropriate, competent authorities of the issuing State and of the executing State may consult each other with a view to facilitating the smooth and efficient application of this Framework Decision. Article 16 Obligations of the authorities involved where the executing State has jurisdiction for subsequent decisions 1. The competent authority of the executing State shall without delay inform the competent authority of the issuing State, by any means which leaves a written record, of all decisions on the: (a) modification of the probation measure or alternative sanction; (b) revocation of the suspension of the execution of the judgment or revocation of the decision on conditional release; (c) enforcement of a custodial sentence or measure involving deprivation of liberty, because of non-compliance with a probation measure or alternative sanction; (d) lapsing of the probation measure or alternative sanction. 2. If so requested by the competent authority of the issuing State, the competent authority of the executing State shall inform it of the maximum duration of deprivation of liberty that is foreseen in the national law of the executing State for the offence which gave rise to the judgment and that could be imposed on the sentenced person in case of breach of the probation measure or alternative sanction. This information shall be provided immediately after reception of the judgment and, where applicable, the probation decision, together with the certificate referred to in Article 6(1). 3. The competent authority of the issuing State shall immediately inform the competent authority of the executing State, by any means which leaves a written record, of any circumstances or findings which, in its opinion, could entail one or more of the decisions referred to in paragraph 1(a), (b) or (c) being taken. Article 17 Obligations of the authorities involved where the issuing State has jurisdiction for subsequent decisions 1. If the competent authority of the issuing State has jurisdiction for the subsequent decisions mentioned in Article 14(1) pursuant to the application of Article 14(3), the competent authority of the executing State shall immediately notify it of: (a) any finding which is likely to result in revocation of the suspension of the execution of the judgment or revocation of the decision on conditional release; (b) any finding which is likely to result in the imposition of a custodial sentence or measure involving deprivation of liberty; (c) all further facts and circumstances which the competent authority of the issuing State requests to be provided and which are essential in order to allow it to take subsequent decisions in accordance with its national law. 2. When a Member State has made use of the possibility referred to in Article 11(4), the competent authority of that State shall inform the competent authority of the issuing State in case of non-compliance by the sentenced person with a probation measure or alternative sanction. 3. Notice of the findings mentioned in paragraph 1(a) and (b) and in paragraph 2 shall be given using the standard form set out in Annex II. Notice of the facts and circumstances mentioned in paragraph 1(c) shall be given by any means which leaves a written record, including, where possible, through the form set out in Annex II. 4. If, under the national law of the issuing State, the sentenced person must be given a judicial hearing before a decision is taken on the imposition of a sentence, this requirement may be met by following mutatis mutandis the procedure contained in instruments of international or European Union law that provide the possibility of using video links for hearing persons. 5. The competent authority of the issuing State shall without delay inform the competent authority of the executing State of all decisions on: (a) the revocation of the suspension of the execution of the judgment or the revocation of the decision on conditional release; (b) the enforcement of the custodial sentence or measure involving deprivation of liberty, where such measure is contained in the judgment; (c) the imposition of a custodial sentence or measure involving deprivation of liberty, where such measure is not contained in the judgment; (d) the lapsing of the probation measure or alternative sanction. Article 18 Information from the executing State in all cases The competent authority of the executing State shall without delay inform the competent authority of the issuing State, by any means which leaves a written record of: 1. the transmission of the judgment and, where applicable, the probation decision, together with the certificate referred to in Article 6(1) to the competent authority responsible for its recognition and for taking the ensuing measures for the supervision of the probation measures or alternative sanctions in accordance with Article 6(7); 2. the fact that it is in practice impossible to supervise the probation measures or alternative sanctions for the reason that, after transmission of the judgment and, where applicable, the probation decision, together with the certificate referred to in Article 6(1) to the executing State, the sentenced person cannot be found in the territory of the executing State, in which case the executing State shall be under no obligation to supervise the probation measures or alternative sanctions; 3. the final decision to recognise the judgment and, where applicable, the probation decision and to assume responsibility for supervising the probation measures or alternative sanctions; 4. any decision not to recognise the judgment and, where applicable, the probation decision and to assume responsibility for supervising the probation measures or alternative sanctions in accordance with Article 11, together with the reasons for the decision; 5. any decision to adapt the probation measures or alternative sanctions in accordance with Article 9, together with the reasons for the decision; 6. any decision on amnesty or pardon which leads to not supervising the probation measures or alternative sanctions for the reasons referred to in Article 19(1), together, where applicable, with the reasons for the decision. Article 19 Amnesty, pardon, review of judgment 1. An amnesty or pardon may be granted by the issuing State and also by the executing State. 2. Only the issuing State may decide on applications for review of the judgment which forms the basis for the probation measures or alternative sanctions to be supervised under this Framework Decision. Article 20 End of jurisdiction of the executing State 1. If the sentenced person absconds or no longer has a lawful and ordinary residence in the executing State, the competent authority of the executing State may transfer the jurisdiction in respect of the supervision of the probation measures or alternative sanctions and in respect of all further decisions relating to the judgment back to the competent authority of the issuing State. 2. If new criminal proceedings against the person concerned are taking place in the issuing State, the competent authority of the issuing State may request the competent authority of the executing State to transfer jurisdiction in respect of the supervision of the probation measures or alternative sanctions and in respect of all further decisions relating to the judgment back to the competent authority of the issuing State. In such a case, the competent authority of the executing State may transfer jurisdiction back to the competent authority of the issuing State. 3. When, in application of this Article, jurisdiction is transferred back to the issuing State, the competent authority of that State shall resume jurisdiction. For the further supervision of the probation measures or alternative sanctions, the competent authority of the issuing State shall take account of the duration and degree of compliance with the probation measures or alternative sanctions in the executing State, as well as of any decisions taken by the executing State in accordance with Article 16(1). Article 21 Languages The certificate referred to in Article 6(1) shall be translated into the official language or one of the official languages of the executing State. Any Member State may, on adoption of this Framework Decision or later, state in a declaration deposited with the General Secretariat of the Council that it will accept a translation in one or more other official languages of the institutions of the European Union. Article 22 Costs Costs resulting from the application of this Framework Decision shall be borne by the executing State, except for costs arising exclusively within the territory of the issuing State. Article 23 Relationship with other agreements and arrangements 1. This Framework Decision shall, in relations between the Member States, from 6 December 2011, replace the corresponding provisions of the Council of Europe Convention of 30 November 1964 on the Supervision of Conditionally Sentenced or Conditionally Released Offenders. 2. Member States may continue to apply bilateral or multilateral agreements or arrangements in force after 6 December 2008, in so far as they allow the objectives of this Framework Decision to be extended or enlarged and help to simplify or facilitate further the procedures for the supervision of probation measures and alternative sanctions. 3. Member States may conclude bilateral or multilateral agreements or arrangements after 6 December 2008, in so far as such agreements or arrangements allow the provisions of this Framework Decision to be extended or enlarged and help to simplify or facilitate further the procedures for the supervision of probation measures and alternative sanctions. 4. Member States shall, by 6 March 2009, notify the Council and the Commission of the existing agreements and arrangements referred to in paragraph 2 which they wish to continue applying. Member States shall also notify the Council and the Commission of any new agreement or arrangement as referred to in paragraph 3, within three months of signing it. Article 24 Territorial application This Framework Decision shall apply to Gibraltar. Article 25 Implementation 1. Member States shall take the necessary measures to comply with the provisions of this Framework Decision by 6 December 2011. 2. Member States shall transmit to the General Secretariat of the Council and to the Commission the text of the provisions transposing into their national law the obligations imposed on them under this Framework Decision. Article 26 Review 1. By 6 December 2014, the Commission shall draw up a report on the basis of the information received from Member States under Article 25(2). 2. On the basis of this report, the Council shall assess: (a) the extent to which the Member States have taken the necessary measures in order to comply with this Framework Decision; and (b) the application of this Framework Decision. 3. The report shall be accompanied, if necessary, by legislative proposals. Article 27 Entry into force This Framework Decision shall enter into force on the day of its publication in the Official Journal of the European Union. Done at Brussels, 27 November 2008. For the Council The President M. ALLIOT-MARIE (1) OJ C 147, 30.6.2007, p. 1. (2) Opinion of 25 October 2007 (not yet published in the Official Journal). (3) OJ C 53, 3.3.2005, p. 1. (4) OJ L 327, 5.12.2008, p. 27. (5) OJ L 76, 22.3.2005, p. 16. (6) OJ L 328, 24.11.2006, p. 59. (7) OJ L 191, 7.7.1998, p. 4. (8) OJ C 316, 27.11.1995, p. 49. ANNEX I ANNEX II
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32012L0029
        'status': In Force
        'act_type': Directive
        'treaty': TFEU (2008)

        full_doc :

        14.11.2012 EN Official Journal of the European Union L 315/57 DIRECTIVE 2012/29/EU OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 25 October 2012 establishing minimum standards on the rights, support and protection of victims of crime, and replacing Council Framework Decision 2001/220/JHA THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 82(2) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the Committee of the Regions (2), Acting in accordance with the ordinary legislative procedure (3), Whereas: (1) The Union has set itself the objective of maintaining and developing an area of freedom, security and justice, the cornerstone of which is the mutual recognition of judicial decisions in civil and criminal matters. (2) The Union is committed to the protection of, and to the establishment of minimum standards in regard to, victims of crime and the Council has adopted Framework Decision 2001/220/JHA of 15 March 2001 on the standing of victims in criminal proceedings (4). Under the Stockholm Programme  An open and secure Europe serving and protecting citizens (5), adopted by the European Council at its meeting on 10 and 11 December 2009, the Commission and the Member States were asked to examine how to improve legislation and practical support measures for the protection of victims, with particular attention paid to, support for and recognition of, all victims, including for victims of terrorism, as a priority. (3) Article 82(2) of the Treaty on the Functioning of the European Union (TFEU) provides for the establishment of minimum rules applicable in the Member States to facilitate mutual recognition of judgments and judicial decisions and police and judicial cooperation in criminal matters having a cross-border dimension, in particular with regard to the rights of victims of crime. (4) In its resolution of 10 June 2011 on a roadmap for strengthening the rights and protection of victims, in particular in criminal proceedings (6) (the Budapest roadmap), the Council stated that action should be taken at Union level in order to strengthen the rights of, support for, and protection of victims of crime. To that end and in accordance with that resolution, this Directive aims to revise and supplement the principles set out in Framework Decision 2001/220/JHA and to take significant steps forward in the level of protection of victims throughout the Union, in particular within the framework of criminal proceedings. (5) The resolution of the European Parliament of 26 November 2009 on the elimination of violence against women (7) called on the Member States to improve their national laws and policies to combat all forms of violence against women and to act in order to tackle the causes of violence against women, not least by employing preventive measures, and called on the Union to guarantee the right to assistance and support for all victims of violence. (6) In its resolution of 5 April 2011 on priorities and outline of a new EU policy framework to fight violence against women (8) the European Parliament proposed a strategy to combat violence against women, domestic violence and female genital mutilation as a basis for future legislative criminal-law instruments against gender-based violence including a framework to fight violence against women (policy, prevention, protection, prosecution, provision and partnership) to be followed up by a Union action plan. International regulation within this area includes the United Nations Convention on the Elimination of All Forms of Discrimination against Women (CEDAW) adopted on 18 December 1979, the CEDAW Committee's recommendations and decisions, and the Council of Europe Convention on preventing and combating violence against women and domestic violence adopted on 7 April 2011. (7) Directive 2011/99/EU of the European Parliament and of the Council of 13 December 2011 on the European protection order (9) establishes a mechanism for the mutual recognition of protection measures in criminal matters between Member States. Directive 2011/36/EU of the European Parliament and of the Council of 5 April 2011 on preventing and combating trafficking in human beings and protecting its victims (10) and Directive 2011/93/EU of the European Parliament and of the Council of 13 December 2011 on combating the sexual abuse and sexual exploitation of children and child pornography (11) address, inter alia, the specific needs of the particular categories of victims of human trafficking, child sexual abuse, sexual exploitation and child pornography. (8) Council Framework Decision 2002/475/JHA of 13 June 2002 on combating terrorism (12) recognises that terrorism constitutes one of the most serious violations of the principles on which the Union is based, including the principle of democracy, and confirms that it constitutes, inter alia, a threat to the free exercise of human rights. (9) Crime is a wrong against society as well as a violation of the individual rights of victims. As such, victims of crime should be recognised and treated in a respectful, sensitive and professional manner without discrimination of any kind based on any ground such as race, colour, ethnic or social origin, genetic features, language, religion or belief, political or any other opinion, membership of a national minority, property, birth, disability, age, gender, gender expression, gender identity, sexual orientation, residence status or health. In all contacts with a competent authority operating within the context of criminal proceedings, and any service coming into contact with victims, such as victim support or restorative justice services, the personal situation and immediate needs, age, gender, possible disability and maturity of victims of crime should be taken into account while fully respecting their physical, mental and moral integrity. Victims of crime should be protected from secondary and repeat victimisation, from intimidation and from retaliation, should receive appropriate support to facilitate their recovery and should be provided with sufficient access to justice. (10) This Directive does not address the conditions of the residence of victims of crime in the territory of the Member States. Member States should take the necessary measures to ensure that the rights set out in this Directive are not made conditional on the victim's residence status in their territory or on the victim's citizenship or nationality. Reporting a crime and participating in criminal proceedings do not create any rights regarding the residence status of the victim. (11) This Directive lays down minimum rules. Member States may extend the rights set out in this Directive in order to provide a higher level of protection. (12) The rights set out in this Directive are without prejudice to the rights of the offender. The term offender refers to a person who has been convicted of a crime. However, for the purposes of this Directive, it also refers to a suspected or accused person before any acknowledgement of guilt or conviction, and it is without prejudice to the presumption of innocence. (13) This Directive applies in relation to criminal offences committed in the Union and to criminal proceedings that take place in the Union. It confers rights on victims of extra-territorial offences only in relation to criminal proceedings that take place in the Union. Complaints made to competent authorities outside the Union, such as embassies, do not trigger the obligations set out in this Directive. (14) In applying this Directive, children's best interests must be a primary consideration, in accordance with the Charter of Fundamental Rights of the European Union and the United Nations Convention on the Rights of the Child adopted on 20 November 1989. Child victims should be considered and treated as the full bearers of rights set out in this Directive and should be entitled to exercise those rights in a manner that takes into account their capacity to form their own views. (15) In applying this Directive, Member States should ensure that victims with disabilities are able to benefit fully from the rights set out in this Directive, on an equal basis with others, including by facilitating the accessibility to premises where criminal proceedings are conducted and access to information. (16) Victims of terrorism have suffered attacks that are intended ultimately to harm society. They may therefore need special attention, support and protection due to the particular nature of the crime that has been committed against them. Victims of terrorism can be under significant public scrutiny and often need social recognition and respectful treatment by society. Member States should therefore take particular account of the needs of victims of terrorism, and should seek to protect their dignity and security. (17) Violence that is directed against a person because of that person's gender, gender identity or gender expression or that affects persons of a particular gender disproportionately, is understood as gender-based violence. It may result in physical, sexual, emotional or psychological harm, or economic loss, to the victim. Gender-based violence is understood to be a form of discrimination and a violation of the fundamental freedoms of the victim and includes violence in close relationships, sexual violence (including rape, sexual assault and harassment), trafficking in human beings, slavery, and different forms of harmful practices, such as forced marriages, female genital mutilation and so-called honour crimes. Women victims of gender-based violence and their children often require special support and protection because of the high risk of secondary and repeat victimisation, of intimidation and of retaliation connected with such violence. (18) Where violence is committed in a close relationship, it is committed by a person who is a current or former spouse, or partner or other family member of the victim, whether or not the offender shares or has shared the same household with the victim. Such violence could cover physical, sexual, psychological or economic violence and could result in physical, mental or emotional harm or economic loss. Violence in close relationships is a serious and often hidden social problem which could cause systematic psychological and physical trauma with severe consequences because the offender is a person whom the victim should be able to trust. Victims of violence in close relationships may therefore be in need of special protection measures. Women are affected disproportionately by this type of violence and the situation can be worse if the woman is dependent on the offender economically, socially or as regards her right to residence. (19) A person should be considered to be a victim regardless of whether an offender is identified, apprehended, prosecuted or convicted and regardless of the familial relationship between them. It is possible that family members of victims are also harmed as a result of the crime. In particular, family members of a person whose death has been directly caused by a criminal offence could be harmed as a result of the crime. Such family members, who are indirect victims of the crime, should therefore also benefit from protection under this Directive. However, Member States should be able to establish procedures to limit the number of family members who can benefit from the rights set out in this Directive. In the case of a child, the child or, unless this is not in the best interests of the child, the holder of parental responsibilty on behalf of the child, should be entitled to exercise the rights set out in this Directive. This Directive is without prejudice to any national administrative procedures required to establish that a person is a victim. (20) The role of victims in the criminal justice system and whether they can participate actively in criminal proceedings vary across Member States, depending on the national system, and is determined by one or more of the following criteria: whether the national system provides for a legal status as a party to criminal proceedings; whether the victim is under a legal requirement or is requested to participate actively in criminal proceedings, for example as a witness; and/or whether the victim has a legal entitlement under national law to participate actively in criminal proceedings and is seeking to do so, where the national system does not provide that victims have the legal status of a party to the criminal proceedings. Member States should determine which of those criteria apply to determine the scope of rights set out in this Directive where there are references to the role of the victim in the relevant criminal justice system. (21) Information and advice provided by competent authorities, victim support services and restorative justice services should, as far as possible, be given by means of a range of media and in a manner which can be understood by the victim. Such information and advice should be provided in simple and accessible language. It should also be ensured that the victim can be understood during proceedings. In this respect, the victim's knowledge of the language used to provide information, age, maturity, intellectual and emotional capacity, literacy and any mental or physical impairment should be taken into account. Particular account should be taken of difficulties in understanding or communicating which may be due to a disability of some kind, such as hearing or speech impediments. Equally, limitations on a victim's ability to communicate information should be taken into account during criminal proceedings. (22) The moment when a complaint is made should, for the purposes of this Directive, be considered as falling within the context of the criminal proceedings. This should also include situations where authorities initiate criminal proceedings ex officio as a result of a criminal offence suffered by a victim. (23) Information about reimbursement of expenses should be provided, from the time of the first contact with a competent authority, for example in a leaflet stating the basic conditions for such reimbursement of expenses. Member States should not be required, at this early stage of the criminal proceedings, to decide on whether the victim concerned fulfils the conditions for reimbursement of expenses. (24) When reporting a crime, victims should receive a written acknowledgement of their complaint from the police, stating the basic elements of the crime, such as the type of crime, the time and place, and any damage or harm caused by the crime. This acknowledgement should include a file number and the time and place for reporting of the crime in order to serve as evidence that the crime has been reported, for example in relation to insurance claims. (25) Without prejudice to rules relating to limitation periods, the delayed reporting of a criminal offence due to fear of retaliation, humiliation or stigmatisation should not result in refusing acknowledgement of the victim's complaint. (26) When providing information, sufficient detail should be given to ensure that victims are treated in a respectful manner and to enable them to make informed decisions about their participation in proceedings. In this respect, information allowing the victim to know about the current status of any proceedings is particularly important. This is equally relevant for information to enable a victim to decide whether to request a review of a decision not to prosecute. Unless otherwise required, it should be possible to provide the information communicated to the victim orally or in writing, including through electronic means. (27) Information to a victim should be provided to the last known correspondence address or electronic contact details given to the competent authority by the victim. In exceptional cases, for example due to the high number of victims involved in a case, it should be possible to provide information through the press, through an official website of the competent authority or through a similar communication channel. (28) Member States should not be obliged to provide information where disclosure of that information could affect the proper handling of a case or harm a given case or person, or if they consider it contrary to the essential interests of their security. (29) Competent authorities should ensure that victims receive updated contact details for communication about their case unless the victim has expressed a wish not to receive such information. (30) A reference to a decision in the context of the right to information, interpretation and translation, should be understood only as a reference to the finding of guilt or otherwise ending criminal proceedings. The reasons for that decision should be provided to the victim through a copy of the document which contains that decision or through a brief summary of them. (31) The right to information about the time and place of a trial resulting from the complaint with regard to a criminal offence suffered by the victim should also apply to information about the time and place of a hearing related to an appeal of a judgment in the case. (32) Specific information about the release or the escape of the offender should be given to victims, upon request, at least in cases where there might be a danger or an identified risk of harm to the victims, unless there is an identified risk of harm to the offender which would result from the notification. Where there is an identified risk of harm to the offender which would result from the notification, the competent authority should take into account all other risks when determining an appropriate action. The reference to identified risk of harm to the victims should cover such factors as the nature and severity of the crime and the risk of retaliation. Therefore, it should not be applied to those situations where minor offences were committed and thus where there is only a slight risk of harm to the victim. (33) Victims should receive information about any right to appeal of a decision to release the offender, if such a right exists in national law. (34) Justice cannot be effectively achieved unless victims can properly explain the circumstances of the crime and provide their evidence in a manner understandable to the competent authorities. It is equally important to ensure that victims are treated in a respectful manner and that they are able to access their rights. Interpretation should therefore be made available, free of charge, during questioning of the victim and in order to enable them to participate actively in court hearings, in accordance with the role of the victim in the relevant criminal justice system. For other aspects of criminal proceedings, the need for interpretation and translation can vary depending on specific issues, the role of the victim in the relevant criminal justice system and his or her involvement in proceedings and any specific rights they have. As such, interpretation and translation for these other cases need only be provided to the extent necessary for victims to exercise their rights. (35) The victim should have the right to challenge a decision finding that there is no need for interpretation or translation, in accordance with procedures in national law. That right does not entail the obligation for Member States to provide for a separate mechanism or complaint procedure in which such decision may be challenged and should not unreasonably prolong the criminal proceedings. An internal review of the decision in accordance with existing national procedures would suffice. (36) The fact that a victim speaks a language which is not widely spoken should not, in itself, be grounds to decide that interpretation or translation would unreasonably prolong the criminal proceedings. (37) Support should be available from the moment the competent authorities are aware of the victim and throughout criminal proceedings and for an appropriate time after such proceedings in accordance with the needs of the victim and the rights set out in this Directive. Support should be provided through a variety of means, without excessive formalities and through a sufficient geographical distribution across the Member State to allow all victims the opportunity to access such services. Victims who have suffered considerable harm due to the severity of the crime could require specialist support services. (38) Persons who are particularly vulnerable or who find themselves in situations that expose them to a particularly high risk of harm, such as persons subjected to repeat violence in close relationships, victims of gender-based violence, or persons who fall victim to other types of crime in a Member State of which they are not nationals or residents, should be provided with specialist support and legal protection. Specialist support services should be based on an integrated and targeted approach which should, in particular, take into account the specific needs of victims, the severity of the harm suffered as a result of a criminal offence, as well as the relationship between victims, offenders, children and their wider social environment. A main task of these services and their staff, which play an important role in supporting the victim to recover from and overcome potential harm or trauma as a result of a criminal offence, should be to inform victims about the rights set out in this Directive so that they can take decisions in a supportive environment that treats them with dignity, respect and sensitivity. The types of support that such specialist support services should offer could include providing shelter and safe accommodation, immediate medical support, referral to medical and forensic examination for evidence in cases of rape or sexual assault, short and long-term psychological counselling, trauma care, legal advice, advocacy and specific services for children as direct or indirect victims. (39) Victim support services are not required to provide extensive specialist and professional expertise themselves. If necessary, victim support services should assist victims in calling on existing professional support, such as psychologists. (40) Although the provision of support should not be dependent on victims making a complaint with regard to a criminal offence to a competent authority such as the police, such authorities are often best placed to inform victims of the possibility of support. Member States are therefore encouraged to establish appropriate conditions to enable the referral of victims to victim support services, including by ensuring that data protection requirements can be and are adhered to. Repeat referrals should be avoided. (41) The right of victims to be heard should be considered to have been fulfilled where victims are permitted to make statements or explanations in writing. (42) The right of child victims to be heard in criminal proceedings should not be precluded solely on the basis that the victim is a child or on the basis of that victim's age. (43) The right to a review of a decision not to prosecute should be understood as referring to decisions taken by prosecutors and investigative judges or law enforcement authorities such as police officers, but not to the decisions taken by courts. Any review of a decision not to prosecute should be carried out by a different person or authority to that which made the original decision, unless the initial decision not to prosecute was taken by the highest prosecuting authority, against whose decision no review can be made, in which case the review may be carried out by that same authority. The right to a review of a decision not to prosecute does not concern special procedures, such as proceedings against members of parliament or government, in relation to the exercise of their official position. (44) A decision ending criminal proceedings should include situations where a prosecutor decides to withdraw charges or discontinue proceedings. (45) A decision of the prosecutor resulting in an out-of-court settlement and thus ending criminal proceedings, excludes victims from the right to a review of a decision of the prosecutor not to prosecute, only if the settlement imposes a warning or an obligation. (46) Restorative justice services, including for example victim-offender mediation, family group conferencing and sentencing circles, can be of great benefit to the victim, but require safeguards to prevent secondary and repeat victimisation, intimidation and retaliation. Such services should therefore have as a primary consideration the interests and needs of the victim, repairing the harm done to the victim and avoiding further harm. Factors such as the nature and severity of the crime, the ensuing degree of trauma, the repeat violation of a victim's physical, sexual, or psychological integrity, power imbalances, and the age, maturity or intellectual capacity of the victim, which could limit or reduce the victim's ability to make an informed choice or could prejudice a positive outcome for the victim, should be taken into consideration in referring a case to the restorative justice services and in conducting a restorative justice process. Restorative justice processes should, in principle, be confidential, unless agreed otherwise by the parties, or as required by national law due to an overriding public interest. Factors such as threats made or any forms of violence committed during the process may be considered as requiring disclosure in the public interest. (47) Victims should not be expected to incur expenses in relation to their participation in criminal proceedings. Member States should be required to reimburse only necessary expenses of victims in relation to their participation in criminal proceedings and should not be required to reimburse victims' legal fees. Member States should be able to impose conditions in regard to the reimbursement of expenses in national law, such as time limits for claiming reimbursement, standard rates for subsistence and travel costs and maximum daily amounts for loss of earnings. The right to reimbursement of expenses in criminal proceedings should not arise in a situation where a victim makes a statement on a criminal offence. Expenses should only be covered to the extent that the victim is obliged or requested by the competent authorities to be present and actively participate in the criminal proceedings. (48) Recoverable property which is seized in criminal proceedings should be returned as soon as possible to the victim of the crime, subject to exceptional circumstances, such as in a dispute concerning the ownership or where the possession of the property or the property itself is illegal. The right to have property returned should be without prejudice to its legitimate retention for the purposes of other legal proceedings. (49) The right to a decision on compensation from the offender and the relevant applicable procedure should also apply to victims resident in a Member State other than the Member State where the criminal offence was committed. (50) The obligation set out in this Directive to transmit complaints should not affect Member States' competence to institute proceedings and is without prejudice to the rules of conflict relating to the exercise of jurisdiction, as laid down in Council Framework Decision 2009/948/JHA of 30 November 2009 on prevention and settlement of conflicts of exercise of jurisdiction in criminal proceedings (13). (51) If the victim has left the territory of the Member State where the criminal offence was committed, that Member State should no longer be obliged to provide assistance, support and protection except for what is directly related to any criminal proceedings it is conducting regarding the criminal offence concerned, such as special protection measures during court proceedings. The Member State of the victim's residence should provide assistance, support and protection required for the victim's need to recover. (52) Measures should be available to protect the safety and dignity of victims and their family members from secondary and repeat victimisation, from intimidation and from retaliation, such as interim injunctions or protection or restraining orders. (53) The risk of secondary and repeat victimisation, of intimidation and of retaliation by the offender or as a result of participation in criminal proceedings should be limited by carrying out proceedings in a coordinated and respectful manner, enabling victims to establish trust in authorities. Interaction with competent authorities should be as easy as possible whilst limiting the number of unnecessary interactions the victim has with them through, for example, video recording of interviews and allowing its use in court proceedings. As wide a range of measures as possible should be made available to practitioners to prevent distress to the victim during court proceedings in particular as a result of visual contact with the offender, his or her family, associates or members of the public. To that end, Member States should be encouraged to introduce, especially in relation to court buildings and police stations, feasible and practical measures enabling the facilities to include amenities such as separate entrances and waiting areas for victims. In addition, Member States should, to the extent possible, plan the criminal proceedings so that contacts between victims and their family members and offenders are avoided, such as by summoning victims and offenders to hearings at different times. (54) Protecting the privacy of the victim can be an important means of preventing secondary and repeat victimisation, intimidation and retaliation and can be achieved through a range of measures including non-disclosure or limitations on the disclosure of information concerning the identity and whereabouts of the victim. Such protection is particularly important for child victims, and includes non-disclosure of the name of the child. However, there might be cases where, exceptionally, the child can benefit from the disclosure or even widespread publication of information, for example where a child has been abducted. Measures to protect the privacy and images of victims and of their family members should always be consistent with the right to a fair trial and freedom of expression, as recognised in Articles 6 and 10, respectively, of the European Convention for the Protection of Human Rights and Fundamental Freedoms. (55) Some victims are particularly at risk of secondary and repeat victimisation, of intimidation and of retaliation by the offender during criminal proceedings. It is possible that such a risk derives from the personal characteristics of the victim or the type, nature or circumstances of the crime. Only through individual assessments, carried out at the earliest opportunity, can such a risk be effectively identified. Such assessments should be carried out for all victims to determine whether they are at risk of secondary and repeat victimisation, of intimidation and of retaliation and what special protection measures they require. (56) Individual assessments should take into account the personal characteristics of the victim such as his or her age, gender and gender identity or expression, ethnicity, race, religion, sexual orientation, health, disability, residence status, communication difficulties, relationship to or dependence on the offender and previous experience of crime. They should also take into account the type or nature and the circumstances of the crime such as whether it is a hate crime, a bias crime or a crime committed with a discriminatory motive, sexual violence, violence in a close relationship, whether the offender was in a position of control, whether the victim's residence is in a high crime or gang dominated area, or whether the victim's country of origin is not the Member State where the crime was committed. (57) Victims of human trafficking, terrorism, organised crime, violence in close relationships, sexual violence or exploitation, gender-based violence, hate crime, and victims with disabilities and child victims tend to experience a high rate of secondary and repeat victimisation, of intimidation and of retaliation. Particular care should be taken when assessing whether such victims are at risk of such victimisation, intimidation and of retaliation and there should be a strong presumption that those victims will benefit from special protection measures. (58) Victims who have been identified as vulnerable to secondary and repeat victimisation, to intimidation and to retaliation should be offered appropriate measures to protect them during criminal proceedings. The exact nature of such measures should be determined through the individual assessment, taking into account the wish of the victim. The extent of any such measure should be determined without prejudice to the rights of the defence and in accordance with rules of judicial discretion. The victims' concerns and fears in relation to proceedings should be a key factor in determining whether they need any particular measure. (59) Immediate operational needs and constraints may make it impossible to ensure, for example, that the same police officer consistently interview the victim; illness, maternity or parental leave are examples of such constraints. Furthermore, premises specially designed for interviews with victims may not be available due, for example, to renovation. In the event of such operational or practical constraints, a special measure envisaged following an individual assessment may not be possible to provide on a case-by-case basis. (60) Where, in accordance with this Directive, a guardian or a representative is to be appointed for a child, those roles could be performed by the same person or by a legal person, an institution or an authority. (61) Any officials involved in criminal proceedings who are likely to come into personal contact with victims should be able to access and receive appropriate initial and ongoing training, to a level appropriate to their contact with victims, so that they are able to identify victims and their needs and deal with them in a respectful, sensitive, professional and non-discriminatory manner. Persons who are likely to be involved in the individual assessment to identify victims' specific protection needs and to determine their need for special protection measures should receive specific training on how to carry out such an assessment. Member States should ensure such training for police services and court staff. Equally, training should be promoted for lawyers, prosecutors and judges and for practitioners who provide victim support or restorative justice services. This requirement should include training on the specific support services to which victims should be referred or specialist training where their work focuses on victims with specific needs and specific psychological training, as appropriate. Where relevant, such training should be gender sensitive. Member States' actions on training should be complemented by guidelines, recommendations and exchange of best practices in accordance with the Budapest roadmap. (62) Member States should encourage and work closely with civil society organisations, including recognised and active non-governmental organisations working with victims of crime, in particular in policymaking initiatives, information and awareness-raising campaigns, research and education programmes and in training, as well as in monitoring and evaluating the impact of measures to support and protect victims of crime. For victims of crime to receive the proper degree of assistance, support and protection, public services should work in a coordinated manner and should be involved at all administrative levels  at Union level, and at national, regional and local level. Victims should be assisted in finding and addressing the competent authorities in order to avoid repeat referrals. Member States should consider developing sole points of access or one-stop shops, that address victims' multiple needs when involved in criminal proceedings, including the need to receive information, assistance, support, protection and compensation. (63) In order to encourage and facilitate reporting of crimes and to allow victims to break the cycle of repeat victimisation, it is essential that reliable support services are available to victims and that competent authorities are prepared to respond to victims' reports in a respectful, sensitive, professional and non-discriminatory manner. This could increase victims' confidence in the criminal justice systems of Member States and reduce the number of unreported crimes. Practitioners who are likely to receive complaints from victims with regard to criminal offences should be appropriately trained to facilitate reporting of crimes, and measures should be put in place to enable third-party reporting, including by civil society organisations. It should be possible to make use of communication technology, such as e-mail, video recordings or online electronic forms for making complaints. (64) Systematic and adequate statistical data collection is recognised as an essential component of effective policymaking in the field of rights set out in this Directive. In order to facilitate evaluation of the application of this Directive, Member States should communicate to the Commission relevant statistical data related to the application of national procedures on victims of crime, including at least the number and type of the reported crimes and, as far as such data are known and are available, the number and age and gender of the victims. Relevant statistical data can include data recorded by the judicial authorities and by law enforcement agencies and, as far as possible, administrative data compiled by healthcare and social welfare services and by public and non-governmental victim support or restorative justice services and other organisations working with victims of crime. Judicial data can include information about reported crime, the number of cases that are investigated and persons prosecuted and sentenced. Service-based administrative data can include, as far as possible, data on how victims are using services provided by government agencies and public and private support organisations, such as the number of referrals by police to victim support services, the number of victims that request, receive or do not receive support or restorative justice. (65) This Directive aims to amend and expand the provisions of Framework Decision 2001/220/JHA. Since the amendments to be made are substantial in number and nature, that Framework Decision should, in the interests of clarity, be replaced in its entirety in relation to Member States participating in the adoption of this Directive. (66) This Directive respects fundamental rights and observes the principles recognised by the Charter of Fundamental Rights of the European Union. In particular, it seeks to promote the right to dignity, life, physical and mental integrity, liberty and security, respect for private and family life, the right to property, the principle of non-discrimination, the principle of equality between women and men, the rights of the child, the elderly and persons with disabilities, and the right to a fair trial. (67) Since the objective of this Directive, namely to establish minimum standards on the rights, support and protection of victims of crime, cannot be sufficiently achieved by the Member States, and can therefore, by reason of its scale and potential effects, be better achieved at Union level, the Union may adopt measures in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty on European Union (TEU). In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary in order to achieve that objective. (68) Personal data processed when implementing this Directive should be protected in accordance with Council Framework Decision 2008/977/JHA of 27 November 2008 on the protection of personal data processed in the framework of police and judicial cooperation in criminal matters (14) and in accordance with the principles laid down in the Council of Europe Convention of 28 January 1981 for the Protection of Individuals with regard to Automatic Processing of Personal Data, which all Member States have ratified. (69) This Directive does not affect more far reaching provisions contained in other Union acts which address the specific needs of particular categories of victims, such as victims of human trafficking and victims of child sexual abuse, sexual exploitation and child pornography, in a more targeted manner. (70) In accordance with Article 3 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the Area of Freedom, Security and Justice, annexed to the TEU and to the TFEU, those Member States have notified their wish to take part in the adoption and application of this Directive. (71) In accordance with Articles 1 and 2 of Protocol No 22 on the position of Denmark, annexed to the TEU and to the TFEU, Denmark is not taking part in the adoption of this Directive and is not bound by it or subject to its application. (72) The European Data Protection Supervisor delivered an opinion on 17 October 2011 (15) based on Article 41(2) of Regulation (EC) No 45/2001 of the European Parliament and of the Council of 18 December 2000 on the protection of individuals with regard to the processing of personal data by the Community institutions and bodies and on the free movement of such data (16), HAVE ADOPTED THIS DIRECTIVE: CHAPTER 1 GENERAL PROVISIONS Article 1 Objectives 1. The purpose of this Directive is to ensure that victims of crime receive appropriate information, support and protection and are able to participate in criminal proceedings. Member States shall ensure that victims are recognised and treated in a respectful, sensitive, tailored, professional and non-discriminatory manner, in all contacts with victim support or restorative justice services or a competent authority, operating within the context of criminal proceedings. The rights set out in this Directive shall apply to victims in a non-discriminatory manner, including with respect to their residence status. 2. Member States shall ensure that in the application of this Directive, where the victim is a child, the child's best interests shall be a primary consideration and shall be assessed on an individual basis. A child-sensitive approach, taking due account of the child's age, maturity, views, needs and concerns, shall prevail. The child and the holder of parental responsibility or other legal representative, if any, shall be informed of any measures or rights specifically focused on the child. Article 2 Definitions 1. For the purposes of this Directive the following definitions shall apply: (a) victim means: (i) a natural person who has suffered harm, including physical, mental or emotional harm or economic loss which was directly caused by a criminal offence; (ii) family members of a person whose death was directly caused by a criminal offence and who have suffered harm as a result of that person's death; (b) family members means the spouse, the person who is living with the victim in a committed intimate relationship, in a joint household and on a stable and continuous basis, the relatives in direct line, the siblings and the dependants of the victim; (c) child means any person below 18 years of age; (d) restorative justice means any process whereby the victim and the offender are enabled, if they freely consent, to participate actively in the resolution of matters arising from the criminal offence through the help of an impartial third party. 2. Member States may establish procedures: (a) to limit the number of family members who may benefit from the rights set out in this Directive taking into account the individual circumstances of each case; and (b) in relation to paragraph (1)(a)(ii), to determine which family members have priority in relation to the exercise of the rights set out in this Directive. CHAPTER 2 PROVISION OF INFORMATION AND SUPPORT Article 3 Right to understand and to be understood 1. Member States shall take appropriate measures to assist victims to understand and to be understood from the first contact and during any further necessary interaction they have with a competent authority in the context of criminal proceedings, including where information is provided by that authority. 2. Member States shall ensure that communications with victims are given in simple and accessible language, orally or in writing. Such communications shall take into account the personal characteristics of the victim including any disability which may affect the ability to understand or to be understood. 3. Unless contrary to the interests of the victim or unless the course of proceedings would be prejudiced, Member States shall allow victims to be accompanied by a person of their choice in the first contact with a competent authority where, due to the impact of the crime, the victim requires assistance to understand or to be understood. Article 4 Right to receive information from the first contact with a competent authority 1. Member States shall ensure that victims are offered the following information, without unnecessary delay, from their first contact with a competent authority in order to enable them to access the rights set out in this Directive: (a) the type of support they can obtain and from whom, including, where relevant, basic information about access to medical support, any specialist support, including psychological support, and alternative accommodation; (b) the procedures for making complaints with regard to a criminal offence and their role in connection with such procedures; (c) how and under what conditions they can obtain protection, including protection measures; (d) how and under what conditions they can access legal advice, legal aid and any other sort of advice; (e) how and under what conditions they can access compensation; (f) how and under what conditions they are entitled to interpretation and translation; (g) if they are resident in a Member State other than that where the criminal offence was committed, any special measures, procedures or arrangements, which are available to protect their interests in the Member State where the first contact with the competent authority is made; (h) the available procedures for making complaints where their rights are not respected by the competent authority operating within the context of criminal proceedings; (i) the contact details for communications about their case; (j) the available restorative justice services; (k) how and under what conditions expenses incurred as a result of their participation in the criminal proceedings can be reimbursed. 2. The extent or detail of information referred to in paragraph 1 may vary depending on the specific needs and personal circumstances of the victim and the type or nature of the crime. Additional details may also be provided at later stages depending on the needs of the victim and the relevance, at each stage of proceedings, of such details. Article 5 Right of victims when making a complaint 1. Member States shall ensure that victims receive written acknowledgement of their formal complaint made by them to the competent authority of a Member State, stating the basic elements of the criminal offence concerned. 2. Member States shall ensure that victims who wish to make a complaint with regard to a criminal offence and who do not understand or speak the language of the competent authority be enabled to make the complaint in a language that they understand or by receiving the necessary linguistic assistance. 3. Member States shall ensure that victims who do not understand or speak the language of the competent authority, receive translation, free of charge, of the written acknowledgement of their complaint provided for in paragraph 1, if they so request, in a language that they understand. Article 6 Right to receive information about their case 1. Member States shall ensure that victims are notified without unnecessary delay of their right to receive the following information about the criminal proceedings instituted as a result of the complaint with regard to a criminal offence suffered by the victim and that, upon request, they receive such information: (a) any decision not to proceed with or to end an investigation or not to prosecute the offender; (b) the time and place of the trial, and the nature of the charges against the offender. 2. Member States shall ensure that, in accordance with their role in the relevant criminal justice system, victims are notified without unnecessary delay of their right to receive the following information about the criminal proceedings instituted as a result of the complaint with regard to a criminal offence suffered by them and that, upon request, they receive such information: (a) any final judgment in a trial; (b) information enabling the victim to know about the state of the criminal proceedings, unless in exceptional cases the proper handling of the case may be adversely affected by such notification. 3. Information provided for under paragraph 1(a) and paragraph 2(a) shall include reasons or a brief summary of reasons for the decision concerned, except in the case of a jury decision or a decision where the reasons are confidential in which cases the reasons are not provided as a matter of national law. 4. The wish of victims as to whether or not to receive information shall bind the competent authority, unless that information must be provided due to the entitlement of the victim to active participation in the criminal proceedings. Member States shall allow victims to modify their wish at any moment, and shall take such modification into account. 5. Member States shall ensure that victims are offered the opportunity to be notified, without unnecessary delay, when the person remanded in custody, prosecuted or sentenced for criminal offences concerning them is released from or has escaped detention. Furthermore, Member States shall ensure that victims are informed of any relevant measures issued for their protection in case of release or escape of the offender. 6. Victims shall, upon request, receive the information provided for in paragraph 5 at least in cases where there is a danger or an identified risk of harm to them, unless there is an identified risk of harm to the offender which would result from the notification. Article 7 Right to interpretation and translation 1. Member States shall ensure that victims who do not understand or speak the language of the criminal proceedings concerned are provided, upon request, with interpretation in accordance with their role in the relevant criminal justice system in criminal proceedings, free of charge, at least during any interviews or questioning of the victim during criminal proceedings before investigative and judicial authorities, including during police questioning, and interpretation for their active participation in court hearings and any necessary interim hearings. 2. Without prejudice to the rights of the defence and in accordance with rules of judicial discretion, communication technology such as videoconferencing, telephone or internet may be used, unless the physical presence of the interpreter is required in order for the victims to properly exercise their rights or to understand the proceedings. 3. Member States shall ensure that victims who do not understand or speak the language of the criminal proceedings concerned are provided, in accordance with their role in the relevant criminal justice system in criminal proceedings, upon request, with translations of information essential to the exercise of their rights in criminal proceedings in a language that they understand, free of charge, to the extent that such information is made available to the victims. Translations of such information shall include at least any decision ending the criminal proceedings related to the criminal offence suffered by the victim, and upon the victim's request, reasons or a brief summary of reasons for such decision, except in the case of a jury decision or a decision where the reasons are confidential in which cases the reasons are not provided as a matter of national law. 4. Member States shall ensure that victims who are entitled to information about the time and place of the trial in accordance with Article 6(1)(b) and who do not understand the language of the competent authority, are provided with a translation of the information to which they are entitled, upon request. 5. Victims may submit a reasoned request to consider a document as essential. There shall be no requirement to translate passages of essential documents which are not relevant for the purpose of enabling victims to actively participate in the criminal proceedings. 6. Notwithstanding paragraphs 1 and 3, an oral translation or oral summary of essential documents may be provided instead of a written translation on condition that such oral translation or oral summary does not prejudice the fairness of the proceedings. 7. Member States shall ensure that the competent authority assesses whether victims need interpretation or translation as provided for under paragraphs 1 and 3. Victims may challenge a decision not to provide interpretation or translation. The procedural rules for such a challenge shall be determined by national law. 8. Interpretation and translation and any consideration of a challenge of a decision not to provide interpretation or translation under this Article shall not unreasonably prolong the criminal proceedings. Article 8 Right to access victim support services 1. Member States shall ensure that victims, in accordance with their needs, have access to confidential victim support services, free of charge, acting in the interests of the victims before, during and for an appropriate time after criminal proceedings. Family members shall have access to victim support services in accordance with their needs and the degree of harm suffered as a result of the criminal offence committed against the victim. 2. Member States shall facilitate the referral of victims, by the competent authority that received the complaint and by other relevant entities, to victim support services. 3. Member States shall take measures to establish free of charge and confidential specialist support services in addition to, or as an integrated part of, general victim support services, or to enable victim support organisations to call on existing specialised entities providing such specialist support. Victims, in accordance with their specific needs, shall have access to such services and family members shall have access in accordance with their specific needs and the degree of harm suffered as a result of the criminal offence committed against the victim. 4. Victim support services and any specialist support services may be set up as public or non-governmental organisations and may be organised on a professional or voluntary basis. 5. Member States shall ensure that access to any victim support services is not dependent on a victim making a formal complaint with regard to a criminal offence to a competent authority. Article 9 Support from victim support services 1. Victim support services, as referred to in Article 8(1), shall, as a minimum, provide: (a) information, advice and support relevant to the rights of victims including on accessing national compensation schemes for criminal injuries, and on their role in criminal proceedings including preparation for attendance at the trial; (b) information about or direct referral to any relevant specialist support services in place; (c) emotional and, where available, psychological support; (d) advice relating to financial and practical issues arising from the crime; (e) unless otherwise provided by other public or private services, advice relating to the risk and prevention of secondary and repeat victimisation, of intimidation and of retaliation. 2. Member States shall encourage victim support services to pay particular attention to the specific needs of victims who have suffered considerable harm due to the severity of the crime. 3. Unless otherwise provided by other public or private services, specialist support services referred to in Article 8(3), shall, as a minimum, develop and provide: (a) shelters or any other appropriate interim accommodation for victims in need of a safe place due to an imminent risk of secondary and repeat victimisation, of intimidation and of retaliation; (b) targeted and integrated support for victims with specific needs, such as victims of sexual violence, victims of gender-based violence and victims of violence in close relationships, including trauma support and counselling. CHAPTER 3 PARTICIPATION IN CRIMINAL PROCEEDINGS Article 10 Right to be heard 1. Member States shall ensure that victims may be heard during criminal proceedings and may provide evidence. Where a child victim is to be heard, due account shall be taken of the child's age and maturity. 2. The procedural rules under which victims may be heard during criminal proceedings and may provide evidence shall be determined by national law. Article 11 Rights in the event of a decision not to prosecute 1. Member States shall ensure that victims, in accordance with their role in the relevant criminal justice system, have the right to a review of a decision not to prosecute. The procedural rules for such a review shall be determined by national law. 2. Where, in accordance with national law, the role of the victim in the relevant criminal justice system will be established only after a decision to prosecute the offender has been taken, Member States shall ensure that at least the victims of serious crimes have the right to a review of a decision not to prosecute. The procedural rules for such a review shall be determined by national law. 3. Member States shall ensure that victims are notified without unnecessary delay of their right to receive, and that they receive sufficient information to decide whether to request a review of any decision not to prosecute upon request. 4. Where the decision not to prosecute is taken by the highest prosecuting authority against whose decision no review may be carried out under national law, the review may be carried out by the same authority. 5. Paragraphs 1, 3 and 4 shall not apply to a decision of the prosecutor not to prosecute, if such a decision results in an out-of-court settlement, in so far as national law makes such provision. Article 12 Right to safeguards in the context of restorative justice services 1. Member States shall take measures to safeguard the victim from secondary and repeat victimisation, from intimidation and from retaliation, to be applied when providing any restorative justice services. Such measures shall ensure that victims who choose to participate in restorative justice processes have access to safe and competent restorative justice services, subject to at least the following conditions: (a) the restorative justice services are used only if they are in the interest of the victim, subject to any safety considerations, and are based on the victim's free and informed consent, which may be withdrawn at any time; (b) before agreeing to participate in the restorative justice process, the victim is provided with full and unbiased information about that process and the potential outcomes as well as information about the procedures for supervising the implementation of any agreement; (c) the offender has acknowledged the basic facts of the case; (d) any agreement is arrived at voluntarily and may be taken into account in any further criminal proceedings; (e) discussions in restorative justice processes that are not conducted in public are confidential and are not subsequently disclosed, except with the agreement of the parties or as required by national law due to an overriding public interest. 2. Member States shall facilitate the referral of cases, as appropriate to restorative justice services, including through the establishment of procedures or guidelines on the conditions for such referral. Article 13 Right to legal aid Member States shall ensure that victims have access to legal aid, where they have the status of parties to criminal proceedings. The conditions or procedural rules under which victims have access to legal aid shall be determined by national law. Article 14 Right to reimbursement of expenses Member States shall afford victims who participate in criminal proceedings, the possibility of reimbursement of expenses incurred as a result of their active participation in criminal proceedings, in accordance with their role in the relevant criminal justice system. The conditions or procedural rules under which victims may be reimbursed shall be determined by national law. Article 15 Right to the return of property Member States shall ensure that, following a decision by a competent authority, recoverable property which is seized in the course of criminal proceedings is returned to victims without delay, unless required for the purposes of criminal proceedings. The conditions or procedural rules under which such property is returned to the victims shall be determined by national law. Article 16 Right to decision on compensation from the offender in the course of criminal proceedings 1. Member States shall ensure that, in the course of criminal proceedings, victims are entitled to obtain a decision on compensation by the offender, within a reasonable time, except where national law provides for such a decision to be made in other legal proceedings. 2. Member States shall promote measures to encourage offenders to provide adequate compensation to victims. Article 17 Rights of victims resident in another Member State 1. Member States shall ensure that their competent authorities can take appropriate measures to minimise the difficulties faced where the victim is a resident of a Member State other than that where the criminal offence was committed, particularly with regard to the organisation of the proceedings. For this purpose, the authorities of the Member State where the criminal offence was committed shall, in particular, be in a position: (a) to take a statement from the victim immediately after the complaint with regard to the criminal offence is made to the competent authority; (b) to have recourse to the extent possible to the provisions on video conferencing and telephone conference calls laid down in the Convention on Mutual Assistance in Criminal Matters between the Member States of the European Union of 29 May 2000 (17) for the purpose of hearing victims who are resident abroad. 2. Member States shall ensure that victims of a criminal offence committed in Member States other than that where they reside may make a complaint to the competent authorities of the Member State of residence, if they are unable to do so in the Member State where the criminal offence was committed or, in the event of a serious offence, as determined by national law of that Member State, if they do not wish to do so. 3. Member States shall ensure that the competent authority to which the victim makes a complaint transmits it without delay to the competent authority of the Member State in which the criminal offence was committed, if the competence to institute the proceedings has not been exercised by the Member State in which the complaint was made. CHAPTER 4 PROTECTION OF VICTIMS AND RECOGNITION OF VICTIMS WITH SPECIFIC PROTECTION NEEDS Article 18 Right to protection Without prejudice to the rights of the defence, Member States shall ensure that measures are available to protect victims and their family members from secondary and repeat victimisation, from intimidation and from retaliation, including against the risk of emotional or psychological harm, and to protect the dignity of victims during questioning and when testifying. When necessary, such measures shall also include procedures established under national law for the physical protection of victims and their family members. Article 19 Right to avoid contact between victim and offender 1. Member States shall establish the necessary conditions to enable avoidance of contact between victims and their family members, where necessary, and the offender within premises where criminal proceedings are conducted, unless the criminal proceedings require such contact. 2. Member States shall ensure that new court premises have separate waiting areas for victims. Article 20 Right to protection of victims during criminal investigations Without prejudice to the rights of the defence and in accordance with rules of judicial discretion, Member States shall ensure that during criminal investigations: (a) interviews of victims are conducted without unjustified delay after the complaint with regard to a criminal offence has been made to the competent authority; (b) the number of interviews of victims is kept to a minimum and interviews are carried out only where strictly necessary for the purposes of the criminal investigation; (c) victims may be accompanied by their legal representative and a person of their choice, unless a reasoned decision has been made to the contrary; (d) medical examinations are kept to a minimum and are carried out only where strictly necessary for the purposes of the criminal proceedings. Article 21 Right to protection of privacy 1. Member States shall ensure that competent authorities may take during the criminal proceedings appropriate measures to protect the privacy, including personal characteristics of the victim taken into account in the individual assessment provided for under Article 22, and images of victims and of their family members. Furthermore, Member States shall ensure that competent authorities may take all lawful measures to prevent public dissemination of any information that could lead to the identification of a child victim. 2. In order to protect the privacy, personal integrity and personal data of victims, Member States shall, with respect for freedom of expression and information and freedom and pluralism of the media, encourage the media to take self-regulatory measures. Article 22 Individual assessment of victims to identify specific protection needs 1. Member States shall ensure that victims receive a timely and individual assessment, in accordance with national procedures, to identify specific protection needs and to determine whether and to what extent they would benefit from special measures in the course of criminal proceedings, as provided for under Articles 23 and 24, due to their particular vulnerability to secondary and repeat victimisation, to intimidation and to retaliation. 2. The individual assessment shall, in particular, take into account: (a) the personal characteristics of the victim; (b) the type or nature of the crime; and (c) the circumstances of the crime. 3. In the context of the individual assessment, particular attention shall be paid to victims who have suffered considerable harm due to the severity of the crime; victims who have suffered a crime committed with a bias or discriminatory motive which could, in particular, be related to their personal characteristics; victims whose relationship to and dependence on the offender make them particularly vulnerable. In this regard, victims of terrorism, organised crime, human trafficking, gender-based violence, violence in a close relationship, sexual violence, exploitation or hate crime, and victims with disabilities shall be duly considered. 4. For the purposes of this Directive, child victims shall be presumed to have specific protection needs due to their vulnerability to secondary and repeat victimisation, to intimidation and to retaliation. To determine whether and to what extent they would benefit from special measures as provided for under Articles 23 and 24, child victims shall be subject to an individual assessment as provided for in paragraph 1 of this Article. 5. The extent of the individual assessment may be adapted according to the severity of the crime and the degree of apparent harm suffered by the victim. 6. Individual assessments shall be carried out with the close involvement of the victim and shall take into account their wishes including where they do not wish to benefit from special measures as provided for in Articles 23 and 24. 7. If the elements that form the basis of the individual assessment have changed significantly, Member States shall ensure that it is updated throughout the criminal proceedings. Article 23 Right to protection of victims with specific protection needs during criminal proceedings 1. Without prejudice to the rights of the defence and in accordance with rules of judicial discretion, Member States shall ensure that victims with specific protection needs who benefit from special measures identified as a result of an individual assessment provided for in Article 22(1), may benefit from the measures provided for in paragraphs 2 and 3 of this Article. A special measure envisaged following the individual assessment shall not be made available if operational or practical constraints make this impossible, or where there is a an urgent need to interview the victim and failure to do so could harm the victim or another person or could prejudice the course of the proceedings. 2. The following measures shall be available during criminal investigations to victims with specific protection needs identified in accordance with Article 22(1): (a) interviews with the victim being carried out in premises designed or adapted for that purpose; (b) interviews with the victim being carried out by or through professionals trained for that purpose; (c) all interviews with the victim being conducted by the same persons unless this is contrary to the good administration of justice; (d) all interviews with victims of sexual violence, gender-based violence or violence in close relationships, unless conducted by a prosecutor or a judge, being conducted by a person of the same sex as the victim, if the victim so wishes, provided that the course of the criminal proceedings will not be prejudiced. 3. The following measures shall be available for victims with specific protection needs identified in accordance with Article 22(1) during court proceedings: (a) measures to avoid visual contact between victims and offenders including during the giving of evidence, by appropriate means including the use of communication technology; (b) measures to ensure that the victim may be heard in the courtroom without being present, in particular through the use of appropriate communication technology; (c) measures to avoid unnecessary questioning concerning the victim's private life not related to the criminal offence; and (d) measures allowing a hearing to take place without the presence of the public. Article 24 Right to protection of child victims during criminal proceedings 1. In addition to the measures provided for in Article 23, Member States shall ensure that where the victim is a child: (a) in criminal investigations, all interviews with the child victim may be audiovisually recorded and such recorded interviews may be used as evidence in criminal proceedings; (b) in criminal investigations and proceedings, in accordance with the role of victims in the relevant criminal justice system, competent authorities appoint a special representative for child victims where, according to national law, the holders of parental responsibility are precluded from representing the child victim as a result of a conflict of interest between them and the child victim, or where the child victim is unaccompanied or separated from the family; (c) where the child victim has the right to a lawyer, he or she has the right to legal advice and representation, in his or her own name, in proceedings where there is, or there could be, a conflict of interest between the child victim and the holders of parental responsibility. The procedural rules for the audiovisual recordings referred to in point (a) of the first subparagraph and the use thereof shall be determined by national law. 2. Where the age of a victim is uncertain and there are reasons to believe that the victim is a child, the victim shall, for the purposes of this Directive, be presumed to be a child. CHAPTER 5 OTHER PROVISIONS Article 25 Training of practitioners 1. Member States shall ensure that officials likely to come into contact with victims, such as police officers and court staff, receive both general and specialist training to a level appropriate to their contact with victims to increase their awareness of the needs of victims and to enable them to deal with victims in an impartial, respectful and professional manner. 2. Without prejudice to judicial independence and differences in the organisation of the judiciary across the Union, Member States shall request that those responsible for the training of judges and prosecutors involved in criminal proceedings make available both general and specialist training to increase the awareness of judges and prosecutors of the needs of victims. 3. With due respect for the independence of the legal profession, Member States shall recommend that those responsible for the training of lawyers make available both general and specialist training to increase the awareness of lawyers of the needs of victims. 4. Through their public services or by funding victim support organisations, Member States shall encourage initiatives enabling those providing victim support and restorative justice services to receive adequate training to a level appropriate to their contact with victims and observe professional standards to ensure such services are provided in an impartial, respectful and professional manner. 5. In accordance with the duties involved, and the nature and level of contact the practitioner has with victims, training shall aim to enable the practitioner to recognise victims and to treat them in a respectful, professional and non-discriminatory manner. Article 26 Cooperation and coordination of services 1. Member States shall take appropriate action to facilitate cooperation between Member States to improve the access of victims to the rights set out in this Directive and under national law. Such cooperation shall be aimed at least at: (a) the exchange of best practices; (b) consultation in individual cases; and (c) assistance to European networks working on matters directly relevant to victims' rights. 2. Member States shall take appropriate action, including through the internet, aimed at raising awareness of the rights set out in this Directive, reducing the risk of victimisation, and minimising the negative impact of crime and the risks of secondary and repeat victimisation, of intimidation and of retaliation, in particular by targeting groups at risk such as children, victims of gender-based violence and violence in close relationships. Such action may include information and awareness raising campaigns and research and education programmes, where appropriate in cooperation with relevant civil society organisations and other stakeholders. CHAPTER 6 FINAL PROVISIONS Article 27 Transposition 1. Member States shall bring into force the laws, regulations and administrative provisions necessary to comply with this Directive by 16 November 2015. 2. When Member States adopt those provisions they shall contain a reference to this Directive or be accompanied by such a reference on the occasion of their official publication. Member States shall determine how such a reference is to be made. Article 28 Provision of data and statistics Member States shall, by 16 November 2017 and every three years thereafter, communicate to the Commission available data showing how victims have accessed the rights set out in this Directive. Article 29 Report The Commission shall, by 16 November 2017, submit a report to the European Parliament and to the Council, assessing the extent to which the Member States have taken the necessary measures in order to comply with this Directive, including a description of action taken under Articles 8, 9 and 23, accompanied, if necessary, by legislative proposals. Article 30 Replacement of Framework Decision 2001/220/JHA Framework Decision 2001/220/JHA is hereby replaced in relation to Member States participating in the adoption of this Directive, without prejudice to the obligations of the Member States relating to the time limits for transposition into national law. In relation to Member States participating in the adoption of this Directive, references to that Framework Decision shall be construed as references to this Directive. Article 31 Entry into force This Directive shall enter into force on the day following that of its publication in the Official Journal of the European Union. Article 32 Addressees This Directive is addressed to the Member States in accordance with the Treaties. Done at Strasbourg, 25 October 2012. For the European Parliament The President M. SCHULZ For the Council The President A. D. MAVROYIANNIS (1) OJ C 43, 15.2.2012, p. 39. (2) OJ C 113, 18.4.2012, p. 56. (3) Position of the European Parliament of 12 September 2012 (not yet published in the Official Journal) and decision of the Council of 4 October 2012. (4) OJ L 82, 22.3.2001, p. 1. (5) OJ C 115, 4.5.2010, p. 1. (6) OJ C 187, 28.6.2011, p. 1. (7) OJ C 285 E, 21.10.2010, p. 53. (8) OJ C 296 E, 2.10.2012, p. 26. (9) OJ L 338, 21.12.2011, p. 2. (10) OJ L 101, 15.4.2011, p. 1. (11) OJ L 335, 17.12.2011, p. 1. (12) OJ L 164, 22.6.2002, p. 3. (13) OJ L 328, 15.12.2009, p. 42. (14) OJ L 350, 30.12.2008, p. 60. (15) OJ C 35, 9.2.2012, p. 10. (16) OJ L 8, 12.1.2001, p. 1. (17) OJ C 197, 12.7.2000, p. 3.
============================== "END OF DOC" ==============================
        

Bad pipe message: %s [b' q=0.9, image/avif, image/webp, image/apng, */*; q=0.8, application/signed-exchange; v=b3; q=0.7\r\nHost: loc', b'host:40221\r\nUser-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, lik']
Bad pipe message: %s [b'Gecko) Chrome/145.0.0.0 Safari/537.36\r\nAccept-Encoding: gzip, deflate, br, zstd\r\nAccept-Language: en']
Bad pipe message: %s [b'S, en; q=0.9\r\nCache-Control: max-age=0\r\nRefe', b'r: https://github.com/\r\nX-Request-ID: c6a0d9ead2d3484aeaae1d709e77696d\r\nX-Real-IP: 10.241.0.36\r\nX-Forwarded-Port:', b'43\r\nX-Forwarded-Scheme: https\r\n']
Bad pipe message: %s [b'Original-URI: /\r\nX-Scheme: https\r\nDNT: 1\r\nsec-fetch-site: cross-site\r\nsec-fetch-mode: n', b'igate\r\nsec-fetch-dest: document\r\nsec-ch-ua: "Not:A-Brand";v="99", "Google Chrome";v="145", "Chro']
Bad pipe message: %s [b'um";v="145"\r\nsec-ch-ua-mobile: ?0\r\nsec-ch-ua-platform: "Windows"\r\npriority: u=0, i\r\nX-Forwarded-Proto: https', b'X-Forwarded-']


In [12]:
display(Markdown(search_docs("Drug dealing Sentences")))

 

        doc 0 :

        'celex': 31997R2046
        'status': Not in Force
        'act_type': Regulation
        'treaty': TEC (1992)

        full_doc :

        Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social development and the attainment of the goals of Community policy in the sphere of development cooperation as defined in Article 130u of the Treaty;Whereas as part of the effort to combat the supply of drugs it is essential, in particular, to effect a radical reduction of poverty in the south and to offer the population a lawful alternative to the growing of illegal crops;Whereas institutional support should be given to those developing countries which so request so that they can combat drugs more effectively;Whereas in a communication to the Parliament and to the Council dated 23 June 1994, the Commission presented its guidelines for a European Union plan of action on drugs for 1995-1999, including measures at international level;Whereas Parliament stated its views on the guidelines in its Opinion on the communication, adopted on 15 June 1995;Whereas the Fourth ACP-EC Convention and the cooperation, association and partnership agreements concluded by the European Community with developing countries contain clauses on cooperation to curb drug abuse and drug trafficking, the monitoring of trade in precursors, chemical products and psychotropic substances and the exchange of relevant information, including measures in the field of money laundering; whereas there is a relationship between the campaign against drugs and drug addiction and the aims of the cooperation policy pursued by the Community and its developing-country partners;Whereas the international community's strategy to curb drug abuse and drug trafficking is based on universal accession to the Single Convention on narcotic drugs of 1961, as amended by the Protocol of 1972, the Convention on psychotropic substances of 1971 and the International Convention against illicit traffic in narcotic drugs and psychotropic substances of 1988, and on the systematic implementation of those conventions at national and international level;Whereas the European Community is a party to the Convention of 1988, in particular by virtue of Article 12 of that Convention, and has adopted Community legislation based on the recommendations of the Chemicals Action Task Force set up by the G7 and the President of the Commission in 1989, the effectiveness of which would be generally enhanced by the adoption of the relevant legislation and procedures in other parts of the world;Whereas effective action against drugs must also encompass measures against the laundering of money from drug trafficking, such as the adoption of a suitable legal framework and appropriate mechanisms in the countries concerned;Whereas human rights must be duly respected in implementing measures under this Regulation;Whereas the Member States of the European Community have endorsed the policy statement and general plan of action adopted by the UN General Assembly at its 17th special session;Whereas a financial reference amount within the meaning of point 2 of the Declaration by the European Parliament, the Council and the Commission of 6 March 1995 is included in this Regulation for the period 1998-2000, without thereby affecting the powers of the budgetary authority as they are defined by the Treaty,HAS ADOPTED THIS REGULATION:Article 1 In the framework of its development cooperation policy and taking account of the harmful effects on development efforts of the production, trading and consumption of drugs, the Community shall carry out cooperation activities in the field of drugs and drug addiction in developing countries, giving priority to those which have demonstrated political will at the highest level to solve their drug problem. The existence of such will may be demonstrated inter alia by ratification of the Single Convention of 1961 as amended by the Protocol of 1972, the Convention of 1971 and the Convention of 1988. Commitment on the part of developing countries shall take the form, inter alia, of the implementation of domestic legislation against laundering of money generated through illicit drugs.Article 2 The assistance provided under this Regulation shall complement and reinforce assistance provided under other instruments of development cooperation.Article 3 The Community shall give priority at the request of a partner country to supporting the preparation of a national drug control master plan, in close consultation with the United Nations International Drug Control Programme (UNDCP). These plans will identify objectives, strategies and priorities in the campaign against drugs and the related resource requirements (including financial requirements), thus establishing an integrated, multidisciplinary and multisectoral approach designed to maximize the efficiency of national drug control programmes and international assistance.The prevention of drug addiction, together with demand reduction, shall be addressed in a consistent policy comprising education and objective information about the consequences of addiction, targeted especially at young people.Community cooperation shall take place in a spirit of dialogue reflecting the genuine cultural differences which affect the perception of drug-related problems, this being crucial to ensure the social and political viability of drug control strategies.Article 4 Preferably operating within the strategic framework established by the national plans, the Community shall also support specific operations capable of a measurable impact (i.e. effective and tangible results within a time-limit set in advance) in the following areas:- development of institutional capacity, in particular for the implementation of:- National Drug Control Plans by developing countries,- agreements between the Community and certain developing countries, in particular to combat the diversion of chemical precursors and to curb money laundering,- demand reduction, in particular through analysis of local patterns, the introduction of measures to control trade in and consumption of narcotics and psychotropic substances, treatment and reintegration of drug addicts, as well as risk limitation. These measures must be integrated into policies on health and education, development and combating poverty and social and economic exclusion,- promotion of pilot alternative development projects, conceived as a process by which the production of illicit drug crops is eventually both combatted and eliminated through appropriate rural development measures in the context of sustained national economic growth. These projects shall comprise social and economic measures which take into account factors contributing to illicit production as well as measures which may facilitate improved use of commercial preferences. In this connection, it shall be systematically examined whether it is possible to make more use of other Community financial instruments (e.g. ALA) and the European Development Fund for alternative development projects,- financing of studies, seminars and fora for the exchange of experience in the above fields.Particular attention shall be paid to the participation of local people and target groups in identifying, planning and carrying out operations.The Community shall only finance projects in which respect for human rights is guaranteed.Article 5 The cooperation partners eligible for financial support under this Regulation shall be regional and international organizations, in particular UNDCP, local- and Member State-based non-governmental organizations, national, provincial and local government departments and agencies, community-based organizations, institutes and public and private operators.Article 6 1. The instruments to be employed in the course of the operations referred to in Articles 3 and 4 shall include studies, technical assistance, training or other services, supplies and works, along with audits and evaluation and monitoring missions.2. According to the needs of the operations concerned, Community financing may cover both capital investment, other than the purchase of real estate, and operating costs in foreign or local currency. However, with the exception of training programmes, operating costs may normally be covered only during the start-up phase and on a degressive basis.3. A financial contribution from the partners defined in Article 5 shall be sought for each cooperation operation. Their contribution will be requested within the limits of the possibilities available to the parties concerned and depending on the nature of the operation concerned.4. A financial contribution from the local partners, particularly in respect of operating costs, shall be sought as a matter of priority in the case of projects intended to launch long term activities, so as to ensure the viability of such projects once Community funding comes to an end.5. Opportunities may be sought for cofinancing with other fund providers, and especially with Member States.6. The Commission will ensure that the Community character of the aid provided under this Regulation is highlighted.7. In order to achieve the objectives of consistency and complementarity referred to in the Treaty and with the aim of guaranteeing optimum effectiveness of all these operations, the Commission may take all necessary coordination measures, including in particular:(a) a system for the systematic exchange and analysis of information on operations financed and those which the Community and the Member States propose to finance;(b) on-the-spot coordination of the implementation of operations through regular meetings and exchange of information between representatives of the Commission and of the Member States in the beneficiary country.8. In order to obtain the greatest possible impact globally and nationally, the Commission, in liaison with the Member States, shall take any initiative necessary for ensuring proper coordination and close collaboration with the beneficiary countries and the providers of funds and other international agencies involved, in particular those forming part of the United Nations system and more specifically the UNDCP.Article 7 Financial support under this Regulation shall take the form of grants.Article 8 The financial reference amount for the implementation of this programme during the period 1998-2000 shall be ECU 30 million.Annual appropriations shall be authorized by the budgetary authority within the limits of the financial perspectives.Article 9 1. The Commission shall be responsible for appraising, approving and managing operations covered by this Regulation in accordance with the budgetary and other procedures in force, and in particular those laid down in the Financial Regulation applicable to the general budget of the European Communities.2. Projects and programme appraisal shall take into account the following factors:- effectiveness and viability of operations,- cultural, social, gender and environmental aspects,- institutional development necessary to achieve project goals,- experience gained from operations of the same kind.3. Decisions relating to grants of more than ECU 2 million for individual operations financed under this Regulation and any changes resulting in an increase of more than 20 % in the sum initially approved for such an operation shall be adopted under the procedure laid down in Article 10.The Commission shall inform the Committee referred to in Article 10 succinctly of the financing decisions which it intends to take with regard to projects and programmes of less than ECU 2 million in value. The information shall be made available not later than one week before the decision is taken.4. The Commission shall be authorized to approve, without recourse to the opinion of the Committee provided for in Article 10, any supplementary commitments needed for covering expected or real cost overruns in connection with the operations, where the overrun or additional requirement is less than or equal to 20 % of the initial commitment fixed by the financing decision.5. All financing agreements or contracts concluded under this Regulation shall provide for the Commission and the Court of Auditors to conduct on-the-spot checks according to the usual procedures laid down by the Commission under the rules in force, and in particular those of the Financial Regulation applicable to the general budget of the European Communities.6. Where operations are the subject of financing agreements between the Community and the recipient country, such agreement shall stipulate that the payment of taxes, duties or any other charges is not to be covered by the Community.7. Participation in invitations to tender and the award of contracts shall be open on equal terms to natural and legal persons of the Member States and of the recipient country. It may be extended to other developing countries.8. Supplies shall originate in the Member States, the recipient country or other developing countries. In exceptional cases, where circumstances warrant, supplies may originate elsewhere.9. Particular attention will be given to:- the pursuit of cost-effectiveness and sustainable impact in project design,- the clear definition and monitoring of objectives and indications of achievement for all projects.Article 10 1. The Commission shall be assisted by the geographically-determined committee competent for development.2. The representative of the Commission shall submit to the committee a draft of the measures to be taken. The committee shall deliver its opinion on the draft, within a time limit which the chairman may lay down according to the urgency of the matter. The opinion shall be delivered by the majority laid down in Article 148 (2) of the Treaty in the case of decisions which the Council is required to adopt on a proposal from the Commission. The votes of the representatives of the Member States within the committee shall be weighted in the manner set out in that Article. The Chairman shall not vote.The Commission shall adopt the measures envisaged if they are in accordance with the opinion of the committee.If the measures envisaged are not in accordance with the opinion of the committee, or if no opinion is delivered, the Commission shall without delay submit to the Council a proposal relating to the measures to be taken. The Council shall act by a qualified majority.If, on the expiry of a period of three months from the date of referral to the Council, the Council has not acted, the proposed measures shall be adopted by the Commission.3. An exchange of views shall take place once a year on the basis of a presentation by the representative of the Commission of the general guidelines for the operations to be carried out in the year ahead, in the framework of a joint meeting of the committees referred to in paragraph 1.Article 11 1. At the end of each budget year, the Commission shall present a report to Parliament and the Council summarizing the operations financed in the course of that year and evaluating the implementation of this Regulation over that period.The summary shall in particular contain information about those with whom contracts have been concluded.2. The Commission shall regularly assess operations financed by the Community with a view to establishing whether the objectives aimed at by such operations have been achieved and to providing guidelines for improving the effectiveness of future operations. The Commission shall submit to the Committee referred to in Article 10 a summary of the assessments made which, if appropriate, may be examined by the Committee. The assessment reports shall be made available to any Member States requesting them.3. The Commission shall inform the Member States, at the latest one month after its decision, of the operations and projects approved, stating their cost and nature, the recipient country and partners.Article 12 1. This Regulation shall enter into force on the third day following that of its publication in the Official Journal of the European Communities.2. Three years after this Regulation enters into force, the Commission shall submit to the European Parliament and the Council an overall assessment of operations financed by the Community under this Regulation together with suggestions regarding the future of this Regulation and, where necessary, proposals for amending or terminating it.This Regulation shall be binding in its entirety and directly applicable in all Member States.Done at Luxembourg, 13 October 1997.For the CouncilThe PresidentJ.-C. JUNCKER(1) OJ C 242, 19. 9. 1995, p. 8.(2) Opinion of the European Parliament of 19 April 1996 (OJ C 141, 13. 5. 1996, p. 252), Council Common Position of 22 November 1996, (OJ C 6, 9. 1. 1997, p. 1) and Decision of the European Parliament of 13 March 1997 (OJ C 115/97, 14. 4. 1997, p. 127).
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32014D0688
        'status': Not in Force
        'act_type': Decision_IMPL
        'treaty': TFEU (2008)

        full_doc :

        1.10.2014 EN Official Journal of the European Union L 287/22 COUNCIL IMPLEMENTING DECISION of 25 September 2014 on subjecting 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl)phenethylamine (25I-NBOMe), 3,4-dichloro-N-[[1-(dimethylamino)cyclohexyl]methyl]benzamide (AH-7921), 3,4-methylenedioxypyrovalerone (MDPV) and 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine) to control measures (2014/688/EU) THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, Having regard to Council Decision 2005/387/JHA of 10 May 2005 on the information exchange, risk-assessment and control of new psychoactive substances (1), and in particular Article 8(3) thereof, Having regard to the proposal from the European Commission, Whereas: (1) Risk assessment reports on the new psychoactive substances 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl)phenethylamine (25I-NBOMe), 3,4-dichloro-N-[[1-(dimethylamino)cyclohexyl]methyl]benzamide (AH-7921), 3,4-methylenedioxypyrovalerone (MDPV) and 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine) were drawn up in compliance with Decision 2005/387/JHA by a special session of the extended Scientific Committee of the European Monitoring Centre for Drugs and Drug Addiction (EMCDDA), and were subsequently submitted to the Commission and to the Council on 23 April 2014. (2) 25I-NBOMe, AH-7921, MDPV and methoxetamine had not been under assessment at the United Nations' level by the time the risk assessment was requested at Union level, but they were evaluated in June 2014 by the Expert Committee on Drug Dependence of the World Health Organization. (3) 25I-NBOMe, AH-7921, MDPV and methoxetamine have no established or acknowledged medical use (human or veterinary). Apart from their use in analytical reference materials, and in scientific research investigating their chemistry, pharmacology and toxicology as a result of their emergence on the drug market  and, in the case of 25I-NBOMe, also in the field of neurochemistry  there is no indication that they are being used for other purposes. (4) 25I-NBOMe is a potent synthetic derivative of 2,5-dimethoxy-4-iodophenethylamine (2C-I), a classical serotonergic hallucinogen, which was subject to risk assessment and to control measures and criminal sanctions at Union level from 2003 by Council Decision 2003/847/JHA (2). (5) The specific physical effects of 25I-NBOMe are difficult to determine because there are no published studies assessing its acute and chronic toxicity, its psychological and behavioural effects, and dependence potential, and because of the limited information and data available. Clinical observations of individuals who have used this substance suggest that it has hallucinogenic effects and has the potential for inducing severe agitation, confusion, intense auditory and visual hallucinations, aggression, violent accidents and self-induced trauma. (6) There have been four deaths associated with 25I-NBOMe registered in three Member States. Severe toxicity associated with its use has been reported in four Member States, which notified 32 non-fatal intoxications. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. There is no information available on the social risks associated with 25I-NBOMe. (7) 22 Member States and Norway have reported to the EMCDDA and European Police Office (Europol) that they detected 25I-NBOMe. No prevalence data is available on the use of 25I-NBOMe, but the limited information that exists suggests that it may be consumed in a wide range of settings, such as at home, in bars, nightclubs and at music festivals. (8) 25I-NBOMe is openly marketed and sold on the internet as a research chemical and information from seizures, collected samples, user websites and internet retailers suggests that it is being sold as a drug in its own right and also marketed as a legal replacement for LSD. EMCDDA identified more than 15 internet retailers selling this substance, who may be based within the Union and China. (9) The risk assessment report reveals that there is limited scientific evidence available on 25I-NBOMe and points out that further research would be needed to determine the health and social risks that it poses. However, the available evidence and information provides sufficient ground for subjecting 25I-NBOMe to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it and of the lack of medical value or use of the substance, 25I-NBOMe should be subjected to control measures across the Union. (10) Since six Member States control 25I-NBOMe under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances, and seven Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles to cross-border law enforcement and judicial cooperation, and would help protect against the risks that its availability and use can pose. (11) AH-7921 is a structurally atypical synthetic opioid analgesic commonly known by internet suppliers, user websites and media as doxylam. It can be easily confused with doxylamine, an antihistaminic medicine with sedative-hypnotic properties, which could lead to unintentional overdoses. (12) The specific physical effects of AH-7921 are difficult to determine because there are no published studies assessing its acute and chronic toxicity, its psychological, behavioural effects, and dependence potential, as well as the limited information and data available. Based on user reports, the effects of AH-7921 appear to resemble those of classical opioids with the feeling of mild euphoria, itchiness and relaxation; nausea appears to be a typical adverse effect. In addition to self-experimentation with AH-7921, as well as recreational use, some of the users report self-medicating with this new drug to relieve pain, others to alleviate withdrawal symptoms due to cessation of the use of other opioids. This may indicate a potential of AH-7921 to spread among the injecting opioid population. (13) There is no prevalence data on the use of AH-7921, but the information available suggests that it is not widely used, and that when it is used, that use is in the home environment. (14) 15 fatalities were recorded in three Member States between December 2012 and September 2013 where AH-7921, alone or in combination with other substances, was detected in post-mortem samples. While it is not possible to determine with certainty the role of AH-7921 in all of those fatalities, in some cases it has been specifically noted in the cause of death. One Member State reported six non-fatal intoxications associated with AH-7921. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. There is no information available on the social risks associated with AH-7921. (15) The risk assessment report reveals that there is limited scientific evidence available on AH-7921 and points out that further research would be needed to determine the health and social risks that it poses. However, the available evidence and information provides sufficient ground for subjecting AH-7921 to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use of the substance, AH-7921 should be subjected to control measures across the Union. (16) Since one Member State controls AH-7921 under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and five Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would help protect against the risks that its availability and use can pose. (17) MDPV is a ring-substituted synthetic derivative of cathinone chemically related to pyrovalerone, which are both subject to control under the 1971 United Nations Convention on Psychotropic Substances. (18) Information on the chronic and acute toxicity associated with MDPV, as well as on psychological and behavioural effects, and on dependence potential, is not collected uniformly across the Union. Information from published studies, confirmed by clinical cases, suggests that the psychopharmacological profile observed for MDPV is similar to that for cocaine and methamphetamine, albeit more potent and longer lasting. Furthermore, MDPV was found to be 10 times more potent in its ability to induce locomotor activation, tachycardia and hypertension. (19) Users' websites indicate that its acute toxicity can provoke adverse effects on humans, similar to those associated with other stimulants. These include paranoid psychosis, tachycardia, hypertension, diaphoresis, breathing problems, severe agitation, auditory and visual hallucinations, profound anxiety, hyperthermia, violent outbursts and multiple organ dysfunctions. (20) 108 fatalities were registered in eight Member States and Norway between September 2009 and August 2013, where MDPV has been detected in post-mortem biological samples or implicated in the cause of death. A total of 525 non-fatal intoxications associated with MDPV have been reported by eight Member States. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. (21) The detection of MDPV has also been reported in biological samples related to fatal and non-fatal road traffic accidents, or driving under the influence of drugs, in four Member States since 2009. (22) MDPV has been present in the Union drug market since November 2008 and 27 Member States, Norway and Turkey reported multi-kilogram seizures of the substance. MDPV is being sold as a substance in its own right, but it has also been detected in combination with other substances. It is widely available from internet suppliers and retailers, head shops and street-level dealers. There are some indications that suggest a degree of organisation in the tableting and distribution of this substance in the Union. (23) The risk assessment report reveals that further research would be needed to determine the health and social risks posed by MDPV. However, the available evidence and information provides sufficient ground for subjecting MDPV to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use of the substance, MDPV should be subjected to control measures across the Union. (24) Since 21 Member States control MDPV under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and four Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would protect against the risks that its availability and use can pose. (25) Methoxetamine is an arylcyclohexylamine substance which is chemically similar to ketamine and the internationally controlled substance phencyclidine (PCP). Like ketamine and PCP, it has dissociative properties. (26) There are no studies assessing the chronic and acute toxicity associated with methoxetamine, as well as its psychological and behavioural effects, and dependence potential. Self-reported experiences from user websites suggest adverse effects similar to ketamine intoxication. These include nausea and severe vomiting, difficulty in breathing, seizures, disorientation, anxiety, catatonia, aggression, hallucination, paranoia and psychosis. In addition, acute methoxetamine intoxications may include stimulant effects (agitation, tachycardia and hypertension) and cerebral features, which are not expectable with acute ketamine intoxication. (27) Twenty deaths associated with methoxetamine were reported by six Member States that detected the substance in post-mortem samples. Used alone or in combination with other substances, methoxetamine was detected in 20 non-fatal intoxications reported by five Member States. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. (28) 23 Member States, Turkey and Norway have reported that they detected methoxetamine, since November 2010. Information suggests that it is sold and used as a substance in its own right, but it is also sold as a legal replacement for ketamine by internet retailers, head shops and street-level drug dealers. (29) Multi-kilogram quantities in powder form were seized within the Union, but there is no information on the possible involvement of organised crime. The manufacture of methoxetamine does not require sophisticated equipment. (30) Prevalence data are limited to non-representative studies in two Member States. Those studies suggest that the prevalence of the use of methoxetamine is lower than that of ketamine. The available information suggests that it may be consumed in a wide range of settings, including at home, in bars, nightclubs and at music festivals. (31) The risk assessment report reveals that further research would be needed to determine the health and social risks posed by methoxetamine. However, the available evidence and information provides sufficient grounds for subjecting methoxetamine to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use, methoxetamine should be subjected to control measures across the Union. (32) Since nine Member States control methoxetamine under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and nine Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would protect against the risks that its availability and use can pose. (33) Decision 2005/387/JHA reserves to the Council implementing powers with a view to giving a quick and expertise-based response at the Union level to the emergence of new psychoactive substances detected and reported by the Member States, by submitting those substances to control measures across the Union. As the conditions and procedure for triggering the exercise of such implementing powers have been met, an implementing decision should be adopted in order to put 25I-NBOMe, AH-7921, MDPV and methoxetamine under control across the Union, HAS ADOPTED THIS DECISION: Article 1 The following new psychoactive substances shall be subjected to control measures across the Union: (a) 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl) phenethylamine(25I-NBOMe); (b) 3,4-dichloro-N-[[1-dimethylamino) cyclohexyl]methyl] benzamide (AH-7921); (c) 3,4-methylenedioxypyrovalerone (MDPV); (d) 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine). Article 2 By 2 October 2015, Member States shall subject in accordance with their national legislation, the new psychoactive substances referred to in Article 1 to control measures and criminal penalties, as provided for under their legislation complying with their obligations under the 1971 United Nations Convention on Psychotropic Substances. Article 3 This Decision shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union. Done at Brussels, 25 September 2014. For the Council The President F. GUIDI (1) OJ L 127, 20.5.2005, p. 32. (2) Council Decision 2003/847/JHA of 27 November 2003 concerning control measures and criminal sanctions in respect of the new synthetic drugs 2C-I, 2C-T-2, 2C-T-7 and TMA-2 (OJ L 321, 6.12.2003, p. 64).
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32004F0757
        'status': In Force
        'act_type': Decision_FRAMW
        'treaty': TEU (1992)

        full_doc :

        11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Article 31(e) and Article 34(2)(b) thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the European Parliament (2), Whereas: (1) Illicit drug trafficking poses a threat to health, safety and the quality of life of citizens of the European Union, and to the legal economy, stability and security of the Member States. (2) The need for legislative action to tackle illicit drug trafficking has been recognised in particular in the Action Plan of the Council and the Commission on how best to implement the provisions of the Amsterdam Treaty on an area of freedom, security and justice (3), adopted by the Justice and Home Affairs Council in Vienna on 3 December 1998, the conclusions of the Tampere European Council of 15 and 16 October 1999, in particular point 48 thereof, the European Union's Drugs Strategy (2000-2004) endorsed by the Helsinki European Council from 10 to 12 December 1999 and the European Union's Action Plan on Drugs (2000-2004) endorsed by the European Council in Santa Maria da Feira on 19 and 20 June 2000. (3) It is necessary to adopt minimum rules relating to the constituent elements of the offences of illicit trafficking in drugs and precursors which will allow a common approach at European Union level to the fight against such trafficking. (4) By virtue of the principle of subsidiarity, European Union action should focus on the most serious types of drug offence. The exclusion of certain types of behaviour as regards personal consumption from the scope of this Framework Decision does not constitute a Council guideline on how Member States should deal with these other cases in their national legislation. (5) Penalties provided for by the Member States should be effective, proportionate and dissuasive, and include custodial sentences. To determine the level of penalties, factual elements such as the quantities and the type of drugs trafficked, and whether the offence was committed within the framework of a criminal organisation, should be taken into account. (6) Member States should be allowed to make provision for reducing the penalties when the offender has supplied the competent authorities with valuable information. (7) It is necessary to take measures to enable the confiscation of the proceeds of the offences referred to in this Framework Decision. (8) Measures should be taken to ensure that legal persons can be held liable for the criminal offences referred to by this Framework Decision which are committed for their benefit. (9) The effectiveness of the efforts made to tackle illicit drug trafficking depends essentially on the harmonisation of the national measures implementing this Framework Decision, HAS DECIDED AS FOLLOWS: Article 1 Definitions For the purposes of this Framework Decision: 1. drugs: shall mean any of the substances covered by the following United Nations Conventions: (a) the 1961 Single Convention on Narcotic Drugs (as amended by the 1972 Protocol); (b) the 1971 Vienna Convention on Psychotropic Substances. It shall also include the substances subject to controls under Joint Action 97/396/JHA of 16 June 1997 concerning the information exchange risk assessment and the control of new synthetic drugs (4); 2. precursors: shall mean any substance scheduled in the Community legislation giving effect to the obligations deriving from Article 12 of the United Nations Convention against Illicit Traffic in Narcotic Drugs and Psychotropic Substances of 20 December 1988; 3. legal person: shall mean any legal entity having such status under the applicable national law, except for States or other public bodies acting in the exercise of their sovereign rights and for public international organisations. Article 2 Crimes linked to trafficking in drugs and precursors 1. Each Member State shall take the necessary measures to ensure that the following intentional conduct when committed without right is punishable: (a) the production, manufacture, extraction, preparation, offering, offering for sale, distribution, sale, delivery on any terms whatsoever, brokerage, dispatch, dispatch in transit, transport, importation or exportation of drugs; (b) the cultivation of opium poppy, coca bush or cannabis plant; (c) the possession or purchase of drugs with a view to conducting one of the activities listed in (a); (d) the manufacture, transport or distribution of precursors, knowing that they are to be used in or for the illicit production or manufacture of drugs. 2. The conduct described in paragraph 1 shall not be included in the scope of this Framework Decision when it is committed by its perpetrators exclusively for their own personal consumption as defined by national law. Article 3 Incitement, aiding and abetting and attempt 1. Each Member State shall take the necessary measures to make incitement to commit, aiding and abetting or attempting one of the offences referred to in Article 2 a criminal offence. 2. A Member State may exempt from criminal liability the attempt to offer or prepare drugs referred to in Article 2(1)(a) and the attempt to possess drugs referred to in Article 2(1)(c). Article 4 Penalties 1. Each Member State shall take the measures necessary to ensure that the offences defined in Articles 2 and 3 are punishable by effective, proportionate and dissuasive criminal penalties. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2 are punishable by criminal penalties of a maximum of at least between one and three years of imprisonment. 2. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2(1)(a), (b) and (c) are punishable by criminal penalties of a maximum of at least between 5 and 10 years of imprisonment in each of the following circumstances: (a) the offence involves large quantities of drugs; (b) the offence either involves those drugs which cause the most harm to health, or has resulted in significant damage to the health of a number of persons. 3. Each Member State shall take the necessary measures to ensure that the offences referred to in paragraph 2 are punishable by criminal penalties of a maximum of at least 10 years of deprivation of liberty, where the offence was committed within the framework of a criminal organisation as defined in Joint Action 98/733/JHA of 21 December 1998 on making it a criminal offence to participate in a criminal organisation in the Member States of the European Union (5). 4. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2(1)(d) are punishable by criminal penalties of a maximum of at least between 5 and 10 years of deprivation of liberty, where the offence was committed within the framework of a criminal organisation as defined in Joint Action 98/733/JHA and the precursors are intended to be used in or for the production or manufacture of drugs under the circumstances referred to in paragraphs 2(a) or (b). 5. Without prejudice to the rights of victims and of other bona fide third parties, each Member State shall take the necessary measures to enable the confiscation of substances which are the object of offences referred to in Articles 2 and 3, instrumentalities used or intended to be used for these offences and proceeds from these offences or the confiscation of property the value of which corresponds to that of such proceeds, substances or instrumentalities. The terms confiscation, instrumentalities, proceeds and property shall have the same meaning as in Article 1 of the 1990 Council of Europe Convention on Laundering, Search, Seizure and Confiscation of the Proceeds from Crime. Article 5 Particular circumstances Notwithstanding Article 4, each Member State may take the necessary measures to ensure that the penalties referred to in Article 4 may be reduced if the offender: (a) renounces criminal activity relating to trafficking in drugs and precursors, and (b) provides the administrative or judicial authorities with information which they would not otherwise have been able to obtain, helping them to: (i) prevent or mitigate the effects of the offence, (ii) identify or bring to justice the other offenders, (iii) find evidence, or (iv) prevent further offences referred to in Articles 2 and 3. Article 6 Liability of legal persons 1. Each Member State shall take the necessary measures to ensure that legal persons can be held liable for any of the criminal offences referred to in Articles 2 and 3 committed for their benefit by any person, acting either individually or as a member of an organ of the legal person in question, who has a leading position within the legal person, based on one of the following: (a) a power of representation of the legal person; (b) an authority to take decisions on behalf of the legal person; (c) an authority to exercise control within the legal person. 2. Apart from the cases provided for in paragraph 1, each Member State shall take the necessary measures to ensure that legal persons can be held liable where the lack of supervision or control by a person referred to in paragraph 1 has made possible the commission of any of the offences referred to in Articles 2 and 3 for the benefit of that legal person by a person under its authority. 3. Liability of legal persons under paragraphs 1 and 2 shall not exclude criminal proceedings against natural persons who are perpetrators, instigators or accessories in any of the offences referred to in Articles 2 and 3. Article 7 Sanctions for legal persons 1. Member States shall take the necessary measures to ensure that a legal person held liable pursuant to Article 6(1) is punishable by effective, proportionate and dissuasive sanctions, which shall include criminal or non-criminal fines and may include other sanctions, such as: (a) exclusion from entitlement to tax relief or other benefits or public aid; (b) temporary or permanent disqualification from the pursuit of commercial activities; (c) placing under judicial supervision; (d) a judicial winding-up order; (e) temporary or permanent closure of establishments used for committing the offence; (f) in accordance with Article 4(5), the confiscation of substances which are the object of offences referred to in Articles 2 and 3, instrumentalities used or intended to be used for these offences and proceeds from these offences or the confiscation of property the value of which corresponds to that of such proceeds, substances or instrumentalities. 2. Each Member State shall take the necessary measures to ensure that a legal person held liable pursuant to Article 6(2) is punishable by effective, proportionate and dissuasive sanctions or measures. Article 8 Jurisdiction and prosecution 1. Each Member State shall take the necessary measures to establish its jurisdiction over the offences referred to in Articles 2 and 3 where: (a) the offence is committed in whole or in part within its territory; (b) the offender is one of its nationals; or (c) the offence is committed for the benefit of a legal person established in the territory of that Member State. 2. A Member State may decide that it will not apply, or that it will apply only in specific cases or circumstances, the jurisdiction rules set out in paragraphs 1(b) and 1(c) where the offence is committed outside its territory. 3. A Member State which, under its laws, does not extradite its own nationals shall take the necessary measures to establish its jurisdiction over and to prosecute, where appropriate, an offence referred to in Articles 2 and 3 when it is committed by one of its own nationals outside its territory. 4. Member States shall inform the General Secretariat of the Council and the Commission when they decide to apply paragraph 2, where appropriate with an indication of the specific cases or circumstances in which the decision applies. Article 9 Implementation and reports 1. Member States shall take the necessary measures to comply with the provisions of this Framework Decision by 12 May 2006. 2. By the deadline referred to in paragraph 1, Member States shall transmit to the General Secretariat of the Council and to the Commission the text of the provisions transposing into their national law the obligations imposed on them under this Framework Decision. The Commission shall, by 12 May 2009, submit a report to the European Parliament and to the Council on the functioning of the implementation of the Framework Decision, including its effects on judicial cooperation in the field of illicit drug trafficking. Following this report, the Council shall assess, at the latest within six months after submission of the report, whether Member States have taken the necessary measures to comply with this Framework Decision. Article 10 Territorial application This Framework Decision shall apply to Gibraltar. Article 11 Entry into force This Framework Decision shall enter into force on the day following its publication in the Official Journal of the European Union. Done at Luxembourg, 25 October 2004. For the Council The President R. VERDONK (1) OJ C 304 E, 30.10.2001, p. 172. (2) Opinion of 9 March 2004 (not yet published in the Official Journal). (3) OJ C 19, 23.1.1999, p. 1. (4) OJ L 167, 25.6.1997, p. 1. (5) OJ L 351, 29.12.1998, p. 1.
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32013R1382
        'status': In Force
        'act_type': Regulation
        'treaty': TFEU (2008)

        full_doc :

        28.12.2013 EN Official Journal of the European Union L 354/73 REGULATION (EU) No 1382/2013 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 17 December 2013 establishing a Justice Programme for the period 2014 to 2020 (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 81(1) and (2), Article 82(1) and Article 84 thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the Committee of the Regions (2), Acting in accordance with the ordinary legislative procedure (3), Whereas: (1) The Treaty on the Functioning of the European Union (TFEU) provides for the creation of an area of freedom, security and justice, in which persons are free to move. To that end, the Union may adopt measures to develop judicial cooperation in civil and criminal matters and to promote and support the action of Member States in the field of crime prevention. Respect for fundamental rights as well as for common principles, such as non-discrimination, gender equality, effective access to justice for all, the rule of law and a well-functioning independent judicial system should be ensured in the further development of a European area of justice. (2) In the Stockholm Programme (4) the European Council reaffirmed the priority of developing an area of freedom, security and justice and specified as a political priority the achievement of a Europe of law and justice. Financing was identified as one of the important tools for the successful implementation of the Stockholm Programme's political priorities. The ambitious goals set by the Treaties and by the Stockholm Programme should be attained inter alia by establishing, for the period 2014 to 2020, a flexible and effective Justice Programme (the "Programme") which should facilitate planning and implementation. The general and specific objectives of the Programme should be interpreted in line with the relevant strategic guidelines defined by the European Council. (3) The Commission Communication of 3 March 2010 on the Europe 2020 Strategy sets out a strategy for smart, sustainable and inclusive growth. A well-functioning area of justice, where obstacles in cross-border judicial proceedings and access to justice in cross-border situations are eliminated, should be developed as a key element to support the specific objectives and flagship initiatives of the Europe 2020 Strategy and to facilitate mechanisms designed to promote growth. (4) For the purposes of this Regulation, the term "judiciary and judicial staff" should be interpreted so as to include judges, prosecutors and court officers, as well as other legal practitioners associated with the judiciary, such as lawyers, notaries, bailiffs, probation officers, mediators and court interpreters. (5) Judicial training is central to building mutual trust and improves cooperation between judicial authorities and practitioners in the various Member States. Judicial training should be seen as an essential element in promoting a genuine European judicial culture in the context of the Commission Communication of 13 September 2011 entitled "Building trust in EU-wide justice. A new dimension to European judicial training", the Council Resolution on the training of judges, prosecutors and judicial staff in the European Union (5), the Council conclusions of 27 and 28 October 2011 on European judicial training and the European Parliament resolution of 14 March 2012 on judicial training. (6) Judicial training can involve different actors, such as Member States' legal, judicial and administrative authorities, academic institutions, national bodies responsible for judicial training, European-level training organisations or networks, or networks of court coordinators of Union law. Bodies and entities pursuing a general European interest in the field of training of the judiciary, such as the European Judicial Training Network (EJTN), the Academy of European Law (ERA), the European Network of Councils for the Judiciary (ENCJ), the Association of the Councils of State and Supreme Administrative Jurisdictions of the European Union (ACA-Europe), the Network of the Presidents of Supreme Judicial Courts of the European Union (RPCSJUE) and the European Institute of Public Administration (EIPA), should continue to play their role in promoting training programmes with a genuine European dimension for the judiciary and judicial staff, and could therefore be granted adequate financial support in accordance with the procedures and the criteria set out in the annual work programmes adopted by the Commission pursuant to this Regulation. (7) The Union should facilitate training activities on the implementation of Union law by considering the salaries of participating judiciary and judicial staff incurred by the Member States' authorities as eligible costs or co-financing in kind, in accordance with Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council (6) (the "Financial Regulation"). (8) Access to justice should include, in particular, access to courts, to alternative methods of dispute settlement and to public office-holders obliged by the law to provide parties with independent and impartial legal advice. (9) In December 2012 the Council endorsed the EU Drugs Strategy (2013-20) (7), which aims to take a balanced approach based on simultaneous reduction of drug demand and drug supply, acknowledging that drug demand reduction and drug supply reduction are mutually reinforcing elements in illicit drugs policy. That Strategy maintains as one of its main objectives the aim of contributing to a measurable reduction of drug demand, of drug dependence and of drug-related health and social risks and harms. Whereas the Drug prevention and information programme established by Decision No 1150/2007/EC of the European Parliament and of the Council (8) was based on a public health legal basis and covered those aspects, the Programme is founded on a different legal basis and should aim at the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation. Thus, in responding to the need for simplification and in line with the legal basis of each programme, the Health for Growth Programme can support measures to complement the Member's States action in attaining the objective of reducing drug-related health damage, including information and prevention. (10) Another important element of the EU Drugs Strategy (2013-20) is drug supply reduction. Whereas the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, should support actions aimed at preventing and combating the trafficking of drugs and other types of crime, and in particular measures targeting the production, manufacture, extraction, sale, transport, importation and exportation of illegal drugs, including possession and purchase with a view to engaging in drug trafficking activities, the Programme should cover those aspects of drugs policy that are not covered by the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, or by the Health for Growth Programme and are closely linked to its general objective. (11) In any case, the continued financing of the priorities under the 2007-2013 programming period that have been maintained as objectives under the new EU Drugs Strategy (2013-20) should be ensured, and funds should therefore be available from the Health for Growth Programme, the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, and the Programme in accordance with their respective priorities and legal bases while avoiding any duplicate financing. (12) Pursuant to Article 3(3) of the Treaty on European Union (TEU), Article 24 of the Charter of Fundamental Rights of the European Union (the "Charter") and the 1989 United Nations Convention on the Rights of the Child, the Programme should support the protection of the rights of the child, including the right to due process, the right to understand the proceedings, the right to respect for private and family life and the right to integrity and dignity. The Programme should aim, in particular, to increase child protection within justice systems and access to justice for children, and should mainstream the promotion of the rights of the child in the implementation of all of its actions. (13) Pursuant to Articles 8 and 10 TFEU, the Programme should support the mainstreaming of equality between women and men and non-discrimination objectives in all its activities. Regular monitoring and evaluation should be carried out to assess the way in which gender equality and non-discrimination issues are addressed in the Programme's activities. (14) Experience of action at Union level has shown that achieving the objectives of the Programme in practice calls for a combination of instruments, including legal acts, policy initiatives and funding. Funding is an important tool complementing legislative measures. (15) In its conclusions of 22 and 23 September 2011 on improving the efficiency of future Union financial programmes supporting judicial cooperation, the Council stressed the important role played by Union financing programmes in the efficient implementation of the Union acquis and reiterated the need for more transparent, flexible, coherent and streamlined access to those programmes. (16) The Commission Communication of 29 June 2011 entitled 'A budget for Europe 2020' stresses the need for the rationalisation and simplification of Union funding. Especially in view of the current economic crisis, it is of the utmost importance that Union funds be structured and managed in the most diligent manner. Meaningful simplification and efficient management of funding can be achieved through a reduction in the number of programmes and through the rationalisation, simplification and harmonisation of funding rules and procedures. (17) In responding to the need for simplification, efficient management and easier access to funding, the Programme should continue and develop activities previously carried out on the basis of three programmes established by Council Decision 2007/126/JHA (9), Decision No 1149/2007/EC of the European Parliament and of the Council (10), and Decision No 1150/2007/EC. The mid-term evaluations of those programmes include recommendations aimed at improving the implementation of those programmes. The findings of those mid-term evaluations, as well as the findings of the respective ex-post evaluations, need to be taken into account in the implementation of the Programme. (18) The Commission Communication of 19 October 2010 entitled 'The EU Budget Review' and the Commission Communication of 29 June 2011 entitled 'A budget for Europe 2020' underline the importance of focusing funding on activities with clear European added value, i.e. where Union intervention can bring additional value compared to the action of Member States alone. Actions covered by this Regulation should contribute to the creation of a European area of justice by promoting the principle of mutual recognition, developing mutual trust between the Member States, increasing cross-border cooperation and networking and achieving the correct, coherent and consistent application of Union law. Funding activities should also contribute to achieving effective and better knowledge of Union law and policies by all concerned, and should provide a sound analytical basis for the support and the development of Union law and policies, in so doing contributing to their enforcement and proper implementation. Union intervention allows for those actions to be pursued consistently across the Union and brings economies of scale. Moreover, the Union is in a better position than Member States to address cross-border situations and to provide a European platform for mutual learning. (19) In selecting actions for funding under the Programme, the Commission should assess the proposals against pre-identified criteria. Those criteria should include an assessment of the European added value of the proposed actions. National projects and small-scale projects can also have European added value. (20) Bodies and entities that have access to the Programme should include national, regional and local authorities. (21) This Regulation lays down a financial envelope for the entire duration of the Programme which is to constitute the prime reference amount, within the meaning of point 17 of the Interinstitutional Agreement of 2 December 2013 between the European Parliament, the Council and the Commission on budgetary discipline, on cooperation in budgetary matters and on sound financial management (11), for the European Parliament and the Council during the annual budgetary procedure. (22) In order to ensure that the Programme is sufficiently flexible to respond to changing needs and corresponding policy priorities throughout its duration, the power to adopt acts in accordance with Article 290 TFEU should be delegated to the Commission concerning modification of the percentages set out in the Annex to this Regulation for each specific objective that would exceed those percentages by more than 5 percentage points. To assess the need for such a delegated act, those percentages should be calculated on the basis of the financial envelope of the Programme for its entire duration, and not on the basis of annual appropriations. It is of particular importance that the Commission carry out appropriate consultations during its preparatory work, including at expert level. The Commission, when preparing and drawing up delegated acts, should ensure a simultaneous, timely and appropriate transmission of relevant documents to the European Parliament and to the Council. (23) This Regulation should be implemented in full compliance with the Financial Regulation. In particular with regard to the eligibility conditions of value added tax (VAT) paid by grant beneficiaries, the eligibility of VAT should not depend on the legal status of the beneficiaries for activities which can be carried out by private and public bodies and entities under the same legal conditions. Taking into account the specific nature of the objectives and activities covered by this Regulation, it should be made clear, in calls for proposals, that, for activities which can be carried out by both public and private bodies and entities, the non-deductible VAT incurred by public bodies and entities is to be eligible, in so far as it is paid in respect of the implementation of activities, such as training or awareness-raising, which cannot be considered as the exercise of public authority. This Regulation should also make use of the simplification tools introduced by the Financial Regulation. Moreover, the criteria for identifying actions to be supported should aim at allocating the available financial resources to actions generating the highest impact in relation to the policy objective pursued. (24) In order to ensure uniform conditions for the implementation of this Regulation, implementing powers should be conferred on the Commission in respect of the adoption of annual work programmes. Those powers should be exercised in accordance with Regulation (EU) No 182/2011 of the European Parliament and of the Council (12). (25) The annual work programmes adopted by the Commission pursuant to this Regulation should ensure appropriate distribution of funds between grants and public procurement contracts. The Programme should primarily allocate funds to grants, while maintaining sufficient funding levels for procurement. The minimum percentage of annual expenditure to be allocated to grants should be established in the annual work programmes and should be not less than 65 %. To facilitate project planning and co-financing by stakeholders, the Commission should establish a clear timetable for the calls for proposals, selection of projects and award decisions. (26) In order to ensure efficient allocation of funds from the general budget of the Union, consistency, complementarity and synergies should be sought between funding programmes supporting policy areas with close links to each other, in particular between the Programme and the Rights, Equality and Citizenship Programme established by Regulation (EU) No 1381/2013 of the European Parliament and of the Council (13), the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, the Health for Growth Programme, the Erasmus+ Programme established by Regulation (EU) No 1288/2013 of the European Parliament and of the Council (14), the Horizon 2020 Framework Programme established by Regulation (EU) No 1291/2013 of the European Parliament and of the Council (15) and the Instrument for Pre-accession Assistance (IPA II). (27) The financial interests of the Union should be protected through proportionate measures throughout the expenditure cycle, including the prevention, detection and investigation of irregularities, the recovery of funds lost, wrongly paid or incorrectly used and, where appropriate, the imposition of administrative and financial penalties in accordance with the Financial Regulation. (28) In order to implement the principle of sound financial management, this Regulation should provide for appropriate tools to assess its performance. To that end, it should define general and specific objectives. To measure the achievement of those specific objectives, a set of concrete and quantifiable indicators should be established which should remain valid for the whole duration of the Programme. The Commission should submit annually to the European Parliament and to the Council a monitoring report which should be based inter alia on the indicators set out in this Regulation and which should give information on the use of available funds. (29) The Programme should be implemented in an effective manner, respecting sound financial management, while also allowing potential applicants to have effective access to the Programme. In order to support effective access to the Programme, the Commission should use its best endeavours to simplify and harmonise the application procedures and documents, the administrative formalities and the financial management requirements, to remove administrative burdens and to encourage grant applications from entities located in Member States which are under-represented in the Programme. The Commission should publish on a dedicated webpage information about the Programme, its objectives, the various calls for proposals and their time schedules. Basic documents and guidelines relating to the calls for proposals should be available in all the official languages of the institutions of the Union. (30) In accordance with point (l) of Article 180(1) of Commission Delegated Regulation (EU) No 1268/2012 (16) ('the Rules of Application'), the grant agreements should lay down provisions governing the visibility of the Union financial support, except in duly justified cases where public display is not possible or appropriate. (31) In accordance with Article 35(2) and (3) of the Financial Regulation and Article 21 of the Rules of Application, the Commission should make available, in an appropriate and timely manner, information concerning recipients and concerning the nature and purpose of the measures financed from the general budget of the Union. That information should be made available with due observance of the requirements of confidentiality and security, in particular the protection of personal data. (32) Since the objective of this Regulation, namely to contribute to the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation in civil and criminal matters, cannot be sufficiently achieved by the Member States but can rather, by reason of its scale and effects, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 TEU. In accordance with the principle of proportionality, as set out in that Article, this Regulation does not go beyond what is necessary in order to achieve that objective. (33) In accordance with Article 3 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the Area of Freedom, Security and Justice, annexed to the TEU and to the TFEU, Ireland has notified its wish to take part in the adoption and application of this Regulation. (34) In accordance with Articles 1 and 2 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the Area of Freedom, Security and Justice, annexed to the TEU and to the TFEU, and without prejudice to Article 4 of that Protocol, the United Kingdom is not taking part in the adoption of this Regulation and is not bound by it or subject to its application. (35) In accordance with Articles 1 and 2 of Protocol No 22 on the position of Denmark, annexed to the TEU and to the TFEU, Denmark is not taking part in the adoption of this Regulation and is not bound by it or subject to its application. (36) In order to ensure the continuity of funding of activities previously carried out on the basis of Decision 2007/126/JHA, Decision No 1149/2007/EC and Decision No 1150/2007/EC, this Regulation should enter into force on the day following that of its publication, HAVE ADOPTED THIS REGULATION: Article 1 Establishment and duration of the Programme 1. This Regulation establishes a Justice programme ('the Programme'). 2. The Programme shall cover the period from 1 January 2014 to 31 December 2020. Article 2 European added value 1. The Programme shall finance actions with European added value which contribute to the further development of a European area of justice. To that end, the Commission shall ensure that the actions selected for funding are intended to produce results with European added value. 2. The European added value of actions, including that of small-scale and national actions, shall be assessed in the light of criteria such as their contribution to the consistent and coherent implementation of Union law and to wide public awareness about the rights deriving from it, their potential to develop mutual trust among Member States and to improve cross-border cooperation, their transnational impact, their contribution to the elaboration and dissemination of best practices or their potential to create practical tools and solutions that address cross-border or Union-wide challenges. Article 3 General objective The general objective of the Programme shall be to contribute to the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation in civil and criminal matters. Article 4 Specific objectives 1. To achieve the general objective set out in Article 3, the Programme shall have the following specific objectives: (a) to facilitate and support judicial cooperation in civil and criminal matters; (b) to support and promote judicial training, including language training on legal terminology, with a view to fostering a common legal and judicial culture; (c) to facilitate effective access to justice for all, including to promote and support the rights of victims of crime, while respecting the rights of the defence; (d) to support initiatives in the field of drugs policy as regards judicial cooperation and crime prevention aspects closely linked to the general objective of the Programme, in so far as they are not covered by the Internal security fund for financial support for police cooperation, preventing and combating crime, and crisis management or by the Health for Growth Programme; 2. The specific objectives of the Programme shall be pursued through, in particular: (a) enhancing public awareness and knowledge of Union law and policies; (b) with a view to ensuring efficient judicial cooperation in civil and criminal matters, improving knowledge of Union law, including substantive and procedural law, of judicial cooperation instruments and of the relevant case-law of the Court of Justice of the European Union, and of comparative law; (c) supporting the effective, comprehensive and consistent implementation and application of Union instruments in the Member States and the monitoring and evaluation thereof; (d) promoting cross-border cooperation, improving mutual knowledge and understanding of the civil and criminal law and the legal and judicial systems of the Member States and enhancing mutual trust; (e) improving knowledge and understanding of potential obstacles to the smooth functioning of a European area of justice; (f) improving the efficiency of judicial systems and their cooperation by means of information and communication technology, including the cross-border interoperability of systems and applications. Article 5 Mainstreaming In the implementation of all of its actions, the Programme shall seek to promote equality between women and men and to promote the rights of the child, inter alia by means of child-friendly justice. It shall also comply with the prohibition of discrimination based on any of the grounds listed in Article 21 of the Charter, in accordance with and within the limits set by Article 51 of the Charter. Article 6 Types of actions 1. The Programme shall finance inter alia the following types of actions: (a) analytical activities, such as the collection of data and statistics; the development of common methodologies and, where appropriate, indicators or benchmarks; studies, researches, analyses and surveys; evaluations; the elaboration and publication of guides, reports and educational material; workshops, seminars, experts meetings and conferences; (b) training activities, such as staff exchanges, workshops, seminars, train-the-trainer events, including language training on legal terminology, and the development of online training tools or other training modules for members of the judiciary and judicial staff; (c) mutual learning, cooperation, awareness-raising and dissemination activities, such as the identification of, and exchanges concerning, good practices, innovative approaches and experiences; the organisation of peer reviews and mutual learning; the organisation of conferences, seminars, information campaigns, including institutional communication on the political priorities of the Union as far as they relate to the objectives of the Programme; the compilation and publication of materials to disseminate information about the Programme and its results; the development, operation and maintenance of systems and tools, using information and communication technologies, including the further development of the European e-Justice portal as a tool to improve citizens' access to justice; (d) support for main actors whose activities contribute to the implementation of the objectives of the Programme, such as support for Member States in the implementation of Union law and policies, support for key European actors and European-level networks, including in the field of judicial training; and support for networking activities at European level among specialised bodies and entities as well as national, regional and local authorities and non-governmental organisations. 2. The European Judicial Training Network shall receive an operating grant to co-finance expenditure associated with its permanent work programme. Article 7 Participation 1. Access to the Programme shall be open to all bodies and entities legally established in: (a) Member States; (b) European Free Trade Association (EFTA) countries which are parties to the Agreement on the European Economic Area, in accordance with that Agreement; (c) candidate countries, potential candidates and countries acceding to the Union, in accordance with the general principles and the general terms and conditions laid down for the participation of those countries in the Union programmes established in the respective Framework Agreements and Association Council decisions, or similar agreements. 2. Bodies and entities which are profit-oriented shall have access to the Programme only in conjunction with non-profit or public organisations. 3. Bodies and entities legally established in third countries, other than those participating in the Programme in accordance with points (b) and (c) of paragraph 1, in particular countries where the European Neighbourhood Policy applies, may be associated to the actions of the Programme at their own cost, if this serves the purpose of those actions. 4. The Commission may cooperate with international organisations under the conditions laid down in the relevant annual work programme. Access to the Programme shall be open to international organisations active in the areas covered by the Programme in accordance with the Financial Regulation and the relevant annual work programme. Article 8 Budget 1. The financial envelope for the implementation of the Programme for the period 2014 to 2020 is set at EUR 377 604 000. 2. The financial allocation of the Programme may also cover expenses pertaining to preparatory, monitoring, control, audit and evaluation activities which are required for the management of the Programme and the assessment of the achievement of its objectives. The financial allocation may cover expenses relating to the necessary studies, meetings of experts, information and communication actions, including institutional communication of the political priorities of the Union, in so far as they are related to the general objectives of this Regulation, as well as expenses linked to information technology networks focusing on information processing and exchange and other technical and administrative assistance needed in connection with the management of the Programme by the Commission. 3. The annual appropriations shall be authorised by the European Parliament and the Council within the limits of the multiannual financial framework established by Council Regulation (EU, Euratom) No 1311/2013 (17). 4. Within the financial envelope for the Programme, amounts shall be allocated to each specific objective in accordance with the percentages set out in the Annex. 5. The Commission shall not depart from the allocated percentages of the financial envelope, as set out in the Annex, by more than 5 percentage points for each specific objective. Should it prove necessary to exceed that limit, the Commission shall be empowered to adopt delegated acts in accordance with Article 9 to modify each of the figures in the Annex by more than 5 and up to 10 percentage points. Article 9 Exercise of the delegation 1. The power to adopt delegated acts is conferred on the Commission subject to the conditions laid down in this Article. 2. The power to adopt delegated acts referred to in Article 8(5) shall be conferred on the Commission for the duration of the Programme. 3. The delegation of power referred to in Article 8(5) may be revoked at any time by the European Parliament or by the Council. A decision to revoke shall put an end to the delegation of the power specified in that decision. It shall take effect the day following the publication of the decision in the Official Journal of the European Union or at a later date specified therein. It shall not affect the validity of any delegated acts already in force. 4. As soon as it adopts a delegated act, the Commission shall notify it simultaneously to the European Parliament and to the Council. 5. A delegated act adopted pursuant to Article 8(5) shall enter into force only if no objection has been expressed either by the European Parliament or the Council within a period of two months of notification of that act to the European Parliament and the Council or if, before the expiry of that period, the European Parliament and the Council have both informed the Commission that they will not object. That period shall be extended by two months at the initiative of the European Parliament or the Council. Article 10 Implementing measures 1. The Commission shall implement the Programme in accordance with the Financial Regulation. 2. In order to implement the Programme, the Commission shall adopt annual work programmes in the form of implementing acts. Those implementing acts shall be adopted in accordance with the examination procedure referred to in Article 11(2). 3. Each annual work programme shall implement the objectives of the Programme by determining the following: (a) the actions to be undertaken, in accordance with the general and specific objectives set out in Article 3 and Article 4(1), including the indicative allocation of financial resources; (b) the essential eligibility, selection and award criteria to be used to select the proposals which are to receive financial contributions, in accordance with Article 84 of the Financial Regulation and with Article 94 of its Rules of Application; (c) the minimum percentage of annual expenditure to be allocated to grants. 4. Appropriate and fair distribution of financial support between different areas covered by this Regulation shall be ensured. When deciding on the allocation of funds to those areas in the annual work programmes, the Commission shall take into consideration the need to maintain sufficient funding levels for both civil justice and criminal justice, as well as for judicial training and initiatives in the field of drugs policy within the scope of the Programme. 5. Calls for proposals shall be published on an annual basis. 6. In order to facilitate judicial training activities, the costs associated with the participation of judiciary and judicial staff in those activities and incurred by the Member States' authorities shall be taken into account in accordance with the Financial Regulation when providing corresponding funding. Article 11 Committee procedure 1. The Commission shall be assisted by a committee. That committee shall be a committee within the meaning of Regulation (EU) No 182/2011. 2. Where reference is made to this paragraph, Article 5 of Regulation (EU) No 182/2011 shall apply. Article 12 Complementarity 1. The Commission, in cooperation with the Member States, shall ensure overall consistency, complementarity and synergies with other Union instruments including, inter alia, the Rights, Equality and Citizenship Programme, the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, the Health for Growth Programme, the Erasmus+ Programme, the Horizon 2020 Framework Programme and the Instrument for Pre-accession Assistance (IPA II). 2. The Commission shall also ensure overall consistency, complementarity and synergies with the work of the Union bodies, offices and agencies operating in areas covered by the objectives of the Programme, such as Eurojust established by Council Decision 2002/187/JHA (18) and the European Monitoring Centre for Drugs and Drug Addiction (EMCDDA) established by Regulation (EC) No 1920/2006 of the European Parliament and of the Council (19). 3. The Programme may share resources with other Union instruments, in particular the Rights, Equality and Citizenship Programme, in order to implement actions meeting the objectives of both programmes. An action for which funding has been awarded from the Programme may also give rise to the award of funding from the Rights, Equality and Citizenship Programme, provided that the funding does not cover the same cost items. Article 13 Protection of the financial interests of the Union 1. The Commission shall take appropriate measures ensuring that, when actions financed under the Programme are implemented, the financial interests of the Union are protected by the application of preventive measures against fraud, corruption and any other illegal activities, by effective checks and, if irregularities are detected, by the recovery of amounts wrongly paid and, where appropriate, by effective, proportionate and dissuasive administrative and financial penalties. 2. The Commission or its representatives and the Court of Auditors shall have the power of audit, both on the basis of documents and on the spot, over all grant beneficiaries, contractors and subcontractors who have received Union funds under the Programme. 3. The European Anti-Fraud Office (OLAF) may carry out investigations, including on-the-spot checks and inspections, in accordance with the provisions and procedures laid down in Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council (20) and in Council Regulation (Euratom, EC) No 2185/96 (21) with a view to establishing whether fraud, corruption or any other illegal activity has occurred affecting the financial interests of the Union in connection with a grant agreement or grant decision or a contract funded under the Programme. 4. Without prejudice to paragraphs 1, 2 and 3, cooperation agreements with third countries and with international organisations, grant agreements, grant decisions and contracts resulting from the implementation of the Programme shall contain provisions expressly empowering the Commission, the Court of Auditors and OLAF to conduct the audits and investigations referred to in those paragraphs, in accordance with their respective competences. Article 14 Monitoring and evaluation 1. The Commission shall monitor the Programme annually in order to follow the implementation of actions carried out under it and the achievement of the specific objectives set out in Article 4. The monitoring shall also provide a means of assessing the way in which gender equality and non-discrimination issues have been addressed across the Programme's actions. 2. The Commission shall provide the European Parliament and the Council with: (a) an annual monitoring report based on the indicators set out in Article 15(2) and on the use of the available funds; (b) an interim evaluation report by 30 June 2018; (c) an ex-post evaluation report by 31 December 2021. 3. The interim evaluation report shall assess the achievement of the Programme's objectives, the efficiency of the use of resources and the Programme's European added value with a view to determining whether funding in areas covered by the Programme should be renewed, modified or suspended after 2020. It shall also address the scope for any simplification of the Programme, its internal and external coherence, and the continued relevance of all objectives and actions. It shall take into account the results of the ex-post evaluations of the previous 2007-2013 programmes established by the Decisions referred to in Article 16. 4. The ex-post evaluation report shall assess the long-term impact of the Programme and the sustainability of the effects of the Programme, with a view to informing a decision on a subsequent programme. 5. The evaluations shall also assess the way in which gender equality and non-discrimination issues have been addressed across the Programme's actions. Article 15 Indicators 1. In accordance with Article 14, the indicators set out in paragraph 2 of this Article shall serve as a basis for monitoring and evaluating the extent to which each of the Programme's specific objectives set out in Article 4 has been achieved through the actions provided for in Article 6. They shall be measured against pre-defined baselines reflecting the situation before implementation. Where relevant, indicators shall be broken down by, inter alia, sex, age and disability. 2. The indicators referred to in paragraph 1 shall include, inter alia, the following: (a) the number and percentage of persons in a target group reached by awareness-raising activities funded by the Programme; (b) the number and percentage of members of the judiciary and judicial staff in a target group that participated in training activities, staff exchanges, study visits, workshops and seminars funded by the Programme; (c) the improvement in the level of knowledge of Union law and policies in the groups participating in activities funded by the Programme compared to the entire target group; (d) the number of cases, activities and outputs of cross-border cooperation, including cooperation by means of information technology tools and procedures established at Union level; (e) participants' assessment of the activities in which they participated and of their (expected) sustainability; (f) the geographical coverage of the activities funded by the Programme. 3. In addition to the indicators set out in paragraph 2, the interim and ex-post evaluation report of the Programme shall assess, inter alia: (a) the perceived impact of the Programme on access to justice based on qualitative and quantitative data collected at European level; (b) the number and quality of instruments and tools developed through actions funded by the Programme; (c) the European added value of the Programme, including an evaluation of the Programme's activities in the light of similar initiatives which have been developed at national or European level without support from Union funding, and their (expected) results and the advantages and/or disadvantages of Union funding compared to national funding for the type of activity in question; (d) the level of funding in relation to the outcomes achieved (efficiency); (e) the possible administrative, organisational and/or structural obstacles to the smoother, more effective and efficient implementation of the Programme (scope for simplification). Article 16 Transitional measures Actions initiated on the basis of Decision 2007/126/JHA, Decision 1149/2007/EC or Decision 1150/2007/EC shall continue to be governed by the provisions of those Decisions until their completion. In respect of those actions, reference to the committees provided for in Article 9 of Decision 2007/126/JHA, in Articles 10 and 11 of Decision 1149/2007/EC and in Article 10 of Decision 1150/2007/EC shall be interpreted as references to the committee provided for in Article 11(1) of this Regulation. Article 17 Entry into force This Regulation shall enter into force on the day following that of its publication in the Official Journal of the European Union. This Regulation shall be binding in its entirety and directly applicable in the Member States in accordance with the Treaties. Done at Brussels, 17 December 2013. For the European Parliament The President M. SCHULZ For the Council The President L. LINKEVIÃ IUS (1) OJ C 299, 4.10.2012, p. 103. (2) OJ C 277, 13.9.2012, p. 43. (3) Position of the European Parliament of 11 December 2013 (not yet published in the Official Journal) and decision of the Council of 16 December. (4) OJ C 115, 4.5.2010, p. 1. (5) OJ C 299, 22.11.2008, p. 1. (6) Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council of 25 October 2012 on the financial rules applicable to the general budget of the Union and repealing Council Regulation (EC, Euratom) No 1605/2002 (OJ L 298, 26.10.2012, p. 1). (7) OJ C 402, 29.12.2012, p. 1. (8) Decision No 1150/2007/EC of the European Parliament and of the Council of 25 September 2007 establishing for the period 2007-2013 the Specific Programme 'Drug prevention and information' as part of the General Programme 'Fundamental Rights and Justice' (OJ L 257, 3.10.2007, p. 23). (9) Council Decision 2007/126/JHA of 12 February 2007 establishing for the period 2007-2013, as part of the General Programme on Fundamental Rights and Justice, the Specific Programme 'Criminal Justice' (OJ L 58, 24.2.2007, p. 13). (10) Decision No 1149/2007/EC of the European Parliament and of the Council of 25 September 2007 establishing for the period 2007-2013 the Specific Programme 'Civil Justice' as part of the General Programme 'Fundamental Rights and Justice (OJ L 257, 3.10.2007, p. 16). (11) OJ C 373, 20.12.2013, p. 1. (12) Regulation (EU) No 182/2011 of the European Parliament and of the Council of 16 February 2011 laying down the rules and general principles concerning mechanisms for control by Member States of the Commission's exercise of implementing powers (OJ L 55, 28.2.2011, p. 13). (13) Regulation (EU) No 1381/2013 of the European Parliament and of the Council of 17 December 2013 establishing a Rights, Equality and Citizenship Programme for the period 2014 to 2020 (see page 62 of this Official Journal). (14) Regulation (EU) No 1288/2013 of the European Parliament and of the Council of 11 December 2013 establishing "Erasmus+": the Union programme for education, training, youth and sport and repealing Decisions No 1719/2006/EC, No 1720/2006/EC and No 1298/2008/EC (OJ L 347, 20.12.2013, p. 50). (15) Regulation (EU) No 1291/2013 of the European Parliament and of the Council of 11 December 2013 establishing Horizon 2020 - the Framework Programme for Research and Innovation (2014-2020) and repealing Decision No 1982/2006/EC (OJ L 347, 20.12.2013, p. 104). (16) Commission Delegated Regulation (EU) No 1268/2012 of 29 October 2012 on the rules of application of Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council on the financial rules applicable to the general budget of the Union (OJ L 362, 31.12.2012, p. 1). (17) Council Regulation (EU, Euratom) No 1311/2013 of 2 December 2013 laying down the multiannual financial framework for the years 2014-2020 (OJ L 347, 20.12.2013, p. 884). (18) Council Decision 2002/187/JHA of 28 February 2002 setting up Eurojust with a view to reinforcing the fight against serious crime (OJ L 63, 6.3.2002, p. 1). (19) Regulation (EC) No 1920/2006 of the European Parliament and of the Council of 12 December 2006 on the European Monitoring Centre for Drugs and Drug Addiction (OJ L 376, 27.12.2006, p. 1). (20) Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council of 11 September 2013 concerning investigations conducted by the European Anti-Fraud Office (OLAF) and repealing Regulation (EC) No 1073/1999 of the European Parliament and of the Council and Council Regulation (Euratom) No 1074/1999 (OJ L 248, 18.9.2013, p. 1). (21) Council Regulation (Euratom, EC) No 2185/96 of 11 November 1996 concerning on-the-spot checks and inspections carried out by the Commission in order to protect the European Communities' financial interests against fraud and other irregularities (OJ L 292, 15.11.1996, p. 2). ANNEX ALLOCATION OF FUNDS Within the financial envelope for the Programme, amounts shall be allocated as follows to each specific objective set out in Article 4(1): Specific objectives Share of the financial envelope (in %) (a) to facilitate and support judicial cooperation in civil and criminal matters 30 % (b) to support and promote judicial training, including language training on legal terminology, with a view to fostering a common legal and judicial culture 35 % (c) to facilitate effective access to justice for all, including to promote and support the rights of victims of crime, while respecting the rights of the defence 30 % (d) to support initiatives in the field of drugs policy as regards judicial cooperation and crime prevention aspects closely linked to the general objective of the Programme, in so far as they are not covered by the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, or by the Health for Growth Programme 5 %.
============================== "END OF DOC" ==============================
        

In [6]:
import pandas as pd

In [7]:
laws = pd.read_csv('dataset/act_raw_text_with_4meta.csv')

In [17]:
laws.columns


Index(['Unnamed: 0', 'CELEX', 'Status', 'Act_type', 'Treaty', 'act_raw_text'], dtype='str')

In [8]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.embed_query(enhanced_query)

    return query_embedding

In [9]:
query_embedding = get__embeddings("driving without license penalty")

In [10]:
weaviate_client.is_live()

True

In [28]:
eu = weaviate_client.collections.use("Euro_Laws")
response = eu.query.near_vector(
    near_vector= query_embedding, 
    limit=5
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=10))  # Inspect the results

KeyboardInterrupt: 

In [11]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("driving without license penalty")
docs

ValueError: Error during query: Query call with protocol GRPC search failed with message Deadline Exceeded.

```
'celex': , 'status': 

'act_type' , 'treaty':

```

In [15]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(i.metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids

get_celex_ids(docs)  

['32015L0413', '32006L0126', '32009R1072', '31980L1263']

In [22]:
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]

'In Force'

In [18]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):

        full_doc_info += f"""

        doc {i} :

        'celex': {laws[laws['CELEX'] == celex_id]['CELEX'].iloc[0]}
        'status': {laws[laws['CELEX'] == celex_id]['Status'].iloc[0]}
        'act_type': {laws[laws['CELEX'] == celex_id]['Act_type'].iloc[0]}
        'treaty': {laws[laws['CELEX'] == celex_id]['Treaty'].iloc[0]}

        full_doc :

        {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]}
{"=="*15} "END OF DOC" {"=="*15}
        """

    return full_doc_info    



In [ ]:
display(Markdown(search_docs("driving without license penalty"))) 

 

        doc 0 :

        'celex': 31980L1263
        'status': Not in Force
        'act_type': Directive
        'treaty': TEEC

        full_doc :

        Avis juridique important|31980L1263First Council Directive 80/1263/EEC of 4 December 1980 on the introduction of a Community driving licence Official Journal L 375 , 31/12/1980 P. 0001 - 0015 Finnish special edition: Chapter 7 Volume 2 P. 0171 Greek special edition: Chapter 13 Volume 10 P. 0089 Swedish special edition: Chapter 7 Volume 2 P. 0171 Spanish special edition: Chapter 07 Volume 2 P. 0259 Portuguese special edition Chapter 07 Volume 2 P. 0259 FIRST COUNCIL DIRECTIVE of 4 December 1980 on the introduction of a Community driving licence (80/1263/EEC) THE COUNCIL OF THE EUROPEAN COMMUNITIES, Having regard to the Treaty establishing the European Economic Community, and in particular Article 75 (1) (c) thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Parliament (1), Having regard to the opinion of the Economic and Social Committee (2), Whereas, for the purposes of the common transport policy, as a contribution to improving road traffic safety, and to assist the movement of persons settling in a Member State other than that in which they have passed a driving test, or moving within the Community, it is desirable that a Community driving licence be introduced; Whereas the introduction of a Community driving licence presupposes the harmonization of existing national driving test arrangements, which can only be achieved gradually ; whereas the first stage of this harmonization could culminate in the establishment of a Community model national licence and the mutual recognition by Member States of national driving licences and the exchange of licences by holders transferring their place of residence or place of employment from one Member State to another; Whereas the Community model national licence should be based on that defined by the Final Act of the Convention on Road Traffic drawn up in Vienna in November 1968 by the United Nations Road Traffic Conference; Whereas the mutual recognition of driving licences issued by the different Member States and the exchange of a licence by a holder moving from one Community country to reside or work in another will only be possible further to an initial harmonization of the regulations governing the issue and validity of licences; Whereas, without prejudice to the final provisions to be adopted by the Council on vehicle categories, it is necessary to establish common standards in respect of the validity of the licence for driving the different categories of vehicles, so that the Community model licence can be issued throughout the Community under comparable conditions; Whereas, however, at this initial harmonization stage and pending the introduction of the final system, Member States should be allowed to lay down the conditions with regard to age and the period of validity of licences arid also, under certain specific conditions, to derogate from the categories, speeds and conditions of validity laid down by this (1)OJ No C 238, 11.10.1976, p. 43. (2)OJ No C 197, 23.8.1976, p. 32. Directive ; and, where appropriate, to check the additional conditions laid down for the exchange of driving licences of certain categories of vehicles; Whereas it is desirable that the standards for testing drivers and issuing licences should be further harmonized as soon as possible, HAS ADOPTED THIS DIRECTIVE: Article 1 The Member States shall introduce a national driving licence based on the Community model provided for in Article 2. A Community model driving licence shall, subject to Article 8, entitle the holder to drive, both on national and international journeys, vehicles of the categories for which it has been granted. Community model driving licences shall be issued by the Member States in accordance with this Directive. Article 2 The driving licence provided for in Article 1 shall conform to the model in Annex I. The oval on page 1 of the model shall contain the distinguishing sign of the State issuing the licence. After consulting the Commission, Member States may adapt the model in the Annex in any way necessary to enable them to: - process the driving licence by computer, - enter in the licence any categories of vehicle which, pursuant to Article 9, differ from those provided for in Article 3. Member States shall take all necessary steps to avoid any risk of forgery of driving licences. Article 3 1. Without prejudice to the final provisions to be adopted by the Council concerning vehicle categories, the driving licence provided for in Article I shall authorize the driving on public roads of vehicles in the following categories: category A : motorcycles with or without side-car; category B : motor vehicles, other than those in category A, with a permissible maximum weight not exceeding 3 500 kg and not more than eight seats in addition to the driver's seat; category C : motor vehicles used for the carriage of goods and whose permissible maximum weight exceeds 3 500 kg; category D : motor vehicles used for the carriage of passengers, with more than eight seats in addition to the driver's seat: category E : combinations of vehicles of which the tractor vehicle is in a category or categories for which the driver is licensed (B and/or C and/or D), but which are not themselves in that category or categories. 2. For the purposes of paragraph 1: (a) a trailer with a permissible maximum weight not exceeding 750 kg may be coupled to a motor vehicle in category B above ; a trailer with a permissible maximum weight exceeding 750 kg may likewise be coupled to such vehicle, provided that the following two conditions are fulfilled: - the permissible maximum weight of the trailer does not exceed the unladen weight of the motor vehicle, and - the total permissible maximum weight of the combination of vehicles does not exceed 3 500 kg; (b) a motor vehicle in category C or D may be coupled to a trailer the permissible maximum weight of which does not exceed 750 kg. 3. For the purposes of this Article: - "motorcycle" means any two or three-wheeled vehicle with a maximum design speed exceeding 50 kph (33 mph) or, if it is powered by a heat propulsion engine, with a cylinder capacity exceeding 50 cc. In addition, in the case of a three-wheeled vehicle, the unladen weight shall not exceed 400 kg; - "power-driven vehicle" means any self-propelled vehicle running on a road, other than a railborne vehicle; - "motor vehicle" means any power-driven vehicle, other than a motorcycle, which is normally used for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage of persons or goods. This term shall include trolley buses, i.e. vehicles connected to an electric conductor and not rail borne. It shall not include agricultural or forestry tractors; - "agricultural or forestry tractor" means any power-driven vehicle running on wheels or tracks, having at least two axles, the principal function of which lies in its tractive power, which is specially designed to pull, push, carry or operate certain tools, machines or trailers used in connection with agricultural or forestry operations, and the use of which for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage or persons or goods is only a secondary function. Article 4 1. The validity of the driving licence provided for in Article 1 shall be determined as follows: (a) licences granted for categories C and D shall also be valid for the driving of vehicles in category B; (b) licences granted for category E shall, without prejudice to the provisions of (c), be valid for the driving of combinations of vehicles; (c) licences for category E shall be granted only to drivers already entitled to drive vehicles in category B, C or D. 2. Licences issued to disabled drivers shall specifically mention the conditions under which such drivers are entitled to drive. Article 5 1. Without prejudice to Article 5 of Council Regulation (EEC) No 543/69 of 25 March 1969 on the harmonization of certain social legislation relating to road transport (1), Member States shall fix the minimum age at which driving licences may be issued. 2. Member States may refuse to recognize the validity on their territory of driving licences issued to drivers under the age of 18 years. Article 6 1. A driving licence shall, moreover, be issued only to those applicants: (a) who have passed a practical and theoretical test and who meet medical standards, the minimum requirements of which may not be substantially less stringent than those set out in Annexes II and III; (b) who have their normal residence in the territory of the Member State issuing the licence, if the legislation of the Member State concerned so requires. 2. Member States may apply to the issue of driving licences the provisions of their national legislation relating thereto which are concerned with conditions other than those referred to in paragraph 1. Article 7 Without prejudice to the provisions which may be adopted by the Council in this regard, each Member State shall retain the right to fix, on the basis of national criteria, the period of validity of the driving licences (Community model) which it issues or exchanges pursuant to Article 8. Article 8 1. The Member States shall provide that, if the holder of a valid national driving licence or valid Community model licence issued by a Member State takes up normal residence in another Member State his licence shall remain valid there for up to a maximum of a year following the taking up of residence At the request of the holder within that period, and against surrender of his licence, the State in which he has taken up normal residence shall issue him with a driving licence (Community model) for the corresponding category or categories without subjecting him to the conditions laid down in Article 6. However, that Member State may refuse to exchange the licence if its national regulations, including medical standards, preclude the issue of the licence. The exchange must be preceded by the submission of a statement by the applicant to the effect that his (1)OJ No L 77, 29.3.1969, p. 49. driving licence is currently valid. It shall be for the Member State effecting the exchange to check the veracity of his statement if necessary. The Member State effecting the exchange shall return the old licence to the authorities of the Member State which issued it. 2. Member States which, pursuant to Article 9, do not apply categories C, D and E as defined in Article 3 (1) may: - exchange category C, D and E driving licences in accordance with paragraph 1 of this Article or, - require the applicant to furnish proof of driving experience and in this case issue a licence entitling him to drive vehicles in the national category in respect of which he furnished proof of adequate experience, or vehicles in a lower category. In any event, such States shall issue to the applicant at least a licence to drive vehicles in the lowest of the national categories corresponding to categories C, D and E as defined in Article 3 (1). During the year following the taking up of residence by drivers who have not applied for a licence exchange, such States shall recognize such drivers' licences as being equivalent at least to licences for the lowest relevant national category. 3. Where a Member State exchanges a licence, issued by a third country, for a Community model driving licence, such exchange shall be recorded in the licence, as shall any subsequent renewal or replacement of that licence. In the event of subsequent exchange of the said licence, Member States shall not be obliged to apply paragraph 1. A Community model driving licence may in any event be issued only if the licence issued by the third country has been surrendered to the competent authorities of the Member State issuing the Community licence. Article 9 After consulting the Commission, Member States may, pending introduction of the final system and provided that the fact is recorded on the licence, derogate from: - the categories defined in Article 3 (1); - the speeds indicated in the first indent of Article 3 (3), provided that the speeds which they prescribe are lower; - the conditions of validity provided for in Article 4. Furthermore, Member States shall, pursuant to the procedure laid down in Article 12, establish equivalent definitions in so far as their national categories differ. Article 10 The Council, acting on a proposal from the Commission, shall carry out as soon as possible a more detailed harmonization of the standards for driving tests and licensing with a view to inter alia subsequent improvements in road safety throughout the Community. Article 11 The Member States shall determine the arrangements for replacing currently valid national driving licences issued by them with Community model driving licences for the corresponding category or categories. This operation shall take place without the need for the tests provided for under Article 6, on submission of and in exchange for the old licences. Article 12 1. After consulting the Commission, Member States shall, in good time and at the latest by 30 June 1982, adopt the laws, regulations or administrative provisions necessary for the implementation of this Directive from 1 January 1983. 2. However, a Member State may, without prejudice to the application of the other provisions in this Directive, decide not to issue Community model driving licences until a later date, which may not be later than 1 January 1986. 3. Member States shall assist one another in the implementation of this Directive. Article 13 This Directive is addressed to the Member States. Done at Brussels, 4 December 1980. For the Council The President J. BARTHEL ANNEX I >PIC FILE= "T0014238">Comments on the model driving licence shown on page 1 1. The colour of the Community driving licence shall be pink. 2. On the cover page: - mention of the name of the Member State issuing the licence shall be optional, - the distinguishing sign of the Member State issuing the licence shall be entered in the oval, - the words "driving licence" shall be printed in large type in the language or languages of the Member State issuing the licence. They shall appear, alter a suitable space, in small type in the other languages of the European Communities, - the words "European Communities model" shall be printed in the language or languages of the Member State issuing the licence. 3. The printed entries on the other pages shall be in the language or languages of the Member State issuing the licence. 4. The page entitled "Additional information" is designed for details of any restriction or extension of the conditions governing the validity of the licence. This page may also be used for showing the period of validity of the licence where this varies. >PIC FILE= "T0014239"> 5. Other comments may be entered on the remaining blank pages. Where appropriate, Member States may enter on them categories of vehicles not covered by this Directive or may subdivide categories A, B, C, D and E in the corresponding page. 6. Member States shall have the right to: - dispense with the photograph requirement; - replace the permanent place of residence by the postal address; - delete the date of issue and indicate the date of commencement of validity of the licence. SPECIMEN COMMUNITY MODEL LICENCE : BELGIAN LICENCE (FOR INFORMATION) >PIC FILE= "T0014240"> ANNEX II MINIMUM REQUIREMENTS FOR DRIVING TESTS THEORETICAL TEST Form 1. The form chosen shall be such as to establish whether the candidate has the required knowledge and understanding of the subjects listed in paragraphs 2 and 3 of this Annex. Content 2. Knowledge and understanding of the regulations, and more especially of the rules applicable to the use of vehicles of the category corresponding to the type of licence applied for: 2.1. Knowledge and understanding of traffic rules and regulations, signs, signals and road markings and of their meaning; 2.2. Basic knowledge and understanding of the technical regulations relating to vehicle safety in traffic; 2.3. Knowledge and understanding of rules relating to the driver, in so far as they concern road safety, including, for drivers of category C and D vehicles only, rules relating to hours of work and rest periods; 2.4. Knowledge and understanding of the rules on what a driver should do in the event of an accident. 3. Knowledge and understanding of other subjects: 3.1. Adequate knowledge and understanding of the importance of road safety matters, and especially of the following accident factors: 3.1.1. Driving hazards, such as the danger of overtaking, misjudgement of speed (effects on braking and safety distances), influence of the weather (snow, rain, fog, side-winds, aquaplaning), behaviour of other road users, and in particular of elderly people and children; 3.1.2. Factors likely to reduce the driver's vigilance and his physical and mental fitness to drive, such as fatigue, illness, alcohol and other drugs, etc.; 3.1.3. Safety factors relating to vehicle loading and to passengers carried. 3.2. Category A and B vehicles only : basic knowledge of those items of the vehicle which are vital to the protection of its occupants and to road safety, such as brakes, tyres, oil levels, safety belts, etc.; Category C, D and E vehicles only : knowledge of the function and simple maintenance of the items mentioned above and of all other vehicle parts and devices of particular importance to safety; 3.3. Knowledge of the action which may be required in order to assist road accident victims. PRACTICAL TEST The vehicle and its equipment 4. If a candidate takes the test on a vehicle with automatic transmission, this shall be recorded on any licence issued on the basis of such a test; - Category C vehicles : the permissible maximum weight shall be not less than 7 000 kg; - Category D vehicles : the vehicle shall have not less than 28 seats and shall be not less than 7 m in length; - Category E vehicles : when the towing vehicle belongs to category C, and except in the case of a semi-trailer, the trailer shall have at least two axles, the distance between which shall be greater than 1 m. Contents 5. The principal manoeuvres to be carried out to check the candidate's ability to control the vehicle are as follows: 5.1. Starting on upgrades; 5.2. Category B, C, D and E vehicles only : reversing and reverse turning: 5.3. Braking and stopping at various speeds, including emergency stops if road and traffic conditions permit; 5.4. Category B, C, D and E vehicles only : oblique parking, parking on upgrades and down-grades; 5.5. Turning in a restricted space; 5.6. Category A vehicles only : riding at a slow speed. 6. Behaviour in traffic The main checks to which the candidate will be subjected are: 6.1. Correct positioning on the carriageway; 6.2. Proper negotiation of right and left-hand bends; 6.3. Correct execution of the manoeuvres of changing lanes and turning off at junctions; 6.4. Alertness to other traffic; 6.5. Correct behaviour at intersections, taking due account of all movements of other road users, with special regard to right-of-way; 6.6. Driving at appropriate speeds; 6.7. Use of rear-view mirrors; 6.8. Correct signalling of intended manoeuvres; 6.9. Correct operation of vehicle lighting and warning devices and other ancillary controls; 6.10. Driving with due care and consideration for pedestrians and other road users; 6.11. Correct behaviour with regard to public transport vehicles; 6.12. Compliance with traffic-light signals and instructions given by authorized officials on point duty; 6.13. Appropriate reaction to legally specified signals given by other road users; 6.14. Observance of traffic signs and signals, road markings and pedestrian crossings; 6.15. Observance of appropriate following and lateral distances; 6.16. Correct overtaking; 6.17 Correct use of safety belts if national legislation requires that they be fitted to the vehicle. Sequence of the parts of the test 7. Whenever possible, the part of the test described in paragraph 5 should be carried out before the part described in paragraph 6. Duration of the test 8. The duration of the test and the distance covered shall be sufficient for the checks prescribed in paragraphs 5 and 6 to be carried out. The duration of the part of the test described in paragraph 6 should be more than 30 minutes, but shall not in any case be less than 20 minutes. Location of the test 9. The part of the test described in paragraph 5 may be conducted on a special testing ground, in which case precise criteria should be laid down for measuring objectively the candidate's ability to handle the vehicle. The part of the test described in paragraph 6 shall, wherever possible, be conducted on roads outside built-up areas and on motorways as well as in urban traffic. ANNEX III MINIMUM STANDARDS OF PHYSICAL AND MENTAL FITNESS DEFINITIONS 1. For the purpose of this Annex, drivers are classified into two groups: 1.1. Group 1 : drivers of vehicles of categories A and B; 1.2. Group 2 : drivers of vehicles of categories C, D and E. 2. Similarly, applicants for a first driving licence or for the renewal of a driving licence are classified in the group to which they will belong once the licence has been granted or renewed. MEDICAL EXAMINATIONS 3. Group 1 : applicants shall be required to undergo a medical examination if it becomes apparent, when the necessary formalities are being completed or during the tests which they have to undergo prior to obtaining a driving licence, that they have one or more of the medical disabilities mentioned in this Annex in respect of this group. 4. Group 2 : applicants shall undergo a medical examination before a driving licence is first granted to them and thereafter drivers shall undergo such periodic examinations as may be prescribed by national laws. Eyesight 5. An examination conducted by suitably trained personnel shall be undergone by all applicants for a driving licence. In doubtful cases the applicant shall be referred to a competent medical authority. At the medical examination, attention should be paid to visual acuity, field of vision, night vision, progressive eye diseases, etc. When the wearing of corrective lenses is recognized by the issuing authority as necessary for driving, this shall be recorded on the driving licence. 6. Group 1 : drivers in this group should have their eyesight tested not later than at the age of 70 and preferably earlier, and thereafter at appropriate intervals. If applicants or drivers aged 40 years or more have sub-normal vision after correction but nevertheless meet the minimum requirements given in paragraphs 6.1 and 6.2 below, the cause of loss of vision shall be investigated before driving licences are granted or renewed. Where a disease of the eye is discovered or suspected, the periodic tests should be frequent. 6.1. Applicants for a driving licence or for the renewal of such a licence shall have a visual acuity, with corrective lenses if necessary, of at least 0 74, and preferably of a higher standard in the better eye or of at least 0 75 in both eyes together and, on medical examination, of at least 0 72 in the worse eye. Driving licences shall not be granted or renewed if, on examination, it is shown that there is more than 20 º loss in the temporal part of the applicant's or the driver's field of vision, or if the applicant or driver has diplopia or defective binocular vision. 6.2. Applicants or drivers with sight only in one eye may obtain a driving licence or the renewal of such a licence if the monocular vision is certified by a competent medical authority as having existed for sufficient time to allow adaptation and the visual acuity, with corrective lenses if necessary, is at least 0 78. Such persons must have unrestricted field of vision in their good eye. 7. Group 2 : applicants or drivers in this group shall have their eyesight tested on application for a driving licence and preferably periodically thereafter. If applicants or drivers aged 40 years or more have sub-normal vision after correction but nevertheless meet the minimum requirements given in paragraph 7.1 below, the cause of visual loss shall be investigated before driving licences are granted or renewed. 7.1. Applicants for a driving licence or for the renewal of such a licence must have binocular vision with a visual acuity, with corrective lenses if necessary, of at least 0 775 in the better eye and of at least 0 75 in the worse eye. If corrective lenses are used, the uncorrected vision must not be less than 0 71 and the correction must be tolerated. Driving licences shall not be granted or renewed if the applicant or driver has a restricted field of vision or if he has diplopia or defective binocular vision. 7.2. The use of contact lenses by drivers in this group may be permitted if approved by a competent medical authority. Hearing 8. Driving licences shall not be granted or renewed for applicants or drivers in group 2 if their hearing is so bad that it interferes with the proper discharge of their duties. General physique and physical disabilities 9. Group 1 : unrestricted driving licences shall not be granted or renewed for physically disabled applicants or drivers, unless a driving test has established their ability to operate vehicles with conventional controls. 9.1. Restricted driving licences may be granted or renewed for physically disabled applicants or drivers if the vehicles they drive are adapted to suit the requirements of their disablement. Any restriction on the driving licence shall state the adaptation required on the vehicle. 9.2. In cases of doubt, a practical test shall be made of driving abilities after medical examination by a competent authority and, where appropriate, a driving licence for a limited duration may be issued so as to keep a case under observation. The assessment of physical disablement shall primarily be based on mechanical considerations which make it possible to ascertain whether the disablement is likely to interfere for prolonged periods with efficient and rapid manoeuvring and the handling of controls under all driving conditions, especially in an emergency. 10. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who have any disablement which is likely to prevent proper and safe control of a vehicle. 10.1. Medical examination of applicants or drivers shall cover the full range of body movements - strength, control and coordination - and, in particular, movements of the upper and lower limbs. 10.2. If disablement which is likely to hinder proper and safe control of a vehicle occurs after a driving licence has been granted, the disabled person must give up driving and undergo an examination by a competent medical authority. Cardiovascular diseases 11. Driving licences shall not be granted or renewed for applicants or drivers with cardiovascular diseases unless their request is supported by authorized medical opinion. 12. With regard to applicants or drivers in group 2, the competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. Endocrine disorders 13. In cases of severe endocrine disorders other than diabetes, appropriate provisions in respect of the granting or renewal of driving licences shall be established by the laws of the Member States. 14. Group 1 : driving licences shall not be granted or renewed for applicants or drivers suffering from diabetes who are affected by ocular, nervous or cardiovascular complications or uncompensated acidosis. 14.1. Driving licences may be granted or renewed for a restricted period for applicants or drivers suffering from diabetes who are not affected by any of the complications mentioned in paragraph 14 above, subject to their remaining under authorized medical supervision. 15. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who are diabetics needing insulin treatment. Diseases of the nervous system 16. Driving licences shall not be granted or renewed for applicants or drivers suffering from (a) encephalitis, multiple sclerosis, myasthenia gravis or hereditary diseases of the nervous system associated with progressive muscular atrophy and congenital myotonic disorders; (b) diseases of the peripheral nervous system ; or (c) trauma of the central or peripheral nervous system, unless their application is supported by authorized medical opinion and they are able to handle the controls of a vehicle safely and to comply with traffic regulations. Such cases shall be reviewed at regular intervals. 17. Group 1 : driving licences shall not be granted or renewed for applicants or drivers suffering from epilepsy. National legislation may provide that, subject to authorized medical opinion, licences be granted to persons who have suffered from epilepsy in the past but who have been free from attacks for a long time (e.g. two years). 17.1. Driving licences shall not be granted or renewed for applicants or drivers suffering from cerebrovascular diseases, unless their application is supported by authorized medical opinion and provided that, where necessary, the controls of the vehicle they drive are suitably re-arranged or modified, or that suitable special types of vehicles are used. The duration of the validity of driving licences granted or renewed in such cases shall be limited in accordance with authorized medical opinion. 17.2. Driving licences shall not be granted or renewed for applicants or drivers who have suffered a lesion with damage to the spinal cord and resultant paraplegia unless the vehicle they drive is fitted with special controls. 18. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who suffer or have suffered in the past from epilepsy, a cerebrovascular disease or a lesion with damage to the spinal cord and resulting paraplegia. Mental disorders 19. Driving licences shall not be granted or renewed for applicants or drivers who: (a) suffer from mental disturbance due to disease or trauma of, or operations upon, the central nervous system; (b) suffer from severe mental retardation; (c) suffer from psychosis, which in particular has caused general paralysis ; or (d) suffer from psychoneurosis or personality disorders. unless their application is supported by authorized medical opinion. 20. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Alcohol 21. Driving licences shall not be granted or renewed for applicants or drivers who suffer from chronic alcoholism. If the application is supported by an authorized medical opinion, driving licences may be granted or renewed for a limited period for applicants or drivers who suffered from chronic alcoholism in the past. Such cases shall be reviewed at regular intervals. 22. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Drugs and medicaments 23. Drug abuse : driving licences shall not be granted or renewed for applicants or drivers who are dependant on psycho-active drugs. 24. Drugs or medicaments taken on a regular basis : driving licences shall not be granted or renewed for applicants or drivers who regularly take drugs or medicaments which can hamper the ability to drive safely, unless their application is supported by authorized medical opinion. 24.1. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Diseases of the blood 25. Driving licences shall not be granted or renewed for applicants or drivers suffering from serious diseases of the blood unless the application is supported by authorized medical opinion. Diseases of the genito-urinary system 26. Driving licences shall not be granted or renewed for applicants or drivers suffering from severe renal deficiency. WITHDRAWAL OF DRIVING LICENCES 27. National laws shall include provisions to the effect that, subject to authorized medical opinion, a driving licence shall be withdrawn where the authorities concerned have become aware that the holder's state of health is such that his application for a licence or for its renewal would have been refused. OTHER PROVISIONS (i) The provisions of the Annex shall not prevent a Member State from providing that a driver who has obtained a driving licence before 1 January 1983 under less stringent conditions than those provided for herein may have this licence regularly renewed under the conditions pertaining when he obtained it. (ii) Member States may derogate from the provisions of the Annex where the development of medical science makes such derogations fully compatible with the standards laid down herein. These derogations shall apply only to applicants who have undergone a medical examination and whose application is supported by authorized medical opinion.
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32009R1072
        'status': In Force
        'act_type': Regulation
        'treaty': TEC (1992)

        full_doc :

        14.11.2009 EN Official Journal of the European Union L 300/72 REGULATION (EC) No 1072/2009 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 21 October 2009 on common rules for access to the international road haulage market (recast) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 71 thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the procedure laid down in Article 251 of the Treaty (2), Whereas: (1) A number of substantial changes are to be made to Council Regulation (EEC) No 881/92 of 26 March 1992 on access to the market in the carriage of goods by road within the Community to or from the territory of a Member State or passing across the territory of one or more Member States (3), to Council Regulation (EEC) No 3118/93 of 25 October 1993 laying down the conditions under which non-resident carriers may operate national road haulage services within a Member State (4), and to Directive 2006/94/EC of the European Parliament and of the Council of 12 December 2006 on the establishment of common rules for certain types of carriage of goods by road (5). In the interests of clarity and simplification, those legal acts should be recast and incorporated into one single regulation. (2) The establishment of a common transport policy entails, inter alia, laying down common rules applicable to access to the market in the international carriage of goods by road within the territory of the Community, as well as laying down the conditions under which non-resident hauliers may operate transport services within a Member State. Those rules must be laid down in such a way as to contribute to the smooth operation of the internal transport market. (3) To ensure a coherent framework for international road haulage throughout the Community, this Regulation should apply to all international carriage on Community territory. Carriage from Member States to third countries is still largely covered by bilateral agreements between the Member States and those third countries. Therefore, this Regulation should not apply to that part of the journey within the territory of the Member State of loading or unloading as long as the necessary agreements between the Community and the third countries concerned have not been concluded. It should, however, apply to the territory of a Member State crossed in transit. (4) The establishment of a common transport policy implies the removal of all restrictions against the person providing transport services on the grounds of nationality or the fact that he is established in a different Member State from the one in which the services are to be provided. (5) In order to achieve this smoothly and flexibly, provision should be made for a transitional cabotage regime as long as harmonisation of the road haulage market has not yet been completed. (6) The gradual completion of the single European market should lead to the elimination of restrictions on access to the domestic markets of Member States. Nevertheless, this should take into account the effectiveness of controls and the evolution of employment conditions in the profession, the harmonisation of the rules in the fields of, inter alia, enforcement and road user charges, and social and safety legislation. The Commission should closely monitor the market situation as well as the harmonisation mentioned above and propose, if appropriate, the further opening of domestic road transport markets, including cabotage. (7) Under Directive 2006/94/EC, a certain number of types of carriage are exempt from Community authorisation and from any other carriage authorisation. Within the framework of the organisation of the market provided for by this Regulation, a system of exemption from the Community licence and from any other carriage authorisation should be maintained for some of those types of carriage, because of their special nature. (8) Under Directive 2006/94/EC, the carriage of goods with vehicles of a maximum laden weight of between 3,5 tonnes and 6 tonnes was exempt from the requirement for a Community licence. Community rules in the field of road transport of goods, however, apply in general to vehicles with a maximum laden mass of more than 3,5 tonnes. Thus, the provisions of this Regulation should be aligned with the general scope of application of Community road transport rules and should only provide for an exemption for vehicles with a maximum laden mass of up to 3,5 tonnes. (9) The international carriage of goods by road should be conditional on the possession of a Community licence. Hauliers should be required to carry a certified true copy of the Community licence aboard each of their vehicles in order to facilitate effective controls by enforcement authorities, especially those outside the Member State in which the haulier is established. To this end, it is necessary to lay down more detailed specifications as regards the layout and other features of the Community licence and the certified copies. (10) Roadside checks should be carried out without direct or indirect discrimination on grounds of the nationality of the road transport operator or the country of establishment of the road transport operator or of registration of the vehicle. (11) The conditions governing the issue and withdrawal of Community licences and the types of carriage to which they apply, their periods of validity and the detailed rules for their use should be determined. (12) A driver attestation should also be established in order to allow Member States to check effectively whether drivers from third countries are lawfully employed or at the disposal of the haulier responsible for a given transport operation. (13) Hauliers who are holders of Community licences provided for in this Regulation and hauliers authorised to operate certain categories of international haulage service should be permitted to carry out national transport services within a Member State on a temporary basis in conformity with this Regulation, without having a registered office or other establishment therein. When such cabotage operations are performed, they should be subject to Community legislation such as Regulation (EC) No 561/2006 of the European Parliament and of the Council of 15 March 2006 on the harmonisation of certain social legislation relating to road transport (6) and to national law in force in specified areas in the host Member State. (14) Provisions should be adopted to allow action to be taken in the event of serious disturbance of the transport markets affected. For that purpose it is necessary to introduce a suitable decision-making procedure and for the required statistical data to be collected. (15) Without prejudice to the provisions of the Treaty on the right of establishment, cabotage operations consist of the provision of services by hauliers within a Member State in which they are not established and should not be prohibited as long as they are not carried out in a way that creates a permanent or continuous activity within that Member State. To assist the enforcement of this requirement, the frequency of cabotage operations and the period in which they can be performed should be more clearly defined. In the past, such national transport services were permitted on a temporary basis. In practice, it has been difficult to ascertain which services are permitted. Clear and easily enforceable rules are thus needed. (16) This Regulation is without prejudice to the provisions concerning the incoming or outgoing carriage of goods by road as one leg of a combined transport journey as laid down in Council Directive 92/106/EEC of 7 December 1992 on the establishment of common rules for certain types of combined transport of goods between Member States (7). National journeys by road within a host Member State which are not part of a combined transport operation as laid down in Directive 92/106/EEC fall within the definition of cabotage operations and should accordingly be subject to the requirements of this Regulation. (17) The provisions of Directive 96/71/EC of the European Parliament and of the Council of 16 December 1996 concerning the posting of workers in the framework of the provision of services (8) apply to transport undertakings performing a cabotage operation. (18) In order to perform efficient controls of cabotage operations, the enforcement authorities of the host Member States should, at least, have access to data from consignment notes and from recording equipment, in accordance with Council Regulation (EEC) No 3821/85 of 20 December 1985 on recording equipment in road transport (9). (19) Member States should grant each other mutual assistance with a view to the sound application of this Regulation. (20) Administrative formalities should be reduced as far as possible without abandoning the controls and penalties that guarantee the correct application and effective enforcement of this Regulation. To this end, the existing rules on the withdrawal of the Community licence should be clarified and strengthened. The current rules should be adapted to allow the effective sanctioning of serious infringements committed in a host Member State. Penalties should be non-discriminatory and proportionate to the seriousness of the infringements. It should be possible to lodge an appeal in respect of any penalties imposed. (21) Member States should enter in their national electronic register of road transport undertakings all serious infringements committed by hauliers which have led to the imposition of a penalty. (22) In order to facilitate and strengthen the exchange of information between national authorities, Member States should exchange the relevant information through the national contact points set up pursuant to Regulation (EC) No 1071/2009 of the European Parliament and of the Council of 21 October 2009 establishing common rules concerning the conditions to be complied with to pursue the occupation of road transport operator (10). (23) The measures necessary for the implementation of this Regulation should be adopted in accordance with Council Decision 1999/468/EC of 28 June 1999 laying down the procedures for the exercise of implementing powers conferred on the Commission (11). (24) In particular, the Commission should be empowered to adapt Annexes I, II and III to this Regulation to technical progress. Since those measures are of general scope and are designed to amend non-essential elements of this Regulation, they must be adopted in accordance with the regulatory procedure with scrutiny provided for in Article 5a of Decision 1999/468/EC. (25) Member States should take the necessary measures to implement this Regulation, in particular as regards effective, proportionate and dissuasive penalties. (26) Since the objective of this Regulation, namely to ensure a coherent framework for international road haulage throughout the Community, cannot be sufficiently achieved by the Member States and can therefore, by reason of its scale and effects, be better achieved at Community level, the Community may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty. In accordance with the principle of proportionality, as set out in that Article, this Regulation does not go beyond what is necessary in order to achieve that objective, HAVE ADOPTED THIS REGULATION: CHAPTER I GENERAL PROVISIONS Article 1 Scope 1. This Regulation shall apply to the international carriage of goods by road for hire or reward for journeys carried out within the territory of the Community. 2. In the event of carriage from a Member State to a third country and vice versa, this Regulation shall apply to the part of the journey on the territory of any Member State crossed in transit. It shall not apply to that part of the journey on the territory of the Member State of loading or unloading, as long as the necessary agreement between the Community and the third country concerned has not been concluded. 3. Pending the conclusion of the agreements referred to in paragraph 2, this Regulation shall not affect: (a) provisions relating to the carriage from a Member State to a third country and vice versa included in bilateral agreements concluded by Member States with those third countries; (b) provisions relating to the carriage from a Member State to a third country and vice versa included in bilateral agreements concluded between Member States which, under either bilateral authorisations or liberalisation arrangements, allow loading and unloading in a Member State by hauliers not established in that Member State. 4. This Regulation shall apply to the national carriage of goods by road undertaken on a temporary basis by a non-resident haulier as provided for in Chapter III. 5. The following types of carriage and unladen journeys made in conjunction with such carriage shall not require a Community licence and shall be exempt from any carriage authorisation: (a) carriage of mail as a universal service; (b) carriage of vehicles which have suffered damage or breakdown; (c) carriage of goods in motor vehicles the permissible laden mass of which, including that of trailers, does not exceed 3,5 tonnes; (d) carriage of goods in motor vehicles provided the following conditions are fulfilled: (i) the goods carried are the property of the undertaking or have been sold, bought, let out on hire or hired, produced, extracted, processed or repaired by the undertaking; (ii) the purpose of the journey is to carry the goods to or from the undertaking or to move them, either inside or outside the undertaking for its own requirements; (iii) motor vehicles used for such carriage are driven by personnel employed by, or put at the disposal of, the undertaking under a contractual obligation; (iv) the vehicles carrying the goods are owned by the undertaking, have been bought by it on deferred terms or have been hired provided that in the latter case they meet the conditions of Directive 2006/1/EC of the European Parliament and of the Council of 18 January 2006 on the use of vehicles hired without drivers for the carriage of goods by road (12); and (v) such carriage is no more than ancillary to the overall activities of the undertaking; (e) carriage of medicinal products, appliances, equipment and other articles required for medical care in emergency relief, in particular for natural disasters. Point (d)(iv) of the first subparagraph shall not apply to the use of a replacement vehicle during a short breakdown of the vehicle normally used. 6. The provisions of paragraph 5 shall not affect the conditions under which a Member State authorises its nationals to engage in the activities referred to in that paragraph. Article 2 Definitions For the purposes of this Regulation: 1. vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods; 2. international carriage means: (a) a laden journey undertaken by a vehicle the point of departure and the point of arrival of which are in two different Member States, with or without transit through one or more Member States or third countries; (b) a laden journey undertaken by a vehicle from a Member State to a third country or vice versa, with or without transit through one or more Member States or third countries; (c) a laden journey undertaken by a vehicle between third countries, with transit through the territory of one or more Member States; or (d) an unladen journey in conjunction with the carriage referred to in points (a), (b) and (c); 3. host Member State means a Member State in which a haulier operates other than the hauliers Member State of establishment; 4. non-resident haulier means a road haulage undertaking which operates in a host Member State; 5. driver means any person who drives the vehicle even for a short period, or who is carried in a vehicle as part of his duties to be available for driving if necessary; 6. cabotage operations means national carriage for hire or reward carried out on a temporary basis in a host Member State, in conformity with this Regulation; 7. serious infringement of Community road transport legislation means an infringement which may lead to the loss of good repute in accordance with Article 6(1) and (2) of Regulation (EC) No 1071/2009 and/or to the temporary or permanent withdrawal of a Community licence. CHAPTER II INTERNATIONAL CARRIAGE Article 3 General principle International carriage shall be carried out subject to possession of a Community licence and, if the driver is a national of a third country, in conjunction with a driver attestation. Article 4 Community licence 1. The Community licence shall be issued by a Member State, in accordance with this Regulation, to any haulier carrying goods by road for hire or reward who: (a) is established in that Member State in accordance with Community legislation and the national legislation of that Member State; and (b) is entitled in the Member State of establishment, in accordance with Community legislation and the national legislation of that Member State concerning admission to the occupation of road haulage operator, to carry out the international carriage of goods by road. 2. The Community licence shall be issued by the competent authorities of the Member State of establishment for renewable periods of up to 10 years. Community licences and certified copies issued before the date of application of this Regulation shall remain valid until their date of expiry. The Commission shall adapt the period of validity of the Community licence to technical progress, in particular the national electronic registers of road transport undertakings as provided for in Article 16 of Regulation (EC) No 1071/2009. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 3. The Member State of establishment shall issue the holder with the original of the Community licence, which shall be kept by the haulier, and the number of certified true copies corresponding to the number of vehicles at the disposal of the holder of the Community licence, whether those vehicles are wholly owned or, for example, held under a hire purchase, hire or leasing contract. 4. The Community licence and the certified true copies shall correspond to the model set out in Annex II, which also lays down the conditions governing its use. They shall contain at least two of the security features listed in Annex I. The Commission shall adapt Annexes I and II to technical progress. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 5. The Community licence and the certified true copies thereof shall bear the seal of the issuing authority as well as a signature and a serial number. The serial numbers of the Community licence and of the certified true copies shall be recorded in the national electronic register of road transport undertakings as part of the data relating to the haulier. 6. The Community licence shall be issued in the name of the haulier and shall be non-transferable. A certified true copy of the Community licence shall be kept in each of the hauliers vehicles and shall be presented at the request of any authorised inspecting officer. In the case of a coupled combination of vehicles, the certified true copy shall accompany the motor vehicle. It shall cover the coupled combination of vehicles even where the trailer or semi-trailer is not registered or authorised to use the roads in the name of the licence holder or where it is registered or authorised to use the roads in another State. Article 5 Driver attestation 1. A driver attestation shall be issued by a Member State, in accordance with this Regulation, to any haulier who: (a) is the holder of a Community licence; and (b) in that Member State, either lawfully employs a driver who is neither a national of a Member State nor a long-term resident within the meaning of Council Directive 2003/109/EC of 25 November 2003 concerning the status of third-country nationals who are long-term residents (13), or lawfully uses a driver who is neither a national of a Member State nor a long-term resident within the meaning of that Directive and who is put at the disposal of that haulier in accordance with the conditions of employment and of vocational training laid down in that Member State: (i) by laws, regulations or administrative provisions; and, as appropriate; (ii) by collective agreements, in accordance with the rules applicable in that Member State. 2. The driver attestation shall be issued by the competent authorities of the Member State of establishment of the haulier, at the request of the holder of the Community licence, for each driver who is neither a national of a Member State nor a long-term resident within the meaning of Directive 2003/109/EC whom that haulier lawfully employs, or for each driver who is neither a national of a Member State nor a long-term resident within the meaning of that Directive and who is put at the disposal of the haulier. Each driver attestation shall certify that the driver named therein is employed in accordance with the conditions laid down in paragraph 1. 3. The driver attestation shall correspond to the model set out in Annex III. It shall contain at least two of the security features listed in Annex I. 4. The Commission shall adapt Annex III to technical progress. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 5. The driver attestation shall bear the seal of the issuing authority as well as a signature and a serial number. The serial number of the driver attestation may be recorded in the national electronic register of road transport undertakings as part of the data relating to the haulier who puts it at the disposal of the driver designated therein. 6. The driver attestation shall belong to the haulier, who puts it at the disposal of the driver designated therein when that driver drives a vehicle using a Community licence issued to that haulier. A certified true copy of the driver attestation issued by the competent authorities of the hauliers Member State of establishment shall be kept at the hauliers premises. The driver attestation shall be presented at the request of any authorised inspecting officer. 7. A driver attestation shall be issued for a period to be determined by the issuing Member State, subject to a maximum validity of 5 years. Driver attestations issued before the date of application of this Regulation shall remain valid until their date of expiry. The driver attestation shall be valid only as long as the conditions under which it was issued are satisfied. Member States shall take appropriate measures to ensure that if those conditions are no longer met, the haulier returns the attestation immediately to the issuing authorities. Article 6 Verification of conditions 1. Whenever an application for a Community licence or an application for renewal of a Community licence in accordance with Article 4(2) is lodged, the competent authorities of the Member State of establishment shall verify whether the haulier satisfies or continues to satisfy the conditions laid down in Article 4(1). 2. The competent authorities of the Member State of establishment shall regularly verify, by carrying out checks each year covering at least 20 % of the valid driver attestations issued in that Member State, whether the conditions, referred to in Article 5(1), under which a driver attestation has been issued are still satisfied. Article 7 Refusal to issue and withdrawal of Community licence and driver attestation 1. If the conditions laid down in Article 4(1) or those referred to in Article 5(1) are not satisfied, the competent authorities of the Member State of establishment shall reject an application for the issue or renewal of a Community licence or the issue of a driver attestation, by means of a reasoned decision. 2. The competent authorities shall withdraw a Community licence or a driver attestation where the holder: (a) no longer satisfies the conditions laid down in Article 4(1) or those referred to in Article 5(1); or (b) has supplied incorrect information in relation to an application for a Community licence or for a driver attestation. CHAPTER III CABOTAGE Article 8 General principle 1. Any haulier for hire or reward who is a holder of a Community licence and whose driver, if he is a national of a third country, holds a driver attestation, shall be entitled, under the conditions laid down in this Chapter, to carry out cabotage operations. 2. Once the goods carried in the course of an incoming international carriage have been delivered, hauliers referred to in paragraph 1 shall be permitted to carry out, with the same vehicle, or, in the case of a coupled combination, the motor vehicle of that same vehicle, up to three cabotage operations following the international carriage from another Member State or from a third country to the host Member State. The last unloading in the course of a cabotage operation before leaving the host Member State shall take place within 7 days from the last unloading in the host Member State in the course of the incoming international carriage. Within the time limit referred to in the first subparagraph, hauliers may carry out some or all of the cabotage operations permitted under that subparagraph in any Member State under the condition that they are limited to one cabotage operation per Member State within 3 days of the unladen entry into the territory of that Member State. 3. National road haulage services carried out in the host Member State by a non-resident haulier shall only be deemed to conform with this Regulation if the haulier can produce clear evidence of the incoming international carriage and of each consecutive cabotage operation carried out. Evidence referred to in the first subparagraph shall comprise the following details for each operation: (a) the name, address and signature of the sender; (b) the name, address and signature of the haulier; (c) the name and address of the consignee as well as his signature and the date of delivery once the goods have been delivered; (d) the place and the date of taking over of the goods and the place designated for delivery; (e) the description in common use of the nature of the goods and the method of packing, and, in the case of dangerous goods, their generally recognised description, as well as the number of packages and their special marks and numbers; (f) the gross mass of the goods or their quantity otherwise expressed; (g) the number plates of the motor vehicle and trailer. 4. No additional document shall be required in order to prove that the conditions laid down in this Article have been met. 5. Any haulier entitled in the Member State of establishment, in accordance with that Member States legislation, to carry out the road haulage operations for hire or reward specified in Article 1(5)(a), (b) and (c) shall be permitted, under the conditions set out in this Chapter, to carry out, as the case may be, cabotage operations of the same kind or cabotage operations with vehicles in the same category. 6. Permission to carry out cabotage operations, within the framework of the types of carriage referred to in Article 1(5)(d) and (e), shall be unrestricted. Article 9 Rules applicable to cabotage operations 1. The performance of cabotage operations shall be subject, save as otherwise provided in Community legislation, to the laws, regulations and administrative provisions in force in the host Member State with regard to the following: (a) the conditions governing the transport contract; (b) the weights and dimensions of road vehicles; (c) the requirements relating to the carriage of certain categories of goods, in particular dangerous goods, perishable foodstuffs and live animals; (d) the driving time and rest periods; (e) the value added tax (VAT) on transport services. The weights and dimensions referred to in point (b) of the first subparagraph may, where appropriate, exceed those applicable in the hauliers Member State of establishment, but they may under no circumstances exceed the limits set by the host Member State for national traffic or the technical characteristics mentioned in the proofs referred to in Article 6(1) of Council Directive 96/53/EC of 25 July 1996 laying down for certain road vehicles circulating within the Community the maximum authorised dimensions in national and international traffic and the maximum authorised weights in international traffic (14). 2. The laws, regulations and administrative provisions referred to in paragraph 1 shall be applied to non-resident hauliers under the same conditions as those imposed on hauliers established in the host Member State, so as to prevent any discrimination on grounds of nationality or place of establishment. Article 10 Safeguard procedure 1. In the event of serious disturbance of the national transport market in a given geographical area due to, or aggravated by, cabotage, any Member State may refer the matter to the Commission with a view to the adoption of safeguard measures and shall provide the Commission with the necessary information and notify it of the measures it intends to take as regards resident hauliers. 2. For the purposes of paragraph 1: serious disturbance of the national transport market in a given geographical area means the existence on the market of problems specific to it, such that there is a serious and potentially enduring excess of supply over demand, implying a threat to the financial stability and survival of a significant number of hauliers, geographical area means an area covering all or part of the territory of a Member State or extending to all or part of the territory of other Member States. 3. The Commission shall examine the situation on the basis in particular of the relevant data and, after consulting the committee referred to in Article 15(1), shall decide within 1 month of receipt of the Member States request whether or not safeguard measures are necessary and shall adopt them if they are necessary. Such measures may involve the temporary exclusion of the area concerned from the scope of this Regulation. Measures adopted in accordance with this Article shall remain in force for a period not exceeding 6 months, renewable once within the same limits of validity. The Commission shall without delay notify the Member States and the Council of any decision taken pursuant to this paragraph. 4. If the Commission decides to adopt safeguard measures concerning one or more Member States, the competent authorities of the Member States involved shall be required to take measures of equivalent scope in respect of resident hauliers and shall inform the Commission thereof. Those measures shall be applied at the latest as from the same date as the safeguard measures adopted by the Commission. 5. Any Member State may refer to the Council a decision taken by the Commission pursuant to paragraph 3 within 30 days of its notification. The Council, acting by a qualified majority may, within 30 days of that referral, or, if there are referrals by several Member States, of the first referral, take a different decision. The limits of validity laid down in the third subparagraph of paragraph 3 shall apply to the Councils decision. The competent authorities of the Member States concerned shall be required to take measures of equivalent scope in respect of resident hauliers, and shall inform the Commission thereof. If the Council takes no decision within the period referred to in the first subparagraph, the Commission decision shall become final. 6. Where the Commission considers that the measures referred to in paragraph 3 need to be prolonged, it shall submit a proposal to the Council, which shall take a decision by qualified majority. CHAPTER IV MUTUAL ASSISTANCE AND PENALTIES Article 11 Mutual assistance Member States shall assist one another in ensuring the application and monitoring of this Regulation. They shall exchange information via the national contact points established pursuant to Article 18 of Regulation (EC) No 1071/2009. Article 12 Sanctioning of infringements by the Member State of establishment 1. In the event of a serious infringement of Community road transport legislation committed or ascertained in any Member State, the competent authorities of the Member State of establishment of the haulier who has committed such infringement shall take the appropriate action which may include a warning, if provided for by national law, to pursue the matter which may lead, inter alia, to the imposition of the following administrative penalties: (a) temporary or permanent withdrawal of some or all of the certified true copies of the Community licence; (b) temporary or permanent withdrawal of the Community licence. These penalties may be determined after the final decision on the matter has been taken and shall have regard to the seriousness of the infringement committed by the holder of the Community licence and to the total number of certified true copies of that licence that he holds in respect of international traffic. 2. In the event of a serious infringement regarding any misuse whatsoever of driver attestations, the competent authorities of the Member State of establishment of the haulier who committed such infringement shall impose appropriate penalties, such as: (a) suspending the issue of driver attestations; (b) withdrawing driver attestations; (c) making the issue of driver attestations subject to additional conditions in order to prevent misuse; (d) withdrawing, temporarily or permanently, some or all of the certified true copies of the Community licence; (e) withdrawing, temporarily or permanently, the Community licence. These penalties may be determined after the final decision on the matter has been taken and shall have regard to the seriousness of the infringement committed by the holder of the Community licence. 3. The competent authorities of the Member State of establishment shall communicate to the competent authorities of the Member State in which the infringement was ascertained, as soon as possible and at the latest within 6 weeks of their final decision on the matter, which, if any, of the penalties provided for in paragraphs 1 and 2 have been imposed. If such penalties are not imposed, the competent authorities of the Member State of establishment shall state the reasons therefor. 4. The competent authorities shall ensure that the penalties imposed on the haulier concerned are, as a whole, proportionate to the infringement or infringements which gave rise to such penalties, taking into account any penalty for the same infringement imposed in the Member State in which the infringement was ascertained. 5. The competent authorities of the hauliers Member State of establishment may also, pursuant to national law, bring proceedings against the haulier before a competent national court or tribunal. They shall inform the competent authority of the host Member State of any decisions taken to this effect. 6. Member States shall ensure that hauliers have the right to appeal against any administrative penalty imposed on them pursuant to this Article. Article 13 Sanctioning of infringements by the host Member State 1. Where the competent authorities of a Member State are aware of a serious infringement of this Regulation or of Community road transport legislation attributable to a non-resident haulier, the Member State within the territory of which the infringement is ascertained shall transmit to the competent authorities of the hauliers Member State of establishment, as soon as possible and at the latest within 6 weeks of their final decision on the matter, the following information: (a) a description of the infringement and the date and time when it was committed; (b) the category, type and seriousness of the infringement; and (c) the penalties imposed and the penalties executed. The competent authorities of the host Member State may request the competent authorities of the Member State of establishment to impose administrative penalties in accordance with Article 12. 2. Without prejudice to any criminal prosecution, the competent authorities of the host Member State shall be empowered to impose penalties on a non-resident haulier who has committed infringements of this Regulation or of national or Community road transport legislation in their territory during a cabotage operation. They shall impose such penalties on a non-discriminatory basis. These penalties may, inter alia, consist of a warning, or, in the event of a serious infringement, a temporary ban on cabotage operations on the territory of the host Member State where the infringement was committed. 3. Member States shall ensure that hauliers have the right to appeal against any administrative penalty imposed on them pursuant to this Article. Article 14 Entry in the national electronic registers Member States shall ensure that serious infringements of Community road transport legislation committed by hauliers established in their territory, which have led to the imposition of a penalty by any Member State, as well as any temporary or permanent withdrawal of the Community licence or of the certified true copy thereof, are recorded in the national electronic register of road transport undertakings. Entries in the register which concern a temporary or permanent withdrawal of a Community licence shall remain in the database for 2 years from the time of the expiry of the period of withdrawal, in the case of temporary withdrawal, or from the date of withdrawal, in the case of permanent withdrawal. CHAPTER V IMPLEMENTATION Article 15 Committee procedure 1. The Commission shall be assisted by the committee established by Article 18(1) of Regulation (EEC) No 3821/85. 2. Where reference is made to this paragraph, Article 5a(1) to (4) and Article 7 of Decision 1999/468/EC shall apply, having regard to the provisions of Article 8 thereof. Article 16 Penalties Member States shall lay down the rules on penalties applicable to infringements of the provisions of this Regulation, and shall take all the measures necessary to ensure that they are implemented. The penalties provided for must be effective, proportionate and dissuasive. Member States shall notify those provisions to the Commission by 4 December 2011, and shall notify it without delay of any subsequent amendment affecting them. Member States shall ensure that all such measures are taken without discrimination as to the nationality or place of establishment of the haulier. Article 17 Reporting 1. Every 2 years Member States shall inform the Commission of the number of hauliers possessing Community licences on 31 December of the previous year and of the number of certified true copies corresponding to the vehicles in circulation at that date. 2. Member States shall also inform the Commission of the number of driver attestations issued in the previous calendar year as well as the number of driver attestations in circulation on 31 December of that same year. 3. The Commission shall draw up a report on the state of the Community road transport market by the end of 2013. The report shall contain an analysis of the market situation, including an evaluation of the effectiveness of controls and the evolution of employment conditions in the profession, as well as an assessment as to whether harmonisation of the rules in the fields, inter alia, of enforcement and road user charges, as well as social and safety legislation, has progressed to such an extent that the further opening of domestic road transport markets, including cabotage, could be envisaged. CHAPTER VI FINAL PROVISIONS Article 18 Repeals Regulations (EEC) No 881/92 and (EEC) No 3118/93 and Directive 2006/94/EC are hereby repealed. References to the repealed Regulations and Directive shall be construed as references to this Regulation and shall be read in accordance with the correlation table set out in Annex IV. Article 19 Entry into force This Regulation shall enter into force on the 20th day following its publication in the Official Journal of the European Union. It shall apply from 4 December 2011, with the exception of Articles 8 and 9, which shall apply from 14 May 2010. This Regulation shall be binding in its entirety and directly applicable in all Member States. Done at Strasbourg, 21 October 2009. For the European Parliament The President J. BUZEK For the Council The President C. MALMSTRÃ M (1) OJ C 204, 9.8.2008, p. 31. (2) Opinion of the European Parliament of 21 May 2008 (not yet published in the Official Journal), Council Common Position of 9 January 2009 (OJ C 62 E, 17.3.2009, p. 46), Position of the European Parliament of 23 April 2009 (not yet published in the Official Journal) and Council Decision of 24 September 2009. (3) OJ L 95, 9.4.1992, p. 1. (4) OJ L 279, 12.11.1993, p. 1. (5) OJ L 374, 27.12.2006, p. 5. (6) OJ L 102, 11.4.2006, p. 1. (7) OJ L 368, 17.12.1992, p. 38. (8) OJ L 18, 21.1.1997, p. 1. (9) OJ L 370, 31.12.1985, p. 8. (10) See page 51 of this Official Journal. (11) OJ L 184, 17.7.1999, p. 23. (12) OJ L 33, 4.2.2006, p. 82. (13) OJ L 16, 23.1.2004, p. 44. (14) OJ L 235, 17.9.1996, p. 59. ANNEX I Security features of the Community licence and the driver attestation The Community licence and the driver attestation must have at least two of the following security features:  a hologram,  special fibres in the paper which become visible under UV-light,  at least one microprint line (printing visible only with a magnifying glass and not reproduced by photocopying machines),  tactile characters, symbols or patterns,  double numbering: serial number of the Community licence, of the certified copy thereof or of the driver attestation as well as, in each case, the issue number,  a security design background with fine guilloche patterns and rainbow printing. ANNEX II Community licence model EUROPEAN COMMUNITY (a) (Colour Pantone light blue, format DIN A4 cellulose paper 100 g/m2 or more) (First page of the licence) (Text in (one of) the official language(s) of the Member State issuing the licence) (b) (Second page of the licence) (Text in (one of) the official language(s) of the Member State issuing the licence) GENERAL PROVISIONS This licence is issued under Regulation (EC) No 1072/2009. It entitles the holder to engage in the international carriage of goods by road for hire or reward by any route for journeys or parts of journeys carried out within the territory of the Community and, where appropriate, subject to the conditions laid down herein:  where the point of departure and the point of arrival are situated in two different Member States, with or without transit through one or more Member States or third countries,  from a Member State to a third country or vice versa, with or without transit through one or more Member States or third countries,  between third countries with transit through the territory of one or more Member States, and unladen journeys in connection with such carriage. In the case of carriage from a Member State to a third country or vice versa, this licence is valid for that part of the journey carried out within the territory of the Community. It shall be valid in the Member State of loading or unloading only after the conclusion of the necessary agreement between the Community and the third country in question in accordance with Regulation (EC) No 1072/2009. The licence is personal to the holder and is non-transferable. It may be withdrawn by the competent authority of the Member State which issued it, notably where the holder has:  not complied with all the conditions for using the licence,  supplied incorrect information with regard to the data needed for the issue or extension of the licence. The original of the licence must be kept by the haulage undertaking. A certified copy of the licence must be kept in the vehicle (1). In the case of a coupled combination of vehicles it must accompany the motor vehicle. It covers the coupled combination of vehicles even if the trailer or semi-trailer is not registered or authorised to use the roads in the name of the licence holder or if it is registered or authorised to use the roads in another State. The licence must be presented at the request of any authorised inspecting officer. Within the territory of each Member State, the holder must comply with the laws, regulations and administrative provisions in force in that State, in particular with regard to transport and traffic. (1) Vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods. ANNEX III Driver attestation model EUROPEAN COMMUNITY (a) (Colour Pantone pink, format DIN A4 cellulose paper 100g/m2 or more) (First page of the attestation) (Text in (one of) the official language(s) of the Member State issuing the attestation) (b) (Second page of the attestation) (Text in (one of) the official language(s) of the Member State issuing the attestation) GENERAL PROVISIONS This attestation is issued under Regulation (EC) No 1072/2009. It certifies that the driver named therein is employed, in accordance with the laws, regulations or administrative provisions and, as appropriate, the collective agreements, in accordance with the rules applicable in the Member State mentioned on the attestation, on the conditions of employment and of vocational training of drivers applicable in that Member State to carry out road operations in that State. The driver attestation shall belong to the haulier, who puts it at the disposal of the driver designated therein when that driver drives a vehicle (1) engaged in carriage using a Community licence issued to that haulier. The driver attestation is not transferable. The driver attestation shall be valid only as long as the conditions under which it was issued are still satisfied and must be returned immediately by the haulier to the issuing authorities if these conditions are no longer met. It may be withdrawn by the competent authority of the Member State which issued it, in particular where the holder has:  not complied with all the conditions for using the attestation,  supplied incorrect information with regard to the data needed for the issue or extension of the attestation. A certified true copy of the attestation must be kept by the haulage undertaking. An original attestation must be kept in the vehicle and must be presented by the driver at the request of any authorised inspecting officer. (1) Vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods. ANNEX IV Correlation Table Regulation (EEC) No 881/92 Regulation (EEC) No 3118/93 Directive 2006/94/EC This Regulation Article 1(1) Article 1(1) Article 1(2) Article 1(2) Article 1(3) Article 1(3) Annex II Article 1(1) and (2), Annex I; Article 2 Article 1(5) Article 2 Article 1(6) Article 2 Article 2 Article 3(1) Article 3 Article 3(2) Article 4(1) Article 3(3) Article 5(1) Article 4 Article 5(1) Article 4(2) Article 5(2) Article 4(3) Article 5(3) Article 4(4) Article 4(5) Article 5(4), Annex I Article 4(6) Article 5(5) Article 4(2) Article 6(1) Article 5(2) Article 6(2) Article 5(2) Article 6(3) Article 5(3) Article 6(4) Article 5(6) Article 6(5) Article 5(7) Article 7 Article 6 Article 8(1) Article 7(1) Article 8(2) Article 7(2) Article 8(3) Article 12(1) Article 8(4) Article 12(2) Article 9(1) and (2) Article 12(6) Article 1(1) Article 8(1) Article 1(2) Article 8(5) Article 1(3) and (4) Article 8(6) Article 2 Article 3 Article 4 Article 5 Article 6(1) Article 9(1) Article 6(2) Article 6(3) Article 9(2) Article 6(4) Article 7 Article 10 Article 10 Article 17(1) Article 11(1) Article 8(1) Article 11 Article 11(2) Article 13(1) Article 11(3) Article 12(4) Article 11a Article 8(2) and (3) Article 13(2) Article 8(4), first and third subparagraphs Article 8(4), second subparagraph Article 12(4) Article 8(4), fourth and fifth subparagraphs Article 12(5) Article 9 Article 13(3) Article 12 Article 18 Article 13 Article 14 Article 10 Article 11 Article 15 Article 12 Article 4 Article 19 Article 3 Article 5 Annex II, III Annex I Annex II Annex III Annex III Annex I Annex II Annex III Annex IV
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32015L0413
        'status': In Force
        'act_type': Directive
        'treaty': TFEU (2008)

        full_doc :

        13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important element of that policy is the consistent enforcement of sanctions for road traffic offences committed in the Union which considerably jeopardise road safety. (2) However, due to a lack of appropriate procedures and notwithstanding existing possibilities under Council Decision 2008/615/JHA (3) and Council Decision 2008/616/JHA (4) (the PrÃ ¼m Decisions), sanctions in the form of financial penalties for certain road traffic offences are often not enforced if those offences are committed with a vehicle which is registered in a Member State other than the Member State where the offence took place. This Directive aims to ensure that even in such cases, the effectiveness of the investigation of road-safety-related traffic offences should be ensured. (3) In its communication of 20 July 2010 entitled Towards a European road safety area: policy orientations on road safety 2011-2020, the Commission emphasised that enforcement of road traffic rules remains a key factor in creating the conditions for a considerable reduction in the number of deaths and injuries. In its conclusions of 2 December 2010 on road safety, the Council called for consideration of the need for further strengthening of enforcement of road traffic rules by Member States and, where appropriate, at Union level. It invited the Commission to examine the possibilities of harmonising traffic rules at Union level where appropriate and adopting further measures on facilitating cross-border enforcement with regard to road traffic offences, in particular those related to serious traffic accidents. (4) On 19 March 2008, the Commission adopted a proposal for a Directive of the European Parliament and of the Council facilitating cross-border enforcement in the field of road safety on the basis of Article 71(1)(c) of the Treaty establishing the European Community (now Article 91 of Treaty on the Functioning of the European Union (TFEU)). Directive 2011/82/EU of the European Parliament and of the Council (5) was, however, adopted on the basis of Article 87(2) TFEU. The judgment of the Court of Justice of 6 May 2014 in Case C-43/12 (6) annulled Directive 2011/82/EU on the grounds that it could not validly be adopted on the basis of Article 87(2) TFEU. The judgment maintained the effects of Directive 2011/82/EU until the entry into force within a reasonable period of time  which is not to exceed 12 months as from the date of delivery of the judgment  of a new directive based on Article 91(1)(c) TFEU. Therefore a new Directive should be adopted on the basis of that Article. (5) Greater convergence of control measures between Member States should be encouraged and the Commission should examine in this respect the need for developing common standards for automatic checking equipment for road safety controls. (6) The awareness of Union citizens should be raised as regards the road safety traffic rules in force in different Member States and as regards the implementation of this Directive, in particular through appropriate measures guaranteeing the provision of sufficient information on the consequences of not respecting the road safety traffic rules when travelling in a Member State other than the Member State of registration. (7) In order to improve road safety throughout the Union and to ensure equal treatment of drivers, namely resident and non-resident offenders, enforcement should be facilitated irrespective of the Member State of registration of the vehicle. To this end, a system of cross-border exchange of information should be used for certain identified road-safety-related traffic offences, regardless of their administrative or criminal nature under the law of the Member State concerned, granting the Member State of the offence access to vehicle registration data (VRD) of the Member State of registration. (8) A more efficient cross-border exchange of VRD, which should facilitate the identification of persons suspected of committing a road-safety-related traffic offence, might increase the deterrent effect and induce more cautious behaviour by the driver of a vehicle that is registered in a Member State other than the Member State of the offence, thereby preventing casualties due to road traffic accidents. (9) The road-safety-related traffic offences covered by this Directive are not subject to homogeneous treatment in the Member States. Some Member States qualify such offences under national law as administrative offences while others qualify them as criminal offences. This Directive should apply regardless of how those offences are qualified under national law. (10) Member States should grant each other the right of access to their VRD in order to improve the exchange of information and to speed up the procedures in force. To this end, the provisions concerning the technical specifications and the availability of automated data exchange set out in the PrÃ ¼m Decisions should, as far as possible, be included in this Directive. (11) Decision 2008/616/JHA specifies the security features for existing software applications and the related technical requirements for the exchange of vehicle registration data. Without prejudice to the general applicability of that Decision, those security features and technical requirements should, for reasons of regulatory and practical efficiency, be used for the purposes of this Directive. (12) Existing software applications should be the basis for the data exchange under this Directive and should, at the same time, also facilitate the reporting by Member States to the Commission. Such applications should provide for the expeditious, secure and confidential exchange of specific VRD between Member States. Advantage should be taken of the European Vehicle and Driving Licence Information System (Eucaris) software application, which is mandatory for Member States under the PrÃ ¼m Decisions as regards VRD. The Commission should assess and report on the functioning of the software applications used for the purposes of this Directive. (13) The scope of those software applications should be limited to the processes used in the exchange of information between the national contact points in the Member States. Procedures and automated processes in which the information is to be used are outside the scope of such applications. (14) The Information Management Strategy for EU internal security aims to find the simplest and most easily traceable and cost-effective solutions for data exchange. (15) Member States should be able to contact the owner, the holder of the vehicle or the otherwise identified person suspected of committing the road-safety-related traffic offence in order to keep the person concerned informed of the applicable procedures and the legal consequences under the law of the Member State of the offence. In doing so, Member States should consider sending the information concerning road-safety-related traffic offences in the language of the registration documents, or in the language most likely to be understood by the person concerned, to ensure that that person has a clear understanding of the information which is being shared with the person concerned. Member States should apply the appropriate procedures to ensure that only the person concerned is informed and not a third party. To that effect, Member States should use detailed arrangements similar to those adopted for following up such offences including means such as, where appropriate, registered delivery. This will allow that person to respond to the information letter in an appropriate way, in particular by asking for more information, by settling the fine or by exercising his/her rights of defence, especially in the case of mistaken identity. Further proceedings are covered by applicable legal instruments, including instruments on mutual assistance and on mutual recognition, for example Council Framework Decision 2005/214/JHA (7). (16) Member States should provide equivalent translation with respect to the information letter sent by the Member State of the offence, as provided for in Directive 2010/64/EU of the European Parliament and of the Council (8). (17) With a view to pursuing a road safety policy that aims to provide a high level of protection for all road users in the Union, and taking into account the widely differing circumstances pertaining within the Union, Member States should act, without prejudice to more restrictive policies and laws, in order to ensure greater convergence of road traffic rules and of their enforcement between Member States. In the framework of its report to the European Parliament and to the Council on the application of this Directive, the Commission should examine the need to develop common standards in order to establish comparable methods, practices and minimum standards at Union level taking into account international cooperation and existing agreements in the field of road safety, in particular the Vienna Convention on Road Traffic of 8 November 1968. (18) In its report to the European Parliament and to the Council on the application of this Directive by the Member States, the Commission should examine the need for common criteria for follow-up procedures by Member States in the event of non-payment of a financial penalty, in accordance with Member States' laws and procedures. In that report, the Commission should address issues such as the procedures between the competent authorities of the Member States for the transmission of the final decision to impose a sanction and/or financial penalty as well as the recognition and enforcement of the final decision. (19) In preparing the review of this Directive, the Commission should consult the relevant stakeholders, such as road safety and law enforcement authorities or competent bodies, victims' associations and other non-governmental organisations active in the field of road safety. (20) Closer cooperation between law enforcement authorities should go hand in hand with respect for fundamental rights, in particular the right to respect for privacy and to the protection of personal data, guaranteed by special data protection arrangements. Those arrangements should take particular account of the specific nature of cross-border online access to databases. It is necessary that the software applications to be set up enable the exchange of information to be carried out in secure conditions and ensure the confidentiality of the data transmitted. The data collected under this Directive should not be used for purposes other than those of this Directive. Member States should comply with the obligations on the conditions of use and of temporary storage of the data. (21) The processing of personal data provided by this Directive is appropriate for attaining the legitimate aims pursued by this Directive in the field of road safety, namely to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences and, thereby, the enforcement of sanctions, and does not exceed what is appropriate and necessary in order to achieve those objectives. (22) Data relating to the identification of an offender are personal data. Directive 95/46/EC of the European Parliament and of the Council (9) should apply to the processing activities carried out in application of this Directive. Without prejudice to the procedural requirements for appeal and the redress mechanisms of the Member State concerned, the data subject should accordingly be informed, when notified of the offence, of the right to access and the right to rectification and deletion of personal data, as well as of the maximum legal storage period of the data. In this context, the data subject should also have the right to obtain the correction of any inaccurate personal data or the immediate deletion of any data recorded unlawfully. (23) In the framework of the PrÃ ¼m Decisions, the processing of VRD containing personal data is subject to the specific provisions on data protection set out in Decision 2008/615/JHA. In that respect, Member States have the possibility to apply those specific provisions to personal data which are also processed for the purposes of this Directive provided that they ensure that the processing of data related to all of the offences covered by this Directive complies with the national provisions implementing Directive 95/46/EC. (24) It should be possible for third countries to participate in the exchange of VRD provided that they have concluded an agreement with the Union to this effect. Such an agreement would have to include necessary provisions on data protection. (25) This Directive upholds the fundamental rights and principles recognised by the Charter of Fundamental Rights of the European Union, including the respect for private and family life, the protection of personal data, the right to a fair trial, the presumption of innocence and the right of defence. (26) In order to achieve the objective of the exchange of information between Member States through interoperable means, the power to adopt acts in accordance with Article 290 TFEU should be delegated to the Commission in respect of the taking into account of relevant changes to PrÃ ¼m Decisions or where required by legal acts of the Union directly relevant for the updating of Annex I. It is of particular importance that the Commission follow its usual practice and carry out appropriate consultations during its preparatory work, including at expert level. The Commission, when preparing and drawing up delegated acts, should ensure a simultaneous, timely and appropriate transmission of relevant documents to the European Parliament and to the Council. (27) The Commission should analyse the application of this Directive with a view to identifying further effective and efficient measures to improve road safety. Without prejudice to obligations to transpose this Directive, Denmark, Ireland and the United Kingdom should also cooperate with the Commission in this work, where appropriate, to ensure timely and complete reporting on this matter. (28) Since the objective of this Directive, namely to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences, where they are committed with a vehicle registered in a Member State other than the Member State where the offence took place, cannot be sufficiently achieved by the Member States, but can rather, by reason of the scale and effects of the action, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity, as set out in Article 5 of the Treaty on European Union. In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary in order to achieve that objective. (29) Given that Denmark, Ireland and the United Kingdom were not subject to Directive 2011/82/EU and therefore have not transposed it, it is appropriate to allow those Member States sufficient additional time to do so. (30) The European Data Protection Supervisor was consulted in accordance with Article 28(2) of Regulation (EC) No 45/2001 of the European Parliament and of the Council (10) and delivered an opinion on 3 October 2014, HAVE ADOPTED THIS DIRECTIVE: Article 1 Objective This Directive aims to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences, and thereby facilitating the enforcement of sanctions, where those offences are committed with a vehicle registered in a Member State other than the Member State in which the offence took place. Article 2 Scope This Directive applies to the following road-safety-related traffic offences: (a) speeding; (b) failing to use a seat-belt; (c) failing to stop at a red traffic light; (d) drink-driving; (e) driving while under the influence of drugs; (f) failing to wear a safety helmet; (g) the use of a forbidden lane; (h) illegally using a mobile telephone or any other communication devices while driving. Article 3 Definitions For the purposes of this Directive, the following definitions apply: (a) vehicle means any power-driven vehicle, including motorcycles, which is normally used for carrying persons or goods by road; (b) Member State of the offence means the Member State where the offence was committed; (c) Member State of registration means the Member State where the vehicle with which the offence was committed is registered; (d) speeding means exceeding speed limits in force in the Member State of offence for the road or type of vehicle concerned; (e) failing to use a seat-belt means not complying with the requirement to wear a seat-belt or to use a child restraint in accordance with Council Directive 91/671/EEC (11) and the law of the Member State of the offence; (f) failing to stop at a red traffic light means driving through a red traffic light or any other relevant stop signal, as defined in the law of the Member State of the offence; (g) drink-driving means driving while impaired by alcohol, as defined in the law of the Member State of the offence; (h) driving under the influence of drugs means driving while impaired by drugs or other substances having a similar effect, as defined in the law of the Member State of the offence; (i) failing to wear a safety helmet means not wearing a safety helmet, as defined in the law of the Member State of the offence; (j) use of a forbidden lane means illegally using part of a road section, such as an emergency lane, public transport lane or temporary closed lane for reasons of congestion or road works, as defined in the law of the Member State of the offence; (k) illegally using a mobile telephone or any other communication devices while driving means illegally using a mobile telephone or any other communication devices while driving, as defined in the law of the Member State of the offence; (l) national contact point means a designated competent authority for the exchange of VRD; (m) automated search means an online access procedure for consulting the databases of one, more than one, or all of the Member States or of the participating countries; (n) holder of the vehicle means the person in whose name the vehicle is registered, as defined in the law of the Member State of registration. Article 4 Procedure for the exchange of information between Member States 1. For the investigation of the road-safety-related traffic offences referred to in Article 2, the Member State shall grant other Member States' national contact points, referred to in paragraph 2 of this Article, access to the following national VRD, with the power to conduct automated searches thereon: (a) data relating to vehicles; and (b) data relating to owners or holders of the vehicle. The data elements referred to in points (a) and (b) which are necessary to conduct a search shall be in compliance with Annex I. 2. For the purposes of the exchange of data referred to in paragraph 1, each Member State shall designate a national contact point. The powers of the national contact points shall be governed by the applicable law of the Member State concerned. 3. When conducting a search in the form of an outgoing request, the national contact point of the Member State of the offence shall use a full registration number. Those searches shall be conducted in compliance with the procedures as described in Chapter 3 of the Annex to Decision 2008/616/JHA, except for point 1 of Chapter 3 of the Annex to Decision 2008/616/JHA, for which Annex I to this Directive shall apply. The Member State of the offence shall, under this Directive, use the data obtained in order to establish who is personally liable for road-safety-related traffic offences listed in Article 2 of this Directive. 4. Member States shall take all necessary measures to ensure that the exchange of information is carried out by interoperable electronic means without exchange of data involving other databases which are not used for the purposes of this Directive. Member States shall ensure that such exchange of information is conducted in a cost-efficient and secure manner. Member States shall ensure the security and protection of the data transmitted, as far as possible using existing software applications such as the one referred to in Article 15 of Decision 2008/616/JHA and amended versions of those software applications, in compliance with Annex I to this Directive and with points 2 and 3 of Chapter 3 of the Annex to Decision 2008/616/JHA. The amended versions of the software applications shall provide for both online real-time exchange mode and batch exchange mode, the latter allowing for the exchange of multiple requests or responses within one message. 5. Each Member State shall bear its own costs arising from the administration, use and maintenance of the software applications referred to in paragraph 4. Article 5 Information letter on the road-safety-related traffic offences 1. The Member State of the offence shall decide whether or not to initiate follow-up proceedings in relation to the road-safety-related traffic offences listed in Article 2. Where the Member State of the offence decides to initiate such proceedings, that Member State shall, in accordance with its national law, inform the owner, the holder of the vehicle or the otherwise identified person suspected of committing the road-safety-related traffic offence. This information shall, as applicable under national law, include the legal consequences thereof within the territory of the Member State of the offence under the law of that Member State. 2. When sending the information letter to the owner, the holder of the vehicle or to the otherwise identified person suspected of committing the road-safety-related traffic offence, the Member State of the offence shall, in accordance with its law, include any relevant information, notably the nature of this road-safety-related traffic offence, the place, date and time of the offence, the title of the texts of the national law infringed and the sanction and, where appropriate, data concerning the device used for detecting the offence. For that purpose, the Member State of the offence may use the template set out in Annex II. 3. Where the Member State of the offence decides to initiate follow-up proceedings in relation to the road-safety-related traffic offences listed in Article 2, the Member State of the offence, for the purpose of ensuring the respect of fundamental rights, sends the information letter in the language of the registration document of the vehicle, if available, or in one of the official languages of the Member State of registration. Article 6 Reporting by Member States to the Commission Each Member State shall send a comprehensive report to the Commission by 6 May 2016 and every two years thereafter. The comprehensive report shall indicate the number of automated searches conducted by the Member State of the offence addressed to the national contact point of the Member State of registration, following offences committed on its territory, together with the type of offences for which requests were addressed and the number of failed requests. The comprehensive report shall also include a description of the situation at national level in relation to the follow-up given to the road-safety-related traffic offences, based on the proportion of such offences which have been followed up by information letters. Article 7 Data protection 1. The provisions on data protection set out in Directive 95/46/EC shall apply to personal data processed under this Directive. 2. In particular, each Member State shall ensure that personal data processed under this Directive are, within an appropriate time period, rectified if inaccurate, or erased or blocked when they are no longer required, in accordance with Articles 6 and 12 of Directive 95/46/EC, and that a time limit for the storage of data is established in accordance with Article 6 of that Directive. Member States shall ensure that all personal data processed under this Directive are only used for the objective set out in Article 1 of this Directive, and that the data subjects have the same rights to information, to access, to rectification, erasure and blocking, to compensation and to judicial redress as those adopted under national law in implementation of the relevant provisions of Directive 95/46/EC. 3. Any person concerned shall have the right to obtain information on which personal data recorded in the Member State of registration were transmitted to the Member State of the offence, including the date of the request and the competent authority of the Member State of the offence. Article 8 Information for road users in the Union 1. The Commission shall make available on its website a summary in all official languages of the institutions of the Union of the rules in force in Member States in the field covered by this Directive. Member States shall provide information on these rules to the Commission. 2. Member States shall provide road users with the necessary information about the rules applicable in their territory and the measures implementing this Directive in association with, among other organisations, road safety bodies, non-governmental organisations active in the field of road safety and automobile clubs. Article 9 Delegated acts The Commission shall be empowered to adopt delegated acts, in accordance with Article 10, updating Annex I in the light of technical progress to take into account relevant changes to PrÃ ¼m Decisions or where this is required by legal acts of the Union directly relevant to the updating of Annex I. Article 10 Exercise of the delegation 1. The power to adopt delegated acts is conferred on the Commission subject to the conditions laid down in this Article. 2. The power to adopt delegated acts referred to in Article 9 shall be conferred on the Commission for a period of five years from 13 March 2015. The Commission shall draw up a report in respect of the delegation of power not later than nine months before the end of the five-year period. The delegation of power shall be tacitly extended for periods of an identical duration, unless the European Parliament or the Council opposes such extension not later than three months before the end of each period. 3. The delegation of power referred to in Article 9 may be revoked at any time by the European Parliament or by the Council. A decision to revoke shall put an end to the delegation of the power specified in that decision. It shall take effect on the day following the publication of the decision in the Official Journal of the European Union or at a later date specified therein. It shall not affect the validity of any delegated acts already in force. 4. It is of particular importance that the Commission follow its usual practice and carry out consultations with experts, including Member States' experts, before adopting those delegated acts. As soon as it adopts a delegated act, the Commission shall notify it simultaneously to the European Parliament and to the Council. 5. A delegated act adopted pursuant to Article 9 shall enter into force only if no objection has been expressed either by the European Parliament or the Council within a period of two months of notification of that act to the European Parliament and the Council or if, before the expiry of that period, the European Parliament and the Council have both informed the Commission that they will not object. That period shall be extended by two months at the initiative of the European Parliament or of the Council. Article 11 Revision of the Directive Without prejudice to the provisions laid down in the second subparagraph of Article 12(1), the Commission shall, by 7 November 2016, submit a report to the European Parliament and to the Council on the application of this Directive by the Member States. In its report, the Commission shall focus in particular on, and shall, as appropriate, make proposals to cover, the following aspects:  an assessment of whether other road-safety-related traffic offences should be added to the scope of this Directive,  an assessment of the effectiveness of this Directive on the reduction in the number of fatalities on Union roads,  an assessment of the need for developing common standards for automatic checking equipment and for procedures. In this context, the Commission is invited to develop at Union level road safety guidelines within the framework of the common transport policy in order to ensure greater convergence of the enforcement of road traffic rules by Member States through comparable methods and practices. These guidelines may cover at least the offences listed in points (a) to (d) of Article 2,  an assessment of the need to strengthen the enforcement of sanctions with regard to road-safety-related traffic offences and to propose common criteria concerning the follow-up procedures in the case of non-payment of a financial penalty, within the framework of all relevant Union policies, including the common transport policy,  the possibilities for harmonising traffic rules where appropriate,  an assessment of the software applications as referred to in Article 4(4), with a view to ensuring proper implementation of this Directive as well as guaranteeing an effective, expeditious, secure and confidential exchange of specific VRD. Article 12 Transposition 1. Member States shall bring into force the laws, regulations and administrative provisions necessary to comply with this Directive by 6 May 2015. They shall forthwith communicate to the Commission the text of those provisions. When Member States adopt those provisions, they shall contain a reference to this Directive or be accompanied by such a reference on the occasion of their official publication. Member States shall determine how such reference is to be made. By way of derogation from the first subparagraph, the Kingdom of Denmark, Ireland and the United Kingdom of Great Britain and Northern Ireland may postpone the deadline referred to in the first subparagraph until 6 May 2017. 2. Member States shall communicate to the Commission the text of the main provisions of national law which they adopt in the field covered by this Directive. Article 13 Entry into force This Directive shall enter into force on the fourth day following that of its publication in the Official Journal of the European Union. Article 14 Addressees This Directive is addressed to the Member States. Done at Strasbourg, 11 March 2015. For the European Parliament The President M. SCHULZ For the Council The President Z. KALNIÃ A-LUKAÃ EVICA (1) OJ C 12, 15.1.2015, p. 115. (2) Position of the European Parliament of 11 February 2015 (not yet published in the Official Journal) and Decision of the Council of 2 March 2015. (3) Council Decision 2008/615/JHA of 23 June 2008 on the stepping up of cross-border cooperation, particularly in combating terrorism and cross-border crime (OJ L 210, 6.8.2008, p. 1). (4) Council Decision 2008/616/JHA of 23 June 2008 on the implementation of Decision 2008/615/JHA on the stepping up of cross-border cooperation, particularly in combating terrorism and cross-border crime (OJ L 210, 6.8.2008, p. 12). (5) Directive 2011/82/EU of the European Parliament and of the Council of 25 October 2011 facilitating the cross-border exchange of information on road safety related traffic offences (OJ L 288, 5.11.2011, p. 1). (6) Judgment in Commission v Parliament and Council, C-43/12, EU:C:2014:298. (7) Council Framework Decision 2005/214/JHA of 24 February 2005 on the application of the principle of mutual recognition to financial penalties (OJ L 76, 22.3.2005, p. 16). (8) Directive 2010/64/EU of the European Parliament and of the Council of 20 October 2010 on the right to interpretation and translation in criminal proceedings (OJ L 280, 26.10.2010, p. 1). (9) Directive 95/46/EC of the European Parliament and of the Council of 24 October 1995 on the protection of individuals with regard to the processing of personal data and on the free movement of such data (OJ L 281, 23.11.1995, p. 31). (10) Regulation (EC) No 45/2001 of the European Parliament and of the Council of 18 December 2000 on the protection of individuals with regard to the processing of personal data by the Community institutions and bodies and on the free movement of such data (OJ L 8, 12.1.2001, p. 1). (11) Council Directive 91/671/EEC of 16 December 1991 relating to the compulsory use of safety belts and child-restraint systems in vehicles (OJ L 373, 31.12.1991, p. 26). ANNEX I Data elements necessary to conduct the search referred to in Article 4(1) Item M/O (1) Remarks Data relating to the vehicle M Member State of registration M Registration number M (A (2)) Data relating to the offence M Member State of the offence M Reference date of the offence M Reference time of the offence M Purpose of the search M Code indicating the type of offence as listed in Article 2 1. = Speeding 2. = Drink-driving 3. = Failing to use a seat belt 4. = Failing to stop at a red traffic light 5. = Use of a forbidden lane 10. = Driving under the influence of drugs 11. = Failing to wear a safety helmet 12. = Illegally using a mobile phone or any other communication devices while driving Data elements provided as a result of the search conducted pursuant to Article 4(1) Part I. Data relating to vehicles Item M/O (3) Remarks Registration number M Chassis number/VIN M Member State of registration M Make M (D.1 (4)) e.g. Ford, Opel, Renault Commercial type of the vehicle M (D.3) e.g. Focus, Astra, Megane EU Category Code M (J) e.g. mopeds, motorbikes, cars Part II. Data relating to owners or holders of the vehicles Item M/O (5) Remarks Data relating to holders of the vehicle (C.1 (6)) The data refer to the holder of the specific registration certificate. Registration holders' (company) name M (C.1.1) Separate fields shall be used for surname, infixes, titles, etc., and the name in printable format shall be communicated. First name M (C.1.2) Separate fields for first name(s) and initials shall be used, and the name in printable format shall be communicated. Address M (C.1.3) Separate fields shall be used for street, house number and annex, post code, place of residence, country of residence, etc., and the address in printable format shall be communicated. Gender O Male, female Date of birth M Legal entity M Individual, association, company, firm, etc. Place of Birth O ID Number O An identifier that uniquely identifies the person or the company. Data relating to owners of the vehicle (C.2) The data refer to the owner of the vehicle. Owners' (company) name M (C.2.1) First name M (C.2.2) Address M (C.2.3) Gender O Male, female Date of birth M Legal entity M Individual, association, company, firm, etc. Place of Birth O ID Number O An identifier that uniquely identifies the person or the company. In case of scrap vehicles, stolen vehicles or number plates, or outdated vehicle registration no owner/holder information shall be provided. Instead, the message Information not disclosed shall be returned. (1) M = mandatory when available in national register, O = optional. (2) Harmonised code, see Council Directive 1999/37/EC of 29 April 1999 on the registration documents for vehicles (OJ L 138, 1.6.1999, p. 57). (3) M = mandatory when available in national register, O = optional. (4) Harmonised code, see Directive 1999/37/EC. (5) M = mandatory when available in national register, O = optional. (6) Harmonised code, see Directive 1999/37/EC. ANNEX II Text of image TEMPLATE FOR THE INFORMATION LETTER referred to in Article 5 [Cover page] [Name, address and telephone number of sender] [Name and address of addressee] INFORMATION LETTER regarding a road-safety-related traffic offence committed in [name of the Member State of the offence] Text of image Page 2 On a road-safety-related traffic offence committed with the vehicle with registration [date] number make model was detected by [name of the responsible body] [Option 1] (1) You are registered as the holder of the registration certificate of the abovementioned vehicle. [Option 2] (1) The holder of the registration certificate of the abovementioned vehicle indicated that you were driving that vehicle when the road-safety-related traffic offence was committed. The relevant details of the offence are described on page 3 below. The amount of the financial penalty due for this offence is EUR/national currency. Deadline for the payment is You are advised to complete the attached reply form (page 4) and send it to the address shown, if you do not pay this financial penalty. This letter shall be processed in accordance with the national law of [name of the Member State of the offence]. Text of image Page 3 Relevant details concerning the offence (a) Data concerning the vehicle with which the offence was committed: Registration number: Member State of registration: Make and model: (b) Data concerning the offence: Place, date and time where the offence was committed: Nature and legal classification of the offence: speeding, failing to use a seatbelt, failing to stop at a red traffic light, drink-driving, driving under the influence of drugs, failing to wear a safety helmet, use of a forbidden lane, illegally using a mobile telephone or any other communication devices while driving (1) Detailed description of the offence: Reference to the relevant legal provision(s): Description of or reference to the evidence for the offence: Text of image (c) Data concerning the device that was used for detecting the offence (2): Type of device for detection of speeding, failing to use a seatbelt, failing to stop at a red traffic light, drink-driving, driving under the influence of drugs, failing to wear a safety helmet, use of a forbidden lane, illegally using a mobile telephone or any other communication devices while driving (1): Specification of the device: Identification number of the device: Expiry date for the last gauging: (d) The result of the application of the device: [example for speeding; other offences to be added:] The maximum speed: The measured speed: The measured speed corrected for margin of error: (1) Delete if not applicable. (2) Not applicable if no device has been used. Text of image Page 4 Reply form (please complete using block capitals) A. Identity of the driver:  Full name:  Place and date of birth:  Number of driving licence: delivered (date): and at (place):  Address: B. List of questions: 1. Is the vehicle, make , registration number , registered in your name? yes/no (1) If not, the holder of the registration certificate is: (name, first name, address) 2. Do you acknowledge that you committed the offence? yes/no (1) 3. If you do not acknowledge this, please explain why: Please send the completed form within 60 days from the date of this information letter to the following authority: at the following address: INFORMATION This case will be examined by the competent authority of [name of the Member State of the offence] If this case is not pursued, you will be informed within 60 days after receipt of the reply form. (1) Delete if not applicable. Text of image If this case is pursued, the following procedure applies: [to be filled in by the Member State of the offence  what the further procedure will be, including details of the possibility and procedure of appeal against the decision to pursue the case. These details shall in any event include: name and address of the authority in charge of pursuing the case; deadline for payment; name and address of the body of appeal concerned; deadline for appeal]. This letter as such does not lead to legal consequences.
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32006L0126
        'status': In Force
        'act_type': Directive
        'treaty': TEC (1992)

        full_doc :

        30.12.2006 EN Official Journal of the European Union L 403/18 DIRECTIVE 2006/126/EC OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 20 December 2006 on driving licences (Recast) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 71 thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the procedure laid down in Article 251 of the Treaty (2), Whereas: (1) Council Directive 91/439/EEC of 29 July 1991 on driving licences (3) has been significantly amended on several occasions. Now that new amendments are being made to the said Directive, it is desirable, in order to clarify matters, that the provisions in question should be recast. (2) The rules on driving licences are essential elements of the common transport policy, contribute to improving road safety, and facilitate the free movement of persons taking up residence in a Member State other than the one issuing the licence. Given the importance of individual means of transport, possession of a driving licence duly recognised by a host Member State promotes free movement and freedom of establishment of persons. Despite the progress achieved with harmonising the rules on driving licences, significant differences have persisted between Member States in the rules on periodicity of licences renewal and on subcategories of vehicles, which needed to be harmonised more fully, in order to contribute to the implementation of Community policies. (3) The possibility of laying down national provisions with regard to the period of validity provided for in Directive 91/439/EEC leads to the co-existence of different rules in different Member States and over 110 different models of driving licences valid in the Member States. This creates problems of transparency for citizens, police forces and the administrations responsible for the administration of driving licences and leads to the falsification of documents which sometimes date back several decades. (4) In order to prevent the single European driving licence model from becoming an additional model to the 110 already in circulation, Member States should take all necessary measures to issue this single model to all licence holders. (5) This Directive should not prejudice existing entitlements to drive granted or acquired before its date of application. (6) Driving licences are mutually recognised. Member States should be able to apply the period of validity prescribed by this Directive to a licence without a limited administrative validity issued by another Member State and whose holder has resided on their territory for more than two years. (7) The introduction of a period of administrative validity for new driving licences should make it possible to apply at the time of periodic renewal the most recent counter-falsification measures and the medical examinations or other measures provided for by the Member States. (8) On road safety grounds, the minimum requirements for the issue of a driving licence should be laid down. Standards for driving tests and licensing need to be harmonised. To this end the knowledge, skills and behaviour connected with driving motor vehicles should be defined, the driving test should be based on these concepts and the minimum standards of physical and mental fitness for driving such vehicles should be redefined. (9) Proof of fulfilment of compliance with minimum standards of physical and mental fitness for driving by drivers of vehicles used for the transport of persons or goods should be provided when the driving licence is issued and periodically thereafter. Such regular control in accordance with national rules of compliance with minimum standards will contribute to the free movement of persons, avoid distortions of competition and better take into account the specific responsibility of drivers of such vehicles. Member States should be allowed to impose medical examinations as a guarantee of compliance with the minimum standards of physical and mental fitness for driving other motor vehicles. For reasons of transparency, such examinations should coincide with a renewal of driving licences and therefore be determined by the period of validity of the licence. (10) It is necessary to strengthen further the principle of progressive access to the categories of two-wheeled vehicles and to the categories of vehicles used for the transport of passengers and goods. (11) Nevertheless, Member States should be allowed to set a higher age limit for the driving of certain categories of vehicles in order to further promote road safety; Member States should in exceptional circumstances be allowed to set lower age limits in order to take account of national circumstances. (12) The definitions of the categories should reflect to a greater extent the technical characteristics of the vehicles concerned and the skills needed to drive a vehicle. (13) Introducing a category of driving licences for mopeds will, in particular, increase road safety as regards the youngest drivers who, according to the statistics, are the hardest hit by road accidents. (14) Specific provisions should be adopted to make it easier for physically disabled persons to drive vehicles. (15) For reasons connected with road safety, Member States should be able to apply their national provisions on the withdrawal, suspension, renewal and cancellation of driving licences to all licence holders having acquired normal residence in their territory. (16) The model driving licence as set out in Directive 91/439/EEC should be replaced by a single model in the form of a plastic card. At the same time, this model driving licence needs to be adapted on account of the introduction of a new category of driving licences for mopeds and of a new category of driving licences for motorcycles. (17) The introduction of an optional microchip in the new plastic card model driving licence should enable the Member States to further improve the level of anti-fraud protection. Member States should have flexibility to include national data on the chip provided that it does not interfere with commonly accessible data. The technical requirements for the microchip should be determined by the Commission, assisted by the committee on driving licences. (18) Minimum standards concerning access to the profession of examiner and examiner training requirements should be established in order to improve the knowledge and skills of examiners thereby ensuring a more objective evaluation of driving licence applicants and achieving greater harmonisation of driving tests. (19) The Commission should be allowed to undertake the adaptation of Annexes I to VI to scientific and technical progress. (20) The measures necessary for the implementation of this Directive should be adopted in accordance with Council Decision 1999/468/EC of 28 June 1999 laying down the procedures for the exercise of implementing powers conferred on the Commission (4). (21) In particular, the Commission should be empowered to establish the criteria necessary for the application of this Directive. Since those measures are of general scope and are designed to amend non-essential elements of this Directive, they should be adopted in accordance with the regulatory procedure with scrutiny provided for in Article 5a of Decision 1999/468/EC. (22) Since the objectives of this Directive cannot be sufficiently achieved by the Member States and can therefore, by reason of their scale and their effects, be better achieved at Community level, the Community may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty. In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary in order to achieve those objectives. (23) This Directive should not prejudice the obligations of the Member States relating to the deadlines for transposition into national law and application of the Directives listed in Annex VII, Part B, HAVE ADOPTED THIS DIRECTIVE: Article 1 Model licence 1. Member States shall introduce a national driving licence based on the Community model set out in Annex I, in accordance with the provisions of this Directive. The emblem on page 1 of the Community model driving licences shall contain the distinguishing sign of the Member State issuing the licence. 2. Without prejudice to data protection rules, Member States may introduce a storage medium (microchip) as part of the driving licence, as soon as the requirements concerning the microchip referred to in Annex I, which are designed to amend non-essential elements of this Directive, by supplementing it, are laid down by the Commission in accordance with the procedure referred to in Article 9(2). These requirements shall provide for EC type-approval, which shall only be granted when the ability to resist attempts to tamper with or alter data is demonstrated. 3. The microchip shall incorporate the harmonised driving licence data specified in Annex I. After consulting the Commission, Member States may store additional data, provided that it does not in any way interfere with the implementation of this Directive. In accordance with the procedure referred to in Article 9(2), the Commission may amend Annex I in order to guarantee future interoperability. 4. With the agreement of the Commission, Member States may make to the model set out in Annex I such adjustments as are necessary for computer processing of the driving licence. Article 2 Mutual recognition 1. Driving licences issued by Member States shall be mutually recognised. 2. When the holder of a valid national driving licence without the administrative validity period set out in Article 7(2) takes up normal residence in a Member State other than that which issued the driving licence, the host Member State may apply to the licence the administrative validity periods set out in that Article by renewing the driving licence, as from 2 years after the date on which the holder has taken up normal residence on its territory. Article 3 Anti-forgery measures 1. Member States shall take all necessary steps to avoid any risk of forgery of driving licences, including that of model driving licences issued before the entry into force of this Directive. They shall inform the Commission thereof. 2. The material used for the driving licence, as set out in Annex I, shall be made secure against forgery in application of specifications designed to amend non-essential elements of this Directive, by supplementing it, which are to be laid down by the Commission in accordance with the procedure referred to in Article 9(2). Member States are free to introduce additional security features. 3. Member States shall ensure that, by 19 January 2033, all driving licences issued or in circulation fulfil all the requirements of this Directive. Article 4 Categories, definitions and minimum ages 1. The driving licence provided for in Article 1 shall authorise the driving of power-driven vehicles in the categories defined hereafter. It may be issued from the minimum age indicated for each category. A power-driven vehicle means any self-propelled vehicle running on a road under its own power, other than a rail-borne vehicle. 2. mopeds: Category AM:  Two-wheel vehicles or three-wheel vehicles with a maximum design speed of not more than 45 km/h, as defined in Article 1(2)(a) of Directive 2002/24/EC of the European Parliament and of the Council of 18 March 2002 relating to the type-approval of two or three-wheel motor vehicles (5) (excluding those with a maximum design speed under or equal to 25 km/h), and light quadricycles as defined in Article 1(3)(a) of Directive 2002/24/EC,  the minimum age for category AM is fixed at 16 years; 3. motorcycles with or without a sidecar and motor tricycles:  motorcycle means two-wheel vehicles with or without a sidecar, as defined in Article 1(2)(b) of Directive 2002/24/EC,  motor tricycle means vehicles with three symmetrically arranged wheels, as defined in Article 1(2)(c) of Directive 2002/24/EC; (a) Category A1:  motorcycles with a cylinder capacity not exceeding 125 cubic centimetres, of a power not exceeding 11 kW and with a power/weight ratio not exceeding 0,1 kW/kg,  motor tricycles with a power not exceeding 15 kW,  the minimum age for category A1 is fixed at 16 years; (b) Category A2:  motorcycles of a power not exceeding 35 kW and with a power/weight ratio not exceeding 0,2 kW/kg and not derived from a vehicle of more than double its power,  the minimum age for category A2 is fixed at 18 years; (c) Category A: (i) motorcycles  The minimum age for category A is fixed at 20 years. However, access to the driving of motorcycles of this category shall be subject to a minimum of two years' experience on motorcycles under an A2 licence. This requirement as to previous experience may be waived if the candidate is at least 24 years old. (ii) motor tricycles with a power exceeding 15 kW  The minimum age for motor tricycles exceeding 15 kW is fixed at 21 years. 4. motor vehicles:  motor vehicle means any power-driven vehicle, which is normally used for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage of persons or goods. This term shall include trolleybuses, i.e. vehicles connected to an electric conductor and not rail-borne. It shall not include agricultural or forestry tractors,  Agricultural or forestry tractor means any power-driven vehicle running on wheels or tracks, having at least two axles, the principal function of which lies in its tractive power, which is specially designed to pull, push, carry or operate certain tools, machines or trailers used in connection with agricultural or forestry operations, and the use of which for carrying persons or goods by road or drawing, on the road, vehicles used for the carriage of persons or goods is only a secondary function; (a) Category B1:  quadricycles, as defined in Article 1(3)(b) of Directive 2002/24/EC,  the minimum age for category B1 is fixed at 16 years,  category B1 is optional; in Member States which do not introduce this category of driving licence, a driving licence for category B shall be required to drive such vehicles; (b) Category B: motor vehicles with a maximum authorised mass not exceeding 3 500 kg and designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg. Without prejudice to the provisions of type-approval rules for the vehicles concerned, motor vehicles in this category may be combined with a trailer with a maximum authorised mass exceeding 750 kg, provided that the maximum authorised mass of this combination does not exceed 4 250 kg. In case such a combination exceeds 3 500 kg, Member States shall, in accordance with the provisions of Annex V, require that this combination shall only be driven after:  a training has been completed, or  a test of skills and behaviour has been passed. Member States may also require both such a training and the passing of a test of skills and behaviour. Member States shall indicate the entitlement to drive such a combination on the driving licence by means of the relevant Community code. The minimum age for category B is fixed at 18 years; (c) Category BE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combination of vehicles consisting of a tractor vehicle in category B and a trailer or semi-trailer where the maximum authorised mass of the trailer or semi-trailer does not exceed 3 500 kg,  the minimum age for category BE is fixed at 18 years; (d) Category C1: motor vehicles other than those in categories D1 or D, the maximum authorised mass of which exceeds 3 500 kg, but does not exceed 7 500 kg, and which are designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass not exceeding 750 kg; (e) Category C1E:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category C1 and its trailer or semi-trailer has a maximum authorised mass of over 750 kg provided that the authorised mass of the combination does not exceed 12 000 kg,  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category B and its trailer or semi-trailer has an authorised mass of over 3 500 kg, provided that the authorised mass of the combination does not exceed 12 000 kg,  the minimum age for categories C1 and C1E is fixed at the age of 18 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC of the European Parliament and of the Council of 15 July 2003 on the initial qualification and periodic training of drivers of certain road vehicles for the carriage of goods or passengers (6); (f) Category C: motor vehicles other than those in categories D1 or D, whose maximum authorised mass is over 3 500 kg and which are designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg; (g) Category CE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category C and its trailer or semi-trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories C and CE is fixed at 21 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; (h) Category D1: motor vehicles designed and constructed for the carriage of no more than 16 passengers in addition to the driver and with a maximum length not exceeding 8 m; motor vehicles in this category may be combined with a trailer having a maximum authorised mass not exceeding 750 kg; (i) Category D1E:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category D1 and its trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories D1 and D1E is fixed at 21 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; (j) Category D: motor vehicles designed and constructed for the carriage of more than eight passengers in addition to the driver; motor vehicles which may be driven with a category D licence may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg; (k) Category DE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category D and its trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories D and DE is fixed at 24 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; 5. With the agreement of the Commission, Member States may exclude from the application of this Article certain specific types of power-driven vehicle such as special vehicles for disabled persons. Member States may exclude from the application of this Directive vehicles used by, or under the control of, the armed forces and civil defence. 6. Member States may raise or lower the minimum age for issuing a driving licence: (a) for category AM down to 14 years or up to 18 years; (b) for category B1 up to 18 years; (c) for category A1 up to 17 or 18 years,  if there is a two years difference between the minimum age for category A1 and the minimum age for category A2, and  there is a requirement of a minimum of two years experience on motorcycles of category A2 before access to the driving of motorcycles for category A can be granted, as referred to in Article 4(3)(c)(i); (d) for categories B and BE down to 17 years. Member States may lower the minimum age for category C to 18 years and for category D to 21 years with regard to: (a) vehicles used by the fire service and vehicles used for maintaining public order; (b) vehicles undergoing road tests for repair or maintenance purposes. Driving licences issued to persons at a lower age than set out in paragraphs 2 to 4 in accordance with this paragraph shall only be valid on the territory of the issuing Member State until the licence holder has reached the minimum age limit set out in paragraphs 2 to 4. Member States may recognise the validity on their territory of driving licences issued to drivers under the minimum ages set out in paragraphs 2 to 4. Article 5 Conditions and restrictions 1. Driving licences shall state the conditions under which the driver is authorised to drive. 2. If, because of a physical disability, driving is authorised only for certain types of vehicle or for adapted vehicles, the test of skills and behaviour provided for in Article 7 shall be taken in such a vehicle. Article 6 Staging and equivalences between categories 1. The issue of driving licences shall be subject to the following conditions: (a) licences for categories C1, C, D1 and D shall be issued only to drivers already entitled to drive vehicles in category B; (b) licences for categories BE, C1E, CE, D1E and DE shall be issued only to drivers already entitled to drive vehicles in categories B, C1, C, D1 and D respectively. 2. The validity of driving licences shall be determined as follows: (a) licences granted for categories C1E, CE, D1E or DE shall be valid for combinations of vehicles in category BE; (b) licences granted for category CE shall be valid for category DE as long as their holders are entitled to drive vehicles in category D; (c) licences granted for category CE and DE shall be valid for combinations of vehicles in categories C1E and D1E respectively; (d) licences granted for any category shall be valid for vehicles in category AM. However, for driving licences issued on its territory, a Member State may limit the equivalences for category AM to categories A1, A2 and A, if that Member State imposes a practical test as a condition for obtaining category AM; (e) licences issued for category A2 shall also be valid for category A1; (f) licences granted for categories A, B, C or D shall be valid for categories A1, A2, B1, C1, or D1 respectively. 3. For driving on their territory, Member States may grant the following equivalences: (a) motor tricycles under a licence for category B, for motor tricycles with a power exceeding 15 kW provided that the holder of the licence for category B is at least 21 years old; (b) category A1 motorcycles under a licence for category B. As this paragraph is only valid on their territories, Member States shall not indicate on the driving licence that a holder is entitled to drive these vehicles. 4. Member States may, after consulting the Commission, authorise the driving on their territory of: (a) vehicles of category D1 (with a maximum authorised mass of 3 500 kg, excluding any specialised equipment intended for the carriage of disabled passengers) by holders over 21 years old of a driving licence for category B which was obtained at least two years earlier provided that the vehicles are being used by non-commercial bodies for social purposes and that the driver provides his services on a voluntary basis; (b) vehicles of a maximum authorised mass exceeding 3 500 kg by holders over 21 years old of a driving licence for category B which was obtained at least two years before, provided that the main purpose of the vehicles is to be used only when stationary as an instructional or recreational area, and that they are being used by non-commercial bodies for social purposes and that vehicles have been modified so that they may not be used either for the transport of more than nine persons or for the transport of any goods other than those strictly necessary for their purposes. Article 7 Issue, validity and renewal 1. Driving licences shall be issued only to those applicants: (a) who have passed a test of skills and behaviour and a theoretical test and who meet medical standards, in accordance with the provisions of Annexes II and III; (b) who have passed a theory test only as regards category AM; Member States may require applicants to pass a test of skills and behaviour and a medical examination for this category. For tricycles and quadricycles within this category, Member States may impose a distinctive test of skills and behaviour. For the differentiation of vehicles in category AM, a national code may be inserted on the driving licence; (c) who have, as regards category A2 or category A, on the condition of having acquired a minimum of 2 years' experience on a motorcycle in category A1 or in category A2 respectively, passed a test of skills and behaviour only, or completed a training pursuant to Annex VI; (d) who have completed a training or passed a test of skills and behaviour, or completed a training and passed a test of skills and behaviour pursuant to Annex V as regards category B for driving a vehicle combination as defined in the second subparagraph of Article 4(4)(b); (e) who have their normal residence in the territory of the Member State issuing the licence, or can produce evidence that they have been studying there for at least six months. 2. (a) As from 19 January 2013, licences issued by Member States for categories AM, A1, A2, A, B, B1 and BE shall have an administrative validity of 10 years. A Member State may choose to issue such licences with an administrative validity of up to 15 years; (b) As from 19 January 2013, licences issued by Member States for categories C, CE, C1, C1E, D, DE, D1, D1E shall have an administrative validity of 5 years; (c) The renewal of a driving licence may trigger a new administrative validity period for another category or categories the licence holder is entitled to drive, insofar as this is in conformity with the conditions laid down in this Directive; (d) The presence of a microchip pursuant to Article 1 shall not be a prerequisite for the validity of a driving licence. The loss or unreadability of the microchip, or any other damage thereto, shall not affect the validity of the document. 3. The renewal of driving licences when their administrative validity expires shall be subject to: (a) continuing compliance with the minimum standards of physical and mental fitness for driving set out in Annex III for driving licences in categories C, CE, C1, C1E, D, DE, D1, D1E; and (b) normal residence in the territory of the Member State issuing the licence, or evidence that applicants have been studying there for at least six months. Member States may, when renewing driving licences in categories AM, A, A1, A2, B, B1 and BE, require an examination applying the minimum standards of physical and mental fitness for driving set out in Annex III. Member States may limit the period of administrative validity set out in paragraph 2 of driving licences issued to novice drivers for any category in order to apply specific measures to such drivers, aiming at improving road safety. Member States may limit the period of administrative validity of the first licence issued to novice drivers for categories C and D to 3 years in order to be able to apply specific measures to such drivers, so as to improve their road safety. Member States may limit the period of administrative validity set out in paragraph 2 of individual driving licences for any category in case it is found necessary to apply an increased frequency of medical checks or other specific measures such as restrictions for traffic offenders. Member States may reduce the period of administrative validity set out in paragraph 2 of driving licences of holders residing on their territory having reached the age of 50 years in order to apply an increased frequency of medical checks or other specific measures such as refresher courses. This reduced period of administrative validity can only be applied upon renewing the driving licence. 4. Without prejudice to national criminal and police laws, Member States may, after consulting the Commission, apply to the issuing of driving licences the provisions of their national rules relating to conditions other than those referred to in this Directive. 5. (a) No person may hold more than one driving licence; (b) A Member State shall refuse to issue a licence where it establishes that the applicant already holds a driving licence; (c) Member States shall take the necessary measures pursuant to point (b). The necessary measures as regards the issue, replacement, renewal or exchange of a driving licence shall be to verify with other Member States where there are reasonable grounds to suspect that the applicant is already the holder of another driving licence; (d) In order to facilitate the checks pursuant to point (b), Member States shall use the EU driving licence network once it is operational. Without prejudice to Article 2, a Member State issuing a licence shall apply due diligence to ensure that a person fulfils the requirements set out in paragraph 1 of this Article and shall apply its national provisions on the cancellation or withdrawal of the right to drive if it is established that a licence has been issued without the requirements having been met. Article 8 Adaptation to scientific and technical progress The amendments necessary to adapt Annexes I to VI to scientific and technical progress shall be adopted in accordance with the procedure referred to in Article 9(2). Article 9 Committee 1. The Commission shall be assisted by the committee on driving licences. 2. Where reference is made to this paragraph, Article 5a(1) to (4), and Article 7 of Decision 1999/468/EC shall apply, having regard to the provisions of Article 8 thereof. Article 10 Examiners From the entry into force of this Directive, driving examiners shall meet the minimum standards set out in Annex IV. Driving examiners already working in that capacity before 19 January 2013 shall be subject only to the requirements concerning quality assurance and regular periodic training measures. Article 11 Various provisions concerning the exchange, the withdrawal, the replacement and the recognition of driving licences 1. Where the holder of a valid national driving licence issued by a Member State has taken up normal residence in another Member State, he may request that his driving licence be exchanged for an equivalent licence. It shall be for the Member State effecting the exchange to check for which category the licence submitted is in fact still valid. 2. Subject to observance of the principle of territoriality of criminal and police laws, the Member State of normal residence may apply its national provisions on the restriction, suspension, withdrawal or cancellation of the right to drive to the holder of a driving licence issued by another Member State and, if necessary, exchange the licence for that purpose. 3. The Member State effecting the exchange shall return the old licence to the authorities of the Member State which issued it and give the reasons for doing so. 4. A Member State shall refuse to issue a driving licence to an applicant whose driving licence is restricted, suspended or withdrawn in another Member State. A Member State shall refuse to recognise the validity of any driving licence issued by another Member State to a person whose driving licence is restricted, suspended or withdrawn in the former State's territory. A Member State may also refuse to issue a driving licence to an applicant whose licence is cancelled in another Member State. 5. A replacement for a driving licence which has, for example, been lost or stolen may only be obtained from the competent authorities of the Member State in which the holder has his normal residence; those authorities shall provide the replacement on the basis of the information in their possession or, where appropriate, proof from the competent authorities of the Member State which issued the original licence. 6. Where a Member State exchanges a driving licence issued by a third country for a Community model driving licence, such exchange shall be recorded on the Community model driving licence as shall any subsequent renewal or replacement. Such an exchange may occur only if the licence issued by the third country has been surrendered to the competent authorities of the Member State making the exchange. If the holder of this licence transfers his normal residence to another Member State, the latter need not apply the principle of mutual recognition set out in Article 2. Article 12 Normal residence For the purpose of this Directive, normal residence means the place where a person usually lives, that is for at least 185 days in each calendar year, because of personal and occupational ties, or, in the case of a person with no occupational ties, because of personal ties which show close links between that person and the place where he is living. However, the normal residence of a person whose occupational ties are in a different place from his personal ties and who consequently lives in turn in different places situated in two or more Member States shall be regarded as being the place of his personal ties, provided that such person returns there regularly. This last condition need not be met where the person is living in a Member State in order to carry out a task of a definite duration. Attendance at a university or school shall not imply transfer of normal residence. Article 13 Equivalences between non-Community model licences 1. With the agreement of the Commission, Member States shall establish equivalences between entitlements obtained before the implementation of this Directive and the categories defined in Article 4. After consulting the Commission, Member States may make to their national legislation such adjustments as are necessary for the purpose of implementing the provisions of Article 11(4), (5) and (6). 2. Any entitlement to drive granted before 19 January 2013 shall not be removed or in any way qualified by the provisions of this Directive. Article 14 Review The Commission shall report on the implementation of this Directive, including its impact on road safety, not earlier than 19 January 2018. Article 15 Mutual Assistance Member States shall assist one another in the implementation of this Directive and shall exchange information on the licences they have issued, exchanged, replaced, renewed or revoked. They shall use the EU driving licence network set up for these purposes, once this network is operational. Article 16 Transposition 1. Member States shall adopt and publish, not later than 19 January 2011, the laws, regulations and administrative provisions necessary to comply with Article 1(1), Article 3, Article 4(1), (2), (3) and (4)(b) to (k), Article 6(1), (2)(a), (c), (d) and (e), Article 7(1)(b), (c) and (d), (2), (3) and (5), Article 8, Article 10, Article 13, Article 14, Article 15, and Annexes I, point 2, II, point 5.2 concerning categories A1, A2 and A, IV, V and VI. They shall forthwith communicate to the Commission the text of those provisions. 2. They shall apply those provisions as from 19 January 2013. 3. When Member States adopt those provisions, they shall contain a reference to this Directive or shall be accompanied by such reference on the occasion of their official publication. They shall also contain an indication that references made, in the laws, regulations or administrative provisions in force, to the repealed Directive shall be construed as being made to this Directive. The methods of making such reference, and its wording, shall be laid down by Member States. 4. Member States shall communicate to the Commission the text of the main provisions of national law which they adopt in the field covered by this Directive. Article 17 Repeal Directive 91/439/EEC shall be repealed with effect from 19 January 2013, without prejudice to the obligations of the Member States with regard to the deadlines indicated in Annex VII, Part B for transposing that Directive into national law. Article 2(4) of Directive 91/439/EEC shall be repealed on 19 January 2007. References made to the repealed Directive shall be construed as being made to this Directive and should be read in accordance with the correlation table in Annex VIII. Article 18 Entry into force This Directive shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union. Article 2(1), Article 5, Article 6(2)(b), Article 7(1)(a), Article 9, Article 11(1), (3), (4), (5) and (6), Article 12, and Annexes I, II and III shall apply from 19 January 2009. Article 19 Addressees This Directive is addressed to the Member States. Done at Brussels, 20 December 2006. For the European Parliament The President J. BORRELL FONTELLES For the Council The President J. KORKEAOJA (1) OJ C 112, 30.4.2004, p. 34. (2) Opinion of the European Parliament of 23 February 2005 (OJ C 304 E, 1.12.2005, p. 202), Council Common Position of 18 September 2006 (OJ C 295 E, 5.12.2006, p. 1) and Position of the European Parliament of 14 December 2006 (not yet published in the Official Journal). Council Decision of 19 December 2006. (3) OJ L 237, 24.8.1991, p. 1. Directive as last amended by Regulation (EC) No 1882/2003 of the European Parliament and of the Council (OJ L 284, 31.10.2003, p. 1). (4) OJ L 184, 17.7.1999, p. 23. Decision as amended by Decision 2006/512/EC (OJ L 200, 22.7.2006, p. 11). (5) OJ L 124, 9.5.2002, p. 1. Directive as last amended by Commission Directive 2005/30/EC (OJ L 106, 27.4.2005, p. 17). (6) OJ L 226, 10.9.2003, p. 4. Directive as amended by Council Directive 2004/66/EC (OJ L 168, 1.5.2004, p. 35). ANNEX I PROVISIONS CONCERNING THE COMMUNITY MODEL DRIVING LICENCE 1. The physical characteristics of the card of the Community model driving licence shall be in accordance with ISO 7810 and ISO 7816-1. The card shall be made of polycarbonate. Methods for testing the characteristics of driving licences for the purpose of confirming their compliance with the international standards shall be in accordance with ISO 10373. 2. Physical security of driving licences The threats to the physical security of driving licences are:  production of false cards: creating a new object which bears great resemblance to the document, either by making it from scratch or by copying an original document,  material alteration: changing a property of an original document, e.g. modifying some of the data printed on the document; The overall security lies in the system in its entirety, consisting of the application process, the transmission of data, the card body material, the printing technique, a minimum set of different security features and the personalisation process. (a) The material used for driving licences shall be made secure against forgery by using the following techniques (mandatory security features):  card bodies shall be UV dull,  a security background pattern designed to be resistant to counterfeit by scanning, printing or copying, using rainbow printing with multicolour security inks and positive and negative guilloche printing. The pattern shall not be composed of the primary colours (CMYK), shall contain complex pattern designs in a minimum of two special colours and shall include micro lettering,  optical variable elements providing adequate protection against copying and tampering of the photograph,  laser engraving,  in the area of the photograph the security design background and photograph should overlap on at least its border (weakening pattern). (b) In addition, the material used for driving licences shall be made secure against forgery by using at least three of the following techniques (additional security features):  colour-shifting inks*,  termochromic ink*,  custom holograms*,  variable laser images*,  ultraviolet fluorescent ink, visible and transparent,  iridescent printing,  digital watermark in the background,  infrared or phosphorescent pigments,  tactile characters, symbols or patterns*. (c) Member States are free to introduce additional security features. As a basis, the techniques indicated with an asterisk are to be preferred as they enable the law enforcement officers to check the validity of the card without any special means. 3. The licence shall have two sides. Page 1 shall contain: (a) the words Driving Licence printed in large type in the language or languages of the Member State issuing the licence; (b) the name of the Member State issuing the licence (optional); (c) the distinguishing sign of the Member State issuing the licence, printed in negative in a blue rectangle and encircled by twelve yellow stars; the distinguishing signs shall be as follows: B : Belgium CZ : Czech Republic DK : Denmark D : Germany EST : Estonia GR : Greece E : Spain F : France IRL : Ireland I : Italy CY : Cyprus LV : Latvia LT : Lithuania L : Luxembourg H : Hungary M : Malta NL : The Netherlands A : Austria PL : Poland P : Portugal SLO : Slovenia SK : Slovakia FIN : Finland S : Sweden UK : The United Kingdom; (d) information specific to the licence issued, numbered as follows: 1. surname of the holder; 2. other name(s) of the holder; 3. date and place of birth; 4. (a) date of issue of the licence; (b) date of expiry of the licence or a dash if the licence is valid indefinitely under the provision of Article 7(2)(c); (c) the name of the issuing authority (may be printed on page 2); (d) a different number from the one under heading 5, for administrative purposes (optional); 5. number of the licence; 6. photograph of the holder; 7. signature of the holder; 8. permanent place of residence, or postal address (optional); 9. category of vehicle(s) the holder is entitled to drive (national categories shall be printed in a different type from harmonised categories); (e) the words European Communities model in the language(s) of the Member State issuing the licence and the words Driving Licence in the other languages of the Community, printed in pink to form the background of the licence: Permiso de ConducciÃ ³n Ã idiÃ skÃ ½ prÃ ¯kaz KÃ ¸rekort FÃ ¼hrerschein Juhiluba Ã Ã ´Ã µÃ ¹Ã ± Ã Ã ´Ã ®Ã ³Ã ·Ã Ã ·Ã  Driving Licence Permis de conduire CeadÃ ºas TiomÃ ¡na Patente di guida VadÃ «tÃ ja apliecÃ «ba Vairuotojo paÃ ¾ymÃ jimas VezetÃ i engedÃ ©ly LiÃ enzja tas-Sewqan Rijbewijs Prawo Jazdy Carta de ConduÃ §Ã £o VodiÃ skÃ ½ preukaz VozniÃ ¡ko dovoljenje Ajokortti KÃ ¶rkort; (f) Colour references:  blue: Pantone Reflex Blue,  yellow: Pantone Yellow. Page 2 shall contain: (a) 9. category of vehicle(s) the holder is entitled to drive (national categories shall be printed in a different type from harmonised categories); 10. date of first issue of each category (this date must be repeated on the new licence in the event of subsequent replacement or exchange); 11. date of expiry of each category; 12. additional information/restriction(s), in code form, facing the (sub)category affected. The codes shall be as follows:  codes 01 to 99 : harmonised Community codes DRIVER (Medical reasons) 01. Sight correction and/or protection 01.01 Glasses 01.02 Contact lense(s) 01.03 Protective glass 01.04 Opaque lense 01.05 Eye cover 01.06 Glasses or contact lenses 02. Hearing aid/communication aid 02.01 Hearing aid for one ear 02.02 Hearing aid for two ears 03. Prosthesis/orthosis for the limbs 03.01 Upper limb prosthesis/orthosis 03.02 Lower limb prosthesis/orthosis 05. Limited use (subcode use obligatory, driving subject to restrictions for medical reasons) 05.01 Limited to day time journeys (for example: one hour after sunrise and one hour before sunset) 05.02 Limited to journeys within a radius of ¦ km from holder's place of residence or only inside city/region 05.03 Driving without passengers 05.04 Limited to journeys with a speed not greater than ¦ km/h 05.05 Driving authorised solely when accompanied by a holder of a driving licence 05.06 Without trailer 05.07 No driving on motorways 05.08 No alcohol VEHICLE ADAPTATIONS 10. Modified transmission 10.01 Manual transmission 10.02 Automatic transmission 10.03 Electronically operated transmission 10.04 Adjusted gear-shift lever 10.05 Without secondary gearbox 15. Modified clutch 15.01 Adjusted gear-shift lever 15.02 Manual clutch 15.03 Automatic clutch 15.04 Partitioning in front of/fold away/detached clutch pedal 20. Modified braking systems 20.01 Adjusted brake pedal 20.02 Enlarged brake pedal 20.03 Brake pedal suitable for use by left foot 20.04 Brake pedal by sole 20.05 Tilted brake pedal 20.06 Manual (adapted) service brake 20.07 Maximum use of reinforced service brake 20.08 Maximum use of emergency brake integrated in the service brake 20.09 Adjusted parking brake 20.10 Electrically operated parking brake 20.11 (Adjusted) foot operated parking brake 20.12 Partitioning in front of/fold away/detached brake pedal 20.13 Brake operated by knee 20.14 Electrically operated service brake 25. Modified accelerator systems 25.01 Adjusted accelerator pedal 25.02 Accelerator pedal by sole 25.03 Tilted accelerator pedal 25.04 Manual accelerator 25.05 Accelerator at knee 25.06 Servo accelerator (electronic, pneumatic, etc.) 25.07 Accelerator pedal on the left of brake pedal 25.08 Accelerator pedal on the left 25.09 Partitioning in front of/fold away/detached accelerator pedal 30. Modified combined braking and accelerator systems 30.01 Parallel pedals 30.02 Pedals at (or almost at) the same level 30.03 Accelerator and brake with sliding 30.04 Accelerator and brake with sliding and orthesis 30.05 Fold away/detached accelerator and brake pedals 30.06 Raised floor 30.07 Partitioning on the side of the brake pedal 30.08 Partitioning for prosthesis on the side of the brake pedal 30.09 Partitioning in front of the accelerator and brake pedals 30.10 Heel/leg support 30.11 Electrically operated accelerator and brake 35. Modified control layouts (Lights switches, windscreen wiper/washer, horn, direction indicators, etc.) 35.01 Control devices operable without negative influence on the steering and handling 35.02 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) 35.03 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) with the left hand 35.04 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) with the right hand 35.05 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) and the combined accelerator and braking mechanismss 40. Modified steering 40.01 Standard assisted steering 40.02 Reinforced assisted steering 40.03 Steering with backup system 40.04 Lengthened steering column 40.05 Adjusted steering wheel (Larger and/or thicker steering wheel section, reduced diameter steering wheel, etc.) 40.06 Tilted steering wheel 40.07 Vertical steering wheel 40.08 Horizontal steering wheel 40.09 Foot operated driving 40.10 Alternative adjusted steering (joy-stick, etc.) 40.11 Knob on the steering wheel 40.12 Hand orthesis on the steering wheel 40.13 With orthesis tenodese 42. Modified rearview mirror(s) 42.01 External (left or) right-side rear-view mirror 42.02 External rear-view mirror set on the wing 42.03 Additional inside rear-view mirror permitting view of traffic 42.04 Panoramic inside rear-view mirror 42.05 Blind spot rear-view mirror 42.06 Electrically operated outside rear-view mirror(s) 43. Modified driver seat 43.01 Driver seat at a good viewing height and in normal distance from the steering wheel and the pedal 43.02 Driver seat adjusted to body shape 43.03 Driver seat with lateral support for good sitting stability 43.04 Driver seat with armrest 43.05 Lengthening of sliding driver's seat 43.06 Seat-belt adjustment 43.07 Harness-type seat-belt 44. Modifications to motorcycles (subcode use obligatory) 44.01 Single operated brake 44.02 (Adjusted) hand operated brake (front wheel) 44.03 (Adjusted) foot operated brake (back wheel) 44.04 (Adjusted) accelerator handle 44.05 (Adjusted) manual transmission and manual clutch 44.06 (Adjusted) rear-view mirror(s) 44.07 (Adjusted) commands (direction indicators, braking light, ¦) 44.08 Seat height allowing the driver, in sitting position, to have two feet on the road at the same time 45. Motorcycle with side-car only 50. Restricted to a specific vehicle/chassis number (vehicle identification number, VIN) 51. Restricted to a specific vehicle/registration plate (vehicle registration number, VRN) ADMINISTRATIVE MATTERS 70. Exchange of licence No ¦ issued by ¦ (EU/UN distinguishing sign in the case of a third country; e.g: 70.0123456789.NL) 71. Duplicate of licence No ¦ (EU/UN distinguishing sign in the case of a third country; e.g: 71.987654321.HR) 72. Restricted to category A vehicles having a maximum cylinder capacity of 125 cc and maximum power of 11 KW (A1) 73. Restricted to category B vehicles of the motor tricycle or quadricycle type (B1) 74. Restricted to category C vehicles the maximum authorised mass of which does not exceed 7 500 kg (C1) 75. Restricted to category D vehicles with not more than 16 seats, excluding the driver's seat (D1) 76. Restricted to category C vehicles the maximum authorised mass of which does not exceed 7 500 kg (C1), attached to a trailer the maximum authorised mass of which exceeds 750 kg, provided that the maximum authorised mass of the vehicle train thus formed does not exceed 12 000 kg, and that the maximum authorised mass of the trailer does not exceed the unladen mass of the drawing vehicle (C1E) 77. Restricted to category D vehicles with not more than 16 passenger seats, excluding the driver's seat (D1), attached to a trailer the maximum authorised mass of which exceeds 750 kg provided that (a) the maximum authorised mass of the vehicle train thus formed does not exceed 12 000 kg and the maximum authorised mass of the trailer does not exceed the unladen mass of the drawing vehicle and (b) the trailer is not used to carry passengers (D1E) 78. Restricted to vehicles with automatic transmission 79. ( ¦) Restricted to vehicles which comply with the specifications indicated in brackets, in the context of the application of Article 10(1) of Directive 91/439/EEC 90.01 : to the left 90.02 : to the right 90.03 : left 90.04 : right 90.05 : hand 90.06 : foot 90.07 : usable 95. Driver holding CPC meeting the obligation of professional aptitude provided for by Directive 2003/59/EC until ¦ [e.g.: 95.01.01.2012] 96. Driver having completed training or having passed a test of skills and behaviour in accordance with the provisions of Annex V.  codes 100 and above: : national codes valid only for driving in the territory of the Member State which issued the licence. Where a code applies to all categories for which the licence is issued, it may be printed under headings 9, 10 and 11; 13. in implementation of section 4(a) of this Annex, a space reserved for the possible entry by the host Member State of information essential for administering the licence; 14. a space reserved for the possible entry by the Member State which issues the licence of information essential for administering the licence or related to road safety (optional). If the information relates to one of the headings defined in this Annex, it should be preceded by the number of the heading in question. With the specific written agreement of the holder, information which is not related to the administration of the driving licence or road safety may also be added in this space; such addition shall not alter in any way the use of the model as a driving licence; (b) an explanation of the numbered items which appear on pages 1 and 2 of the licence (at least items 1, 2, 3, 4 (a), 4 (b), 4 (c), 5, 10, 11 and 12) If a Member State wishes to make the entries in a national language other than one of the following languages: Czech, Danish, Dutch, English, Estonian, Finnish, French, German, Greek, Hungarian, Italian, Latvian, Lithuanian, Maltese, Polish, Portuguese, Slovak, Slovenian, Spanish or Swedish, it shall draw up a bilingual version of the licence using one of the aforementioned languages, without prejudice to the other provisions of this Annex; (c) a space shall be reserved on the Community model licence to allow for the possible introduction of a microchip or similar computer device. 4. Special provisions (a) Where the holder of a driving licence issued by a Member State in accordance with this Annex has his normal place of residence in another Member State, that Member State may enter in the licence such information as is essential for administering it, provided that it also enters this type of information in the licences which it issues and provided that there remains enough space for the purpose. (b) After consulting the Commission, Member States may add colours or markings, such as bar codes and national symbols, without prejudice to the other provisions of this Annex. In the context of mutual recognition of licences, the bar code may not contain information other than what can already be read on the driving licence or which is essential to the process of issuing the licence. COMMUNITY MODEL DRIVING LICENCE Page 1 DRIVING LICENCE [MEMBER STATE] Page 2 1. Name 2. First name 3. Date and place of birth 4a. Date of issue of driving licence 4b. Official date of expiry 4c. Issued by 5. Serial number of licence 8. Place of residence 9. Category (1) 10. Date of issue, by category 11. Date of expiry, by category 12. Restrictions SPECIMEN MODEL LICENCE BELGIAN LICENCE (for information) (1) Note: a pictogram and a line for category AM will be added. Note: the term A2 will be added to the section on motorcycle categories. ANNEX II I. MINIMUM REQUIREMENTS FOR DRIVING TESTS Member States shall take the necessary measures to ensure that applicants for driving licences possess the knowledge and skills and exhibit the behaviour required for driving a motor vehicle. The tests introduced to this effect must consist of:  a theory test, and then  a test of skills and behaviour. The conditions under which these tests shall be conducted are set out below. A. THEORY TEST 1. Form The form chosen shall be such as to make sure that the applicant has the required knowledge of the subjects listed on points 2, 3 and 4. Any applicant for a licence in one category who has passed a theory test for a licence in a different category may be exempt from the common provisions of points 2, 3 and 4. 2. Content of the theory test concerning all vehicle categories 2.1. Questions must be asked on each of the points listed below, the content and form of the questions being left to the discretion of each Member State: 2.1.1. Road traffic regulations:  in particular as regards road signs, markings and signals, rights of way and speed limits; 2.1.2. The driver:  importance of alertness and of attitude to other road users,  perception, judgement and decision-taking, especially reaction time, as well as changes in driving behaviour due to the influence of alcohol, drugs and medicinal products, state of mind and fatigue; 2.1.3. The road:  the most important principles concerning the observance of a safe distance between vehicles, braking distances and road holding under various weather and road conditions,  driving risk factors related to various road conditions, in particular as they change with the weather and the time of day or night,  characteristics of various types of road and the related statutory requirements; 2.1.4. Other road users:  specific risk factors related to the lack of experience of other road users and the most vulnerable categories of users such as children, pedestrians, cyclists and people whose mobility is reduced,  risks involved in the movement and driving of various types of vehicles and of the different fields of view of their drivers; 2.1.5. General rules and regulations and other matters:  rules concerning the administrative documents required for the use of vehicles,  general rules specifying how the driver must behave in the event of an accident (setting warning devices and raising the alarm) and the measures which he can take to assist road accident victims where necessary,  safety factors relating to the vehicle, the load and persons carried; 2.1.6. Precautions necessary when alighting from the vehicle; 2.1.7. Mechanical aspects with a bearing on road safety; applicants must be able to detect the most common faults, in particular in the steering, suspension and braking systems, tyres, lights and direction indicators, reflectors, rear-view mirrors, windscreen and wipers, the exhaust system, seat-belts and the audible warning device; 2.1.8. Vehicle safety equipment and, in particular, the use of seat-belts, head restraints and child safety equipment; 2.1.9. Rules regarding vehicle use in relation to the environment (appropriate use of audible warning devices, moderate fuel consumption, limitation of pollutant emissions, etc.). 3. Specific provisions concerning categories A1, A2 and A 3.1. Compulsory check of general knowledge on: 3.1.1. Use of protective outfit such as gloves, boots, clothes and safety helmet; 3.1.2. Visibility of motorcycle riders for other road users; 3.1.3. Risk factors related to various road conditions as laid down above with additional attention to slippery parts such as drain covers, road markings such as lines and arrows, tram rails; 3.1.4. Mechanical aspects with a bearing on road safety as laid down above with additional attention to the emergency stop switch, the oil levels and the chain. 4. Specific provisions concerning categories C, CE, C1, C1E, D, DE, D1 and D1E 4.1. Compulsory check of general knowledge on: 4.1.1. Rules on driving hours and rest periods as defined by Council Regulation (EEC) No 3820/85 of 20 December 1985 on the harmonisation of certain social legislation relating to road transport (1); use of the recording equipment as defined by Council Regulation (EEC) No 3821/85 of 20 December 1985 on recording equipment in road transport (2), 4.1.2. Rules concerning the type of transport concerned: goods or passengers; 4.1.3. Vehicle and transport documents required for the national and international carriage of goods and passengers; 4.1.4. How to behave in the event of an accident; knowledge of measures to be taken after an accident or similar occurrence, including emergency action such as evacuation of passengers and basic knowledge of first aid; 4.1.5. The precautions to be taken during the removal and replacement of wheels; 4.1.6. Rules on vehicle weights and dimensions; rules on speed limiters; 4.1.7. Obstruction of the field of view caused by the characteristics of their vehicles; 4.1.8. Reading a road map, route planning, including the use of electronic navigation systems (optional); 4.1.9. Safety factors relating to vehicle loading: controlling the load (stowing and fastening), difficulties with different kinds of load (e.g. liquids, hanging loads, ¦), loading and unloading goods and the use of loading equipment (categories C, CE, C1, C1E only); 4.1.10. The driver's responsibility in respect to the carriage of passengers; comfort and safety of passengers; transport of children; necessary checks before driving away; all sorts of buses should be part of the theory test (public service buses and coaches, buses with special dimensions, ¦) (categories D, DE, D1, D1E only). 4.2. Compulsory check of general knowledge on the following additional provisions concerning categories C, CE, D and DE: 4.2.1. The principles of the construction and functioning of: internal combustion engines, fluids (e.g. engine oil, coolant, washer fluid), the fuel system, the electrical system, the ignition system, the transmission system (clutch, gearbox, etc.); 4.2.2. Lubrication and antifreeze protection; 4.2.3. The principles of the construction, the fitting, correct use and care of tyres; 4.2.4. The principles of the types, operation, main parts, connection, use and day-to-day maintenance of brake fittings and speed governors, and use of anti-lock brakes; 4.2.5. The principles of the types, operation, main parts, connection, use and day-to-day maintenance of coupling systems (categories CE, DE only); 4.2.6. Methods of locating causes of breakdowns; 4.2.7. Preventive maintenance of vehicles and necessary running repairs; 4.2.8. The driver's responsibility in respect of the receipt, carriage and delivery of goods in accordance with the agreed conditions (categories C, CE only). B. TEST OF SKILLS AND BEHAVIOUR 5. The vehicle and its equipment 5.1. The driving of a vehicle with manual transmission shall be subject to the passing of a skills and behaviour test taken on a vehicle with manual transmission. If an applicant takes the test of skills and behaviour on a vehicle with automatic transmission this shall be recorded on any licence issued on the basis of such a test. Licences with this indication shall be used only for driving vehicles with automatic transmission. Vehicle with automatic transmission means a vehicle in which the gear ratio between the engine and the wheels can be varied by use only of the accelerator or the brakes 5.2. The vehicles used in tests of skills and behaviour shall comply with the minimum criteria given below. Member States may make provisions for more stringent criteria or add others. Category A1: Category A1 motorcycle without sidecar, with a cubic capacity of at least 120 cm3, and capable of a speed of at least 90 km/h Category A2: Motorcycle without sidecar, with a cylinder capacity of at least 400 cm3, and an engine power of at least 25 kW Category A Motorcycle without sidecar, with a cylinder capacity of at least 600 cm3, and an engine power of at least 40 kW Category B: A four-wheeled category B vehicle capable of a speed of at least 100 km/h; Category BE: A combination, made up of a category B test vehicle and a trailer with a maximum authorised mass of at least 1 000 kg, capable of a speed of at least 100 km/h, which does not fall within category B; the cargo compartment of the trailer shall consist of a closed box body which is at least as wide and as high as the motor vehicle; the closed box body may also be slightly less wide than the motor vehicle provided that the view to the rear is only possible by use of the external rear-view mirrors of the motor vehicle; the trailer shall be presented with a minimum of 800 kg real total mass; Category B1: A motor-powered quadricycle capable of a speed of at least 60 km/h; Category C: A category C vehicle with a maximum authorised mass of at least 12 000 kg, a length of at least 8 m, a width of at least 2,40 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes, equipped with a gearbox having at least eight forward ratios and recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; the vehicle shall be presented with a minimum of 10 000 kg real total mass; Category CE: either an articulated vehicle or a combination of a category C test vehicle and a trailer of at least 7,5 m in length; both the articulated vehicle and the combination shall have a maximum authorised mass of at least 20 000 kg, a length of at least 14 m and a width of at least 2,40 m, shall be capable of a speed of at least 80 km/h, fitted with anti-lock brakes, equipped with a gearbox having at least eight forward ratios and with recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; both the articulated vehicle and the combination shall be presented with a minimum of 15 000 kg real total mass; Category C1: A subcategory C1 vehicle with a maximum authorised mass of at least 4 000 kg, with a length of at least 5 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; Category C1E: A combination made up of a subcategory C1 test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg; this combination shall be at least 8 m in length and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least as wide and as high as the cab; the closed box body may also be slightly less wide than the cab provided that the view to the rear is only possible by use of the external rear-view mirrors of the motor vehicle; the trailer shall be presented with a minimum of 800 kg real total mass; Category D: A category D vehicle with a length of at least 10 m, a width of at least 2,40 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; Category DE: A combination made up of a category D test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg, a width of at least 2,40 m and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least 2 m wide and 2 m high; the trailer shall be presented with a minimum of 800 kg real total mass; Category D1: A subcategory D1 vehicle with a maximum authorised mass of at least 4 000 kg, with a length of at least 5 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; Category D1E: A combination made up of a subcategory D1 test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least 2 m wide and 2 m high; the trailer shall be presented with a minimum of 800 kg real total mass; Testing vehicles for categories BE, C, CE, C1, C1E, D, DE, D1 and D1E which are not in conformity with the minimum criteria given above but which were in use on or before the moment of entry into force of this Directive, may still be used for a period not exceeding ten years after that date. The requirements related to the load to be carried by these vehicles, may be implemented by Member States up to ten years from the moment of entry into force of Commission Directive 2000/56/EC (3). 6. Skills and behaviour to be tested concerning categories A1, A2 and A 6.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to ride safely by satisfying the following requirements: 6.1.1. Adjust the protective outfit, such as gloves, boots, clothes and safety helmet; 6.1.2. Perform a random check on the condition of the tyres, brakes, steering, emergency stop switch (if applicable), chain, oil levels, lights, reflectors, direction indicators and audible warning device. 6.2. Special manoeuvres to be tested with a bearing on road safety 6.2.1. Putting the motorcycle on and off its stand and moving it, without the aid of the engine, by walking alongside the vehicle; 6.2.2. Parking the motorcycle on its stand; 6.2.3. At least two manoeuvres to be executed at slow speed, including a slalom; this should allow competence to be assessed in handling of the clutch in combination with the brake, balance, vision direction and position on the motorcycle and the position of the feet on the foot rests; 6.2.4. At least two manoeuvres to be executed at higher speed, of which one manoeuvre in second or third gear, at least 30 km/h and one manoeuvre avoiding an obstacle at a minimum speed of 50 km/h; this should allow competence to be assessed in the position on the motorcycle, vision direction, balance, steering technique and technique of changing gears; 6.2.5. Braking: at least two braking exercises shall be executed, including an emergency brake at a minimum speed of 50 km/h; this should allow competence to be assessed in handling of the front and rear brake, vision direction and the position on the motorcycle. The special manoeuvres mentioned under points 6.2.3 to 6.2.5 have to be implemented at the latest five years after entry into force of Directive 2000/56/EC. 6.3. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 6.3.1. Riding away: after parking, after a stop in traffic; exiting a driveway; 6.3.2. Riding on straight roads; passing oncoming vehicles, including in confined spaces; 6.3.3. Riding round bends; 6.3.4. Crossroads: approaching and crossing of intersections and junctions; 6.3.5. Changing direction: left and right turns; changing lanes; 6.3.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 6.3.7. Overtaking/passing: overtaking other traffic (if possible); riding alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 6.3.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; riding up-/downhill on long slopes; 6.3.9. Taking the necessary precautions when getting off the vehicle. 7. Skills and behaviour to be tested concerning categories B, B1 and BE 7.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to drive safely by satisfying the following requirements: 7.1.1. Adjusting the seat as necessary to obtain a correct seated position; 7.1.2. Adjusting rear-view mirrors, seat belts and head restraints if available; 7.1.3. Checking that the doors are closed; 7.1.4. Performing a random check on the condition of the tyres, steering, brakes, fluids (e.g. engine oil, coolant, washer fluid), lights, reflectors, direction indicators and audible warning device; 7.1.5. Checking the safety factors relating to vehicle loading: body, sheets, cargo doors, cabin locking, way of loading, securing load (category BE only); 7.1.6. Checking the coupling mechanism and the brake and electrical connections (category BE only). 7.2. Categories B and B1: special manoeuvres to be tested with a bearing on road safety A selection of the following manoeuvres shall be tested (at least two manoeuvres for the four points, including one in reverse gear): 7.2.1. Reversing in a straight line or reversing right or left round a corner while keeping within the correct traffic lane; 7.2.2. Turning the vehicle to face the opposite way, using forward and reverse gears; 7.2.3. Parking the vehicle and leaving a parking space (parallel, oblique or right-angle, forwards or in reverse, on the flat, uphill or downhill); 7.2.4. Braking accurately to a stop; however, performing an emergency stop is optional. 7.3. Category BE: special manoeuvres to be tested with a bearing on road safety 7.3.1. Coupling and uncoupling, or uncoupling and re-coupling a trailer from its motor vehicle; the manoeuvre must involve the towing vehicle being parked alongside the trailer (i.e. not in one line); 7.3.2. Reversing along a curve, the line of which shall be left to the discretion of the Member States; 7.3.3. Parking safely for loading/unloading. 7.4. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 7.4.1. Driving away: after parking, after a stop in traffic; exiting a driveway; 7.4.2. Driving on straight roads; passing oncoming vehicles, including in confined spaces; 7.4.3. Driving round bends; 7.4.4. Crossroads: approaching and crossing of intersections and junctions; 7.4.5. Changing direction: left and right turns; changing lanes; 7.4.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 7.4.7. Overtaking/passing: overtaking other traffic (if possible); driving alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 7.4.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; driving up-/downhill on long slopes; 7.4.9. Taking the necessary precautions when alighting from the vehicle. 8. Skills and behaviour to be tested concerning categories C, CE, C1, C1E, D, DE, D1 and D1E 8.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to drive safely by satisfying the following requirements: 8.1.1. Adjusting the seat as necessary to obtain a correct seated position; 8.1.2. Adjusting rear-view mirrors, seat belts and head restraints if available; 8.1.3. Random checks on the condition of the tyres, steering, brakes, lights, reflectors, direction indicators and audible warning device; 8.1.4. Checking the power-assisted braking and steering systems; checking the condition of the wheels, wheelnuts, mudguards, windscreen, windows and wipers, fluids (e.g. engine oil, coolant, washer fluid); checking and using the instrument panel including the recording equipment as defined in Regulation (EEC) No 3821/85; 8.1.5. Checking the air pressure, air tanks and the suspension; 8.1.6. Checking the safety factors relating to vehicle loading: body, sheets, cargo doors, loading mechanism (if available), cabin locking (if available), way of loading, securing load (categories C, CE, C1, C1E only); 8.1.7. Checking the coupling mechanism and the brake and electrical connections (categories CE, C1E, DE, D1E only); 8.1.8. Being capable of taking special vehicle safety measures; controlling the body, service doors, emergency exits, first aid equipment, fire extinguishers and other safety equipment (categories D, DE, D1, D1E only); 8.1.9. Reading a road map, route planning, including the use of electronic navigation systems (optional). 8.2. Special manoeuvres to be tested with a bearing on road safety 8.2.1. Coupling and uncoupling, or uncoupling and re-coupling a trailer from its motor vehicle; the manoeuvre must involve the towing vehicle being parked alongside the trailer (i.e. not in one line) (categories CE, C1E, DE, D1E only); 8.2.2. Reversing along a curve, the line of which shall be left to the discretion of the Member States; 8.2.3. Parking safely for loading/unloading at a loading ramp/platform or similar installation (categories C, CE, C1, C1E only); 8.2.4. Parking to let passengers on or off the bus safely (categories D, DE, D1, D1E only). 8.3. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 8.3.1. Driving away: after parking, after a stop in traffic; exiting a driveway; 8.3.2. Driving on straight roads; passing oncoming vehicles, including in confined spaces; 8.3.3. Driving round bends; 8.3.4. Crossroads: approaching and crossing of intersections and junctions; 8.3.5. Changing direction: left and right turns; changing lanes; 8.3.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 8.3.7. Overtaking/passing: overtaking other traffic (if possible); driving alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 8.3.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; driving up-/downhill on long slopes; 8.3.9. Taking the necessary precautions when alighting from the vehicle. 9. Marking of the test of skills and behaviour 9.1. For each of the abovementioned driving situations, the assessment must reflect the degree of ease with which the applicant handles the vehicle controls and his demonstrated capacity to drive in traffic in complete safety. The examiner must feel safe throughout the test. Driving errors or dangerous conduct immediately endangering the safety of the test vehicle, its passengers or other road users shall be penalised by failing the test, whether or not the examiner or accompanying person has to intervene. Nonetheless, the examiner shall be free to decide whether or not the skills and behaviour test should be completed. Driving examiners must be trained to assess correctly the applicants' ability to drive safely. The work of driving examiners must be monitored and supervised, by a body authorised by the Member State, to ensure correct and consistent application of fault assessment in accordance with the standards laid down in this Annex. 9.2. During their assessment, driving examiners shall pay special attention to whether an applicant is showing a defensive and social driving behaviour. This should reflect the overall style of driving and the driving examiner should take this into account in the overall picture of the applicant. It includes adapted and determined (safe) driving, taking into account road and weather conditions, taking into account other traffic, taking into account the interests of other road users (particularly the more vulnerable) and anticipation. 9.3. The driving examiner will furthermore assess whether the applicant is: 9.3.1. Controlling the vehicle; taking into account: proper use of safety belts, rear-view mirrors, head restraints; seat; proper use of lights and other equipment; proper use of clutch, gearbox, accelerator, braking systems (including third braking system, if available), steering; controlling the vehicle under different circumstances, at different speeds; steadiness on the road; the weight and dimensions and characteristics of the vehicle; the weight and type of load (categories BE, C, CE, C1, C1E, DE, D1E only); the comfort of the passengers (categories D, DE, D1, D1E only) (no fast acceleration, smoothly driving and no hard braking); 9.3.2. Driving economically and in an environmentally friendly way, taking into account the revolutions per minute, changing gears, braking and accelerating (categories BE, C, CE, C1, C1E, D, DE, D1, D1E only); 9.3.3. Observation: all-round observation; proper use of mirrors; far, middle, near distance vision; 9.3.4. Priority/giving way: priority at crossroads, intersections and junctions; giving way at other occasions (e.g. changing direction, changing lanes, special manoeuvres); 9.3.5. Correct position on the road: proper position on the road, in lanes, on roundabouts, round bends, suitable for the type and the characteristics of the vehicle; pre-positioning; 9.3.6. Keeping distance: keeping adequate distance to the front and the side; keeping adequate distance from other road users; 9.3.7. Speed: not exceeding the maximum allowed speed; adapting speed to weather/traffic conditions and where appropriate up to national speed limits; driving at such a speed that stopping within distance of the visible and free road is possible; adapting speed to general speed of same kind of road users; 9.3.8. Traffic lights, road signs and other indications: acting correctly at traffic lights; obeying instructions from traffic controllers; acting correctly at road signs (prohibitions or commands); take appropriate action at road markings; 9.3.9. Signalling: give signals where necessary, correctly and properly timed; indicating directions correctly; taking appropriate action with regard to all signals made by other road users; 9.3.10. Braking and stopping: decelerating in time, braking or stopping according to circumstances; anticipation; using the various braking systems (only for categories C, CE, D, DE); using speed reduction systems other than the brakes (only for categories C, CE, D, DE). 10. Length of the test The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in paragraph B of this Annex. In no circumstances should the time spent driving on the road be less than 25 minutes for categories A, A1, A2, B, B1 and BE and 45 minutes for the other categories. This does not include the reception of the applicant, the preparation of the vehicle, the technical check of the vehicle with a bearing on road safety, the special manoeuvres and the announcement of the outcome of the practical test. 11. Location of the test The part of the test to assess the special manoeuvres may be conducted on a special testing ground. Wherever practicable, the part of the test to assess behaviour in traffic should be conducted on roads outside built-up areas, expressways and motorways (or similar), as well as on all kinds of urban streets (residential areas, 30 and 50 km/h areas, urban expressways) which should represent the various types of difficulty likely to be encountered by drivers. It is also desirable for the test to take place in various traffic density conditions. The time spent driving on the road should be used in an optimal way to assess the applicant in all the various traffic areas that can be encountered, with a special emphasis on changing between these areas. II. KNOWLEDGE, SKILL AND BEHAVIOUR FOR DRIVING A POWER-DRIVEN VEHICLE Drivers of all power-driven vehicles must at any moment have the knowledge, skills and behaviour described under points 1 to 9, with a view to be able to:  Recognise traffic dangers and assess their seriousness,  Have sufficient command of their vehicle not to create dangerous situations and to react appropriately should such situations occur,  Comply with road traffic regulations, and in particular those intended to prevent road accidents and to maintain the flow of traffic,  Detect any major technical faults in their vehicles, in particular those posing a safety hazard, and have them remedied in an appropriate fashion,  Take account of all the factors affecting driving behaviour (e.g. alcohol, fatigue, poor eyesight, etc.) so as to retain full use of the faculties needed to drive safely,  Help ensure the safety of all road users, and in particular of the weakest and most exposed by showing due respect for others. Member States may implement the appropriate measures to ensure that drivers who have lost the knowledge, skills and behaviour as described under points 1 to 9 can recover this knowledge and these skills and will continue to exhibit such behaviour required for driving a motor vehicle. (1) OJ L 370, 31.12.1985, p. 1. Regulation as repealed by Regulation (EC) No 561/2006 of the European Parliament and of the Council (OJ L 102, 11.4.2006, p. 1). (2) OJ L 370, 31.12.1985, p. 8. Regulation as last amended by Regulation (EC) No 561/2006. (3) Commission Directive 2000/56/EC of 14 September 2000 amending Council Directive 91/439/EEC on driving licences (OJ L 237, 21.9.2000, p. 45). ANNEX III MINIMUM STANDARDS OF PHYSICAL AND MENTAL FITNESS FOR DRIVING A POWER-DRIVEN VEHICLE DEFINITIONS 1. For the purpose of this Annex, drivers are classified in two groups: 1.1. Group 1: drivers of vehicles of categories A, A1, A2, AM, B, B1 and BE. 1.2. Group 2: drivers of vehicles of categories C, CE, C1, C1E, D, DE, D1 and D1E. 1.3. National legislation may provide for the provisions set out in this Annex for Group 2 drivers to apply to drivers of Category B vehicles using their driving licence for professional purposes (taxis, ambulances, etc.). 2. Similarly, applicants for a first driving licence or for the renewal of a driving licence are classified in the group to which they will belong once the licence has been issued or renewed. MEDICAL EXAMINATIONS 3. Group 1: Applicants shall be required to undergo a medical examination if it becomes apparent, when the necessary formalities are being completed or during the tests which they have to undergo prior to obtaining a driving licence, that they have one or more of the medical disabilities mentioned in this Annex. 4. Group 2: Applicants shall undergo medical examinations before a driving licence is first issued to them and thereafter drivers shall be checked in accordance with the national system in place in the Member State of normal residence whenever their driving licence is renewed 5. The standards set by Member States for the issue or any subsequent renewal of driving licences may be stricter than those set out in this Annex. SIGHT 6. All applicants for a driving licence shall undergo an appropriate investigation to ensure that they have adequate visual acuity for driving power-driven vehicles. Where there is reason to doubt that the applicant's vision is adequate, he shall be examined by a competent medical authority. At this examination attention shall be paid the following in particular: visual acuity, field of vision, twilight vision and progressive eye diseases. For the purpose of this Annex, intra-ocular lenses shall not be considered corrective lenses. Group 1: 6.1. Applicants for a driving licence or for the renewal of such a licence shall have a binocular visual acuity, with corrective lenses if necessary, of at least 0,5 when using both eyes together. Driving licences shall not be issued or renewed if, during the medical examination, it is shown that the horizontal field of vision is less than 120o o, apart from exceptional cases duly justified by a favourable medical opinion and a positive practical test, or that the person concerned suffers from any other eye condition that would compromise safe driving. When a progressive eye disease is detected or declared, driving licences may be issued or renewed subject to the applicant undergoing regular examination by a competent medical authority. 6.2. Applicants for a driving licence, or for the renewal of such a licence, who have total functional loss of vision in one eye or who use only one eye (e.g. in the case of diplopia) must have a visual acuity of at least 0,6, with corrective lenses if necessary. The competent medical authority must certify that this condition of monocular vision has existed sufficiently long to allow adaptation and that the field of vision in this eye is normal. Group 2: 6.3. Applicants for a driving licence or for the renewal of such a licence must have a visual acuity, with corrective lenses if necessary, of at least 0,8 in the better eye and at least 0,5 in the worse eye. If corrective lenses are used to attain the values of 0,8 and 0,5, the uncorrected acuity in each eye must reach 0,05, or else the minimum acuity (0,8 and 0,5) must be achieved either by correction by means of glasses with a power not exceeding plus or minus 8 dioptres or with the aid of contact lenses (uncorrected vision = 0,05). The correction must be well tolerated. Driving licences shall not be issued to or renewed for applications or drivers without a normal binocular field of vision or suffering from diplopia. HEARING 7. Driving licences may be issued to or renewed for applicants or drivers in Group 2 subject to the opinion of the competent medical authorities; particular account will be taken in medical examinations of the scope for compensation. PERSONS WITH A LOCOMOTOR DISABILITY 8. Driving licences shall not be issued to or renewed for applicants or drivers suffering from complaints or abnormalities of the locomotor system which make it dangerous to drive a power-driven vehicle. Group 1: 8.1. Driving licences subject to certain restrictions, if necessary, may be issued to physically disabled applicants or drivers following the issuing of an opinion by a competent medical authority. This opinion must be based on a medical assessment of the complaint or abnormality in question and, where necessary, on a practical test. It must also indicate what type of modification to the vehicle is required and whether the driver needs to be fitted with an orthopaedic device, insofar as the test of skills and behaviour demonstrates that with such a device driving would not to be dangerous. 8.2. Driving licences may be issued to or renewed for any applicant suffering from a progressive complaint on condition that the disabled person is regularly examined to check that the person is still capable of driving the vehicle completely safely. Where the disability is static, driving licences may be issued or renewed without the applicant being subject to regular medical examination. Group 2: 8.3. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. CARDIOVASCULAR DISEASES 9. Any disease capable of exposing an applicant for a first licence or a driver applying for renewal to a sudden failure of the cardiovascular system such that there is a sudden impairment of the cerebral functions constitutes a danger to road safety. Group 1: 9.1. Driving licences will not to be issued to, or renewed for, applicants or drivers with serious arrhythmia. 9.2. Driving licences may be issued to, or renewed for, applicants or drivers wearing a pacemaker subject to authorised medical opinion and regular medical check-ups. 9.3. The question of whether to issue or renew a licence for applicants or drivers suffering from abnormal arterial blood pressure shall be assessed with reference to the other results of the examination, any associated complications and the danger they might constitute for road safety. 9.4. Generally speaking, a driving licence shall not be issued to or renewed for applicants or drivers suffering from angina during rest or emotion. The issuing or renewal of a driving licence to any applicant or driver having suffered myocardial infarction shall be subject to authorised medical opinion and, if necessary, regular medical check-ups. Group 2: 9.5. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. DIABETES MELLITUS 10. Driving licences may be issued to, or renewed for, applicants or drivers suffering from diabetes mellitus, subject to authorised medical opinion and regular medical check-ups appropriate to each case. Group 2: 10.1. Only in very exceptional cases may driving licences be issued to, or renewed for, applicants or drivers in this group suffering from diabetes mellitus and requiring insulin treatment, and then only where duly justified by authorised medical opinion and subject to regular medical check-ups. NEUROLOGICAL DISEASES 11. Driving licences shall not be issued to, or renewed for, applicants or drivers suffering from a serious neurological disease, unless the application is supported by authorised medical opinion. Neurological disturbances associated with diseases or surgical intervention affecting the central or peripheral nervous system, which lead to sensory or motor deficiencies and affect balance and coordination, must accordingly be taken into account in relation to their functional effects and the risks of progression. In such cases, the issue or renewal of the licence may be subject to periodic assessment in the event of risk of deterioration. 12. Epileptic seizures or other sudden disturbances of the state of consciousness constitute a serious danger to road safety if they occur in a person driving a power-driven vehicle. Group 1: 12.1. A licence may be issued or renewed subject to an examination by a competent medical authority and to regular medical check-ups. The authority shall decide on the state of the epilepsy or other disturbances of consciousness, its clinical form and progress (no seizure in the last two years, for example), the treatment received and the results thereof. Group 2: 12.2. Driving licences shall not be issued to or renewed for applicants or drivers suffering or liable to suffer from epileptic seizures or other sudden disturbances of the state of consciousness. MENTAL DISORDERS Group 1: 13.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who suffer from:  severe mental disturbance, whether congenital or due to disease, trauma or neurosurgical operations,  severe mental retardation,  severe behavioural problems due to ageing; or personality defects leading to seriously impaired judgment, behaviour or adaptability, unless their application is supported by authorised medical opinion and, if necessary, subject to regular medical check-ups. Group 2: 13.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. ALCOHOL 14. Alcohol consumption constitutes a major danger to road safety. In view of the scale of the problem, the medical profession must be very vigilant. Group 1: 14.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who are dependent on alcohol or unable to refrain from drinking and driving. After a proven period of abstinence and subject to authorised medical opinion and regular medical check-ups, driving licences may be issued to, or renewed for, applicant or drivers who have in the past been dependent on alcohol. Group 2: 14.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. DRUGS AND MEDICINAL PRODUCTS 15. Abuse: Driving licences shall not be issued to or renewed for applicants or drivers who are dependent on psychotropic substances or who are not dependent on such substances but regularly abuse them, whatever category of licence is requested. Regular use: Group 1: 15.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who regularly use psychotropic substances, in whatever form, which can hamper the ability to drive safely where the quantities absorbed are such as to have an adverse effect on driving. This shall apply to all other medicinal products or combinations of medicinal products which affect the ability to drive. Group 2: 15.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definitions of this group. RENAL DISORDERS Group 1: 16.1. Driving licences may be issued or renewed for applicants and drivers suffering from serious renal insufficiency subject to authorised medical opinion and regular medical check-ups. Group 2: 16.2. Save in exceptional cases duly justified by authorised medical opinion, and subject to regular medical check-ups, driving licences shall not be issued to or renewed for applicants or drivers suffering from serious and irreversible renal deficiency. MISCELLANEOUS PROVISIONS Group 1: 17.1. Subject to authorised medical opinion and, if necessary, regular medical check-ups, driving licences may be issued to or renewed for applications or drivers who have had an organ transplant or an artificial implant which affects the ability to drive. Group 2: 17.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. 18. As a general rule, where applicants or drivers suffer from any disorder which is not mentioned in the preceding paragraph but is liable to be, or to result in, a functional incapacity affecting safety at the wheel, driving licences shall not be issued or renewed unless the application is supported by authorised medical opinion and, if necessary, subject to regular medical check-ups. ANNEX IV MINIMUM STANDARDS FOR PERSONS WHO CONDUCT PRACTICAL DRIVING TESTS 1. Competences required by a driving examiner 1.1. A person authorised to conduct practical assessments in a motor vehicle of the driving performance of a candidate must have knowledge, skills and understanding related to the topics listed in points 1.2 to 1.6. 1.2. The competences of an examiner must be relevant to assessing the performance of a candidate seeking the category of driving licence entitlement for which the driving test is being undertaken. 1.3. Knowledge and understanding of driving and assessment:  theory of driving behaviour,  hazard perception and accident avoidance,  the syllabus underpinning driving test standards,  the requirements of the driving test,  relevant road and traffic legislation, including relevant EU and national legislation and interpretative guidelines,  assessment theory and techniques,  defensive driving. 1.4. Assessment skills:  ability to observe accurately, monitor, and evaluate overall candidate performance, in particular:  correct and comprehensive recognition of dangerous situations,  accurate determination of cause and likely effect of such situations,  achievement of competence and recognition of errors,  uniformity and consistency in assessment,  assimilate information quickly and extract key points,  look ahead, identify potential problems, and develop strategies to deal with them,  provide timely and constructive feedback. 1.5. Personal driving skills:  A person authorised to conduct a practical test for a category of driving licence must be able to drive to a consistently high standard that type of motor vehicle. 1.6. Quality of service:  establish and communicate what the candidate can expect during the test,  communicate clearly, choosing content, style and language to suit the audience and context and deal with enquiries from candidates,  provide clear feedback about the test result,  treat candidates with respect and indiscriminately. 1.7. Knowledge about vehicle technique and physics:  knowledge about vehicle technique such as steering, tyres, brakes, lights, specially for motorcycles and heavy vehicles,  loading safety,  knowledge about vehicle physics such as speed, friction, dynamics, energy. 1.8. Driving in a fuel efficient and environmentally friendly way. 2. General conditions 2.1. A category B driving examiner: (a) must have held a category B licence for at least 3 years; (b) must be at least 23 years old; (c) must have successfully completed the initial qualification provided for in point 3 of this Annex and subsequently followed the quality assurance and the periodic training arrangements as provided for in point 4 of this Annex; (d) must have terminated a vocational education that leads at least to a completion of level 3 as defined by Council Decision 85/368/EEC of 16 July 1985 on the comparability of vocational training qualifications between the Member States of the European Community (1); (e) may not be active as a commercial driving instructor in a driving school simultaneously. 2.2. A driving examiner for the other categories: (a) must hold a driving licence in the category concerned or possess equivalent knowledge through adequate professional qualification; (b) must have successfully completed the initial qualification provided for in point 3 of this Annex and subsequently followed the quality assurance and the periodic training arrangements as provided for in point 4 of this Annex; (c) must have been a qualified category B driving examiner for at least 3 years; this period may be waived provided that the examiner in question can provide evidence of:  at least 5 years of driving in the category concerned, or,  a theoretical and practical assessment of driving ability of a standard higher than that needed to obtain a driving licence thus making that requirement unnecessary, (d) must have completed a vocational education that leads at least to a termination of the level 3 as defined by Decision 85/368/EEC; (e) may not be active as a commercial driving instructor in a driving school simultaneously. 2.3. Equivalences 2.3.1. Member States may authorise an examiner to conduct driving tests for categories AM, A1, A2 and A upon passing the initial qualification prescribed in point 3 for one of these categories. 2.3.2. Member States may authorise an examiner to conduct driving tests for categories C1, C, D1 and D upon passing the initial qualification prescribed in point 3 for one of these categories. 2.3.3. Member States may authorise an examiner to conduct driving tests for categories BE, C1E, CE, D1E and DE upon passing the initial qualification prescribed in point 3 for one of these categories. 3. Initial qualification 3.1. Initial training 3.1.1. Before a person may be authorised to conduct driving tests, that person must satisfactorily complete such training programme as a Member State may specify in order to have the competences set out in point 1. 3.1.2. Member States must determine whether the content of any particular training programme will relate to authorisation to conduct driving tests for one driving licence category, or more than one. 3.2. Examinations 3.2.1. Before a person may be authorised to conduct driving tests, that person must demonstrate a satisfactory standard of knowledge, understanding, skills and aptitude in respect of the subjects listed in point 1. 3.2.2. Member States shall operate an examination process that assesses, in a pedagogically appropriate manner, the competences of the person as defined under point 1, in particular point 1.4. The examination process must include both a theoretical element and a practical element. Computer-based assessment may be used where appropriate. The details concerning the nature and duration of any tests and assessments within the examination shall be at the discretion of the individual Member States. 3.2.3. Member States must determine whether the content of any particular examination will relate to authorisation to conduct driving tests for one driving licence category, or more than one. 4. Quality assurance and periodic training 4.1. Quality assurance 4.1.1. Member States shall have in place quality assurance arrangements to provide for the maintenance of standards of driving examiners. 4.1.2. Quality assurance arrangements should involve the supervision of examiners at work, their further training and re-accreditation, their continuing professional development, and by periodic review of the outcomes of the driving tests that they have conducted. 4.1.3. Member States must provide that each examiner is subject to yearly supervision making use of quality assurance arrangements listed in point 4.1.2. Moreover, the Member States must provide that each examiner is observed conducting tests once every 5 years, for a minimum period cumulatively of at least half a day, allowing the observation of several tests. When issues are identified corrective action should be put in place. The person undertaking the supervision must be a person authorised by the Member State for that purpose. 4.1.4 Member States may provide that where an examiner is authorised to conduct driving tests in more than one category, satisfying the supervision requirement in relation to tests for one category satisfies the requirement for more than one category. 4.1.5 The work of driving examination must be monitored and supervised by a body authorised by the Member State, to ensure correct and consistent application of assessment. 4.2. Periodic training 4.2.1. Member States shall provide that, in order to remain authorised, driving examiners, irrespective of the number of categories for which they are accredited, undertake:  a minimum regular periodic training of four days in total per period of two years in order to:  maintain and refresh the necessary knowledge and examining skills,  to develop new competences that have become essential for the exercise of their profession,  ensure that an examiner continues to conduct tests to a fair and uniform standard,  a minimum periodic training of at least five days in total per period of five years,  in order to develop and maintain the necessary practical driving skills. 4.2.2. Member States shall take the appropriate measures for ensuring that specific training is given promptly to those examiners that have found to be seriously malfunctioning by the quality assurance system in place. 4.2.3. The nature of periodic training may take the form of briefing, classroom training, conventional or electronic-based learning, and it may be undertaken on an individual or group basis. It may include such re-accreditation of standards as Member States consider appropriate. 4.2.4. Member States may provide that where an examiner is authorised to conduct driving tests in more than one category, satisfying the periodic training requirement in relation to tests for one category satisfies the requirement for more than one category, provided the condition set out in point 4.2.5 is satisfied. 4.2.5. Where an examiner has not conducted tests for a category within a 24-month period, the examiner shall undertake a suitable reassessment before being allowed to carry out driving tests relating to that category. That re-assessment may be undertaken as part of the requirement set out in point 4.2.1. 5. Acquired rights 5.1. Member States may allow persons authorised to conduct driving tests immediately before these provisions come into force to continue to conduct driving tests, notwithstanding that they were not authorised in accordance with the general conditions in point 2 or the initial qualification process set out in point 3. 5.2. Such examiners are nonetheless subject to the regular supervision and quality assurance arrangements set out in point 4. (1) OJ L 199, 31.7.1985, p. 56. ANNEX V MINIMUM REQUIREMENTS FOR DRIVER TRAINING AND TESTING FOR COMBINATIONS AS DEFINED IN THE SECOND SUBPARAGRAPH OF ARTICLE 4(4)(B) 1. Member States shall take the necessary measures to:  approve and supervise the training provided for in Article 7(1)(d) or,  organise the test of skills and behaviour provided for in Article 7(1)(d). 2.1. Duration of driver training  at least 7 hours. 3. Content of driver training The driver training shall cover the knowledge, skills and behaviour as described in points 2 and 7 of Annex II. Particular attention shall be paid to:  vehicle movement dynamics, safety criteria, tractor vehicle and trailer (coupling mechanism), correct loading and safety fittings; A practical component shall include the following exercises: acceleration, deceleration, reversing, braking, stopping distance, lane-changing, braking/evasive action, trailer swing, uncoupling from and re-coupling a trailer to its motor vehicle, parking;  Each training participant has to perform the practical component and shall demonstrate its skills and behaviour on public roads,  Vehicle combinations used for the training shall fall within the category of driving licence participants have applied for. 4. Duration and contents of the test of skills and behaviour The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in point 3. ANNEX VI MINIMUM REQUIREMENTS FOR DRIVER TRAINING AND TESTING FOR MOTORCYCLES WITHIN CATEGORY A (PROGRESSIVE ACCESS) 1. Member States shall take the necessary measures to:  approve and supervise the training provided for in Article 7(1)(c) or,  organise the test of skills and behaviour provided for in Article 7(1)(c). 2. Duration of driver training  at least 7 hours. 3. Content of driver training  The driver training shall contain all aspects covered in point 6 of Annex II.  Each participant has to perform the practical components of the training and shall demonstrate its skills and behaviour on public roads.  Motorcycles used for the training shall fall within the category of driving licence participants have applied for. 4. Duration and contents of the test of skills and behaviour The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in point 3 of this Annex. ANNEX VII Part A REPEALED DIRECTIVE AS SUCCESSIVELY AMENDED (referred to in Article 17) Council Directive 91/439/EEC (1) (OJ L 237, 24.8.1991, p. 1) Council Directive 94/72/EC (OJ L 337, 24.12.1994, p. 86) Council Directive 96/47/EC (OJ L 235, 17.9.1996, p. 1) Council Directive 97/26/EC (OJ L 150, 7.6.1997, p. 41) Commission Directive 2000/56/EC (OJ L 237, 21.9.2000, p. 45) Directive 2003/59/EC of the European Parliament and of the Council, only Article 10, paragraph 2 (OJ L 226, 10.9.2003, p. 4) Regulation (EC) No 1882/2003 of the European Parliament and of the Council, only Annex II, point 24 (OJ L 284, 31.10.2003, p. 1) Part B DEADLINES FOR TRANSPOSITION INTO NATIONAL LAW AND FOR APPLICATION (referred to in Article 17) Directive Deadline for transposition Date of application Directive 91/439/EEC 1st July 1994 1st July 1996 Directive 94/72/EC - 1st January1995 Decision 96/427/EC - 16 July 1996 Directive 96/47/EC 1st July 1996 >1st July 1996 Directive 97/26/EC 1st January 1998 1st January 1998 Directive 2000/56/EC 30 September 2003 30 September 2003, 30 September 2008 (Annex II, point 6.2.5) and 30 September 2013 (Annex II point 5.2) Directive 2003/59/EC 10 September 2006 10 September 2008 (passenger transport) and 10 September 2009 (goods transport) (1) Directive 91/439/EEC was also amended by the following act which has not been repealed: 1994 Act of accession. ANNEX VIII CORRELATION TABLE Directive 91/439/EEC This Directive Article 1(1), first sentence Article 1(1) first sentence Article 1(1), second sentence  - Article 1(2) Article 1(2) Article 2(1) - Article 2(2) Article 1(3) - Article 2(1) Article 1(1), second sentence Article 2(2) Article 3(1) Article 3(2) Article 3(3) Article 2(3) - Article 2(4) - Article 3(1), first subparagraph, introductory words Article 4(1), first sentence - Article 4(2), first indent - Article 4(2), second indent Article 3(1), first subparagraph, first indent Article 4(3), first indent Article 3(1), first subparagraph, second indent Article 4(4)(b), first subparagraph Article 3(1), first subparagraph, third indent Article 4(4)(b), second subparagraph Article 3(1), first subparagraph, fourth indent Article 4(4)(c) Article 3(1), first subparagraph, fifth indent Article 4(4)(f) Article 3(1), first subparagraph, sixth indent Article 4(4)(g) Article 3(1), first subparagraph, seventh indent Article 4(4)(j) Article 3(1), first subparagraph, eighth indent Article 4(4)(k) Article 3(2), first subparagraph, introductory words - Article 3(2), first subparagraph, first indent Article 4(3)(a) Article 3(2), first subparagraph, second indent Article 4(4)(a) Article 3(2), first subparagraph, third indent Article 4(4)(d) Article 3(2), first subparagraph, fourth indent Article 4(4)(e) Article 3(2), first subparagraph, fifth indent Article 4(4)(h) Article 3(2), first subparagraph, sixth indent, introductory words Article 4(4)(i) Article 3(2), first subparagraph, sixth indent, first sub-indent - Article 3(2), first subparagraph, sixth indent, second sub-indent - Article 3(3), introductory words - Article 3(3), first indent Article 4(1), third sentence Article 3(3), second indent, first subparagraph Article 4(3), second indent Article 3(3), second indent, second subparagraph - Article 3(3), third indent Article 4(3), first indent Article 3(3), fourth indent Article 4(4), first indent Article 3(3), fifth indent Article 4(4), second indent - Article 4(3) Article 3(4) - Article 3(5) - Article 3(6) Article 4(5), first sentence - Article 4(5), second sentence Article 4 Article 5 Article 5(1) Article 6(1) Article 5(1)(a) Article 6(1)(a) Article 5(1)(b) Article 6(1)(b) Article 5(2), introductory words Article 6(2), introductory words Article 5(2)(a) Article 6(2)(a) Article 5(2)(b) Article 6(2)(b) - Article 6(2)(c) - Article 6(2)(d) - Article 6(2)(e) - Article 6(2)(f) Article 5(3) - Article 5(4) Article 6(4) Article 6(1), introductory words Article 4(1), second sentence Article 6(1)(a), first indent Article 4(3)(a), third indent Article 6(1)(a), second indent Article 4(4)(a), second indent Article 6(1)(b), first indent Article 4(3)(b), second indent Article 4(3)(c), second indent Article 6(1)(b), second indent first alternative Article 4(4)(b), fifth subparagraph Article 6(1)(b), second indent second alternative Article 4(4)(c), second indent Article 6(1)(b), third indent first and second alternative Article 4(4)(g), second indent Article 6(1)(b), third indent third and fourth alternative Article 4(4)(e), third indent Article 6(1)(c), first indent first and second alternative Article 4(4)(k), second indent Article 6(1)(c), first indent third and fourth alternative Article 4(4)(i), second indent Article 6(2) Article 4(6), first subparagraph - Article 4(6), second subparagraph Article 6(3) Article 4(6), third and fourth subparagraphs Article 7(1), introductory words Article 7(1), introductory words Article 7(1)(a) Article 7(1)(a) - Article 7(1)(b) - Article 7(1)(c) - Article 7(1)(d) Article 7(1)(b) Article 7(1)(e) Article 7(2) - Article 7(3) - - Article 7(2) - Article 7(3) Article 7(4) Article 7(4) Article 7(5) Article 7(5)(a) - Article 7(5)(b) - Article 7(5)(c) - Article 7(5)(d) Article 7 a(1) - Article 7 a(2) Article 8 Article 7 b Article 9 - Article 10 Article 8 Article 11 Article 9 Article 12 Article 10 Article 13(1) - Article 13(2) Article 11 Article 14 Article 12(1) - Article 12(2) - Article 12(3) Article 15 - Article 16 Article 13 Article 17, first subparagraph - Article 17, second subparagraph - Article 18 Article 14 Article 19 Annex I - Annex Ia Annex I Annex II Annex II Annex III Annex III - Annex IV - Annex V - Annex VI
============================== "END OF DOC" ==============================
        

: 

In [23]:
laws[laws['CELEX'] == '32015L0413']['CELEX'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Act_type'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Treaty'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['act_raw_text'].iloc[0]


"13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important e

In [ ]:
def search_docs(query):
    celex_ids : list[str,str]
    full_doc_info : list[str]

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list[Any](set(celex_ids))    

    
    full_doc_info = f""" Doc{i}:\n 
    
      {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]} 

""" 

    return full_doc_info    



In [ ]:
search_docs("Drug dealing Sentences")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")

retrived = []

for i , doc in enumerate (docs):
    retrived.append([f"Doc{i}:"  doc.metadata['status','treaty','act_type','celex']])


[Document(metadata={'subject_matter': 'sources and branches of the law;  European Union law;  justice;  criminal law', 'status': 'In Force', 'legal_basis': '12002M031; 12002M034', 'additional_info': 'CNS 2001/0114', 'chunk_number': 1, 'eurovoc': 'penal code; Community law - national law; criminal procedure; penalty; drug traffic', 'treaty': 'TEU (1992)', 'act_type': 'Decision_FRAMW', 'cites': 'joint_action/1997/396; 31999Y0123%2801%29; joint_action/1998/733', 'celex': '32004F0757', 'authors': 'European Council', 'act_name': 'Council Framework Decision 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking', 'document_length': 13663, 'total_chunks': 6}, page_content="11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the 

In [ ]:
retrived = []

for doc in docs:
    retrived.append({doc.metadata['status','treaty']})

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI."),
    ("human", "Answer this question: {question}")
])

formatted = prompt.invoke({"question": "What is AI?"})
print(formatted)

messages=[SystemMessage(content='You are a helpful AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Answer this question: What is AI?', additional_kwargs={}, response_metadata={})]


In [9]:
laws[laws['CELEX'] == '31997R2046']['act_raw_text'].iloc[0]   

"Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social de